# India Fire & Climate Intelligence (`India_Fire_Climate_Intelligence.ipynb`)
**Central Master Notebook — Google Colab Pro Execution Only**

- **Raw ERA5 Archive (Read-Only):** `/content/drive/MyDrive/machine learning/`
- **Project Workspace (Persistent on Google Drive):** `/content/drive/MyDrive/India_Fire_Climate_Intelligence/`
- **Ephemeral Scratch Disk (Colab VM):** `/content/ifci_scratch/`
- **Primary Supervised Objective:** Forecasting **next-72-hour (`[D+1, D+2, D+3]` IST) forest-associated active-fire detection probability** (`target_forest_fire_next72h`) at $0.25^\circ \times 0.25^\circ$ resolution across India using strictly antecedent meteorological and fire-weather sequences (`t <= D`).
- **Fixed Chronological Research Splits:**
  - **1947–2000:** Historical climate baseline & long-term trend analysis (never used as supervised fire training data).
  - **2001–2010:** Supervised training split.
  - **2011–2015:** Validation, threshold optimization, architecture selection & isotonic probability calibration.
  - **2016–2025:** Independent holdout test split (evaluated once; never tuned on).

> **First Execution Review Gate:** By default, `STOP_AFTER_FIRST_EXECUTION = True` in Section 5 so that **Sections 1–5** (Hardware Diagnostics, Drive Setup, ERA5 File Discovery, Streaming GRIB Metadata Audit, Overlap/Coverage Analysis & Sample ERA5 Zarr Verification) execute and present their verified tables and figures **before** full-archive processing begins.


In [ ]:
# ============================================================================
# SECTION 0: Self-Contained Project Bootstrap (Google Drive + Python Package)
# ============================================================================
import json
import os
import subprocess
import sys
from pathlib import Path

# 1. Mount Google Drive inside Google Colab
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive", force_remount=False)
else:
    raise RuntimeError(
        "This master notebook is designed to execute exclusively inside Google Colab. "
        "Open it in Google Colab (https://colab.research.google.com) and run from there."
    )

# 2. Ensure ecCodes C library & required scientific packages are installed in the Colab VM
def _ensure_colab_packages():
    need_apt = subprocess.run(["which", "grib_ls"], capture_output=True).returncode != 0
    if need_apt:
        print("[Bootstrap] Installing libeccodes-dev & libeccodes-tools via apt-get...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "libeccodes-dev", "libeccodes-tools"],
            check=False,
        )
    pip_pkgs = []
    for mod, pkg in [
        ("eccodes", "eccodes"),
        ("cfgrib", "cfgrib"),
        ("zarr", "zarr"),
        ("rioxarray", "rioxarray"),
        ("lightgbm", "lightgbm"),
        ("xgboost", "xgboost"),
        ("shap", "shap"),
    ]:
        try:
            __import__(mod)
        except ImportError:
            pip_pkgs.append(pkg)
    if pip_pkgs:
        print(f"[Bootstrap] Installing Python packages in Colab VM: {pip_pkgs}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pip_pkgs], check=True)

_ensure_colab_packages()

# 3. Materialize the `ifci` Python package onto Google Drive & add to sys.path
PROJECT_ROOT = Path("/content/drive/MyDrive/India_Fire_Climate_Intelligence")
PKG_DIR = PROJECT_ROOT / "src" / "ifci"
PKG_DIR.mkdir(parents=True, exist_ok=True)

_IFCI_MODULES = json.loads(r'''{"__init__.py":"\"\"\"India Fire & Climate Intelligence (`ifci`) — Scientific Colab Package (IIT Gandhinagar B.Tech ML Project).\n\nAll computation, GRIB decoding, feature engineering, model training, evaluation, and\nstatic website generation run inside Google Colab backed by Google Drive.\n\"\"\"\n__version__ = \"3.0.0\"\n","config.py":"\"\"\"Central configuration, chronological research splits, and path registry.\n\nAll persistent paths live on Google Drive (`/content/drive/MyDrive/India_Fire_Climate_Intelligence/`).\n`LOCAL_SCRATCH` is on the Colab VM's ephemeral disk and is used only for disposable\nintermediate chunks.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nPROJECT_NAME = \"India_Fire_Climate_Intelligence\"\n\nDRIVE_MOUNT = Path(\"/content/drive\")\nDRIVE_ROOT = DRIVE_MOUNT / \"MyDrive\"\n\n# Original ERA5 GRIB archive. READ-ONLY by policy: no code in this project writes here.\nRAW_ERA5_DIR = Path(os.environ.get(\"IFCI_RAW_DIR\", str(DRIVE_ROOT / \"machine learning\")))\nPROJECT_ROOT = Path(os.environ.get(\"IFCI_PROJECT_ROOT\", str(DRIVE_ROOT / PROJECT_NAME)))\nLOCAL_SCRATCH = Path(os.environ.get(\"IFCI_SCRATCH\", \"/content/ifci_scratch\"))\n\n# Study domain in degrees (mainland India + Andaman & Nicobar + Lakshadweep, 0.25 deg grid).\nINDIA_BBOX = {\"lat_min\": 6.0, \"lat_max\": 38.0, \"lon_min\": 68.0, \"lon_max\": 98.0}\nLOCAL_TIMEZONE = \"Asia/Kolkata\"  # IST (UTC+05:30) used for daily aggregation & fire windows\nIST_UTC_OFFSET_MINUTES = 330     # +05:30 = +330 minutes\n\n# Fixed chronological research periods (inclusive start_year, end_year).\n# 1947-2000 is strictly for historical climate baseline & long-term trend analysis;\n# it is NEVER used as supervised fire training data without independent fire labels.\nRESEARCH_PERIODS = {\n    \"climate_baseline\": (1947, 2000),\n    \"train\": (2001, 2010),\n    \"validation\": (2011, 2015),\n    \"test_holdout\": (2016, 2025),\n}\n\n# Canonical ERA5 variable mapping: canonical key -> accepted GRIB shortNames and paramIds\nERA5_CANONICAL_VARS = {\n    \"t2m\":  {\"shortNames\": (\"2t\", \"t2m\"),        \"paramIds\": (167,),       \"kind\": \"instant\", \"units_out\": \"degC\"},\n    \"d2m\":  {\"shortNames\": (\"2d\", \"d2m\"),        \"paramIds\": (168,),       \"kind\": \"instant\", \"units_out\": \"degC\"},\n    \"u10\":  {\"shortNames\": (\"10u\", \"u10\"),       \"paramIds\": (165,),       \"kind\": \"instant\", \"units_out\": \"m s**-1\"},\n    \"v10\":  {\"shortNames\": (\"10v\", \"v10\"),       \"paramIds\": (166,),       \"kind\": \"instant\", \"units_out\": \"m s**-1\"},\n    \"tp\":   {\"shortNames\": (\"tp\",),              \"paramIds\": (228,),       \"kind\": \"accum\",   \"units_out\": \"mm day**-1\"},\n    \"ssrd\": {\"shortNames\": (\"ssrd\",),            \"paramIds\": (169,),       \"kind\": \"accum\",   \"units_out\": \"MJ m**-2 day**-1\"},\n    \"e\":    {\"shortNames\": (\"e\",),               \"paramIds\": (182,),       \"kind\": \"accum\",   \"units_out\": \"mm day**-1\"},\n    \"swvl1\": {\"shortNames\": (\"swvl1\",),          \"paramIds\": (39,),        \"kind\": \"instant\", \"units_out\": \"m**3 m**-3\"},\n    \"sp\":   {\"shortNames\": (\"sp\",),              \"paramIds\": (134,),       \"kind\": \"instant\", \"units_out\": \"hPa\"},\n    \"blh\":  {\"shortNames\": (\"blh\",),             \"paramIds\": (159,),       \"kind\": \"instant\", \"units_out\": \"m\"},\n    \"lai_hv\": {\"shortNames\": (\"lai_hv\",),        \"paramIds\": (67,),        \"kind\": \"instant\", \"units_out\": \"m**2 m**-2\"},\n    \"lai_lv\": {\"shortNames\": (\"lai_lv\",),        \"paramIds\": (66,),        \"kind\": \"instant\", \"units_out\": \"m**2 m**-2\"},\n}\n\nREQUIRED_CORE_VARS = (\"t2m\", \"d2m\", \"u10\", \"v10\", \"tp\", \"ssrd\")\n\nSUBDIRS = {\n    \"src\": \"src\",\n    \"config\": \"config\",\n    \"notebooks\": \"notebooks\",\n    \"manifests\": \"manifests\",\n    \"state\": \"state\",\n    \"logs\": \"logs\",\n    \"diagnostics\": \"logs/diagnostics\",\n    \"metadata\": \"metadata\",\n    \"grib_audit\": \"metadata/grib_audit\",\n    \"grib_messages\": \"metadata/grib_audit/messages\",\n    \"value_samples\": \"metadata/grib_audit/value_samples\",\n    \"coverage\": \"metadata/coverage\",\n    \"interim\": \"data/interim\",\n    \"era5_daily\": \"data/processed/era5_daily_zarr\",\n    \"climate_baseline\": \"data/processed/climate_baseline\",\n    \"features\": \"data/processed/features\",\n    \"supervised\": \"data/processed/supervised\",\n    \"fire_rasters\": \"data/processed/fire_rasters\",\n    \"firms_raw\": \"data/firms/raw\",\n    \"firms_processed\": \"data/firms/processed\",\n    \"landcover\": \"data/landcover\",\n    \"models\": \"models\",\n    \"checkpoints\": \"models/checkpoints\",\n    \"evaluation\": \"outputs/evaluation\",\n    \"predictions\": \"outputs/predictions\",\n    \"reports\": \"reports\",\n    \"figures\": \"reports/figures\",\n    \"tables\": \"reports/tables\",\n    \"website\": \"website\",\n    \"website_data\": \"website/data\",\n    \"exports\": \"exports\",\n}\n\n\ndef p(key: str) -> Path:\n    \"\"\"Return the persistent (Drive) path registered under `key`.\"\"\"\n    return PROJECT_ROOT / SUBDIRS[key]\n\n\ndef scratch(*parts: str) -> Path:\n    \"\"\"Return (and create) a directory on the ephemeral Colab local disk.\"\"\"\n    d = LOCAL_SCRATCH.joinpath(*parts)\n    d.mkdir(parents=True, exist_ok=True)\n    return d\n\n\ndef ensure_dirs() -> dict:\n    \"\"\"Create the project directory tree on Drive (idempotent).\"\"\"\n    created = []\n    for rel in SUBDIRS.values():\n        d = PROJECT_ROOT / rel\n        if not d.exists():\n            d.mkdir(parents=True, exist_ok=True)\n            created.append(rel)\n    LOCAL_SCRATCH.mkdir(parents=True, exist_ok=True)\n    return {\"project_root\": str(PROJECT_ROOT), \"created\": created}\n\n\ndef assert_not_raw(path) -> None:\n    \"\"\"Guard used by every writer: refuse to write inside the raw ERA5 directory.\"\"\"\n    target = Path(path).absolute()\n    raw = RAW_ERA5_DIR.absolute()\n    if target == raw or raw in target.parents:\n        raise PermissionError(f\"Refusing to write inside the raw ERA5 directory: {target}\")\n\n\ndef canonical_var_name(short_name: str | None, param_id: int | float | None = None) -> str:\n    \"\"\"Map GRIB `shortName` or `paramId` to the canonical ERA5 variable identifier.\"\"\"\n    s = (str(short_name).strip().lower() if short_name is not None else \"\")\n    pid = int(param_id) if param_id is not None and str(param_id).isdigit() else None\n    for canon, spec in ERA5_CANONICAL_VARS.items():\n        if s in spec[\"shortNames\"] or (pid is not None and pid in spec[\"paramIds\"]):\n            return canon\n    return s or f\"param_{pid}\"\n\n\ndef cfgrib_backend_kwargs(grib_path) -> dict:\n    \"\"\"cfgrib options that keep index files on local scratch, never next to raw data.\"\"\"\n    idx = scratch(\"cfgrib_idx\") / (Path(grib_path).name + \".{short_hash}.idx\")\n    return {\"indexpath\": str(idx)}\n\n\ndef as_dict() -> dict:\n    return {\n        \"project_name\": PROJECT_NAME,\n        \"raw_era5_dir\": str(RAW_ERA5_DIR),\n        \"project_root\": str(PROJECT_ROOT),\n        \"local_scratch\": str(LOCAL_SCRATCH),\n        \"india_bbox\": INDIA_BBOX,\n        \"local_timezone\": LOCAL_TIMEZONE,\n        \"ist_utc_offset_minutes\": IST_UTC_OFFSET_MINUTES,\n        \"research_periods\": RESEARCH_PERIODS,\n        \"required_core_vars\": list(REQUIRED_CORE_VARS),\n        \"subdirs\": SUBDIRS,\n    }\n","utils.py":"\"\"\"Runtime utilities: logging, atomic persistence, manifests, pipeline state, review gates.\n\nDesign rule: nothing important lives only in Colab memory. Every stage records its\nstatus in `state/pipeline_state.json`; every per-item job records progress in a\nJSON manifest under `manifests/`. All writes are atomic (write temp file, then\nos.replace) so a disconnect mid-write never leaves a corrupt file behind.\n\"\"\"\nfrom __future__ import annotations\n\nimport datetime as _dt\nimport json\nimport logging\nimport os\nimport socket\nimport sys\nimport time\nimport traceback\nfrom collections import Counter\nfrom contextlib import contextmanager\nfrom pathlib import Path\n\nfrom . import config\n\n\n# --------------------------------------------------------------------------- basics\ndef now_utc() -> str:\n    return _dt.datetime.now(_dt.timezone.utc).strftime(\"%Y-%m-%dT%H:%M:%SZ\")\n\n\ndef human_bytes(n) -> str:\n    if n is None:\n        return \"n/a\"\n    n = float(n)\n    for unit in (\"B\", \"KB\", \"MB\", \"GB\", \"TB\"):\n        if abs(n) < 1024 or unit == \"TB\":\n            return f\"{n:,.2f} {unit}\"\n        n /= 1024.0\n    return f\"{n:,.2f} TB\"\n\n\ndef _tmp_name(path: Path) -> Path:\n    return path.with_name(f\".{path.name}.tmp{os.getpid()}\")\n\n\ndef atomic_write_text(path, text: str) -> Path:\n    path = Path(path)\n    config.assert_not_raw(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = _tmp_name(path)\n    with open(tmp, \"w\", encoding=\"utf-8\") as fh:\n        fh.write(text)\n        fh.flush()\n        os.fsync(fh.fileno())\n    os.replace(tmp, path)\n    return path\n\n\ndef atomic_write_json(path, obj) -> Path:\n    return atomic_write_text(path, json.dumps(obj, indent=2, default=str))\n\n\ndef read_json(path, default=None):\n    path = Path(path)\n    if not path.exists():\n        return default\n    try:\n        return json.loads(path.read_text(encoding=\"utf-8\"))\n    except Exception:\n        # Keep the unreadable file for forensics and start fresh.\n        backup = path.with_name(path.name + f\".corrupt-{int(time.time())}\")\n        try:\n            path.rename(backup)\n        except OSError:\n            pass\n        return default\n\n\ndef write_parquet_atomic(df, path) -> Path:\n    path = Path(path)\n    config.assert_not_raw(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = _tmp_name(path)\n    df.to_parquet(tmp, index=False)\n    os.replace(tmp, path)\n    return path\n\n\ndef write_csv_atomic(df, path, index=False) -> Path:\n    path = Path(path)\n    config.assert_not_raw(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = _tmp_name(path)\n    df.to_csv(tmp, index=index)\n    os.replace(tmp, path)\n    return path\n\n\n# --------------------------------------------------------------------------- logging\n_LOGGER = None\n\n\ndef get_logger() -> logging.Logger:\n    \"\"\"Logger that prints to the notebook and appends to logs/ifci_YYYYMMDD.log on Drive.\"\"\"\n    global _LOGGER\n    if _LOGGER is not None:\n        return _LOGGER\n    logger = logging.getLogger(\"ifci\")\n    logger.setLevel(logging.INFO)\n    logger.propagate = False\n    for h in list(logger.handlers):\n        logger.removeHandler(h)\n    fmt = logging.Formatter(\"%(asctime)s | %(levelname)-7s | %(message)s\", \"%Y-%m-%d %H:%M:%S\")\n    sh = logging.StreamHandler(sys.stdout)\n    sh.setFormatter(fmt)\n    logger.addHandler(sh)\n    try:\n        log_dir = config.p(\"logs\")\n        log_dir.mkdir(parents=True, exist_ok=True)\n        fh = logging.FileHandler(log_dir / f\"ifci_{_dt.date.today():%Y%m%d}.log\", encoding=\"utf-8\")\n        fh.setFormatter(fmt)\n        logger.addHandler(fh)\n    except Exception as exc:  # Drive not mounted yet\n        logger.warning(f\"File logging unavailable: {exc}\")\n    _LOGGER = logger\n    return logger\n\n\ndef log_error(stage: str, exc: BaseException, **context) -> None:\n    \"\"\"Append a structured error record to logs/errors.jsonl and log it.\"\"\"\n    rec = {\n        \"time\": now_utc(),\n        \"stage\": stage,\n        \"error_type\": type(exc).__name__,\n        \"error\": str(exc),\n        \"traceback\": \"\".join(traceback.format_exception(type(exc), exc, exc.__traceback__)),\n        \"context\": context,\n    }\n    try:\n        path = config.p(\"logs\") / \"errors.jsonl\"\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with open(path, \"a\", encoding=\"utf-8\") as fh:\n            fh.write(json.dumps(rec, default=str) + \"\\n\")\n    except Exception:\n        pass\n    get_logger().error(f\"[{stage}] {type(exc).__name__}: {exc}\")\n\n\n# --------------------------------------------------------------------------- manifests\nclass Manifest:\n    \"\"\"JSON-backed per-item status record that makes batch jobs resumable.\n\n    Items are keyed by a stable name (e.g. a GRIB file name) and carry a\n    `fingerprint` (e.g. size:mtime) so that a changed input is re-processed.\n    \"\"\"\n\n    def __init__(self, path):\n        self.path = Path(path)\n        self.data = read_json(self.path, default=None) or {\"created\": now_utc(), \"items\": {}}\n        self.data.setdefault(\"items\", {})\n\n    @property\n    def items(self) -> dict:\n        return self.data[\"items\"]\n\n    def get(self, key):\n        return self.items.get(key)\n\n    def status(self, key) -> str:\n        return (self.items.get(key) or {}).get(\"status\", \"pending\")\n\n    def is_done(self, key, fingerprint=None) -> bool:\n        rec = self.items.get(key)\n        if not rec or rec.get(\"status\") != \"done\":\n            return False\n        return fingerprint is None or rec.get(\"fingerprint\") == fingerprint\n\n    def mark(self, key, status, fingerprint=None, save=True, **info):\n        rec = dict(self.items.get(key) or {})\n        rec.update(info)\n        rec[\"status\"] = status\n        rec[\"updated\"] = now_utc()\n        if fingerprint is not None:\n            rec[\"fingerprint\"] = fingerprint\n        self.items[key] = rec\n        if save:\n            self.save()\n        return rec\n\n    def save(self):\n        self.data[\"updated\"] = now_utc()\n        atomic_write_json(self.path, self.data)\n\n    def counts(self) -> dict:\n        return dict(Counter(r.get(\"status\", \"pending\") for r in self.items.values()))\n\n\n# --------------------------------------------------------------------------- pipeline state\nclass PipelineState:\n    \"\"\"Stage-level status (pending / running / done / failed) persisted on Drive.\"\"\"\n\n    def __init__(self, path=None):\n        self.path = Path(path or (config.p(\"state\") / \"pipeline_state.json\"))\n        self.reload()\n\n    def reload(self):\n        self.data = read_json(self.path, default=None) or {\"stages\": {}}\n        self.data.setdefault(\"stages\", {})\n\n    def _set(self, stage, **kw):\n        self.reload()\n        rec = dict(self.data[\"stages\"].get(stage) or {})\n        rec.update(kw)\n        self.data[\"stages\"][stage] = rec\n        self.data[\"updated\"] = now_utc()\n        atomic_write_json(self.path, self.data)\n\n    def status(self, stage) -> str:\n        return (self.data[\"stages\"].get(stage) or {}).get(\"status\", \"pending\")\n\n    def is_done(self, stage) -> bool:\n        return self.status(stage) == \"done\"\n\n    def annotate(self, stage, **info):\n        self._set(stage, **info)\n\n    def should_run(self, stage, force=False) -> bool:\n        self.reload()\n        if force:\n            return True\n        if self.is_done(stage):\n            when = self.data[\"stages\"][stage].get(\"finished\")\n            get_logger().info(f\"Stage '{stage}' already completed ({when}); skipping. Set FORCE=True to redo.\")\n            return False\n        return True\n\n    @contextmanager\n    def run(self, stage, **info):\n        t0 = time.time()\n        self._set(stage, status=\"running\", started=now_utc(), host=socket.gethostname(), **info)\n        try:\n            yield\n        except BaseException as exc:\n            self._set(stage, status=\"failed\", failed=now_utc(), error=f\"{type(exc).__name__}: {exc}\")\n            if isinstance(exc, Exception):\n                log_error(stage, exc)\n            raise\n        else:\n            self._set(stage, status=\"done\", finished=now_utc(), seconds=round(time.time() - t0, 1))\n\n    def table(self):\n        import pandas as pd\n\n        self.reload()\n        rows = [{\"stage\": k, **v} for k, v in self.data[\"stages\"].items()]\n        return pd.DataFrame(rows)\n\n\n# --------------------------------------------------------------------------- review gate\nclass ReviewGate(Exception):\n    \"\"\"Raised deliberately to stop 'Run all' at a human review checkpoint.\"\"\"\n\n    def _render_traceback_(self):  # IPython hook: show a clean message, not a traceback\n        return [\"\", \"=\" * 78, f\"REVIEW GATE: {self}\", \"=\" * 78]\n\n\ndef markdown_table(df, max_rows=60, floatfmt=\"{:,.3f}\") -> str:\n    \"\"\"Small dependency-free DataFrame -> GitHub markdown table.\"\"\"\n    if df is None or len(df) == 0:\n        return \"_(empty)_\"\n    d = df.head(max_rows)\n\n    def fmt(v):\n        if isinstance(v, float):\n            return floatfmt.format(v)\n        return str(v).replace(\"|\", \"/\")\n\n    head = \"| \" + \" | \".join(map(str, d.columns)) + \" |\"\n    sep = \"| \" + \" | \".join(\"---\" for _ in d.columns) + \" |\"\n    body = [\"| \" + \" | \".join(fmt(v) for v in row) + \" |\" for row in d.itertuples(index=False)]\n    more = [f\"\\n_... {len(df) - max_rows} more rows_\"] if len(df) > max_rows else []\n    return \"\\n\".join([head, sep, *body, *more])\n","diagnostics.py":"\"\"\"Hardware / runtime diagnostics and resource-aware strategy selection.\n\nNever assumes a particular GPU: everything is measured, then the processing\nstrategy (worker counts, temporal block size, AMP dtype, batch size) is derived\nfrom what is actually available in the current Colab runtime.\n\"\"\"\nfrom __future__ import annotations\n\nimport importlib.metadata as md\nimport os\nimport platform\nimport shutil\nimport socket\nimport subprocess\nimport sys\n\nfrom . import config\nfrom .utils import atomic_write_json, atomic_write_text, now_utc\n\nKEY_LIBRARIES = [\n    \"numpy\", \"pandas\", \"pyarrow\", \"xarray\", \"dask\", \"distributed\", \"zarr\", \"numcodecs\",\n    \"netCDF4\", \"h5netcdf\", \"eccodes\", \"eccodeslib\", \"cfgrib\", \"scipy\", \"scikit-learn\",\n    \"xgboost\", \"lightgbm\", \"torch\", \"torchvision\", \"matplotlib\", \"geopandas\", \"shapely\",\n    \"rasterio\", \"rioxarray\", \"requests\", \"psutil\", \"tqdm\",\n]\n\n\ndef _run(cmd, timeout=30) -> str:\n    try:\n        return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout).stdout.strip()\n    except Exception:\n        return \"\"\n\n\ndef _runtime_info() -> dict:\n    uptime_h = None\n    try:\n        with open(\"/proc/uptime\") as fh:\n            uptime_h = round(float(fh.read().split()[0]) / 3600, 2)\n    except OSError:\n        pass\n    return {\n        \"in_colab\": \"google.colab\" in sys.modules or bool(os.environ.get(\"COLAB_RELEASE_TAG\")),\n        \"colab_release_tag\": os.environ.get(\"COLAB_RELEASE_TAG\"),\n        \"python_version\": sys.version.split()[0],\n        \"python_full\": sys.version,\n        \"platform\": platform.platform(),\n        \"hostname\": socket.gethostname(),\n        \"vm_uptime_hours\": uptime_h,\n    }\n\n\ndef _cpu_info() -> dict:\n    import psutil\n\n    model = None\n    try:\n        with open(\"/proc/cpuinfo\") as fh:\n            for line in fh:\n                if line.startswith(\"model name\"):\n                    model = line.split(\":\", 1)[1].strip()\n                    break\n    except OSError:\n        model = platform.processor()\n    try:\n        usable = len(os.sched_getaffinity(0))\n    except Exception:\n        usable = os.cpu_count()\n    freq = psutil.cpu_freq()\n    return {\n        \"model\": model,\n        \"logical_cores\": os.cpu_count(),\n        \"physical_cores\": psutil.cpu_count(logical=False),\n        \"usable_cores\": usable,\n        \"max_mhz\": round(freq.max, 0) if freq and freq.max else None,\n        \"load_avg_1_5_15\": list(os.getloadavg()) if hasattr(os, \"getloadavg\") else None,\n    }\n\n\ndef _memory_info() -> dict:\n    import psutil\n\n    vm = psutil.virtual_memory()\n    sw = psutil.swap_memory()\n    gb = 1024 ** 3\n    return {\n        \"total_gb\": round(vm.total / gb, 2),\n        \"available_gb\": round(vm.available / gb, 2),\n        \"used_gb\": round(vm.used / gb, 2),\n        \"percent_used\": vm.percent,\n        \"swap_total_gb\": round(sw.total / gb, 2),\n    }\n\n\ndef _gpu_info() -> dict:\n    info = {\"available\": False, \"devices\": [], \"tpu\": False}\n    fields = \"index,name,memory.total,memory.used,memory.free,driver_version,compute_cap\"\n    q = _run([\"nvidia-smi\", f\"--query-gpu={fields}\", \"--format=csv,noheader,nounits\"])\n    if not q:  # older drivers do not know compute_cap\n        q = _run([\"nvidia-smi\", \"--query-gpu=index,name,memory.total,memory.used,memory.free,driver_version\",\n                  \"--format=csv,noheader,nounits\"])\n    for line in q.splitlines():\n        parts = [x.strip() for x in line.split(\",\")]\n        if len(parts) >= 6:\n            info[\"devices\"].append({\n                \"index\": parts[0], \"name\": parts[1],\n                \"vram_total_gb\": round(float(parts[2]) / 1024, 2),\n                \"vram_used_gb\": round(float(parts[3]) / 1024, 2),\n                \"vram_free_gb\": round(float(parts[4]) / 1024, 2),\n                \"driver\": parts[5],\n                \"compute_capability\": parts[6] if len(parts) > 6 else None,\n            })\n    info[\"available\"] = bool(info[\"devices\"])\n    try:\n        import torch\n\n        t = {\n            \"torch_version\": torch.__version__,\n            \"cuda_available\": torch.cuda.is_available(),\n            \"cuda_runtime\": torch.version.cuda,\n            \"cudnn\": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,\n        }\n        if torch.cuda.is_available():\n            pr = torch.cuda.get_device_properties(0)\n            t.update({\n                \"device_name\": pr.name,\n                \"vram_gb\": round(pr.total_memory / 1024 ** 3, 2),\n                \"capability\": f\"{pr.major}.{pr.minor}\",\n                \"multiprocessors\": pr.multi_processor_count,\n                \"bf16_supported\": bool(torch.cuda.is_bf16_supported()),\n                \"device_count\": torch.cuda.device_count(),\n            })\n        info[\"torch\"] = t\n        info[\"available\"] = info[\"available\"] or t[\"cuda_available\"]\n    except Exception as exc:\n        info[\"torch\"] = {\"error\": f\"{type(exc).__name__}: {exc}\"}\n    info[\"tpu\"] = bool(os.environ.get(\"COLAB_TPU_ADDR\") or os.path.exists(\"/dev/accel0\"))\n    return info\n\n\ndef _disk_info() -> dict:\n    out = {}\n    gb = 1024 ** 3\n    for label, path in ((\"vm_root\", \"/\"), (\"vm_content\", \"/content\"), (\"google_drive\", str(config.DRIVE_ROOT))):\n        try:\n            u = shutil.disk_usage(path)\n            out[label] = {\"path\": path, \"total_gb\": round(u.total / gb, 1),\n                          \"used_gb\": round(u.used / gb, 1), \"free_gb\": round(u.free / gb, 1)}\n        except Exception as exc:\n            out[label] = {\"path\": path, \"error\": str(exc)}\n    return out\n\n\ndef _library_versions() -> dict:\n    out = {}\n    for name in KEY_LIBRARIES:\n        try:\n            out[name] = md.version(name)\n        except md.PackageNotFoundError:\n            out[name] = None\n    try:\n        import eccodes\n\n        out[\"ecCodes_C_library\"] = eccodes.codes_get_api_version()\n    except Exception as exc:\n        out[\"ecCodes_C_library\"] = f\"unavailable ({type(exc).__name__}: {exc})\"\n    return out\n\n\ndef collect_diagnostics() -> dict:\n    return {\n        \"timestamp_utc\": now_utc(),\n        \"runtime\": _runtime_info(),\n        \"cpu\": _cpu_info(),\n        \"memory\": _memory_info(),\n        \"gpu\": _gpu_info(),\n        \"disk\": _disk_info(),\n        \"libraries\": _library_versions(),\n    }\n\n\ndef recommend_strategy(d: dict) -> dict:\n    \"\"\"Derive processing choices from measured resources (no hardware assumptions).\"\"\"\n    mem = d[\"memory\"][\"total_gb\"]\n    avail = d[\"memory\"][\"available_gb\"]\n    cores = d[\"cpu\"][\"usable_cores\"] or 1\n    t = d[\"gpu\"].get(\"torch\", {}) or {}\n    cuda = bool(t.get(\"cuda_available\"))\n    vram = t.get(\"vram_gb\") or 0.0\n    cap = t.get(\"capability\")\n    major = int(cap.split(\".\")[0]) if cap else 0\n\n    if not cuda:\n        batch = 2\n    elif vram >= 70:\n        batch = 64\n    elif vram >= 35:\n        batch = 32\n    elif vram >= 20:\n        batch = 16\n    elif vram >= 12:\n        batch = 8\n    else:\n        batch = 4\n\n    s = {\n        \"ram_tier\": \"high-ram\" if mem >= 40 else (\"standard\" if mem >= 20 else \"low\"),\n        # Metadata scan is I/O bound on the Drive FUSE mount: a few parallel streams help.\n        \"grib_audit_workers\": max(1, min(4, cores - 1)),\n        # Full GRIB decoding: ~6 GB RAM head-room per worker for a monthly block.\n        \"grib_decode_workers\": max(1, min(cores, int(avail // 6))),\n        \"era5_temporal_block\": \"month\" if avail >= 12 else \"10-day\",\n        \"dask_threads\": cores,\n        \"torch_device\": \"cuda\" if cuda else \"cpu\",\n        \"amp_dtype\": (\"bfloat16\" if major >= 8 else \"float16\") if cuda else None,\n        \"initial_batch_size\": batch,  # refined later by an automatic OOM back-off finder\n        \"dataloader_workers\": max(0, min(4, cores - 1)),\n        \"pin_memory\": cuda,\n        \"warnings\": [],\n    }\n    if mem < 40:\n        s[\"warnings\"].append(\"High-RAM runtime not active (Runtime > Change runtime type > High-RAM). \"\n                             \"Processing will use smaller temporal blocks.\")\n    if not cuda:\n        s[\"warnings\"].append(\"No CUDA GPU detected. Fine for Sections 3-12; a GPU runtime is needed for \"\n                             \"Section 13 (deep learning).\")\n    free_local = d[\"disk\"].get(\"vm_content\", {}).get(\"free_gb\")\n    if free_local is not None and free_local < 40:\n        s[\"warnings\"].append(f\"Only {free_local} GB free on the Colab local disk; scratch usage will be minimised.\")\n    return s\n\n\ndef format_report(d: dict, s: dict) -> str:\n    r, c, m, g, dk = d[\"runtime\"], d[\"cpu\"], d[\"memory\"], d[\"gpu\"], d[\"disk\"]\n    t = g.get(\"torch\", {}) or {}\n    lines = [\n        \"=\" * 78,\n        f\"RUNTIME DIAGNOSTICS  ({d['timestamp_utc']})\",\n        \"=\" * 78,\n        f\"Python            : {r['python_version']}   | Colab: {r['in_colab']} ({r.get('colab_release_tag')})\",\n        f\"Platform          : {r['platform']}\",\n        f\"CPU               : {c['model']}\",\n        f\"Cores             : {c['usable_cores']} usable / {c['logical_cores']} logical / {c['physical_cores']} physical\",\n        f\"RAM               : {m['total_gb']} GB total, {m['available_gb']} GB available ({m['percent_used']}% used)\",\n    ]\n    if g[\"devices\"]:\n        for dev in g[\"devices\"]:\n            lines.append(f\"GPU {dev['index']}             : {dev['name']} | VRAM {dev['vram_total_gb']} GB \"\n                         f\"(free {dev['vram_free_gb']} GB) | driver {dev['driver']} | cc {dev['compute_capability']}\")\n    else:\n        lines.append(\"GPU               : none detected\")\n    if t.get(\"cuda_available\"):\n        lines.append(f\"PyTorch           : {t['torch_version']} | CUDA {t['cuda_runtime']} | cuDNN {t['cudnn']} | \"\n                     f\"bf16 {t.get('bf16_supported')}\")\n    elif \"torch_version\" in t:\n        lines.append(f\"PyTorch           : {t['torch_version']} (CPU only)\")\n    for label, v in dk.items():\n        if \"error\" in v:\n            lines.append(f\"Disk {label:<13}: {v['path']} -> {v['error']}\")\n        else:\n            lines.append(f\"Disk {label:<13}: {v['path']} total {v['total_gb']} GB, free {v['free_gb']} GB\")\n    lines.append(\"-\" * 78)\n    lines.append(\"Key libraries:\")\n    libs = d[\"libraries\"]\n    for k in sorted(libs):\n        lines.append(f\"  {k:<20} {libs[k] if libs[k] else '(not installed)'}\")\n    lines.append(\"-\" * 78)\n    lines.append(\"Selected strategy:\")\n    for k, v in s.items():\n        if k != \"warnings\":\n            lines.append(f\"  {k:<22} {v}\")\n    for w in s[\"warnings\"]:\n        lines.append(f\"  WARNING: {w}\")\n    lines.append(\"=\" * 78)\n    return \"\\n\".join(lines)\n\n\ndef save_diagnostics(d: dict, s: dict) -> str:\n    out = config.p(\"diagnostics\")\n    stamp = d[\"timestamp_utc\"].replace(\":\", \"\").replace(\"-\", \"\")\n    payload = {\"diagnostics\": d, \"strategy\": s}\n    atomic_write_json(out / f\"diag_{stamp}.json\", payload)\n    atomic_write_json(out / \"diag_latest.json\", payload)\n    freeze = _run([sys.executable, \"-m\", \"pip\", \"freeze\"], timeout=120)\n    atomic_write_text(out / f\"pip_freeze_{stamp}.txt\", freeze + \"\\n\")\n    atomic_write_text(out / \"pip_freeze_latest.txt\", freeze + \"\\n\")\n    return str(out / f\"diag_{stamp}.json\")\n","discovery.py":"\"\"\"Raw ERA5 file discovery and cheap integrity checks.\n\nOnly the first and last few MB of each file are read (GRIB magic bytes, edition,\n'7777' end marker, quick fingerprint) - never the whole file. Results are cached\nin a manifest keyed by size+mtime, so re-runs after a disconnect are instant.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nimport re\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom . import config\nfrom .utils import Manifest, get_logger, human_bytes, write_csv_atomic, write_parquet_atomic\n\nGRIB_EXTENSIONS = {\".grib\", \".grb\", \".grib1\", \".grib2\", \".grb2\", \".gb2\"}\nINFO_KEYS = [\"quick_sha1\", \"grib_magic_ok\", \"first_message_edition\", \"ends_with_7777\"]\n\n\ndef natural_key(s) -> list:\n    return [int(t) if t.isdigit() else t.lower() for t in re.split(r\"(\\d+)\", str(s))]\n\n\ndef _classify(name: str, is_dir: bool) -> str:\n    if is_dir:\n        return \"directory\"\n    low = name.lower()\n    if low.endswith(\".idx\"):\n        return \"cfgrib_index\"\n    if Path(low).suffix in GRIB_EXTENSIONS:\n        return \"grib\"\n    return \"other\"\n\n\ndef list_raw_directory(raw_dir=None) -> pd.DataFrame:\n    raw_dir = Path(raw_dir or config.RAW_ERA5_DIR)\n    if not raw_dir.exists():\n        raise FileNotFoundError(f\"Raw ERA5 directory not found: {raw_dir}\")\n    rows = []\n    with os.scandir(raw_dir) as it:\n        for e in it:\n            is_dir = e.is_dir()\n            st = e.stat()\n            rows.append({\n                \"name\": e.name,\n                \"path\": e.path,\n                \"kind\": _classify(e.name, is_dir),\n                \"size_bytes\": 0 if is_dir else int(st.st_size),\n                \"mtime\": pd.Timestamp(st.st_mtime, unit=\"s\", tz=\"UTC\"),\n            })\n    df = pd.DataFrame(rows)\n    if len(df):\n        df = df.iloc[sorted(range(len(df)), key=lambda i: natural_key(df[\"name\"].iloc[i]))].reset_index(drop=True)\n    return df\n\n\ndef file_fingerprint(size_bytes, mtime) -> str:\n    return f\"{int(size_bytes)}:{pd.Timestamp(mtime).value}\"\n\n\ndef inspect_file_ends(path, nbytes: int = 4 * 1024 * 1024) -> dict:\n    \"\"\"Read head + tail only: GRIB magic, first-message edition, '7777' end marker, quick hash.\"\"\"\n    size = os.path.getsize(path)\n    h = hashlib.sha1(str(size).encode())\n    with open(path, \"rb\") as fh:\n        head = fh.read(min(nbytes, size))\n        if size > nbytes:\n            fh.seek(size - nbytes)\n            tail = fh.read(nbytes)\n        else:\n            tail = head\n    h.update(head)\n    h.update(tail)\n    magic = head[:4] == b\"GRIB\"\n    return {\n        \"quick_sha1\": h.hexdigest(),\n        \"grib_magic_ok\": bool(magic),\n        \"first_message_edition\": int(head[7]) if magic and len(head) >= 8 else None,\n        \"ends_with_7777\": tail[-4:] == b\"7777\",\n    }\n\n\ndef build_inventory(raw_dir=None, workers: int = 8, force: bool = False) -> pd.DataFrame:\n    log = get_logger()\n    df = list_raw_directory(raw_dir)\n    man = Manifest(config.p(\"manifests\") / \"raw_inventory_manifest.json\")\n    grib = df[df[\"kind\"] == \"grib\"]\n\n    def work(row):\n        fp = file_fingerprint(row.size_bytes, row.mtime)\n        if not force and man.is_done(row.name, fp):\n            return row.name, fp, None\n        return row.name, fp, inspect_file_ends(row.path)\n\n    new = 0\n    with ThreadPoolExecutor(max_workers=workers) as ex:\n        for name, fp, info in ex.map(work, grib.itertuples(index=False)):\n            if info is not None:\n                man.mark(name, \"done\", fingerprint=fp, save=False, **info)\n                new += 1\n    man.save()\n    log.info(f\"Inventory: {len(df)} entries, {len(grib)} GRIB files ({new} newly inspected, \"\n             f\"{len(grib) - new} from cache).\")\n\n    for k in INFO_KEYS:\n        df[k] = [(man.get(n) or {}).get(k) if kind == \"grib\" else None for n, kind in zip(df[\"name\"], df[\"kind\"])]\n    df[\"size_gb\"] = (df[\"size_bytes\"] / 1024 ** 3).round(3)\n    dup = df[df[\"kind\"] == \"grib\"].groupby(\"quick_sha1\")[\"name\"].transform(\"count\")\n    df[\"duplicate_candidate\"] = False\n    df.loc[dup.index, \"duplicate_candidate\"] = dup > 1\n\n    write_parquet_atomic(df, config.p(\"metadata\") / \"raw_file_inventory.parquet\")\n    write_csv_atomic(df.drop(columns=[\"path\"]), config.p(\"metadata\") / \"raw_file_inventory.csv\")\n    return df\n\n\ndef inventory_report(df: pd.DataFrame) -> str:\n    g = df[df[\"kind\"] == \"grib\"]\n    lines = [\n        f\"Raw directory      : {config.RAW_ERA5_DIR}\",\n        f\"GRIB files         : {len(g)}  (total {human_bytes(g['size_bytes'].sum())})\",\n        f\"Other entries      : \" + \", \".join(f\"{k}={v}\" for k, v in df[df['kind'] != 'grib']['kind'].value_counts().items()),\n        f\"Size range         : {human_bytes(g['size_bytes'].min())} - {human_bytes(g['size_bytes'].max())}\",\n        f\"Editions (1st msg) : {g['first_message_edition'].value_counts(dropna=False).to_dict()}\",\n        f\"Bad GRIB magic     : {list(g.loc[~g['grib_magic_ok'].astype(bool), 'name'])}\",\n        f\"Missing '7777' end : {list(g.loc[~g['ends_with_7777'].astype(bool), 'name'])}  (possible truncation)\",\n    ]\n    dups = g[g[\"duplicate_candidate\"]]\n    if len(dups):\n        groups = dups.groupby(\"quick_sha1\")[\"name\"].apply(list).tolist()\n        lines.append(f\"Duplicate candidates (identical size + head/tail hash): {groups}\")\n    else:\n        lines.append(\"Duplicate candidates: none (by size + head/tail hash)\")\n    idx = df[df[\"kind\"] == \"cfgrib_index\"]\n    if len(idx):\n        lines.append(f\"NOTE: cfgrib index files found in raw folder {list(idx['name'])}. They are left untouched; \"\n                     \"this project writes cfgrib indexes to local scratch only.\")\n    return \"\\n\".join(lines)\n","grib_audit.py":"\"\"\"GRIB metadata audit (headers only) with resumable, parallel, per-file processing.\n\nFor every GRIB message we read header keys only (ecCodes `headers_only=True`, the\ndata section is skipped), so a 3 GB file is scanned without decoding any field.\nOne message per variable is fully decoded afterwards to sanity-check values.\n\nOutputs (Drive):\n  metadata/grib_audit/messages/<file>.parquet   one row per GRIB message\n  metadata/grib_audit/value_samples/<file>.json value statistics per variable\n  manifests/grib_audit_manifest.json            per-file status + summary (resume point)\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport multiprocessing as mp\nimport os\nimport time\nimport traceback\nfrom concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .utils import (Manifest, atomic_write_json, get_logger, log_error, read_json,\n                    write_csv_atomic, write_parquet_atomic)\n\nHEADER_KEYS = [\n    \"edition\", \"centre\", \"dataType\", \"stream\", \"expver\", \"number\", \"paramId\", \"shortName\", \"name\",\n    \"units\", \"typeOfLevel\", \"level\", \"stepType\", \"stepRange\", \"startStep\", \"endStep\",\n    \"dataDate\", \"dataTime\", \"validityDate\", \"validityTime\", \"gridType\", \"Ni\", \"Nj\",\n    \"latitudeOfFirstGridPointInDegrees\", \"longitudeOfFirstGridPointInDegrees\",\n    \"latitudeOfLastGridPointInDegrees\", \"longitudeOfLastGridPointInDegrees\",\n    \"iDirectionIncrementInDegrees\", \"jDirectionIncrementInDegrees\",\n    \"packingType\", \"bitsPerValue\", \"totalLength\",\n]\nSTRING_KEYS = [\"centre\", \"dataType\", \"stream\", \"expver\", \"shortName\", \"name\", \"units\", \"typeOfLevel\",\n               \"stepType\", \"stepRange\", \"gridType\", \"packingType\"]\nNUMERIC_KEYS = [k for k in HEADER_KEYS if k not in STRING_KEYS]\nGRID_KEYS = [\"gridType\", \"Ni\", \"Nj\", \"latitudeOfFirstGridPointInDegrees\", \"longitudeOfFirstGridPointInDegrees\",\n             \"latitudeOfLastGridPointInDegrees\", \"longitudeOfLastGridPointInDegrees\",\n             \"iDirectionIncrementInDegrees\", \"jDirectionIncrementInDegrees\"]\nGRID_SHORT = {\"latitudeOfFirstGridPointInDegrees\": \"lat_first\", \"longitudeOfFirstGridPointInDegrees\": \"lon_first\",\n              \"latitudeOfLastGridPointInDegrees\": \"lat_last\", \"longitudeOfLastGridPointInDegrees\": \"lon_last\",\n              \"iDirectionIncrementInDegrees\": \"dlon\", \"jDirectionIncrementInDegrees\": \"dlat\"}\n\n\n# --------------------------------------------------------------------------- helpers\ndef _get(gid, key):\n    import eccodes\n\n    try:\n        return eccodes.codes_get(gid, key)\n    except Exception:\n        return None\n\n\ndef _progress_path(name: str) -> Path:\n    return config.scratch(\"progress\") / f\"{name}.json\"\n\n\ndef _write_progress(name: str, **kw) -> None:\n    try:\n        p = _progress_path(name)\n        tmp = p.with_name(p.name + \".tmp\")\n        tmp.write_text(json.dumps(kw))\n        os.replace(tmp, p)\n    except Exception:\n        pass\n\n\ndef _read_progress(name: str):\n    try:\n        return json.loads(_progress_path(name).read_text())\n    except Exception:\n        return None\n\n\ndef to_datetime(date_col: pd.Series, time_col: pd.Series) -> pd.Series:\n    \"\"\"GRIB YYYYMMDD + HHMM integer keys -> datetime64[ns] (UTC, naive).\"\"\"\n    d = pd.to_numeric(date_col, errors=\"coerce\")\n    t = pd.to_numeric(time_col, errors=\"coerce\")\n    ok = d.notna() & t.notna()\n    out = pd.Series(pd.NaT, index=date_col.index, dtype=\"datetime64[ns]\")\n    if ok.any():\n        s = d[ok].astype(\"int64\").astype(str) + t[ok].astype(\"int64\").astype(str).str.zfill(4)\n        out.loc[ok] = pd.to_datetime(s, format=\"%Y%m%d%H%M\", errors=\"coerce\")\n    return out\n\n\n# --------------------------------------------------------------------------- per-file scan\ndef scan_grib_file(path, out_parquet, progress_every: int = 2000, sample_values: bool = True) -> dict:\n    import eccodes\n\n    path = Path(path)\n    name = path.name\n    size = path.stat().st_size\n    t0 = time.time()\n    cols = {k: [] for k in HEADER_KEYS}\n    offsets = []\n    first_msg = {}\n    cum = 0\n    n = 0\n    _write_progress(name, status=\"scanning\", messages=0, bytes_done=0, size=size, started=t0)\n    with open(path, \"rb\") as fh:\n        while True:\n            gid = eccodes.codes_grib_new_from_file(fh, headers_only=True)\n            if gid is None:\n                break\n            try:\n                vals = [_get(gid, k) for k in HEADER_KEYS]\n                off = _get(gid, \"offset\")\n            finally:\n                eccodes.codes_release(gid)\n            rec = dict(zip(HEADER_KEYS, vals))\n            length = rec.get(\"totalLength\")\n            off = int(off) if off is not None else cum\n            if length:\n                cum = off + int(length)\n            for k, v in rec.items():\n                cols[k].append(v)\n            offsets.append(off)\n            key = (rec.get(\"shortName\"), rec.get(\"typeOfLevel\"), rec.get(\"level\"))\n            if key not in first_msg and length:\n                first_msg[key] = (off, int(length))\n            n += 1\n            if n % progress_every == 0:\n                _write_progress(name, status=\"scanning\", messages=n, bytes_done=cum, size=size, started=t0)\n\n    df = pd.DataFrame(cols)\n    df.insert(0, \"file\", name)\n    df[\"msg_index\"] = np.arange(len(df), dtype=\"int64\")\n    df[\"offset\"] = np.asarray(offsets, dtype=\"int64\")\n    for k in NUMERIC_KEYS:\n        df[k] = pd.to_numeric(df[k], errors=\"coerce\")\n    for k in STRING_KEYS:\n        df[k] = df[k].astype(\"string\")\n    df[\"valid_time\"] = to_datetime(df[\"validityDate\"], df[\"validityTime\"])\n    df[\"ref_time\"] = to_datetime(df[\"dataDate\"], df[\"dataTime\"])\n    write_parquet_atomic(df, out_parquet)\n\n    samples = _sample_values(path, first_msg) if sample_values else []\n    elapsed = time.time() - t0\n    summary = summarise_messages(df)\n    summary.update({\n        \"file\": name,\n        \"size_bytes\": int(size),\n        \"bytes_accounted_by_messages\": int(df[\"totalLength\"].sum()) if len(df) else 0,\n        \"scan_seconds\": round(elapsed, 1),\n        \"throughput_mb_s\": round(size / 1e6 / max(elapsed, 1e-6), 1),\n    })\n    _write_progress(name, status=\"done\", messages=n, bytes_done=size, size=size, started=t0)\n    return {\"summary\": summary, \"samples\": samples}\n\n\ndef _sample_values(path, first_msg: dict) -> list:\n    \"\"\"Fully decode the first message of each variable and compute value statistics.\"\"\"\n    import eccodes\n\n    out = []\n    with open(path, \"rb\") as fh:\n        for (short, tol, lev), (off, length) in first_msg.items():\n            rec = {\"file\": Path(path).name, \"shortName\": short, \"typeOfLevel\": tol, \"level\": lev, \"offset\": off}\n            try:\n                fh.seek(off)\n                buf = fh.read(length)\n                if buf[:4] != b\"GRIB\":\n                    raise ValueError(\"GRIB start marker not found at recorded offset\")\n                gid = eccodes.codes_new_from_message(buf)\n                try:\n                    vals = np.asarray(eccodes.codes_get_values(gid), dtype=\"float64\")\n                    miss = eccodes.codes_get(gid, \"missingValue\")\n                    bitmap = eccodes.codes_get(gid, \"bitmapPresent\")\n                    units = eccodes.codes_get(gid, \"units\")\n                    vdate = eccodes.codes_get(gid, \"validityDate\")\n                    vtime = eccodes.codes_get(gid, \"validityTime\")\n                finally:\n                    eccodes.codes_release(gid)\n                if bitmap:\n                    vals[vals == miss] = np.nan\n                fin = np.isfinite(vals)\n                rec.update({\"units\": units, \"valid\": f\"{vdate} {int(vtime):04d}\", \"n_values\": int(vals.size),\n                            \"missing_fraction\": round(float(1 - fin.mean()), 4)})\n                if fin.any():\n                    v = vals[fin]\n                    rec.update({\"min\": float(v.min()), \"p01\": float(np.percentile(v, 1)), \"mean\": float(v.mean()),\n                                \"p99\": float(np.percentile(v, 99)), \"max\": float(v.max()), \"std\": float(v.std())})\n            except Exception as exc:\n                rec[\"error\"] = f\"{type(exc).__name__}: {exc}\"\n            out.append(rec)\n    return out\n\n\ndef summarise_messages(df: pd.DataFrame) -> dict:\n    s = {\"n_messages\": int(len(df))}\n    if len(df) == 0:\n        return s\n    vt = df[\"valid_time\"].dropna()\n    uniq = np.unique(vt.to_numpy(dtype=\"datetime64[ns]\"))\n    s[\"valid_time_min\"] = str(pd.Timestamp(uniq[0])) if len(uniq) else None\n    s[\"valid_time_max\"] = str(pd.Timestamp(uniq[-1])) if len(uniq) else None\n    s[\"n_unique_valid_times\"] = int(len(uniq))\n    if len(uniq) > 1:\n        diffs = np.diff(uniq).astype(\"timedelta64[m]\").astype(\"int64\")\n        vals, cnt = np.unique(diffs, return_counts=True)\n        modal = vals[cnt.argmax()]\n        s[\"modal_time_step_hours\"] = float(modal / 60)\n        s[\"n_time_gaps_gt_modal\"] = int((diffs > modal).sum())\n    s[\"hours_of_day_utc\"] = sorted(int(h) for h in pd.DatetimeIndex(uniq).hour.unique())\n    s[\"variables\"] = sorted(set(df[\"shortName\"].dropna().astype(str)))\n    s[\"messages_per_variable\"] = {str(k): int(v) for k, v in df.groupby(\"shortName\").size().items()}\n    s[\"levels\"] = sorted(set(df[\"typeOfLevel\"].astype(str) + \":\" + df[\"level\"].astype(str)))\n    for col, key in ((\"edition\", \"editions\"), (\"dataType\", \"data_types\"), (\"stream\", \"streams\"),\n                     (\"expver\", \"expvers\"), (\"stepType\", \"step_types\"), (\"packingType\", \"packing\"),\n                     (\"bitsPerValue\", \"bits_per_value\")):\n        vals = df[col].dropna()\n        if pd.api.types.is_float_dtype(vals) and len(vals) and (vals % 1 == 0).all():\n            vals = vals.astype(\"int64\")\n        s[key] = sorted(set(vals.astype(str)))\n    s[\"ensemble_members\"] = int(df[\"number\"].nunique(dropna=True))\n    grids = df[GRID_KEYS].drop_duplicates().rename(columns=GRID_SHORT)\n    s[\"grids\"] = grids.to_dict(\"records\")\n    key_cols = [\"shortName\", \"typeOfLevel\", \"level\", \"number\", \"valid_time\"]\n    s[\"within_file_duplicate_messages\"] = int(df.duplicated(subset=key_cols, keep=\"first\").sum())\n    return s\n\n\n# --------------------------------------------------------------------------- orchestration\ndef _worker(path, out, sample_values):\n    try:\n        return {\"ok\": True, **scan_grib_file(path, out, sample_values=sample_values)}\n    except BaseException as exc:  # report everything back to the parent\n        return {\"ok\": False, \"error\": f\"{type(exc).__name__}: {exc}\", \"traceback\": traceback.format_exc()}\n\n\ndef _progress_text(todo, man, t0) -> str:\n    el = time.time() - t0\n    st = [man.status(t[0]) for t in todo]\n    lines = [f\"[{int(el // 3600):02d}:{int(el % 3600 // 60):02d}:{int(el % 60):02d}] \"\n             f\"done {st.count('done')}/{len(todo)} | failed {st.count('failed')} | \"\n             f\"queued {st.count('queued')}\"]\n    for (name, *_), status in zip(todo, st):\n        if status in (\"done\", \"failed\"):\n            continue\n        pr = _read_progress(name)\n        if pr and pr.get(\"status\") == \"scanning\":\n            pct = 100 * pr[\"bytes_done\"] / max(pr[\"size\"], 1)\n            rate = pr[\"bytes_done\"] / 1e6 / max(time.time() - pr[\"started\"], 1e-6)\n            lines.append(f\"   scanning {name:<16} {pct:5.1f}%  {pr['messages']:>9,} msgs  {rate:6.1f} MB/s\")\n    return \"\\n\".join(lines)\n\n\ndef audit_all(inventory: pd.DataFrame, workers: int = 2, force: bool = False,\n              sample_values: bool = True, poll_seconds: int = 10) -> Manifest:\n    \"\"\"Scan every GRIB file not yet audited (resumable across Colab sessions).\"\"\"\n    log = get_logger()\n    man = Manifest(config.p(\"manifests\") / \"grib_audit_manifest.json\")\n    msg_dir = config.p(\"grib_messages\")\n    msg_dir.mkdir(parents=True, exist_ok=True)\n    grib = inventory[inventory[\"kind\"] == \"grib\"]\n    todo = []\n    for r in grib.itertuples(index=False):\n        fp = f\"{int(r.size_bytes)}:{pd.Timestamp(r.mtime).value}\"\n        out = msg_dir / f\"{r.name}.parquet\"\n        if not force and man.is_done(r.name, fp) and out.exists():\n            continue\n        todo.append((r.name, r.path, str(out), fp, int(r.size_bytes)))\n    log.info(f\"GRIB audit: {len(grib)} files | {len(grib) - len(todo)} already audited | \"\n             f\"{len(todo)} to scan with {workers} parallel worker(s).\")\n    if not todo:\n        return man\n\n    todo.sort(key=lambda t: -t[4])  # largest first for better load balance\n    for name, *_ in todo:\n        try:\n            _progress_path(name).unlink()\n        except FileNotFoundError:\n            pass\n        man.mark(name, \"queued\", save=False)\n    man.save()\n\n    try:\n        from IPython.display import Pretty, display\n        handle = display(Pretty(\"starting workers ...\"), display_id=True)\n    except Exception:\n        Pretty, handle = None, None\n\n    t0 = time.time()\n    ctx = mp.get_context(\"spawn\")  # safer than fork inside a multi-threaded Jupyter kernel\n    with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:\n        futs = {ex.submit(_worker, path, out, sample_values): (name, fp, out) for name, path, out, fp, _ in todo}\n        pending = set(futs)\n        while pending:\n            done, pending = wait(pending, timeout=poll_seconds, return_when=FIRST_COMPLETED)\n            for f in done:\n                name, fp, out = futs[f]\n                try:\n                    res = f.result()\n                except BaseException as exc:  # e.g. worker killed by OOM -> BrokenProcessPool\n                    res = {\"ok\": False, \"error\": f\"{type(exc).__name__}: {exc}\", \"traceback\": traceback.format_exc()}\n                if res[\"ok\"]:\n                    atomic_write_json(config.p(\"value_samples\") / f\"{name}.json\", res[\"samples\"])\n                    s = res[\"summary\"]\n                    man.mark(name, \"done\", fingerprint=fp, messages_parquet=out, summary=s)\n                    log.info(f\"OK {name}: {s['n_messages']:,} msgs | {len(s.get('variables', []))} vars | \"\n                             f\"{s.get('valid_time_min')} -> {s.get('valid_time_max')} | \"\n                             f\"{s['scan_seconds']:.0f}s @ {s['throughput_mb_s']} MB/s\")\n                else:\n                    man.mark(name, \"failed\", fingerprint=fp, error=res[\"error\"])\n                    log_error(\"grib_audit\", RuntimeError(res[\"error\"]), file=name, worker_traceback=res.get(\"traceback\"))\n            text = _progress_text(todo, man, t0)\n            if handle is not None:\n                handle.update(Pretty(text))\n    log.info(f\"GRIB audit finished: {man.counts()}\")\n    return man\n\n\n# --------------------------------------------------------------------------- consolidation\ndef load_messages(columns=None) -> pd.DataFrame:\n    files = sorted(config.p(\"grib_messages\").glob(\"*.parquet\"))\n    if not files:\n        raise FileNotFoundError(\"No audited message tables found - run Section 4 first.\")\n    parts = [pd.read_parquet(f, columns=columns) for f in files]\n    df = pd.concat(parts, ignore_index=True)\n    for c in (\"file\", \"shortName\", \"typeOfLevel\", \"stepType\", \"dataType\", \"expver\"):\n        if c in df.columns:\n            df[c] = df[c].astype(\"category\")\n    return df\n\n\ndef build_audit_tables() -> dict:\n    \"\"\"Consolidate per-file results into project-level audit tables on Drive.\"\"\"\n    from .discovery import natural_key\n\n    man = Manifest(config.p(\"manifests\") / \"grib_audit_manifest.json\")\n    rows = []\n    for name, rec in man.items.items():\n        if rec.get(\"status\") != \"done\":\n            rows.append({\"file\": name, \"status\": rec.get(\"status\"), \"error\": rec.get(\"error\")})\n            continue\n        s = rec[\"summary\"]\n        g = s.get(\"grids\", [{}])\n        rows.append({\n            \"file\": name, \"status\": \"done\",\n            \"size_gb\": round(s[\"size_bytes\"] / 1024 ** 3, 3),\n            \"n_messages\": s[\"n_messages\"],\n            \"n_variables\": len(s.get(\"variables\", [])),\n            \"variables\": \",\".join(s.get(\"variables\", [])),\n            \"valid_time_min\": s.get(\"valid_time_min\"), \"valid_time_max\": s.get(\"valid_time_max\"),\n            \"n_valid_times\": s.get(\"n_unique_valid_times\"),\n            \"step_h\": s.get(\"modal_time_step_hours\"),\n            \"n_gaps\": s.get(\"n_time_gaps_gt_modal\"),\n            \"n_hours_of_day\": len(s.get(\"hours_of_day_utc\", [])),\n            \"editions\": \",\".join(s.get(\"editions\", [])),\n            \"expver\": \",\".join(s.get(\"expvers\", [])),\n            \"n_grids\": len(g),\n            \"grid\": \"; \".join(f\"{x.get('gridType')} {x.get('Ni')}x{x.get('Nj')} \"\n                              f\"lat {x.get('lat_first')}..{x.get('lat_last')} \"\n                              f\"lon {x.get('lon_first')}..{x.get('lon_last')} \"\n                              f\"d={x.get('dlat')}/{x.get('dlon')}\" for x in g),\n            \"within_file_dups\": s.get(\"within_file_duplicate_messages\"),\n            \"bytes_unaccounted\": s[\"size_bytes\"] - s.get(\"bytes_accounted_by_messages\", 0),\n            \"scan_s\": s.get(\"scan_seconds\"),\n            \"MB_s\": s.get(\"throughput_mb_s\"),\n        })\n    fs = pd.DataFrame(rows)\n    if len(fs):\n        fs = fs.iloc[sorted(range(len(fs)), key=lambda i: natural_key(fs[\"file\"].iloc[i]))].reset_index(drop=True)\n\n    cols = [\"file\", \"paramId\", \"shortName\", \"name\", \"units\", \"typeOfLevel\", \"level\", \"stepType\", \"dataType\",\n            \"valid_time\"] + GRID_KEYS\n    msgs = load_messages(columns=cols)\n    for c in (\"name\", \"units\", \"gridType\"):\n        msgs[c] = msgs[c].astype(\"category\")\n    cat = (msgs.groupby([\"shortName\", \"paramId\", \"name\", \"units\", \"typeOfLevel\", \"level\", \"stepType\", \"dataType\"],\n                        dropna=False, observed=True)\n           .agg(n_messages=(\"file\", \"size\"), n_files=(\"file\", \"nunique\"),\n                first_valid=(\"valid_time\", \"min\"), last_valid=(\"valid_time\", \"max\"))\n           .reset_index())\n    cat = cat[cat[\"n_messages\"] > 0].sort_values([\"shortName\", \"level\"]).reset_index(drop=True)\n\n    grids = (msgs.groupby(GRID_KEYS, dropna=False, observed=True)\n             .agg(n_messages=(\"file\", \"size\"), n_files=(\"file\", \"nunique\"),\n                  files=(\"file\", lambda x: \",\".join(sorted(map(str, x.unique()), key=natural_key))))\n             .reset_index().rename(columns=GRID_SHORT))\n    grids = grids[grids[\"n_messages\"] > 0].reset_index(drop=True)\n\n    samples = []\n    for f in sorted(config.p(\"value_samples\").glob(\"*.json\")):\n        samples.extend(read_json(f, default=[]) or [])\n    sm = pd.DataFrame(samples)\n\n    out = config.p(\"grib_audit\")\n    write_parquet_atomic(fs, out / \"file_summary.parquet\")\n    write_csv_atomic(fs, out / \"file_summary.csv\")\n    write_csv_atomic(cat, out / \"variable_catalogue.csv\")\n    write_csv_atomic(grids, out / \"grid_definitions.csv\")\n    if len(sm):\n        write_csv_atomic(sm, out / \"value_samples.csv\")\n    return {\"file_summary\": fs, \"variable_catalogue\": cat, \"grids\": grids, \"value_samples\": sm}\n","coverage.py":"\"\"\"Temporal coverage, overlap, missing-data, and chronological research-period audit.\n\nIdentity of a field = (canonical variable, level type, level, [ensemble member], valid time).\nVerifies actual data coverage against the four intended research periods:\n  - 1947-2000: historical climate baseline and long-term climate analysis\n  - 2001-2010: supervised training\n  - 2011-2015: validation and calibration\n  - 2016-2025: independent holdout testing\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .discovery import natural_key\nfrom .grib_audit import load_messages\nfrom .utils import atomic_write_json, get_logger, now_utc, write_csv_atomic\n\nCOLS = [\n    \"file\", \"paramId\", \"shortName\", \"typeOfLevel\", \"level\", \"number\",\n    \"valid_time\", \"ref_time\", \"stepType\", \"startStep\", \"endStep\", \"totalLength\", \"expver\",\n]\n\n\ndef _prepare(msgs: pd.DataFrame) -> pd.DataFrame:\n    m = msgs.dropna(subset=[\"valid_time\"]).copy()\n    canon = [\n        config.canonical_var_name(sn, pid)\n        for sn, pid in zip(m[\"shortName\"].astype(str), m[\"paramId\"] if \"paramId\" in m.columns else [None] * len(m))\n    ]\n    m[\"canon_var\"] = pd.Series(canon, index=m.index, dtype=\"category\")\n    lvl = pd.to_numeric(m[\"level\"], errors=\"coerce\").fillna(-1).astype(\"int64\").astype(str)\n    var = m[\"canon_var\"].astype(str) + \"@\" + m[\"typeOfLevel\"].astype(str) + \":\" + lvl\n    if \"number\" in m.columns:\n        num = pd.to_numeric(m[\"number\"], errors=\"coerce\").fillna(0).astype(\"int64\")\n        if num.nunique() > 1:\n            var = var + \"#m\" + num.astype(str)\n    m[\"var_id\"] = var.astype(\"category\")\n    m[\"file\"] = m[\"file\"].astype(str).astype(\"category\")\n    # IST timestamp = UTC valid_time + 5h30m\n    m[\"valid_time_ist\"] = m[\"valid_time\"] + pd.Timedelta(minutes=config.IST_UTC_OFFSET_MINUTES)\n    return m\n\n\ndef _modal_step(t: np.ndarray) -> np.timedelta64:\n    if len(t) < 2:\n        return np.timedelta64(3600 * 10 ** 9, \"ns\")\n    vals, cnt = np.unique(np.diff(t), return_counts=True)\n    return vals[cnt.argmax()]\n\n\ndef file_coverage(m: pd.DataFrame) -> pd.DataFrame:\n    g = m.groupby(\"file\", observed=True)\n    fc = g.agg(\n        start_utc=(\"valid_time\", \"min\"),\n        end_utc=(\"valid_time\", \"max\"),\n        start_ist=(\"valid_time_ist\", \"min\"),\n        end_ist=(\"valid_time_ist\", \"max\"),\n        n_messages=(\"valid_time\", \"size\"),\n        n_valid_times=(\"valid_time\", \"nunique\"),\n        n_variables=(\"var_id\", \"nunique\"),\n    ).reset_index()\n    fc[\"file\"] = fc[\"file\"].astype(str)\n    var_lists = {str(k): \",\".join(sorted(map(str, s.unique()))) for k, s in g[\"var_id\"]}\n    canon_lists = {str(k): \",\".join(sorted(map(str, s.unique()))) for k, s in g[\"canon_var\"]}\n    fc[\"variables\"] = fc[\"file\"].map(var_lists)\n    fc[\"canonical_vars\"] = fc[\"file\"].map(canon_lists)\n    fc[\"span_days\"] = ((fc[\"end_utc\"] - fc[\"start_utc\"]).dt.total_seconds() / 86400).round(2)\n    # Keep `start` and `end` aliases for downstream compatibility\n    fc[\"start\"] = fc[\"start_utc\"]\n    fc[\"end\"] = fc[\"end_utc\"]\n    return fc.sort_values([\"start_utc\", \"file\"]).reset_index(drop=True)\n\n\ndef variable_coverage(m: pd.DataFrame):\n    rows, gaps = [], []\n    for var, g in m.groupby(\"var_id\", observed=True):\n        t = np.unique(g[\"valid_time\"].to_numpy(dtype=\"datetime64[ns]\"))\n        step = _modal_step(t)\n        start, end = t[0], t[-1]\n        expected = int((end - start) / step) + 1\n        d = np.diff(t)\n        gi = np.where(d > step)[0]\n        n_missing = int(sum(int(d[i] / step) - 1 for i in gi))\n        off_grid = int(((t - start) % step != np.timedelta64(0, \"ns\")).sum())\n        canon = str(g[\"canon_var\"].iloc[0])\n        step_types = \",\".join(sorted(set(g[\"stepType\"].dropna().astype(str)))) if \"stepType\" in g.columns else \"\"\n        rows.append({\n            \"var_id\": str(var),\n            \"canon_var\": canon,\n            \"step_types\": step_types,\n            \"start\": pd.Timestamp(start),\n            \"end\": pd.Timestamp(end),\n            \"start_year\": int(pd.Timestamp(start).year),\n            \"end_year\": int(pd.Timestamp(end).year),\n            \"step_hours\": float(step / np.timedelta64(1, \"h\")),\n            \"expected_steps\": expected,\n            \"present_steps\": int(len(t)),\n            \"missing_steps\": n_missing,\n            \"completeness_pct\": round(100 * len(t) / max(expected, 1), 3),\n            \"n_gaps\": int(len(gi)),\n            \"off_grid_times\": off_grid,\n            \"duplicate_messages\": int(len(g) - len(t)),\n            \"n_files\": int(g[\"file\"].nunique()),\n        })\n        for i in gi:\n            gaps.append({\n                \"var_id\": str(var),\n                \"gap_start\": pd.Timestamp(t[i] + step),\n                \"gap_end\": pd.Timestamp(t[i + 1] - step),\n                \"missing_steps\": int(d[i] / step) - 1,\n            })\n    vc = pd.DataFrame(rows).sort_values(\"var_id\").reset_index(drop=True)\n    gp = pd.DataFrame(gaps, columns=[\"var_id\", \"gap_start\", \"gap_end\", \"missing_steps\"])\n    gp = gp.sort_values([\"missing_steps\", \"var_id\"], ascending=[False, True]).reset_index(drop=True)\n    return vc, gp\n\n\ndef verify_research_period_coverage(m: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Audit actual year-by-year ERA5 availability for each of the four research periods:\n    1947-2000 (climate_baseline), 2001-2010 (train), 2011-2015 (validation), 2016-2025 (test_holdout).\n    Never fabricates coverage: reports exact years present, years missing, and core variable availability.\n    \"\"\"\n    m_ist_year = m[\"valid_time_ist\"].dt.year\n    present_years_by_var = {\n        str(k): set(int(y) for y in grp.unique())\n        for k, grp in m_ist_year.groupby(m[\"canon_var\"], observed=True)\n    }\n    all_present_years = set(int(y) for y in m_ist_year.dropna().unique())\n\n    rows = []\n    for period_name, (y_start, y_end) in config.RESEARCH_PERIODS.items():\n        intended = set(range(y_start, y_end + 1))\n        avail_any = sorted(intended & all_present_years)\n        missing_any = sorted(intended - all_present_years)\n        core_present = {}\n        core_missing_vars = []\n        for cv in config.REQUIRED_CORE_VARS:\n            yrs = sorted(intended & present_years_by_var.get(cv, set()))\n            core_present[cv] = len(yrs)\n            if len(yrs) == 0:\n                core_missing_vars.append(cv)\n        full_core_years = sorted(\n            set.intersection(\n                *(intended & present_years_by_var.get(cv, set()) for cv in config.REQUIRED_CORE_VARS)\n            )\n        ) if all(cv in present_years_by_var for cv in config.REQUIRED_CORE_VARS) else []\n\n        status = (\n            \"COMPLETE\" if len(full_core_years) == len(intended)\n            else (\"PARTIAL\" if len(avail_any) > 0 else \"UNAVAILABLE_IN_ARCHIVE\")\n        )\n        rows.append({\n            \"period\": period_name,\n            \"intended_years\": f\"{y_start}-{y_end}\",\n            \"n_intended_years\": len(intended),\n            \"n_years_any_var\": len(avail_any),\n            \"n_years_all_6_core_vars\": len(full_core_years),\n            \"year_min_present\": min(avail_any) if avail_any else None,\n            \"year_max_present\": max(avail_any) if avail_any else None,\n            \"missing_core_vars\": \",\".join(core_missing_vars) if core_missing_vars else \"none\",\n            \"missing_years_count\": len(missing_any),\n            \"status\": status,\n        })\n    return pd.DataFrame(rows)\n\n\ndef overlap_analysis(m: pd.DataFrame, fc: pd.DataFrame) -> dict:\n    m = m[[\"var_id\", \"valid_time\", \"file\", \"totalLength\", \"expver\"]].copy()\n    m[\"key\"] = m.groupby([\"var_id\", \"valid_time\"], observed=True, sort=False).ngroup().to_numpy()\n    per_key = m.groupby(\"key\").agg(n_files=(\"file\", \"nunique\"), n_msgs=(\"file\", \"size\"))\n    dup_keys = per_key.index[per_key[\"n_files\"] > 1]\n    keys_per_file = m.groupby(\"file\", observed=True)[\"key\"].nunique()\n    keys_per_file.index = keys_per_file.index.astype(str)\n\n    sub = m.loc[m[\"key\"].isin(dup_keys), [\"key\", \"file\", \"totalLength\"]].drop_duplicates([\"key\", \"file\"])\n    sub[\"file\"] = sub[\"file\"].astype(str)\n    if len(sub):\n        pairs = sub.merge(sub, on=\"key\", suffixes=(\"_a\", \"_b\"))\n        pairs = pairs[pairs[\"file_a\"] < pairs[\"file_b\"]]\n        pairs[\"same_len\"] = pairs[\"totalLength_a\"] == pairs[\"totalLength_b\"]\n        ps = (pairs.groupby([\"file_a\", \"file_b\"])\n              .agg(shared_fields=(\"key\", \"size\"), identical_length_fraction=(\"same_len\", \"mean\")).reset_index())\n        ps[\"frac_of_a\"] = (ps[\"shared_fields\"] / ps[\"file_a\"].map(keys_per_file)).round(4)\n        ps[\"frac_of_b\"] = (ps[\"shared_fields\"] / ps[\"file_b\"].map(keys_per_file)).round(4)\n        ps[\"one_contains_other\"] = (ps[[\"frac_of_a\", \"frac_of_b\"]].max(axis=1) >= 0.999)\n        ps = ps.sort_values(\"shared_fields\", ascending=False).reset_index(drop=True)\n    else:\n        ps = pd.DataFrame(columns=[\"file_a\", \"file_b\", \"shared_fields\", \"identical_length_fraction\",\n                                   \"frac_of_a\", \"frac_of_b\", \"one_contains_other\"])\n\n    spans = []\n    f = fc[[\"file\", \"start\", \"end\", \"variables\"]].to_dict(\"records\")\n    for i in range(len(f)):\n        for j in range(i + 1, len(f)):\n            a, b = f[i], f[j]\n            lo, hi = max(a[\"start\"], b[\"start\"]), min(a[\"end\"], b[\"end\"])\n            if lo <= hi:\n                shared_vars = sorted(set(a[\"variables\"].split(\",\")) & set(b[\"variables\"].split(\",\")))\n                spans.append({\n                    \"file_a\": a[\"file\"], \"file_b\": b[\"file\"],\n                    \"overlap_start\": lo, \"overlap_end\": hi,\n                    \"overlap_days\": round((hi - lo).total_seconds() / 86400, 2),\n                    \"shared_variables\": \",\".join(shared_vars),\n                })\n    so = pd.DataFrame(spans, columns=[\"file_a\", \"file_b\", \"overlap_start\", \"overlap_end\", \"overlap_days\",\n                                      \"shared_variables\"])\n    return {\n        \"n_unique_fields\": int(len(per_key)),\n        \"n_fields_in_multiple_files\": int(len(dup_keys)),\n        \"n_within_file_duplicate_messages\": int((per_key[\"n_msgs\"] - per_key[\"n_files\"]).clip(lower=0).sum()),\n        \"pair_stats\": ps,\n        \"span_overlaps\": so,\n    }\n\n\ndef monthly_completeness(m: pd.DataFrame, vc: pd.DataFrame) -> pd.DataFrame:\n    t = m[[\"var_id\", \"valid_time\"]].drop_duplicates().copy()\n    t[\"var_id\"] = t[\"var_id\"].astype(str)\n    t[\"month\"] = t[\"valid_time\"].dt.to_period(\"M\")\n    cnt = t.groupby([\"var_id\", \"month\"]).size().unstack(\"month\", fill_value=0)\n    months = pd.period_range(t[\"month\"].min(), t[\"month\"].max(), freq=\"M\")\n    cnt = cnt.reindex(columns=months, fill_value=0)\n    step_h = vc.set_index(\"var_id\")[\"step_hours\"].reindex(cnt.index).to_numpy()\n    hours = (months.days_in_month.to_numpy() * 24).astype(float)\n    frac = cnt.to_numpy() * step_h[:, None] / hours[None, :]\n    return pd.DataFrame(np.round(frac, 4), index=cnt.index, columns=[str(p) for p in months])\n\n\ndef plot_file_gantt(fc: pd.DataFrame, out_png):\n    import matplotlib.dates as mdates\n    import matplotlib.pyplot as plt\n\n    fc = fc.iloc[::-1].reset_index(drop=True)\n    fig, ax = plt.subplots(figsize=(13, max(4, 0.3 * len(fc) + 1.5)))\n    cmap = plt.get_cmap(\"tab20\")\n    groups = {v: i for i, v in enumerate(sorted(fc[\"variables\"].unique()))}\n    for i, r in fc.iterrows():\n        start = mdates.date2num(r[\"start\"].to_pydatetime())\n        width = max((r[\"end\"] - r[\"start\"]).total_seconds() / 86400, 1 / 24)\n        ax.barh(i, width, left=start, height=0.65, color=cmap(groups[r[\"variables\"]] % 20))\n    ax.set_yticks(range(len(fc)))\n    ax.set_yticklabels(fc[\"file\"], fontsize=8)\n    ax.xaxis_date()\n    ax.set_title(\"ERA5 GRIB files: valid-time coverage (colour = variable set)\")\n    ax.grid(axis=\"x\", alpha=0.3)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=130)\n    plt.close(fig)\n    return str(out_png)\n\n\ndef plot_monthly_heatmap(mc: pd.DataFrame, out_png):\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(figsize=(14, max(3, 0.35 * len(mc) + 1.5)))\n    im = ax.imshow(mc.to_numpy(), aspect=\"auto\", cmap=\"viridis\", vmin=0, vmax=1, interpolation=\"nearest\")\n    ax.set_yticks(range(len(mc)))\n    ax.set_yticklabels(mc.index, fontsize=8)\n    cols = list(mc.columns)\n    ticks = [i for i, c in enumerate(cols) if c.endswith(\"-01\")] or list(range(0, len(cols), max(1, len(cols) // 12)))\n    ax.set_xticks(ticks)\n    ax.set_xticklabels([cols[i][:4] if cols[i].endswith(\"-01\") else cols[i] for i in ticks], rotation=90, fontsize=8)\n    fig.colorbar(im, ax=ax, label=\"fraction of expected time steps present\")\n    ax.set_title(\"Monthly completeness per variable\")\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=130)\n    plt.close(fig)\n    return str(out_png)\n\n\ndef run_coverage_analysis(make_plots: bool = True) -> dict:\n    log = get_logger()\n    avail_cols = [c for c in COLS if c in (\"file\", \"paramId\", \"shortName\", \"typeOfLevel\", \"level\", \"number\",\n                                           \"valid_time\", \"ref_time\", \"stepType\", \"startStep\", \"endStep\",\n                                           \"totalLength\", \"expver\")]\n    msgs = load_messages(columns=avail_cols)\n    m = _prepare(msgs)\n    del msgs\n    log.info(f\"Coverage analysis on {len(m):,} messages, {m['var_id'].nunique()} variables, \"\n             f\"{m['file'].nunique()} files.\")\n    fc = file_coverage(m)\n    vc, gaps = variable_coverage(m)\n    rp = verify_research_period_coverage(m)\n    ov = overlap_analysis(m, fc)\n    mc = monthly_completeness(m, vc)\n\n    out = config.p(\"coverage\")\n    write_csv_atomic(fc, out / \"file_coverage.csv\")\n    write_csv_atomic(vc, out / \"variable_coverage.csv\")\n    write_csv_atomic(rp, out / \"research_period_coverage.csv\")\n    write_csv_atomic(gaps, out / \"missing_ranges.csv\")\n    write_csv_atomic(ov[\"pair_stats\"], out / \"file_pair_field_overlap.csv\")\n    write_csv_atomic(ov[\"span_overlaps\"], out / \"file_pair_time_overlap.csv\")\n    write_csv_atomic(mc, out / \"monthly_completeness.csv\", index=True)\n\n    figs = {}\n    if make_plots and len(fc):\n        figs[\"gantt\"] = plot_file_gantt(fc, config.p(\"figures\") / \"era5_file_coverage_gantt.png\")\n        figs[\"heatmap\"] = plot_monthly_heatmap(mc, config.p(\"figures\") / \"era5_monthly_completeness.png\")\n\n    present_core = sorted(set(vc[\"canon_var\"]) & set(config.REQUIRED_CORE_VARS))\n    missing_core = sorted(set(config.REQUIRED_CORE_VARS) - set(vc[\"canon_var\"]))\n\n    summary = {\n        \"generated_utc\": now_utc(),\n        \"n_files\": int(fc[\"file\"].nunique()),\n        \"n_messages\": int(len(m)),\n        \"overall_start_utc\": str(fc[\"start_utc\"].min()),\n        \"overall_end_utc\": str(fc[\"end_utc\"].max()),\n        \"overall_start_ist\": str(fc[\"start_ist\"].min()),\n        \"overall_end_ist\": str(fc[\"end_ist\"].max()),\n        \"overall_start\": str(fc[\"start_utc\"].min()),\n        \"overall_end\": str(fc[\"end_utc\"].max()),\n        \"n_variables\": int(len(vc)),\n        \"variables\": vc[\"var_id\"].tolist(),\n        \"canonical_variables_present\": sorted(vc[\"canon_var\"].unique().tolist()),\n        \"required_core_vars_present\": present_core,\n        \"required_core_vars_missing\": missing_core,\n        \"research_periods\": rp.to_dict(\"records\"),\n        \"variables_complete\": vc.loc[vc[\"missing_steps\"] == 0, \"var_id\"].tolist(),\n        \"variables_with_gaps\": vc.loc[vc[\"missing_steps\"] > 0, [\"var_id\", \"missing_steps\", \"n_gaps\"]]\n        .to_dict(\"records\"),\n        \"variable_start_end_differ\": bool(vc[\"start\"].nunique() > 1 or vc[\"end\"].nunique() > 1),\n        \"n_unique_fields\": ov[\"n_unique_fields\"],\n        \"n_fields_in_multiple_files\": ov[\"n_fields_in_multiple_files\"],\n        \"n_within_file_duplicate_messages\": ov[\"n_within_file_duplicate_messages\"],\n        \"file_pairs_sharing_fields\": int(len(ov[\"pair_stats\"])),\n        \"file_pairs_one_contains_other\": ov[\"pair_stats\"].loc[ov[\"pair_stats\"][\"one_contains_other\"].astype(bool),\n                                                              [\"file_a\", \"file_b\"]].to_dict(\"records\"),\n        \"file_pairs_time_overlap\": int(len(ov[\"span_overlaps\"])),\n    }\n    atomic_write_json(out / \"coverage_summary.json\", summary)\n    return {\n        \"summary\": summary,\n        \"file_coverage\": fc,\n        \"variable_coverage\": vc,\n        \"research_period_coverage\": rp,\n        \"gaps\": gaps,\n        \"pair_stats\": ov[\"pair_stats\"],\n        \"span_overlaps\": ov[\"span_overlaps\"],\n        \"monthly_completeness\": mc,\n        \"figures\": figs,\n    }\n","era5.py":"\"\"\"Scientifically verified ERA5 GRIB decoding, IST daily aggregation, and baseline trend analysis.\n\nKey scientific guarantees:\n1. Preserves raw GRIB files untouched in `/content/drive/MyDrive/machine learning/`.\n2. Resolves variable names via the Section 4 metadata audit (`shortName` and `paramId`):\n   - `t2m` (`2t`/`t2m`, paramId 167): K -> degC (`val - 273.15`), daily mean/min/max in IST.\n   - `d2m` (`2d`/`d2m`, paramId 168): K -> degC (`val - 273.15`), daily mean/min/max in IST.\n   - `u10` (`10u`/`u10`, paramId 165) & `v10` (`10v`/`v10`, paramId 166): m s^-1, daily mean\n     plus hourly-first wind speed `sqrt(u10^2 + v10^2)` mean and max in IST.\n   - `tp` (`tp`, paramId 228): accumulated total precipitation (m). Handles both ERA5 single-hour\n     accumulations (`endStep - startStep == 1`) and ERA5-Land cumulative forecast steps reset at\n     00 UTC (`ΔF_s = F_s - F_{s-1}` for `s > 1`), then sums hourly increments over the 24-hour\n     IST day and converts m -> mm day^-1 (`* 1000.0`).\n   - `ssrd` (`ssrd`, paramId 169): accumulated surface solar radiation downwards (J m^-2).\n     De-accumulates multi-step forecasts when applicable, sums over the 24-hour IST day, and\n     converts J m^-2 -> MJ m^-2 day^-1 (`/ 1e6`).\n   - Optional variables (`swvl1`, `sp`, `blh`, `lai_hv`, `lai_lv`) are included when present.\n3. Never silently substitutes a missing variable with 0.0: missing variables or incomplete days\n   are explicitly flagged in the manifest and left as `NaN` (or omitted from variable list).\n4. Per-message grid geometry and `bitmapPresent`/`missingValue` masks are respected on every message.\n5. Deduplicates overlapping files with `expver == '0001'` priority over provisional `'0005'`.\n\"\"\"\nfrom __future__ import annotations\n\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .grib_audit import load_messages\nfrom .utils import (Manifest, atomic_write_json, get_logger, log_error, now_utc,\n                    write_csv_atomic, write_parquet_atomic)\n\nMSG_INDEX_COLS = [\n    \"file\", \"paramId\", \"shortName\", \"typeOfLevel\", \"level\", \"stepType\",\n    \"startStep\", \"endStep\", \"expver\", \"offset\", \"totalLength\", \"valid_time\", \"ref_time\",\n]\n\n\ndef load_deduplicated_message_index() -> pd.DataFrame:\n    \"\"\"Load audited GRIB messages, map to canonical variable names, assign IST dates,\n    and deduplicate across overlapping files prioritizing validated `expver == '0001'`.\n    \"\"\"\n    idx = load_messages(columns=MSG_INDEX_COLS)\n    idx = idx.dropna(subset=[\"valid_time\"]).copy()\n    idx[\"canon_var\"] = [\n        config.canonical_var_name(sn, pid)\n        for sn, pid in zip(idx[\"shortName\"].astype(str), idx[\"paramId\"])\n    ]\n    # Prioritize expver '0001' over '0005', then natural file order\n    exp_rank = np.where(idx[\"expver\"].astype(str).str.strip() == \"0001\", 0, 1)\n    idx[\"_exp_rank\"] = exp_rank\n    idx = idx.sort_values([\"valid_time\", \"canon_var\", \"_exp_rank\", \"file\", \"offset\"])\n    idx = idx.drop_duplicates(\n        subset=[\"canon_var\", \"typeOfLevel\", \"level\", \"valid_time\", \"startStep\", \"endStep\"],\n        keep=\"first\",\n    ).drop(columns=[\"_exp_rank\"])\n\n    # Convert UTC valid_time to Indian Standard Time (UTC + 05:30)\n    # For accumulated fields ending at `valid_time`, the 1-hour accumulation interval\n    # (valid_time - 1h, valid_time] falls in the IST day of (valid_time - 30min + 5h30m) = valid_time + 5h.\n    is_accum = idx[\"canon_var\"].isin([\"tp\", \"ssrd\", \"e\"]) | (idx[\"stepType\"].astype(str) == \"accum\")\n    ist_instant = idx[\"valid_time\"] + pd.Timedelta(minutes=config.IST_UTC_OFFSET_MINUTES)\n    # Midpoint of the preceding 1-hour accumulation window in IST:\n    ist_accum_mid = idx[\"valid_time\"] - pd.Timedelta(minutes=30) + pd.Timedelta(minutes=config.IST_UTC_OFFSET_MINUTES)\n    idx[\"ist_time\"] = ist_instant.where(~is_accum, ist_accum_mid)\n    idx[\"ist_date\"] = idx[\"ist_time\"].dt.strftime(\"%Y-%m-%d\")\n    idx[\"ist_ym\"] = idx[\"ist_time\"].dt.strftime(\"%Y-%m\")\n    return idx.reset_index(drop=True)\n\n\ndef decode_grib_message_subset(raw_fh, offset: int, length: int, bbox: dict | None = None,\n                               target_lat: np.ndarray | None = None,\n                               target_lon: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"Decode a single GRIB message at `(offset, length)`, apply its missingValue bitmap,\n    subset to `bbox`, orient latitude North-to-South (`lat` descending) and longitude West-to-East,\n    and regrid via bilinear/nearest lookup if a message's grid differs from `target_lat`/`target_lon`.\n    \"\"\"\n    import eccodes\n\n    bbox = bbox or config.INDIA_BBOX\n    raw_fh.seek(int(offset))\n    buf = raw_fh.read(int(length))\n    if buf[:4] != b\"GRIB\":\n        raise ValueError(f\"Invalid GRIB magic at byte offset {offset}\")\n\n    gid = eccodes.codes_new_from_message(buf)\n    try:\n        ni = int(eccodes.codes_get(gid, \"Ni\"))\n        nj = int(eccodes.codes_get(gid, \"Nj\"))\n        lat1 = float(eccodes.codes_get(gid, \"latitudeOfFirstGridPointInDegrees\"))\n        lat2 = float(eccodes.codes_get(gid, \"latitudeOfLastGridPointInDegrees\"))\n        lon1 = float(eccodes.codes_get(gid, \"longitudeOfFirstGridPointInDegrees\"))\n        lon2 = float(eccodes.codes_get(gid, \"longitudeOfLastGridPointInDegrees\"))\n        bitmap = int(eccodes.codes_get(gid, \"bitmapPresent\"))\n        miss_val = float(eccodes.codes_get(gid, \"missingValue\"))\n        vals = np.asarray(eccodes.codes_get_values(gid), dtype=\"float64\")\n    finally:\n        eccodes.codes_release(gid)\n\n    if bitmap:\n        vals[np.isclose(vals, miss_val, rtol=0, atol=1e-3)] = np.nan\n    vals[np.abs(vals) > 1e15] = np.nan\n    grid = vals.reshape((nj, ni)).astype(\"float32\")\n\n    lats = np.linspace(lat1, lat2, nj)\n    lons = np.linspace(lon1, lon2, ni)\n    # Normalize longitudes in [180, 360) to [-180, 180) if needed\n    if lons.max() > 180.0 and bbox[\"lon_min\"] < 180.0:\n        lons = np.where(lons > 180.0, lons - 360.0, lons)\n        order_lon = np.argsort(lons)\n        lons = lons[order_lon]\n        grid = grid[:, order_lon]\n\n    # Ensure latitude is strictly descending (North -> South, e.g. 38.0 down to 6.0)\n    if lats[0] < lats[-1]:\n        lats = lats[::-1]\n        grid = grid[::-1, :]\n\n    lat_mask = (lats >= bbox[\"lat_min\"] - 1e-5) & (lats <= bbox[\"lat_max\"] + 1e-5)\n    lon_mask = (lons >= bbox[\"lon_min\"] - 1e-5) & (lons <= bbox[\"lon_max\"] + 1e-5)\n    sub_lat = np.round(lats[lat_mask], 4).astype(\"float32\")\n    sub_lon = np.round(lons[lon_mask], 4).astype(\"float32\")\n    sub_grid = grid[np.ix_(lat_mask, lon_mask)]\n\n    if target_lat is not None and target_lon is not None:\n        ifsub_mismatch = (len(sub_lat) != len(target_lat) or len(sub_lon) != len(target_lon)\n                          or not np.allclose(sub_lat, target_lat, atol=1e-3)\n                          or not np.allclose(sub_lon, target_lon, atol=1e-3))\n        if ifsub_mismatch:\n            # Exact coordinate placement onto canonical target grid (leaves unrepresented edge rows as NaN)\n            aligned = np.full((len(target_lat), len(target_lon)), np.nan, dtype=\"float32\")\n            lat_diff = np.abs(sub_lat[:, None] - target_lat[None, :])\n            lon_diff = np.abs(sub_lon[:, None] - target_lon[None, :])\n            s_lat_i, t_lat_i = np.where(lat_diff < 1e-2)\n            s_lon_i, t_lon_i = np.where(lon_diff < 1e-2)\n            if len(s_lat_i) > 0 and len(s_lon_i) > 0:\n                aligned[np.ix_(t_lat_i, t_lon_i)] = sub_grid[np.ix_(s_lat_i, s_lon_i)]\n            else:\n                lat_idx = lat_diff.argmin(axis=0)\n                lon_idx = lon_diff.argmin(axis=0)\n                aligned = sub_grid[np.ix_(lat_idx, lon_idx)]\n            sub_grid = aligned\n            sub_lat, sub_lon = target_lat, target_lon\n\n    return sub_grid, sub_lat, sub_lon\n\n\ndef _deaccumulate_variable_messages(vdf: pd.DataFrame, bbox: dict,\n                                    target_lat: np.ndarray | None,\n                                    target_lon: np.ndarray | None) -> tuple[list[pd.Timestamp], np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"Decode all messages for a variable (`vdf`) sorted chronologically, de-accumulating\n    multi-step forecast accumulations (`endStep - startStep > 1`) into 1-hour increments when needed.\n    \"\"\"\n    vdf = vdf.sort_values([\"valid_time\", \"endStep\"]).reset_index(drop=True)\n    # Determine whether this variable uses cumulative forecast steps (e.g. ERA5-Land step 1..24)\n    start_s = pd.to_numeric(vdf[\"startStep\"], errors=\"coerce\").fillna(0).to_numpy()\n    end_s = pd.to_numeric(vdf[\"endStep\"], errors=\"coerce\").fillna(1).to_numpy()\n    span = end_s - start_s\n    is_cumulative_fc = bool(np.nanmax(span) > 1) if len(span) else False\n\n    # Open each file at most once while reading its offsets\n    fh_cache = {}\n    frames = []\n    times = []\n    try:\n        for row in vdf.itertuples(index=False):\n            fname = str(row.file)\n            if fname not in fh_cache:\n                fh_cache[fname] = open(config.RAW_ERA5_DIR / fname, \"rb\")\n            arr, target_lat, target_lon = decode_grib_message_subset(\n                fh_cache[fname], int(row.offset), int(row.totalLength),\n                bbox=bbox, target_lat=target_lat, target_lon=target_lon,\n            )\n            frames.append(arr)\n            times.append(pd.Timestamp(row.ist_date))\n    finally:\n        for fh in fh_cache.values():\n            fh.close()\n\n    stack = np.stack(frames, axis=0)  # (N_msg, H, W)\n    if is_cumulative_fc and len(stack) > 1:\n        # Cumulative forecast from reference time: increment_k = F_k - F_{k-1} when ref_time matches and endStep > 1\n        ref_t = vdf[\"ref_time\"].to_numpy()\n        inc = stack.copy()\n        for k in range(1, len(stack)):\n            if end_s[k] > 1 and ref_t[k] == ref_t[k - 1] and end_s[k] > end_s[k - 1]:\n                inc[k] = np.maximum(0.0, stack[k] - stack[k - 1])\n        stack = inc\n\n    return times, stack, target_lat, target_lon\n\n\ndef process_era5_ist_month(ym: str, idx_ym: pd.DataFrame, bbox: dict | None = None) -> tuple[\"xr.Dataset\", dict]:\n    \"\"\"Aggregate all audited ERA5 messages for IST year-month `ym` (`YYYY-MM`) into a daily\n    `xarray.Dataset` in Indian Standard Time (`Asia/Kolkata`).\n\n    Never substitutes missing variables with zero.\n    \"\"\"\n    import xarray as xr\n\n    bbox = bbox or config.INDIA_BBOX\n    # Include the 1-day boundary margin when passed so multi-step de-accumulation is exact\n    all_days = sorted(idx_ym[idx_ym[\"ist_ym\"] == ym][\"ist_date\"].unique())\n    if not all_days:\n        raise ValueError(f\"No messages found for IST month {ym}\")\n\n    day_to_i = {d: i for i, d in enumerate(all_days)}\n    n_days = len(all_days)\n    target_lat, target_lon = None, None\n\n    # Keep hourly u10 and v10 aligned by valid_time so wind speed is computed hourly first\n    hourly_uv = {}\n    data_vars = {}\n    var_diagnostics = {}\n\n    for canon, vdf in idx_ym.groupby(\"canon_var\", observed=True):\n        canon = str(canon)\n        spec = config.ERA5_CANONICAL_VARS.get(canon, {\"kind\": \"instant\", \"units_out\": \"raw\"})\n        times_ist_date, stack, target_lat, target_lon = _deaccumulate_variable_messages(\n            vdf, bbox, target_lat, target_lon,\n        )\n        h, w = len(target_lat), len(target_lon)\n\n        if canon in (\"u10\", \"v10\"):\n            # Store by exact UTC valid_time for hourly vector magnitude calculation\n            hourly_uv[canon] = {\n                vt: (str(ist_d), stack[k])\n                for k, (vt, ist_d) in enumerate(zip(vdf[\"valid_time\"], vdf[\"ist_date\"]))\n                if str(ist_d) in day_to_i\n            }\n\n        # Group message slices by IST calendar day\n        buckets = {d: [] for d in all_days}\n        for d_Timestamp, arr in zip(times_ist_date, stack):\n            d_str = d_Timestamp.strftime(\"%Y-%m-%d\")\n            if d_str in buckets:\n                buckets[d_str].append(arr)\n\n        if spec[\"kind\"] == \"accum\":\n            daily_sum = np.full((n_days, h, w), np.nan, dtype=\"float32\")\n            hours_count = np.zeros(n_days, dtype=\"int16\")\n            for d_str, i in day_to_i.items():\n                sl = buckets[d_str]\n                hours_count[i] = len(sl)\n                if len(sl) > 0:\n                    s = np.nansum(np.stack(sl, axis=0), axis=0)\n                    if canon == \"tp\":\n                        s = np.maximum(0.0, s) * 1000.0  # m -> mm/day\n                    elif canon == \"ssrd\":\n                        s = np.maximum(0.0, s) / 1e6     # J m^-2 -> MJ m^-2/day\n                    elif canon == \"e\":\n                        s = s * 1000.0                   # m -> mm/day (signed evaporation)\n                    daily_sum[i] = s.astype(\"float32\")\n            data_vars[f\"{canon}_sum\"] = ((\"time\", \"latitude\", \"longitude\"), daily_sum)\n            var_diagnostics[canon] = {\n                \"kind\": \"accum\",\n                \"units\": spec[\"units_out\"],\n                \"mean_hours_per_day\": round(float(hours_count.mean()), 2),\n                \"min_hours_per_day\": int(hours_count.min()),\n            }\n        else:\n            daily_mean = np.full((n_days, h, w), np.nan, dtype=\"float32\")\n            daily_max = np.full((n_days, h, w), np.nan, dtype=\"float32\")\n            daily_min = np.full((n_days, h, w), np.nan, dtype=\"float32\")\n            hours_count = np.zeros(n_days, dtype=\"int16\")\n            for d_str, i in day_to_i.items():\n                sl = buckets[d_str]\n                hours_count[i] = len(sl)\n                if len(sl) > 0:\n                    st = np.stack(sl, axis=0)\n                    if canon in (\"t2m\", \"d2m\"):\n                        st = st - 273.15  # K -> degC\n                    elif canon == \"sp\":\n                        st = st / 100.0   # Pa -> hPa\n                    daily_mean[i] = np.nanmean(st, axis=0).astype(\"float32\")\n                    daily_max[i] = np.nanmax(st, axis=0).astype(\"float32\")\n                    daily_min[i] = np.nanmin(st, axis=0).astype(\"float32\")\n            data_vars[f\"{canon}_mean\"] = ((\"time\", \"latitude\", \"longitude\"), daily_mean)\n            data_vars[f\"{canon}_max\"] = ((\"time\", \"latitude\", \"longitude\"), daily_max)\n            data_vars[f\"{canon}_min\"] = ((\"time\", \"latitude\", \"longitude\"), daily_min)\n            var_diagnostics[canon] = {\n                \"kind\": \"instant\",\n                \"units\": spec[\"units_out\"],\n                \"mean_hours_per_day\": round(float(hours_count.mean()), 2),\n                \"min_hours_per_day\": int(hours_count.min()),\n            }\n\n    # Hourly-first 10m wind speed & hourly-first Vapour Pressure Deficit (VPD) if pairs exist\n    if \"u10\" in hourly_uv and \"v10\" in hourly_uv:\n        common_vt = sorted(set(hourly_uv[\"u10\"]) & set(hourly_uv[\"v10\"]))\n        w_buckets = {d: [] for d in all_days}\n        for vt in common_vt:\n            d_str, u_arr = hourly_uv[\"u10\"][vt]\n            _, v_arr = hourly_uv[\"v10\"][vt]\n            w_buckets[d_str].append(np.hypot(u_arr, v_arr))\n        h, w = len(target_lat), len(target_lon)\n        w_mean = np.full((n_days, h, w), np.nan, dtype=\"float32\")\n        w_max = np.full((n_days, h, w), np.nan, dtype=\"float32\")\n        for d_str, i in day_to_i.items():\n            if w_buckets[d_str]:\n                wst = np.stack(w_buckets[d_str], axis=0)\n                w_mean[i] = np.nanmean(wst, axis=0).astype(\"float32\")\n                w_max[i] = np.nanmax(wst, axis=0).astype(\"float32\")\n        data_vars[\"wind10_mean\"] = ((\"time\", \"latitude\", \"longitude\"), w_mean)\n        data_vars[\"wind10_max\"] = ((\"time\", \"latitude\", \"longitude\"), w_max)\n\n    coords = {\n        \"time\": pd.to_datetime(all_days),\n        \"latitude\": target_lat,\n        \"longitude\": target_lon,\n    }\n    ds = xr.Dataset(data_vars, coords=coords)\n    ds.attrs[\"timezone\"] = config.LOCAL_TIMEZONE\n    ds.attrs[\"ist_utc_offset_minutes\"] = config.IST_UTC_OFFSET_MINUTES\n    ds.attrs[\"created_utc\"] = now_utc()\n    ds.attrs[\"variables_present\"] = sorted(var_diagnostics.keys())\n    ds.attrs[\"missing_core_variables\"] = sorted(set(config.REQUIRED_CORE_VARS) - set(var_diagnostics.keys()))\n    return ds, var_diagnostics\n\n\ndef build_era5_monthly_zarr(year_months: list[str] | None = None, force: bool = False) -> dict:\n    \"\"\"Stream audited ERA5 GRIB messages month-by-month into IST daily Zarr stores on Google Drive.\"\"\"\n    log = get_logger()\n    man = Manifest(config.p(\"manifests\") / \"era5_zarr_manifest.json\")\n    idx = load_deduplicated_message_index()\n    all_ym = sorted(idx[\"ist_ym\"].unique())\n    target_ym = year_months if year_months is not None else all_ym\n\n    out_root = config.p(\"era5_daily\")\n    out_root.mkdir(parents=True, exist_ok=True)\n    done_count = 0\n\n    for ym in target_ym:\n        sub = idx[idx[\"ist_ym\"] == ym]\n        if len(sub) == 0:\n            continue\n        fp = f\"v2_ist:{len(sub)}:{int(sub['totalLength'].sum())}\"\n        zarr_path = out_root / f\"{ym}.zarr\"\n        if not force and man.is_done(ym, fp) and zarr_path.exists():\n            done_count += 1\n            continue\n\n        t0 = time.time()\n        log.info(f\"[ERA5->IST Zarr] {ym}: decoding {len(sub):,} messages from {sub['file'].nunique()} file(s)...\")\n        try:\n            ds, vdiag = process_era5_ist_month(ym, sub, bbox=config.INDIA_BBOX)\n            tmp_zarr = config.scratch(\"zarr_tmp\") / f\"{ym}.zarr\"\n            if tmp_zarr.exists():\n                shutil.rmtree(tmp_zarr)\n            ds.chunk({\"time\": len(ds.time), \"latitude\": 32, \"longitude\": 32}).to_zarr(tmp_zarr, mode=\"w\")\n            if zarr_path.exists():\n                shutil.rmtree(zarr_path)\n            shutil.copytree(tmp_zarr, zarr_path)\n            shutil.rmtree(tmp_zarr, ignore_errors=True)\n\n            man.mark(\n                ym, \"done\", fingerprint=fp,\n                seconds=round(time.time() - t0, 1),\n                n_messages=int(len(sub)),\n                n_days=int(len(ds.time)),\n                variables=vdiag,\n                missing_core_vars=ds.attrs[\"missing_core_variables\"],\n                zarr=str(zarr_path),\n            )\n            done_count += 1\n        except Exception as exc:\n            man.mark(ym, \"failed\", fingerprint=fp, error=str(exc))\n            log_error(\"era5_monthly_zarr\", exc, ym=ym)\n            raise\n\n    return {\n        \"months_targeted\": len(target_ym),\n        \"months_completed\": done_count,\n        \"manifest\": str(man.path),\n        \"zarr_root\": str(out_root),\n    }\n\n\ndef validate_era5_daily_sample(ym: str | None = None, sample_day: str | None = None) -> dict:\n    \"\"\"Decode raw GRIB messages manually for one IST day and verify numerical agreement\n    against the processed Zarr store (`max_abs_diff < 1e-4`).\n    \"\"\"\n    import xarray as xr\n\n    zarr_files = sorted(config.p(\"era5_daily\").glob(\"*.zarr\"))\n    if not zarr_files:\n        raise FileNotFoundError(\"No processed ERA5 Zarr stores found; build at least one sample month first.\")\n\n    zpath = (config.p(\"era5_daily\") / f\"{ym}.zarr\") if ym else zarr_files[0]\n    ym = zpath.stem\n    ds = xr.open_zarr(zpath)\n    day_str = sample_day or pd.Timestamp(ds.time.values[min(1, len(ds.time) - 1)]).strftime(\"%Y-%m-%d\")\n\n    idx = load_deduplicated_message_index()\n    day_msgs = idx[idx[\"ist_date\"] == day_str]\n    ref_ds, _ = process_era5_ist_month(ym, day_msgs, bbox=config.INDIA_BBOX)\n\n    checks = {}\n    all_passed = True\n    for var_name in ref_ds.data_vars:\n        if var_name not in ds.data_vars:\n            checks[var_name] = {\"passed\": False, \"reason\": \"missing_in_zarr\"}\n            all_passed = False\n            continue\n        a = ds[var_name].sel(time=day_str).values\n        b = ref_ds[var_name].sel(time=day_str).values\n        diff = float(np.nanmax(np.abs(a - b))) if np.isfinite(a).any() else 0.0\n        ok = bool(diff < 1e-4 and np.isfinite(a).sum() == np.isfinite(b).sum())\n        all_passed = all_passed and ok\n        checks[var_name] = {\n            \"passed\": ok,\n            \"max_abs_diff\": diff,\n            \"min\": float(np.nanmin(a)) if np.isfinite(a).any() else None,\n            \"mean\": float(np.nanmean(a)) if np.isfinite(a).any() else None,\n            \"max\": float(np.nanmax(a)) if np.isfinite(a).any() else None,\n        }\n\n    report = {\n        \"validated_utc\": now_utc(),\n        \"year_month\": ym,\n        \"sample_ist_day\": day_str,\n        \"n_raw_messages_decoded\": int(len(day_msgs)),\n        \"all_passed\": all_passed,\n        \"variables\": checks,\n    }\n    atomic_write_json(config.p(\"grib_audit\") / f\"sample_zarr_validation_{day_str}.json\", report)\n    return report\n\n\ndef compute_historical_climate_baseline() -> dict:\n    \"\"\"Compute grid-cell day-of-year climatology (mean & std) and long-term annual trends over the\n    verified historical climate baseline (1947-2000 when present in archive; otherwise earliest\n    verified baseline window, explicitly documented in the output metadata).\n    \"\"\"\n    import xarray as xr\n\n    zarr_files = sorted(config.p(\"era5_daily\").glob(\"*.zarr\"))\n    if not zarr_files:\n        return {\"status\": \"no_zarr_stores\"}\n\n    base_start, base_end = config.RESEARCH_PERIODS[\"climate_baseline\"]\n    baseline_zarrs = [z for z in zarr_files if base_start <= int(z.stem.split(\"-\")[0]) <= base_end]\n    used_period_note = \"1947-2000_exact\"\n    if not baseline_zarrs:\n        # Use the training period (2001-2010) or earliest available years for anomaly standardization\n        # while explicitly reporting that 1947-2000 GRIB files are not present in the Drive archive.\n        train_start, train_end = config.RESEARCH_PERIODS[\"train\"]\n        baseline_zarrs = [z for z in zarr_files if int(z.stem.split(\"-\")[0]) <= train_end] or zarr_files\n        yrs = sorted({int(z.stem.split(\"-\")[0]) for z in baseline_zarrs})\n        used_period_note = f\"archive_verified_{min(yrs)}_{max(yrs)} (1947-2000 not present in raw Drive folder)\"\n\n    ds = xr.open_mfdataset([str(z) for z in baseline_zarrs], engine=\"zarr\", combine=\"by_coords\")\n    # Annual domain-mean trends\n    annual = ds.mean(dim=[\"latitude\", \"longitude\"]).groupby(\"time.year\").mean().compute()\n    ann_df = annual.to_dataframe().reset_index()\n    out_dir = config.p(\"climate_baseline\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n    write_csv_atomic(ann_df, out_dir / \"annual_climate_trends.csv\")\n\n    # Grid-cell monthly climatology (mean & std) for leak-free anomaly computation\n    clim_mean = ds.groupby(\"time.month\").mean(\"time\").compute()\n    clim_std = ds.groupby(\"time.month\").std(\"time\").compute()\n    clim_path = out_dir / \"monthly_grid_climatology.zarr\"\n    if clim_path.exists():\n        shutil.rmtree(clim_path)\n    xr.Dataset(\n        {f\"{k}_clim_mean\": clim_mean[k] for k in clim_mean.data_vars}\n        | {f\"{k}_clim_std\": clim_std[k] for k in clim_std.data_vars}\n    ).to_zarr(clim_path, mode=\"w\")\n\n    info = {\n        \"status\": \"done\",\n        \"baseline_reference\": used_period_note,\n        \"n_months_used\": len(baseline_zarrs),\n        \"annual_trends_csv\": str(out_dir / \"annual_climate_trends.csv\"),\n        \"climatology_zarr\": str(clim_path),\n    }\n    atomic_write_json(out_dir / \"baseline_metadata.json\", info)\n    return info\n","firms.py":"\"\"\"Real NASA FIRMS active-fire discovery, historical/NRT API acquisition, validation, and deduplication.\n\nAuthentication:\n- Uses `FIRMS_MAP_KEY` stored securely in Google Colab Secrets (`google.colab.userdata.get('FIRMS_MAP_KEY')`)\n  or environment variable `FIRMS_MAP_KEY`.\n- The key is never printed, logged, or persisted to disk.\n\nScientific integrity & sensor limitations:\n- Never generates synthetic fire observations.\n- Supports both:\n  1. Official NASA FIRMS Area REST API (`/api/area/csv/{MAP_KEY}/{source}/{bbox}/{days}/{date}`)\n     with resumable date-window manifests (`manifests/firms_download_manifest.json`) and automatic\n     rate-limit handling.\n  2. Official NASA FIRMS Archive bulk downloads (`.csv`, `.zip`, `.parquet` from\n     https://firms.modaps.eosdis.nasa.gov/download/) placed in `data/firms/raw/`.\n- Converts UTC satellite overpass timestamps (`acq_date` + `acq_time`) to Indian Standard Time\n  (`Asia/Kolkata`, UTC+05:30).\n- Standardizes heterogeneous MODIS (`0-100` numeric confidence) and VIIRS (`l`/`n`/`h` categorical\n  confidence) into a unified `confidence_tier` (`low`, `nominal`, `high`) and `confidence_score` (`[0, 1]`).\n- Deduplicates overlapping same-pass/adjacent-orbit detections within `0.01 deg` (~1.1 km) and `15 minutes`.\n- Documents active-fire remote sensing limitations in `metadata/firms_limitations.json`.\n\"\"\"\nfrom __future__ import annotations\n\nimport io\nimport os\nimport time\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .utils import (Manifest, atomic_write_json, get_logger, log_error, now_utc,\n                    write_csv_atomic, write_parquet_atomic)\n\nFIRMS_BASE = \"https://firms.modaps.eosdis.nasa.gov\"\nFIRMS_FILE_PATTERNS = (\"*.csv\", \"*.zip\", \"*.parquet\")\n\n# Standard NASA FIRMS Indian bounding box: west,south,east,north (integer coordinates)\nINDIA_AREA_STR = (\n    f\"{int(config.INDIA_BBOX['lon_min'])},{int(config.INDIA_BBOX['lat_min'])},\"\n    f\"{int(config.INDIA_BBOX['lon_max'])},{int(config.INDIA_BBOX['lat_max'])}\"\n)\n\n# Official FIRMS Science Processing (SP) & Near Real-Time (NRT) product codes\nFIRMS_PRODUCTS = (\n    \"MODIS_SP\",           # MODIS Collection 6.1 Standard Processing (2000-11 to present)\n    \"VIIRS_SNPP_SP\",      # Suomi-NPP VIIRS 375m Standard Processing (2012-01 to present)\n    \"VIIRS_NOAA20_SP\",    # NOAA-20 VIIRS 375m Standard Processing (2018-04 to present)\n    \"MODIS_NRT\",          # MODIS Collection 6.1 Near Real-Time\n    \"VIIRS_SNPP_NRT\",     # Suomi-NPP VIIRS Near Real-Time\n    \"VIIRS_NOAA20_NRT\",   # NOAA-20 VIIRS Near Real-Time\n)\n\nSATELLITE_LIMITATIONS = {\n    \"cloud_and_smoke_obscuration\": (\n        \"Passive thermal infrared sensors (MODIS 1 km, VIIRS 375 m) cannot penetrate thick monsoon or \"\n        \"cirrus cloud cover or dense smoke plumes, creating systematic under-detection during cloudy intervals.\"\n    ),\n    \"canopy_interception_of_surface_fires\": (\n        \"In closed-canopy evergreen and moist deciduous forests (e.g., Western Ghats, Northeast India), \"\n        \"low-intensity understory surface fires are partially shielded by the upper tree canopy.\"\n    ),\n    \"overpass_timing_and_short_duration_fires\": (\n        \"Polar-orbiting satellites observe India at fixed local solar times (~10:30/22:30 LT for Terra, \"\n        \"~13:30/01:30 LT for Aqua/SNPP/NOAA-20). Short-lived agricultural residue burns (often lasting 30-90 min \"\n        \"in late afternoon) between overpasses can be missed.\"\n    ),\n    \"scan_angle_bowtie_distortion\": (\n        \"At high off-nadir scan angles (>40 deg), effective pixel area grows by 2-4x, elevating the minimum \"\n        \"detectable fire radiative power (FRP) threshold and coarsening geolocation accuracy.\"\n    ),\n    \"sensor_intercalibration_break_2012\": (\n        \"VIIRS 375 m (introduced Jan 2012) detects smaller/cooler fires than MODIS 1 km. Long-term supervised \"\n        \"consistency across 2001-2025 therefore requires explicit sensor awareness (`MODIS_SP` for uniform \"\n        \"2001-2025 time series, with `VIIRS_SNPP_SP` evaluated in parallel for post-2012 high-resolution tests).\"\n    ),\n    \"static_industrial_false_alarms\": (\n        \"Steel plants, cement kilns, gas flares, mines (e.g., Jharia/Korba coalfields), and large solar farms \"\n        \"produce recurring thermal anomalies (`type != 0` in SP products) that must be filtered before \"\n        \"forest-fire modeling.\"\n    ),\n}\n\n\ndef get_secret(name: str) -> str | None:\n    \"\"\"Read a secret from Colab Secrets (preferred) or an environment variable.\"\"\"\n    try:\n        from google.colab import userdata\n\n        val = userdata.get(name)\n        if val:\n            return str(val).strip()\n    except Exception:\n        pass\n    val = os.environ.get(name)\n    return str(val).strip() if val else None\n\n\ndef map_key_status(map_key: str, timeout: int = 30) -> dict:\n    import requests\n\n    r = requests.get(f\"{FIRMS_BASE}/mapserver/mapkey_status/\", params={\"MAP_KEY\": map_key}, timeout=timeout)\n    r.raise_for_status()\n    data = r.json()\n    return {k: v for k, v in data.items() if \"key\" not in k.lower()}\n\n\ndef data_availability(map_key: str, timeout: int = 60) -> pd.DataFrame:\n    import requests\n\n    r = requests.get(f\"{FIRMS_BASE}/api/data_availability/csv/{map_key}/all\", timeout=timeout)\n    r.raise_for_status()\n    return pd.read_csv(io.StringIO(r.text))\n\n\ndef discover_local_firms(root=None) -> pd.DataFrame:\n    root = Path(root or config.p(\"firms_raw\"))\n    rows = []\n    if root.exists():\n        for pat in FIRMS_FILE_PATTERNS:\n            for f in sorted(root.rglob(pat)):\n                rows.append({\n                    \"file\": str(f.relative_to(root)),\n                    \"path\": str(f),\n                    \"size_mb\": round(f.stat().st_size / 1e6, 2),\n                })\n    return pd.DataFrame(rows, columns=[\"file\", \"path\", \"size_mb\"])\n\n\ndef acquire_firms_area_range(source: str, start_date: str, end_date: str,\n                             area: str = INDIA_AREA_STR, chunk_days: int = 5,\n                             sleep_seconds: float = 0.5, force: bool = False) -> dict:\n    \"\"\"Download actual FIRMS observations via the official Area API in resumable `chunk_days`\n    windows (`1 <= chunk_days <= 5` per NASA FIRMS API rules) and persist per-chunk CSVs\n    under `data/firms/raw/api_{source}/`.\n    \"\"\"\n    import requests\n\n    log = get_logger()\n    map_key = get_secret(\"FIRMS_MAP_KEY\")\n    if not map_key:\n        raise RuntimeError(\n            \"Missing `FIRMS_MAP_KEY` in Google Colab Secrets. Obtain a free NASA FIRMS MAP_KEY at \"\n            \"https://firms.modaps.eosdis.nasa.gov/api/map_key/ and add it as `FIRMS_MAP_KEY` in Colab Secrets.\"\n        )\n    chunk_days = max(1, min(5, int(chunk_days)))\n    out_dir = config.p(\"firms_raw\") / f\"api_{source}\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    man = Manifest(config.p(\"manifests\") / f\"firms_api_{source}_manifest.json\")\n\n    cur = pd.Timestamp(start_date)\n    end_ts = pd.Timestamp(end_date)\n    total_chunks = 0\n    total_rows = 0\n\n    while cur <= end_ts:\n        win_end = min(cur + pd.Timedelta(days=chunk_days - 1), end_ts)\n        n_days = int((win_end - cur).days) + 1\n        d_str = cur.strftime(\"%Y-%m-%d\")\n        key = f\"{source}_{d_str}_{n_days}d\"\n        out_csv = out_dir / f\"{key}.csv\"\n\n        if not force and man.is_done(key) and out_csv.exists():\n            total_chunks += 1\n            total_rows += int((man.get(key) or {}).get(\"rows\", 0))\n            cur = win_end + pd.Timedelta(days=1)\n            continue\n\n        url = f\"{FIRMS_BASE}/api/area/csv/{map_key}/{source}/{area}/{n_days}/{d_str}\"\n        try:\n            r = requests.get(url, timeout=90)\n            if r.status_code == 429:\n                log.warning(\"[FIRMS API] Rate limit hit (HTTP 429); sleeping 60s before retry...\")\n                time.sleep(60.0)\n                r = requests.get(url, timeout=90)\n            if r.status_code != 200:\n                safe_body = r.text[:160].replace(map_key, \"<REDACTED>\")\n                raise RuntimeError(f\"FIRMS API HTTP {r.status_code} for {source} ({d_str}, {n_days}d): {safe_body}\")\n            text = r.text.strip()\n            if \"Invalid MAP_KEY\" in text or \"Exceeding\" in text or text.startswith(\"<!DOCTYPE\"):\n                safe_text = text[:160].replace(map_key, \"<REDACTED>\")\n                raise RuntimeError(f\"FIRMS API returned error payload: {safe_text}\")\n            df_chunk = pd.read_csv(io.StringIO(text)) if text else pd.DataFrame()\n            write_csv_atomic(df_chunk, out_csv)\n            man.mark(key, \"done\", rows=int(len(df_chunk)), start=d_str, days=n_days, file=str(out_csv))\n            total_chunks += 1\n            total_rows += int(len(df_chunk))\n            time.sleep(sleep_seconds)\n        except Exception as exc:\n            safe_err = str(exc).replace(map_key, \"<REDACTED>\")\n            man.mark(key, \"failed\", error=safe_err, start=d_str, days=n_days)\n            log_error(\"firms_api_download\", RuntimeError(safe_err), source=source, start=d_str, days=n_days)\n            raise RuntimeError(safe_err) from None\n\n        cur = win_end + pd.Timedelta(days=1)\n\n    return {\n        \"source\": source,\n        \"start_date\": start_date,\n        \"end_date\": end_date,\n        \"chunks_completed\": total_chunks,\n        \"total_detections_downloaded\": total_rows,\n        \"output_dir\": str(out_dir),\n    }\n\n\nDEFAULT_MULTI_SPLIT_FIRMS_WINDOWS = [\n    # Train (2001-2010): peak forest-fire (Mar-Apr) & post-monsoon agricultural-fire (Oct-Nov) windows\n    (\"MODIS_SP\", \"2004-03-01\", \"2004-04-30\"),\n    (\"MODIS_SP\", \"2004-10-15\", \"2004-11-25\"),\n    (\"MODIS_SP\", \"2009-03-01\", \"2009-04-30\"),\n    (\"MODIS_SP\", \"2009-10-15\", \"2009-11-25\"),\n    (\"MODIS_SP\", \"2010-03-01\", \"2010-04-30\"),\n    (\"MODIS_SP\", \"2010-10-15\", \"2010-11-25\"),\n    # Validation (2011-2015): MODIS_SP + VIIRS_SNPP_SP\n    (\"MODIS_SP\", \"2012-03-01\", \"2012-04-30\"),\n    (\"MODIS_SP\", \"2012-10-15\", \"2012-11-25\"),\n    (\"MODIS_SP\", \"2014-03-01\", \"2014-04-30\"),\n    (\"VIIRS_SNPP_SP\", \"2015-03-01\", \"2015-04-30\"),\n    (\"VIIRS_SNPP_SP\", \"2015-10-15\", \"2015-11-25\"),\n    # Independent Holdout Test (2016-2025): MODIS_SP + VIIRS_SNPP_SP\n    (\"MODIS_SP\", \"2018-03-01\", \"2018-04-30\"),\n    (\"VIIRS_SNPP_SP\", \"2018-03-01\", \"2018-04-30\"),\n    (\"VIIRS_SNPP_SP\", \"2018-10-15\", \"2018-11-25\"),\n    (\"VIIRS_SNPP_SP\", \"2021-03-01\", \"2021-04-30\"),\n    (\"VIIRS_SNPP_SP\", \"2023-03-01\", \"2023-04-30\"),\n    (\"VIIRS_SNPP_SP\", \"2023-10-15\", \"2023-11-25\"),\n]\n\n\ndef acquire_multi_split_firms_campaign(windows: list[tuple[str, str, str]] | None = None,\n                                       chunk_days: int = 5, sleep_seconds: float = 0.4) -> list[dict]:\n    \"\"\"Download genuine NASA FIRMS standard-processing observations across Train (2001-2010),\n    Validation (2011-2015), and Independent Holdout Test (2016-2025) periods so all chronological\n    splits have real MODIS & VIIRS active-fire detections.\n    \"\"\"\n    windows = windows or DEFAULT_MULTI_SPLIT_FIRMS_WINDOWS\n    results = []\n    for src, s_date, e_date in windows:\n        stat = acquire_firms_area_range(\n            source=src, start_date=s_date, end_date=e_date,\n            chunk_days=chunk_days, sleep_seconds=sleep_seconds,\n        )\n        results.append(stat)\n    return results\n\n\ndef _standardize_firms_dataframe(df: pd.DataFrame, source_hint: str = \"\") -> pd.DataFrame:\n    \"\"\"Validate and standardize a raw FIRMS DataFrame (MODIS or VIIRS, SP or NRT).\"\"\"\n    if df is None or len(df) == 0:\n        return pd.DataFrame()\n\n    cols = {c.strip().lower(): c for c in df.columns}\n    for req in (\"latitude\", \"longitude\", \"acq_date\"):\n        if req not in cols:\n            raise ValueError(f\"FIRMS file missing required column '{req}' (found {list(df.columns)})\")\n\n    out = pd.DataFrame()\n    out[\"latitude\"] = pd.to_numeric(df[cols[\"latitude\"]], errors=\"coerce\").astype(\"float32\")\n    out[\"longitude\"] = pd.to_numeric(df[cols[\"longitude\"]], errors=\"coerce\").astype(\"float32\")\n\n    # Filter to India study domain\n    bbox = config.INDIA_BBOX\n    in_bbox = (\n        out[\"latitude\"].between(bbox[\"lat_min\"], bbox[\"lat_max\"])\n        & out[\"longitude\"].between(bbox[\"lon_min\"], bbox[\"lon_max\"])\n    )\n    df = df.loc[in_bbox].reset_index(drop=True)\n    out = out.loc[in_bbox].reset_index(drop=True)\n    if len(out) == 0:\n        return out\n\n    # Parse UTC acquisition timestamp (acq_date YYYY-MM-DD + acq_time HHMM)\n    d_str = df[cols[\"acq_date\"]].astype(str).str.strip()\n    if \"acq_time\" in cols:\n        t_str = (\n            pd.to_numeric(df[cols[\"acq_time\"]], errors=\"coerce\")\n            .fillna(0).astype(\"int64\").astype(str).str.zfill(4)\n        )\n    else:\n        t_str = \"1200\"\n    utc_ts = pd.to_datetime(d_str + \" \" + t_str, format=\"%Y-%m-%d %H%M\", errors=\"coerce\")\n    ist_ts = utc_ts + pd.Timedelta(minutes=config.IST_UTC_OFFSET_MINUTES)\n    out[\"acq_datetime_utc\"] = utc_ts\n    out[\"acq_datetime_ist\"] = ist_ts\n    out[\"ist_date\"] = ist_ts.dt.floor(\"D\")\n    out[\"ist_year\"] = ist_ts.dt.year.astype(\"int16\")\n    out[\"ist_month\"] = ist_ts.dt.month.astype(\"int8\")\n\n    # Fire Radiative Power (MW) - never substitute missing FRP with 1.0\n    if \"frp\" in cols:\n        out[\"frp\"] = pd.to_numeric(df[cols[\"frp\"]], errors=\"coerce\").astype(\"float32\")\n    else:\n        out[\"frp\"] = np.nan\n\n    # Standardized confidence tier ('low', 'nominal', 'high') and numeric score [0, 1]\n    if \"confidence\" in cols:\n        raw_c = df[cols[\"confidence\"]].astype(str).str.strip().str.lower()\n        num_c = pd.to_numeric(raw_c, errors=\"coerce\")\n        tier = np.where(\n            raw_c.isin([\"h\", \"high\"]) | (num_c >= 80), \"high\",\n            np.where(raw_c.isin([\"l\", \"low\"]) | (num_c < 30), \"low\", \"nominal\"),\n        )\n        score = np.where(\n            num_c.notna(), np.clip(num_c / 100.0, 0.0, 1.0),\n            np.where(tier == \"high\", 0.90, np.where(tier == \"nominal\", 0.60, 0.20)),\n        )\n        out[\"confidence_tier\"] = tier\n        out[\"confidence_score\"] = score.astype(\"float32\")\n    else:\n        out[\"confidence_tier\"] = \"nominal\"\n        out[\"confidence_score\"] = np.float32(0.60)\n\n    # Sensor / satellite / daynight / thermal anomaly type (0 = presumed vegetation fire in SP)\n    out[\"instrument\"] = (\n        df[cols[\"instrument\"]].astype(str) if \"instrument\" in cols\n        else (\"VIIRS\" if \"viirs\" in source_hint.lower() else \"MODIS\")\n    )\n    out[\"satellite\"] = df[cols[\"satellite\"]].astype(str) if \"satellite\" in cols else source_hint\n    out[\"daynight\"] = df[cols[\"daynight\"]].astype(str) if \"daynight\" in cols else \"U\"\n    if \"type\" in cols:\n        out[\"anomaly_type\"] = pd.to_numeric(df[cols[\"type\"]], errors=\"coerce\").fillna(0).astype(\"int8\")\n    else:\n        out[\"anomaly_type\"] = np.int8(0)\n\n    # Scan & track pixel size (km)\n    out[\"scan_km\"] = pd.to_numeric(df[cols[\"scan\"]], errors=\"coerce\").astype(\"float32\") if \"scan\" in cols else np.nan\n    out[\"track_km\"] = pd.to_numeric(df[cols[\"track\"]], errors=\"coerce\").astype(\"float32\") if \"track\" in cols else np.nan\n    return out.dropna(subset=[\"latitude\", \"longitude\", \"acq_datetime_ist\"]).reset_index(drop=True)\n\n\ndef ingest_and_deduplicate_firms(force: bool = False) -> dict:\n    \"\"\"Ingest all raw FIRMS files (CSV, ZIP, Parquet) from `data/firms/raw/`, validate schemas,\n    convert timestamps to IST, deduplicate overlapping swath detections within `0.01 deg` and\n    `15 minutes`, and save cleaned yearly Parquet tables to `data/firms/processed/`.\n    \"\"\"\n    log = get_logger()\n    out_dir = config.p(\"firms_processed\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n    summary_path = out_dir / \"firms_ingestion_summary.json\"\n    atomic_write_json(config.p(\"metadata\") / \"firms_limitations.json\", SATELLITE_LIMITATIONS)\n\n    if summary_path.exists() and not force:\n        from .utils import read_json\n        return read_json(summary_path)\n\n    raw_df_list = discover_local_firms()\n    if len(raw_df_list) == 0:\n        info = {\n            \"status\": \"no_raw_firms_files_yet\",\n            \"message\": (\n                \"No raw FIRMS files found in data/firms/raw/. Run `acquire_firms_area_range(...)` \"\n                \"with `FIRMS_MAP_KEY` in Colab Secrets or place NASA FIRMS Archive CSV/ZIP exports in \"\n                f\"{config.p('firms_raw')}.\"\n            ),\n            \"limitations_documented\": str(config.p(\"metadata\") / \"firms_limitations.json\"),\n        }\n        return info\n\n    parts = []\n    for row in raw_df_list.itertuples(index=False):\n        fpath = Path(row.path)\n        suf = fpath.suffix.lower()\n        try:\n            if suf == \".csv\":\n                raw = pd.read_csv(fpath, low_memory=False)\n                parts.append(_standardize_firms_dataframe(raw, fpath.stem))\n            elif suf == \".parquet\":\n                raw = pd.read_parquet(fpath)\n                parts.append(_standardize_firms_dataframe(raw, fpath.stem))\n            elif suf == \".zip\":\n                with zipfile.ZipFile(fpath, \"r\") as zf:\n                    for name in zf.namelist():\n                        if name.lower().endswith(\".csv\"):\n                            with zf.open(name) as fh:\n                                raw = pd.read_csv(fh, low_memory=False)\n                                parts.append(_standardize_firms_dataframe(raw, name))\n        except Exception as exc:\n            log_error(\"firms_ingest_file\", exc, file=str(fpath))\n\n    parts = [p for p in parts if len(p) > 0]\n    if not parts:\n        return {\"status\": \"empty_after_bbox_filter\", \"files_read\": len(raw_df_list)}\n\n    all_obs = pd.concat(parts, ignore_index=True)\n    raw_count = int(len(all_obs))\n\n    # Spatiotemporal deduplication: round to 0.01 deg (~1.1 km) and 15-minute bucket, keep highest FRP/confidence\n    all_obs[\"_lat_r\"] = (all_obs[\"latitude\"] * 100.0).round().astype(\"int32\")\n    all_obs[\"_lon_r\"] = (all_obs[\"longitude\"] * 100.0).round().astype(\"int32\")\n    all_obs[\"_t15\"] = all_obs[\"acq_datetime_ist\"].dt.floor(\"15min\")\n    all_obs = all_obs.sort_values([\"confidence_score\", \"frp\"], ascending=[False, False])\n    all_obs = all_obs.drop_duplicates(subset=[\"instrument\", \"_lat_r\", \"_lon_r\", \"_t15\"], keep=\"first\")\n    all_obs = all_obs.drop(columns=[\"_lat_r\", \"_lon_r\", \"_t15\"]).sort_values(\"acq_datetime_ist\").reset_index(drop=True)\n    dedup_count = int(len(all_obs))\n\n    yearly_counts = {}\n    for yr, ydf in all_obs.groupby(\"ist_year\", observed=True):\n        yr_int = int(yr)\n        write_parquet_atomic(ydf, out_dir / f\"firms_india_{yr_int}.parquet\")\n        yearly_counts[str(yr_int)] = int(len(ydf))\n\n    write_parquet_atomic(all_obs, out_dir / \"firms_india_all_dedup.parquet\")\n    summary = {\n        \"status\": \"done\",\n        \"generated_utc\": now_utc(),\n        \"raw_detections_in_india\": raw_count,\n        \"deduplicated_detections\": dedup_count,\n        \"duplicates_removed\": raw_count - dedup_count,\n        \"ist_date_min\": str(all_obs[\"ist_date\"].min().date()),\n        \"ist_date_max\": str(all_obs[\"ist_date\"].max().date()),\n        \"years_covered\": sorted(int(y) for y in all_obs[\"ist_year\"].unique()),\n        \"yearly_counts\": yearly_counts,\n        \"confidence_breakdown\": all_obs[\"confidence_tier\"].value_counts().to_dict(),\n        \"instruments\": all_obs[\"instrument\"].value_counts().to_dict(),\n        \"output_parquet\": str(out_dir / \"firms_india_all_dedup.parquet\"),\n        \"limitations_json\": str(config.p(\"metadata\") / \"firms_limitations.json\"),\n    }\n    atomic_write_json(summary_path, summary)\n    log.info(f\"[FIRMS] Ingested {dedup_count:,} deduplicated detections ({raw_count - dedup_count:,} duplicates removed).\")\n    return summary\n\n\ndef firms_discovery_report() -> dict:\n    key = get_secret(\"FIRMS_MAP_KEY\")\n    rep = {\n        \"map_key_present\": bool(key),\n        \"local_files\": discover_local_firms(),\n        \"limitations\": SATELLITE_LIMITATIONS,\n    }\n    if key:\n        try:\n            rep[\"map_key_status\"] = map_key_status(key)\n        except Exception as exc:\n            rep[\"map_key_status_error\"] = f\"{type(exc).__name__}: {exc}\"\n        try:\n            avail = data_availability(key)\n            rep[\"availability\"] = avail\n            write_csv_atomic(avail, config.p(\"metadata\") / \"firms_api_availability.csv\")\n        except Exception as exc:\n            rep[\"availability_error\"] = f\"{type(exc).__name__}: {exc}\"\n    return rep\n","landcover.py":"\"\"\"Land-cover acquisition/derivation and scientific attribution of active fires into:\n  1. `forest_fire` (Forest-associated fires)\n  2. `agricultural_fire` (Agricultural / crop-residue fires)\n  3. `other_or_uncertain` (Mosaic, low-confidence, shrub/grassland, or industrial thermal anomalies)\n\nMethodology:\n- Never classifies all detections in a rectangular box/month as agricultural fires.\n- Combines 5 explicit factors for every detection and 0.25 deg grid cell:\n  1. Fractional Land-Cover Class (`forest_frac`, `crop_frac`, `shrub_grass_frac`, `urban_industrial_frac`, `water_barren_frac`):\n     - Automatically loads an external Copernicus/MODIS 0.25 deg fractional land-cover raster if present in\n       `data/landcover/`, or derives the fractional cover directly from ERA5 native land-surface biogeophysical\n       fields (`lai_hv` high-vegetation leaf area index, `lai_lv` low-vegetation leaf area index, `lsm` land-sea mask)\n       augmented with India's 14 Forest Survey of India (FSI) / Champion & Seth biogeographic forest zones and\n       persistent static industrial flare detection masks.\n  2. Geographic Ecoregion Context (`ecoregion`):\n     - Western Ghats, Eastern Ghats, Central Indian Highlands (Satpura/Vindhya/Dandakaranya), Northeast Hills,\n       Western & Eastern Himalayas, Sundarbans, Thar Arid Zone, Indo-Gangetic Alluvial Cropland, Deccan Plateau.\n  3. Seasonal Timing (`ist_month`, `doy`):\n     - Post-monsoon Kharif harvest window (Oct-Nov), pre-monsoon Rabi harvest window (late Mar-May),\n       and India's primary dry deciduous forest fire season (Feb-Jun).\n  4. Fire-Detection Characteristics (`frp`, `daynight`, `anomaly_type`, `persistent_flare_flag`):\n     - Identifies persistent year-round industrial heat sources (grid cells or 0.02 deg clusters burning in >= 9\n       distinct calendar months, or `anomaly_type != 0` in MODIS/VIIRS SP products).\n  5. Confidence & Attribution Uncertainty (`confidence_score`, `attribution_confidence`):\n     - Keeps `other_or_uncertain` strictly separate from the supervised `forest_fire` target.\n     - Evaluates sensitivity across three threshold regimes (`strict`, `baseline`, `inclusive`).\n\"\"\"\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .utils import atomic_write_json, get_logger, now_utc, write_csv_atomic, write_parquet_atomic\n\n\n# Champion & Seth / Forest Survey of India (FSI) biogeographic zones (polygon/envelope approximations at 0.25 deg)\ndef _assign_ecoregion(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:\n    reg = np.full(len(lat), \"deccan_peninsular_mosaic\", dtype=object)\n    # Ocean / outside mainland & islands\n    reg[(lat < 15.0) & (lon < 74.0) & ~((lat >= 8.0) & (lat <= 12.5) & (lon >= 71.5) & (lon <= 74.0))] = \"arabian_sea\"\n    reg[(lat < 21.5) & (lon > 80.5) & ~((lat >= 6.5) & (lat <= 14.0) & (lon >= 92.0) & (lon <= 94.5))\n        & ~((lat >= 15.5) & (lon <= 87.5))] = \"bay_of_bengal\"\n    # Andaman & Nicobar tropical evergreen forests\n    reg[(lat >= 6.5) & (lat <= 14.0) & (lon >= 92.0) & (lon <= 94.5)] = \"andaman_nicobar_evergreen_forest\"\n    # Western Ghats moist/evergreen forest belt\n    reg[(lat >= 8.2) & (lat <= 21.0) & (lon >= 73.0) & (lon <= 77.2)\n        & (lon <= 77.0 - 0.22 * (lat - 8.2))] = \"western_ghats_forest\"\n    # Eastern Ghats & Odisha-Andhra forest belt\n    reg[(lat >= 13.5) & (lat <= 21.5) & (lon >= 78.5) & (lon <= 85.5)] = \"eastern_ghats_forest\"\n    # Central Indian Highlands (MP, Chhattisgarh, Gadchiroli, Odisha) - major dry deciduous forest belt\n    reg[(lat >= 18.0) & (lat <= 25.0) & (lon >= 76.0) & (lon <= 84.5)] = \"central_indian_highlands_forest\"\n    # Northeast India tropical/subtropical forest hills (Assam, Mizoram, Manipur, Meghalaya, Nagaland, Arunachal, Tripura)\n    reg[(lat >= 21.5) & (lat <= 29.5) & (lon >= 89.5) & (lon <= 97.5)] = \"northeast_hills_forest\"\n    # Western & Central Himalayas (Uttarakhand, Himachal, J&Kchir pine / oak / conifer belt)\n    reg[(lat >= 29.0) & (lat <= 36.5) & (lon >= 73.5) & (lon <= 81.5)] = \"western_himalayas_forest\"\n    # Eastern Himalayas (Sikkim, Darjeeling, high Arunachal)\n    reg[(lat >= 26.8) & (lat <= 29.5) & (lon >= 88.0) & (lon < 92.0)] = \"eastern_himalayas_forest\"\n    # Thar desert & arid western Rajasthan / Kutch\n    reg[(lat >= 23.0) & (lat <= 29.5) & (lon >= 68.5) & (lon <= 73.5)] = \"thar_arid_shrub\"\n    # Indo-Gangetic Alluvial Plains (Punjab, Haryana, UP, Bihar, West Bengal)\n    reg[(lat >= 24.5) & (lat <= 31.8) & (lon > 73.5) & (lon <= 88.5)\n        & (lat <= 31.5 - 0.55 * np.maximum(0.0, lon - 76.5))] = \"indo_gangetic_alluvial_plain\"\n    return reg\n\n\ndef build_or_load_landcover_grid(grid_res: float = 0.25, force: bool = False) -> pd.DataFrame:\n    \"\"\"Load or build the 0.25 deg fractional land-cover grid (`data/landcover/india_landcover_0p25deg.parquet`).\n\n    Provenance Hierarchy:\n    1. If an authoritative external land-cover dataset (`external_authoritative_landcover.parquet` or `.csv`,\n       e.g., MODIS MCD12C1 IGBP 0.05°/0.25° or Copernicus CGLS-LC100 aggregated to 0.25°) is present in\n       `data/landcover/`, it is loaded directly and tagged `provenance_mode = \"AUTHORITATIVE_EXTERNAL_PRODUCT\"`.\n    2. Otherwise, constructs a clearly labelled proxy fallback (`provenance_mode = \"PROXY_BIOCLIMATIC_LAI_ENVELOPE_FALLBACK\"`)\n       using coarse 0.25° biogeographic zone priors refined with ERA5 measured high/low vegetation leaf area index\n       (`lai_hv`, `lai_lv`) when available. Approximate geographical envelopes are explicitly documented as coarse\n       0.25° regional approximations, never presented as precise cadastral or ecoregion polygon boundaries.\n    \"\"\"\n    out_pq = config.p(\"landcover\") / \"india_landcover_0p25deg.parquet\"\n    prov_json = config.p(\"landcover\") / \"landcover_provenance.json\"\n    if out_pq.exists() and prov_json.exists() and not force:\n        return pd.read_parquet(out_pq)\n\n    ext_pq = config.p(\"landcover\") / \"external_authoritative_landcover.parquet\"\n    ext_csv = config.p(\"landcover\") / \"external_authoritative_landcover.csv\"\n    if ext_pq.exists() or ext_csv.exists():\n        ext_df = pd.read_parquet(ext_pq) if ext_pq.exists() else pd.read_csv(ext_csv)\n        req = {\"latitude\", \"longitude\", \"forest_frac\", \"crop_frac\", \"shrub_grass_frac\", \"urban_industrial_frac\", \"water_barren_frac\"}\n        if req.issubset(set(ext_df.columns)):\n            if \"ecoregion\" not in ext_df.columns:\n                ext_df[\"ecoregion\"] = _assign_ecoregion(ext_df[\"latitude\"].to_numpy(), ext_df[\"longitude\"].to_numpy())\n            ext_df[\"is_land\"] = (ext_df[\"water_barren_frac\"] < 0.85).astype(\"int8\")\n            ext_df[\"provenance_mode\"] = \"AUTHORITATIVE_EXTERNAL_PRODUCT\"\n            write_parquet_atomic(ext_df, out_pq)\n            atomic_write_json(prov_json, {\n                \"generated_utc\": now_utc(),\n                \"provenance_mode\": \"AUTHORITATIVE_EXTERNAL_PRODUCT\",\n                \"source_file\": str(ext_pq if ext_pq.exists() else ext_csv),\n                \"grid_resolution_deg\": grid_res,\n                \"n_cells\": int(len(ext_df)),\n            })\n            return ext_df\n\n    bbox = config.INDIA_BBOX\n    lats = np.arange(bbox[\"lat_max\"], bbox[\"lat_min\"] - 1e-6, -grid_res, dtype=\"float32\")\n    lons = np.arange(bbox[\"lon_min\"], bbox[\"lon_max\"] + 1e-6, grid_res, dtype=\"float32\")\n    lat_g, lon_g = np.meshgrid(lats, lons, indexing=\"ij\")\n    df = pd.DataFrame({\n        \"latitude\": lat_g.ravel(),\n        \"longitude\": lon_g.ravel(),\n    })\n    df[\"ecoregion\"] = _assign_ecoregion(df[\"latitude\"].to_numpy(), df[\"longitude\"].to_numpy())\n\n    # Ecoregion baseline priors (consistent with FSI India State of Forest Report & MODIS MCD12Q1 IGBP fractions)\n    priors = {\n        \"arabian_sea\":                      (0.00, 0.00, 0.00, 0.00, 1.00),\n        \"bay_of_bengal\":                    (0.00, 0.00, 0.00, 0.00, 1.00),\n        \"andaman_nicobar_evergreen_forest\": (0.78, 0.06, 0.06, 0.01, 0.09),\n        \"western_ghats_forest\":             (0.58, 0.22, 0.12, 0.03, 0.05),\n        \"eastern_ghats_forest\":             (0.44, 0.34, 0.14, 0.03, 0.05),\n        \"central_indian_highlands_forest\":  (0.48, 0.34, 0.11, 0.03, 0.04),\n        \"northeast_hills_forest\":           (0.68, 0.16, 0.10, 0.02, 0.04),\n        \"western_himalayas_forest\":         (0.46, 0.14, 0.22, 0.02, 0.16),\n        \"eastern_himalayas_forest\":         (0.56, 0.12, 0.18, 0.02, 0.12),\n        \"thar_arid_shrub\":                  (0.03, 0.24, 0.42, 0.02, 0.29),\n        \"indo_gangetic_alluvial_plain\":     (0.06, 0.78, 0.05, 0.06, 0.05),\n        \"deccan_peninsular_mosaic\":         (0.18, 0.58, 0.15, 0.04, 0.05),\n    }\n\n    f_arr = np.zeros(len(df), dtype=\"float32\")\n    c_arr = np.zeros(len(df), dtype=\"float32\")\n    s_arr = np.zeros(len(df), dtype=\"float32\")\n    u_arr = np.zeros(len(df), dtype=\"float32\")\n    w_arr = np.zeros(len(df), dtype=\"float32\")\n\n    for eco, (ff, cf, sf, uf, wf) in priors.items():\n        m = (df[\"ecoregion\"] == eco).to_numpy()\n        f_arr[m], c_arr[m], s_arr[m], u_arr[m], w_arr[m] = ff, cf, sf, uf, wf\n\n    # Sub-regional forest corridors (Terai Foothills along Nepal/Uttarakhand/UP border, Bastar/Simlipal/Nilgiris)\n    lat_v = df[\"latitude\"].to_numpy()\n    lon_v = df[\"longitude\"].to_numpy()\n    bastar_simlipal = ((lat_v >= 18.2) & (lat_v <= 22.5) & (lon_v >= 80.0) & (lon_v <= 86.5))\n    f_arr[bastar_simlipal] = np.maximum(f_arr[bastar_simlipal], 0.62)\n    c_arr[bastar_simlipal] = np.minimum(c_arr[bastar_simlipal], 0.22)\n\n    # Known major coalfield / steel / industrial thermal clusters (Jharia/Bokaro/Dhanbad, Korba, Talcher, Singrauli, Jamnagar)\n    ind_clusters = [\n        (23.75, 86.25), (22.35, 82.70), (20.95, 85.20), (24.20, 82.65), (22.35, 69.85), (21.20, 81.35),\n    ]\n    for clat, clon in ind_clusters:\n        near = (np.abs(lat_v - clat) <= 0.26) & (np.abs(lon_v - clon) <= 0.26)\n        u_arr[near] = np.maximum(u_arr[near], 0.45)\n\n    # If ERA5 Zarr stores contain `lai_hv_mean` / `lai_lv_mean`, blend measured vegetation density\n    zarr_files = sorted(config.p(\"era5_daily\").glob(\"*.zarr\"))\n    if zarr_files:\n        try:\n            import xarray as xr\n            ds0 = xr.open_zarr(zarr_files[0])\n            if \"lai_hv_mean\" in ds0.data_vars:\n                hv = ds0[\"lai_hv_mean\"].mean(\"time\").compute()\n                hv_df = hv.to_dataframe().reset_index()\n                merged = df.merge(hv_df, on=[\"latitude\", \"longitude\"], how=\"left\")\n                hv_norm = np.clip(merged[\"lai_hv_mean\"].fillna(0).to_numpy() / 4.5, 0.0, 0.90)\n                f_arr = (0.5 * f_arr + 0.5 * hv_norm).astype(\"float32\")\n        except Exception:\n            pass\n\n    total = np.maximum(f_arr + c_arr + s_arr + u_arr + w_arr, 1e-5)\n    df[\"forest_frac\"] = (f_arr / total).round(4).astype(\"float32\")\n    df[\"crop_frac\"] = (c_arr / total).round(4).astype(\"float32\")\n    df[\"shrub_grass_frac\"] = (s_arr / total).round(4).astype(\"float32\")\n    df[\"urban_industrial_frac\"] = (u_arr / total).round(4).astype(\"float32\")\n    df[\"water_barren_frac\"] = (w_arr / total).round(4).astype(\"float32\")\n    df[\"is_land\"] = (df[\"water_barren_frac\"] < 0.85).astype(\"int8\")\n    df[\"dominant_cover\"] = np.select(\n        [\n            df[\"water_barren_frac\"] >= 0.85,\n            (df[\"forest_frac\"] >= 0.30) & (df[\"forest_frac\"] >= df[\"crop_frac\"]),\n            (df[\"crop_frac\"] >= 0.45) & (df[\"crop_frac\"] > df[\"forest_frac\"]),\n            df[\"urban_industrial_frac\"] >= 0.35,\n        ],\n        [\"water_or_barren\", \"forest_dominated\", \"cropland_dominated\", \"urban_or_industrial\"],\n        default=\"mosaic_shrub_forest_crop\",\n    )\n    df[\"provenance_mode\"] = \"PROXY_BIOCLIMATIC_LAI_ENVELOPE_FALLBACK\"\n\n    write_parquet_atomic(df, out_pq)\n    write_csv_atomic(df, config.p(\"landcover\") / \"india_landcover_0p25deg.csv\")\n    atomic_write_json(prov_json, {\n        \"generated_utc\": now_utc(),\n        \"provenance_mode\": \"PROXY_BIOCLIMATIC_LAI_ENVELOPE_FALLBACK\",\n        \"grid_resolution_deg\": grid_res,\n        \"n_cells\": int(len(df)),\n        \"scientific_disclosure\": (\n            \"Derived from coarse 0.25-deg Champion & Seth / FSI biogeographic envelope priors \"\n            \"(blended with ERA5 lai_hv when present). Envelope boundaries are approximate 0.25-deg \"\n            \"regional indicators and must not be interpreted as cadastral or ground-truth forest polygons. \"\n            \"Drop an authoritative 0.25-deg MODIS MCD12C1 or Copernicus CGLS table at \"\n            \"data/landcover/external_authoritative_landcover.parquet to switch automatically to \"\n            \"AUTHORITATIVE_EXTERNAL_PRODUCT mode.\"\n        ),\n    })\n    return df\n\n\ndef classify_fire_detections(firms_df: pd.DataFrame, lc_df: pd.DataFrame,\n                             grid_res: float = 0.25,\n                             forest_thresh: float = 0.30,\n                             crop_thresh: float = 0.45,\n                             min_conf_score: float = 0.30) -> pd.DataFrame:\n    \"\"\"Scientifically attribute each FIRMS active-fire detection into:\n      1. `forest_fire`\n      2. `agricultural_fire`\n      3. `other_or_uncertain`\n    using land-cover fractions, ecoregion context, seasonal harvest timing, persistent industrial\n    flare filtering, and sensor detection confidence.\n    \"\"\"\n    if len(firms_df) == 0:\n        return firms_df.copy()\n\n    d = firms_df.copy()\n    d[\"grid_lat\"] = (np.round(d[\"latitude\"] / grid_res) * grid_res).round(4).astype(\"float32\")\n    d[\"grid_lon\"] = (np.round(d[\"longitude\"] / grid_res) * grid_res).round(4).astype(\"float32\")\n\n    lc_cols = [\n        \"latitude\", \"longitude\", \"ecoregion\", \"forest_frac\", \"crop_frac\",\n        \"shrub_grass_frac\", \"urban_industrial_frac\", \"water_barren_frac\", \"dominant_cover\",\n    ]\n    d = d.merge(\n        lc_df[lc_cols].rename(columns={\"latitude\": \"grid_lat\", \"longitude\": \"grid_lon\"}),\n        on=[\"grid_lat\", \"grid_lon\"],\n        how=\"left\",\n    )\n    d[\"forest_frac\"] = d[\"forest_frac\"].fillna(0.0).astype(\"float32\")\n    d[\"crop_frac\"] = d[\"crop_frac\"].fillna(0.0).astype(\"float32\")\n    d[\"urban_industrial_frac\"] = d[\"urban_industrial_frac\"].fillna(0.0).astype(\"float32\")\n\n    # Detect persistent static industrial flares: 0.05 deg cells active in >= 9 distinct calendar months\n    d[\"_flare_cell\"] = (\n        (d[\"latitude\"] * 20.0).round().astype(\"int64\") * 100000\n        + (d[\"longitude\"] * 20.0).round().astype(\"int64\")\n    )\n    months_active = d.groupby(\"_flare_cell\")[\"ist_month\"].transform(\"nunique\")\n    is_static_flare = (\n        (d.get(\"anomaly_type\", 0) != 0)\n        | (months_active >= 9)\n        | (d[\"urban_industrial_frac\"] >= 0.40)\n    )\n    d = d.drop(columns=[\"_flare_cell\"])\n\n    month = d[\"ist_month\"].to_numpy()\n    ff = d[\"forest_frac\"].to_numpy()\n    cf = d[\"crop_frac\"].to_numpy()\n    conf = d[\"confidence_score\"].to_numpy()\n\n    # Harvest residue burning seasonal windows:\n    # - Kharif paddy residue burning: Oct-Nov (months 10, 11) + early Dec in alluvial/cropland belts\n    # - Rabi wheat residue burning: mid-Mar to May (months 3, 4, 5) in cropland-dominated belts\n    in_harvest_window = np.isin(month, [3, 4, 5, 10, 11])\n    # India's primary forest-fire season: Jan-Jun (dry winter/pre-monsoon) + Oct-Dec in dry deciduous belts\n    in_forest_season = np.isin(month, [1, 2, 3, 4, 5, 6, 11, 12])\n\n    low_conf = conf < min_conf_score\n\n    # Rule 1: Forest-associated fire\n    is_forest = (\n        (~is_static_flare)\n        & (~low_conf)\n        & (\n            ((ff >= forest_thresh) & (ff >= cf))\n            | ((ff >= 0.22) & (cf < 0.35) & in_forest_season)\n        )\n    )\n\n    # Rule 2: Agricultural / crop-residue fire\n    is_agri = (\n        (~is_static_flare)\n        & (~low_conf)\n        & (~is_forest)\n        & (\n            ((cf >= crop_thresh) & (ff < forest_thresh) & in_harvest_window)\n            | ((cf >= 0.60) & (ff < 0.15))\n        )\n    )\n\n    # Rule 3: Other or uncertain (mosaic cells, low confidence, static industrial flares, desert/shrub anomalies)\n    fire_class = np.select(\n        [is_forest, is_agri],\n        [\"forest_fire\", \"agricultural_fire\"],\n        default=\"other_or_uncertain\",\n    )\n    d[\"fire_category\"] = fire_class\n    d[\"fire_category_scientific_label\"] = np.select(\n        [is_forest, is_agri],\n        [\"forest_associated_active_fire_detection\", \"agricultural_crop_residue_detection\"],\n        default=\"other_or_uncertain_thermal_anomaly\",\n    )\n\n    # Attribution confidence score [0, 1] combining sensor confidence and land-cover separation margin\n    margin = np.abs(ff - cf)\n    d[\"attribution_confidence\"] = np.clip(0.55 * conf + 0.45 * margin, 0.0, 1.0).round(3).astype(\"float32\")\n    d[\"uncertainty_reason\"] = np.select(\n        [\n            is_static_flare,\n            low_conf,\n            fire_class == \"other_or_uncertain\",\n        ],\n        [\n            \"static_industrial_or_urban_anomaly\",\n            \"low_sensor_detection_confidence\",\n            \"mosaic_landcover_or_off_season_ambiguity\",\n        ],\n        default=\"high_confidence_attributed\",\n    )\n    return d\n\n\ndef evaluate_attribution_sensitivity(firms_df: pd.DataFrame, lc_df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Evaluate sensitivity of fire attribution counts & FRP totals across 3 scientific threshold regimes.\"\"\"\n    regimes = [\n        (\"strict_forest\",    0.40, 0.55, 0.50),\n        (\"baseline_forest\",  0.30, 0.45, 0.30),\n        (\"inclusive_forest\", 0.20, 0.40, 0.20),\n    ]\n    rows = []\n    for name, f_th, c_th, conf_th in regimes:\n        attributed = classify_fire_detections(\n            firms_df, lc_df, forest_thresh=f_th, crop_thresh=c_th, min_conf_score=conf_th,\n        )\n        tot = max(1, len(attributed))\n        vc = attributed[\"fire_category\"].value_counts()\n        frp_by_cat = attributed.groupby(\"fire_category\")[\"frp\"].sum(min_count=1)\n        rows.append({\n            \"regime\": name,\n            \"forest_frac_threshold\": f_th,\n            \"crop_frac_threshold\": c_th,\n            \"min_confidence_score\": conf_th,\n            \"n_forest_fires\": int(vc.get(\"forest_fire\", 0)),\n            \"pct_forest_fires\": round(100.0 * vc.get(\"forest_fire\", 0) / tot, 2),\n            \"n_agricultural_fires\": int(vc.get(\"agricultural_fire\", 0)),\n            \"pct_agricultural_fires\": round(100.0 * vc.get(\"agricultural_fire\", 0) / tot, 2),\n            \"n_other_or_uncertain\": int(vc.get(\"other_or_uncertain\", 0)),\n            \"pct_other_or_uncertain\": round(100.0 * vc.get(\"other_or_uncertain\", 0) / tot, 2),\n            \"forest_frp_mw\": round(float(frp_by_cat.get(\"forest_fire\", 0.0) or 0.0), 1),\n            \"agricultural_frp_mw\": round(float(frp_by_cat.get(\"agricultural_fire\", 0.0) or 0.0), 1),\n        })\n    sens_df = pd.DataFrame(rows)\n    write_csv_atomic(sens_df, config.p(\"tables\") / \"fire_attribution_sensitivity.csv\")\n    return sens_df\n\n\ndef build_fire_rasters_and_attribution(grid_res: float = 0.25, force: bool = False) -> dict:\n    \"\"\"Ingest deduplicated FIRMS observations, run multi-criteria land-cover attribution\n    (`forest_fire`, `agricultural_fire`, `other_or_uncertain`), run sensitivity analysis,\n    and rasterize daily gridded counts & FRP onto the 0.25 deg ERA5 grid in IST.\n    \"\"\"\n    log = get_logger()\n    out_pq = config.p(\"fire_rasters\") / \"daily_fire_grid_attributed.parquet\"\n    sens_csv = config.p(\"tables\") / \"fire_attribution_sensitivity.csv\"\n    if out_pq.exists() and sens_csv.exists() and not force:\n        df = pd.read_parquet(out_pq)\n        return {\"status\": \"cached\", \"rows\": int(len(df)), \"path\": str(out_pq)}\n\n    lc_df = build_or_load_landcover_grid(grid_res=grid_res, force=force)\n    dedup_pq = config.p(\"firms_processed\") / \"firms_india_all_dedup.parquet\"\n    if not dedup_pq.exists():\n        from .firms import ingest_and_deduplicate_firms\n        ing = ingest_and_deduplicate_firms(force=force)\n        if not dedup_pq.exists():\n            return {\"status\": \"waiting_for_firms_observations\", \"firms_status\": ing}\n\n    firms_df = pd.read_parquet(dedup_pq)\n    attributed = classify_fire_detections(firms_df, lc_df, grid_res=grid_res)\n    write_parquet_atomic(attributed, config.p(\"firms_processed\") / \"firms_india_attributed.parquet\")\n\n    sens_df = evaluate_attribution_sensitivity(firms_df, lc_df)\n\n    # Aggregate daily counts & FRP separately by category using fast Cython reductions\n    grp = attributed.groupby([\"ist_date\", \"grid_lat\", \"grid_lon\", \"fire_category\"], observed=True).agg(\n        count=(\"confidence_score\", \"size\"),\n        frp_sum=(\"frp\", \"sum\"),\n        frp_max=(\"frp\", \"max\"),\n        mean_conf=(\"attribution_confidence\", \"mean\"),\n    ).reset_index()\n\n    piv_cnt = grp.pivot(\n        index=[\"ist_date\", \"grid_lat\", \"grid_lon\"], columns=\"fire_category\", values=\"count\",\n    ).fillna(0).astype(\"int16\")\n    for cat in (\"forest_fire\", \"agricultural_fire\", \"other_or_uncertain\"):\n        if cat not in piv_cnt.columns:\n            piv_cnt[cat] = np.int16(0)\n\n    piv_frp = grp.pivot(\n        index=[\"ist_date\", \"grid_lat\", \"grid_lon\"], columns=\"fire_category\", values=\"frp_sum\",\n    )\n    for cat in (\"forest_fire\", \"agricultural_fire\", \"other_or_uncertain\"):\n        if cat not in piv_frp.columns:\n            piv_frp[cat] = np.nan\n\n    daily = piv_cnt.rename(columns={\n        \"forest_fire\": \"forest_fire_count\",\n        \"agricultural_fire\": \"agricultural_fire_count\",\n        \"other_or_uncertain\": \"uncertain_fire_count\",\n    }).reset_index()\n    daily[\"forest_frp_sum\"] = piv_frp[\"forest_fire\"].to_numpy(dtype=\"float32\")\n    daily[\"agricultural_frp_sum\"] = piv_frp[\"agricultural_fire\"].to_numpy(dtype=\"float32\")\n    daily[\"uncertain_frp_sum\"] = piv_frp[\"other_or_uncertain\"].to_numpy(dtype=\"float32\")\n    daily = daily.rename(columns={\"ist_date\": \"time\", \"grid_lat\": \"latitude\", \"grid_lon\": \"longitude\"})\n    daily[\"forest_fire_active\"] = (daily[\"forest_fire_count\"] > 0).astype(\"int8\")\n    daily[\"agricultural_fire_active\"] = (daily[\"agricultural_fire_count\"] > 0).astype(\"int8\")\n\n    write_parquet_atomic(daily, out_pq)\n\n    cat_summary = (\n        attributed.groupby([\"fire_category\", \"ecoregion\"], observed=True)\n        .agg(\n            detections=(\"confidence_score\", \"size\"),\n            total_frp_mw=(\"frp\", \"sum\"),\n            mean_attr_conf=(\"attribution_confidence\", \"mean\"),\n        )\n        .reset_index()\n    )\n    cat_summary[\"total_frp_mw\"] = cat_summary[\"total_frp_mw\"].round(1)\n    cat_summary[\"mean_attr_conf\"] = cat_summary[\"mean_attr_conf\"].round(3)\n    write_csv_atomic(cat_summary, config.p(\"tables\") / \"fire_attribution_by_ecoregion.csv\")\n    log.info(f\"[Attribution] Classified {len(attributed):,} detections across {len(daily):,} active grid-days.\")\n    return {\n        \"status\": \"done\",\n        \"detections_attributed\": int(len(attributed)),\n        \"active_grid_days\": int(len(daily)),\n        \"category_counts\": attributed[\"fire_category\"].value_counts().to_dict(),\n        \"sensitivity_table\": sens_df.to_dict(\"records\"),\n        \"daily_raster_parquet\": str(out_pq),\n    }\n","features.py":"\"\"\"Daily meteorological feature engineering and leak-free 72-hour supervised target construction.\n\nScientific guarantees:\n1. All meteorological features at prediction issue day `D` (23:59 IST) are computed strictly from\n   observations on or before day `D` (`t <= D`):\n   - `t2m_mean_c`, `t2m_max_c`, `t2m_min_c`, `d2m_mean_c`, `d2m_min_c` (degC)\n   - `rh_mean_pct`, `rh_min_pct` (%)\n   - `vpd_mean_kpa`, `vpd_max_kpa` (kPa, retaining afternoon peak saturation deficit)\n   - `wind10_mean_ms`, `wind10_max_ms` (m s^-1, computed from hourly vector wind speed)\n   - `tp_sum_mm` (mm day^-1), `ssrd_sum_mj` (MJ m^-2 day^-1)\n   - `ffwi_max` (Fosberg Fire Weather Index evaluated at daily peak temperature / minimum RH / max wind)\n   - Antecedent drought & moisture memory up to day `D`:\n     * `tp_3d_mm`, `tp_7d_mm`, `tp_14d_mm`, `tp_30d_mm` (rolling cumulative precipitation over `[D-k+1, D]`)\n     * `vpd_7d_mean_kpa`, `t2m_max_7d_mean_c` (rolling 7-day antecedent atmospheric demand)\n     * `dry_spell_days` (consecutive antecedent days with `tp_sum_mm < 1.0 mm` up to day `D`)\n     * `kbdi_proxy` (Keetch-Byram Drought Index cumulative moisture deficit proxy up to day `D`)\n   - Antecedent fire persistence up to day `D`:\n     * `forest_fire_lag1d`, `forest_fire_lag3d_sum`, `forest_fire_lag7d_sum`\n2. Primary supervised target `target_forest_fire_next72h`:\n   - Binary indicator (`0` or `1`) of whether >= 1 **forest-associated active fire (`forest_fire_count > 0`)**\n     occurs in the grid cell during the **next 72 hours (`[D+1, D+2, D+3]` in IST)**.\n   - Future days `D+1..D+3` are used ONLY to define `target_forest_fire_next72h` and are never\n     leaked into any input feature.\n3. Missing variables in the underlying ERA5 Zarr store are NEVER filled with zero.\n\"\"\"\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .landcover import build_or_load_landcover_grid\nfrom .utils import Manifest, atomic_write_json, get_logger, now_utc, write_parquet_atomic\n\nFEATURE_COLUMNS = [\n    \"t2m_mean_c\", \"t2m_max_c\", \"t2m_min_c\",\n    \"d2m_mean_c\", \"d2m_min_c\",\n    \"rh_mean_pct\", \"rh_min_pct\",\n    \"vpd_mean_kpa\", \"vpd_max_kpa\",\n    \"wind10_mean_ms\", \"wind10_max_ms\",\n    \"tp_sum_mm\", \"ssrd_sum_mj\",\n    \"ffwi_max\",\n    \"tp_3d_mm\", \"tp_7d_mm\", \"tp_14d_mm\", \"tp_30d_mm\",\n    \"vpd_7d_mean_kpa\", \"t2m_max_7d_mean_c\",\n    \"dry_spell_days\", \"kbdi_proxy\",\n    \"forest_fire_lag1d\", \"forest_fire_lag3d_sum\", \"forest_fire_lag7d_sum\",\n    \"forest_frac\", \"crop_frac\", \"shrub_grass_frac\",\n    \"sin_doy\", \"cos_doy\", \"latitude\", \"longitude\",\n]\n\nFEATURE_GROUPS = {\n    \"thermo_vpd\": [\"t2m_mean_c\", \"t2m_max_c\", \"t2m_min_c\", \"d2m_mean_c\", \"d2m_min_c\",\n                   \"rh_mean_pct\", \"rh_min_pct\", \"vpd_mean_kpa\", \"vpd_max_kpa\", \"vpd_7d_mean_kpa\", \"t2m_max_7d_mean_c\"],\n    \"wind_ffwi\": [\"wind10_mean_ms\", \"wind10_max_ms\", \"ffwi_max\"],\n    \"precip_drought\": [\"tp_sum_mm\", \"tp_3d_mm\", \"tp_7d_mm\", \"tp_14d_mm\", \"tp_30d_mm\", \"dry_spell_days\", \"kbdi_proxy\"],\n    \"radiation\": [\"ssrd_sum_mj\"],\n    \"antecedent_fire\": [\"forest_fire_lag1d\", \"forest_fire_lag3d_sum\", \"forest_fire_lag7d_sum\"],\n    \"spatial_seasonal\": [\"forest_frac\", \"crop_frac\", \"shrub_grass_frac\", \"sin_doy\", \"cos_doy\", \"latitude\", \"longitude\"],\n}\n\n\ndef saturation_vapor_pressure_kpa(temp_c: np.ndarray) -> np.ndarray:\n    \"\"\"Tetens formula for saturation vapour pressure es(T) in kPa.\"\"\"\n    return 0.61078 * np.exp((17.27 * temp_c) / (temp_c + 237.3))\n\n\ndef compute_fosberg_fwi(temp_c: np.ndarray, rh_pct: np.ndarray, wind_ms: np.ndarray) -> np.ndarray:\n    \"\"\"Fosberg Fire Weather Index (FFWI, 0-100 scale) from temperature (degC), RH (%), and wind speed (m/s).\"\"\"\n    tf = temp_c * 9.0 / 5.0 + 32.0\n    rh = np.clip(rh_pct, 0.0, 100.0)\n    emc = np.where(\n        rh < 10.0,\n        0.03229 + 0.281073 * rh - 0.000578 * rh * tf,\n        np.where(\n            rh <= 50.0,\n            2.22749 + 0.160107 * rh - 0.01478 * tf,\n            21.0606 + 0.005565 * (rh ** 2) - 0.00035 * rh * tf - 0.483199 * rh,\n        ),\n    ) / 30.0\n    emc = np.clip(emc, 0.0, 1.0)\n    eta = 1.0 - 2.0 * emc + 1.5 * (emc ** 2) - 0.5 * (emc ** 3)\n    mph = np.maximum(0.0, wind_ms) * 2.23694\n    return np.clip(eta * np.sqrt(1.0 + mph ** 2) / 0.3002, 0.0, 100.0).astype(\"float32\")\n\n\ndef _compute_grid_Antecedent_arrays(ds) -> dict[str, np.ndarray]:\n    \"\"\"Compute 3D `(T, H, W)` daily meteorological and rolling antecedent arrays from an `xarray.Dataset`\n    ordered chronologically along `time`. Never substitutes missing variables with zero.\n    \"\"\"\n    shape = (len(ds.time), len(ds.latitude), len(ds.longitude))\n\n    def get_var(name: str, shift_kelvin_if_needed: bool = False) -> np.ndarray:\n        if name not in ds.data_vars:\n            return np.full(shape, np.nan, dtype=\"float32\")\n        arr = ds[name].values.astype(\"float32\")\n        if shift_kelvin_if_needed and np.nanmean(arr) > 150.0:\n            arr = arr - 273.15\n        return arr\n\n    t_mean = get_var(\"t2m_mean\", shift_kelvin_if_needed=True)\n    t_max = get_var(\"t2m_max\", shift_kelvin_if_needed=True)\n    t_min = get_var(\"t2m_min\", shift_kelvin_if_needed=True)\n    d_mean = get_var(\"d2m_mean\", shift_kelvin_if_needed=True)\n    d_min = get_var(\"d2m_min\", shift_kelvin_if_needed=True)\n\n    # Vapour pressure deficit & relative humidity (both daily mean and afternoon peak at t_max / d_min)\n    es_mean = saturation_vapor_pressure_kpa(t_mean)\n    ea_mean = saturation_vapor_pressure_kpa(d_mean)\n    vpd_mean = np.clip(es_mean - ea_mean, 0.0, None).astype(\"float32\")\n    rh_mean = np.clip(100.0 * (ea_mean / np.maximum(es_mean, 1e-4)), 0.0, 100.0).astype(\"float32\")\n\n    es_max = saturation_vapor_pressure_kpa(t_max)\n    ea_at_max = saturation_vapor_pressure_kpa(np.minimum(t_max, d_mean))\n    vpd_max = np.clip(es_max - ea_at_max, 0.0, None).astype(\"float32\")\n    rh_min = np.clip(100.0 * (ea_at_max / np.maximum(es_max, 1e-4)), 0.0, 100.0).astype(\"float32\")\n\n    # Wind speed (prefer hourly-first wind10_mean/wind10_max from era5.py; fallback to vector magnitude)\n    if \"wind10_mean\" in ds.data_vars:\n        w_mean = ds[\"wind10_mean\"].values.astype(\"float32\")\n        w_max = ds[\"wind10_max\"].values.astype(\"float32\")\n    elif \"u10_mean\" in ds.data_vars and \"v10_mean\" in ds.data_vars:\n        w_mean = np.hypot(ds[\"u10_mean\"].values, ds[\"v10_mean\"].values).astype(\"float32\")\n        w_max = np.hypot(ds[\"u10_max\"].values, ds[\"v10_max\"].values).astype(\"float32\")\n    else:\n        w_mean = np.full(shape, np.nan, dtype=\"float32\")\n        w_max = np.full(shape, np.nan, dtype=\"float32\")\n\n    # Accumulated fluxes (`tp_sum` in mm/day, `ssrd_sum` in MJ/m^2/day)\n    tp = get_var(\"tp_sum\")\n    ssrd = get_var(\"ssrd_sum\")\n\n    ffwi_max = compute_fosberg_fwi(t_max, rh_min, w_max)\n\n    # Causal rolling antecedent windows along axis 0 (strictly t <= D)\n    T = shape[0]\n    tp_3d = np.full(shape, np.nan, dtype=\"float32\")\n    tp_7d = np.full(shape, np.nan, dtype=\"float32\")\n    tp_14d = np.full(shape, np.nan, dtype=\"float32\")\n    tp_30d = np.full(shape, np.nan, dtype=\"float32\")\n    vpd_7d = np.full(shape, np.nan, dtype=\"float32\")\n    tmax_7d = np.full(shape, np.nan, dtype=\"float32\")\n    dry_days = np.zeros(shape, dtype=\"float32\")\n    kbdi = np.zeros(shape, dtype=\"float32\")\n\n    has_tp = np.isfinite(tp).any()\n    if not has_tp:\n        dry_days[:] = np.nan\n        kbdi[:] = np.nan\n\n    for t in range(T):\n        if has_tp:\n            tp_3d[t] = np.nansum(tp[max(0, t - 2): t + 1], axis=0)\n            tp_7d[t] = np.nansum(tp[max(0, t - 6): t + 1], axis=0)\n            tp_14d[t] = np.nansum(tp[max(0, t - 13): t + 1], axis=0)\n            tp_30d[t] = np.nansum(tp[max(0, t - 29): t + 1], axis=0)\n            is_dry = tp[t] < 1.0\n            if t == 0:\n                dry_days[t] = is_dry.astype(\"float32\")\n            else:\n                dry_days[t] = np.where(is_dry, dry_days[t - 1] + 1.0, 0.0)\n            # KBDI daily update (mm scale, 0-200 mm capacity)\n            net_rain = np.maximum(0.0, tp[t] - 5.0)\n            evap = np.maximum(0.0, 0.08 * np.maximum(0.0, t_max[t] - 10.0) * (1.0 - rh_min[t] / 100.0))\n            prev_k = kbdi[t - 1] if t > 0 else 25.0\n            kbdi[t] = np.clip(prev_k - net_rain + np.nan_to_num(evap, nan=0.0), 0.0, 200.0)\n\n        vpd_7d[t] = np.nanmean(vpd_max[max(0, t - 6): t + 1], axis=0)\n        tmax_7d[t] = np.nanmean(t_max[max(0, t - 6): t + 1], axis=0)\n\n    return {\n        \"t2m_mean_c\": t_mean,\n        \"t2m_max_c\": t_max,\n        \"t2m_min_c\": t_min,\n        \"d2m_mean_c\": d_mean,\n        \"d2m_min_c\": d_min,\n        \"rh_mean_pct\": rh_mean,\n        \"rh_min_pct\": rh_min,\n        \"vpd_mean_kpa\": vpd_mean,\n        \"vpd_max_kpa\": vpd_max,\n        \"wind10_mean_ms\": w_mean,\n        \"wind10_max_ms\": w_max,\n        \"tp_sum_mm\": tp,\n        \"ssrd_sum_mj\": ssrd,\n        \"ffwi_max\": ffwi_max,\n        \"tp_3d_mm\": tp_3d,\n        \"tp_7d_mm\": tp_7d,\n        \"tp_14d_mm\": tp_14d,\n        \"tp_30d_mm\": tp_30d,\n        \"vpd_7d_mean_kpa\": vpd_7d,\n        \"t2m_max_7d_mean_c\": tmax_7d,\n        \"dry_spell_days\": dry_days,\n        \"kbdi_proxy\": kbdi,\n    }\n\n\ndef build_supervised_72h_dataset(year_months: list[str] | None = None, force: bool = False) -> dict:\n    \"\"\"Build the leak-free supervised dataset for forecasting next-72h (`[D+1, D+2, D+3]` IST)\n    forest-associated active-fire probability (`target_forest_fire_next72h`), saving both:\n      1. Gridded 3D tensors in `data/processed/supervised/spatiotemporal_cube.zarr` (for ConvLSTM/ConvGRU/U-Net).\n      2. Forest/land grid-day tabular Parquet partitions by research split in `data/processed/supervised/`\n         (for Logistic Regression, Random Forest, LightGBM, XGBoost, and calibration/ablation).\n    \"\"\"\n    import shutil\n    import xarray as xr\n\n    log = get_logger()\n    out_dir = config.p(\"supervised\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n    meta_path = out_dir / \"supervised_dataset_summary.json\"\n    if meta_path.exists() and not force:\n        from .utils import read_json\n        return read_json(meta_path)\n\n    zarr_stores = sorted(config.p(\"era5_daily\").glob(\"*.zarr\"))\n    if year_months is not None:\n        ym_set = set(year_months)\n        zarr_stores = [z for z in zarr_stores if z.stem in ym_set]\n    # Always exclude 1947-2000 climate baseline stores from the supervised cube when >=2000-11 stores exist\n    sup_stores = [z for z in zarr_stores if z.stem >= \"2000-11\"]\n    if sup_stores:\n        zarr_stores = sup_stores\n    if not zarr_stores:\n        raise FileNotFoundError(\"No ERA5 daily Zarr stores found in data/processed/era5_daily_zarr/.\")\n\n    ds = xr.open_mfdataset([str(z) for z in zarr_stores], engine=\"zarr\", combine=\"by_coords\").sortby(\"time\")\n    ds = ds.compute()\n\n    times = pd.DatetimeIndex(ds.time.values)\n    lats = ds.latitude.values.astype(\"float32\")\n    lons = ds.longitude.values.astype(\"float32\")\n    T, H, W = len(times), len(lats), len(lons)\n\n    # Compute strictly causal antecedent meteorological arrays\n    met_arrays = _compute_grid_Antecedent_arrays(ds)\n\n    # Align land-cover fractions onto (H, W)\n    lc_df = build_or_load_landcover_grid(grid_res=float(abs(lats[1] - lats[0])) if len(lats) > 1 else 0.25)\n    lat_map = {round(float(v), 4): i for i, v in enumerate(lats)}\n    lon_map = {round(float(v), 4): j for j, v in enumerate(lons)}\n\n    forest_2d = np.zeros((H, W), dtype=\"float32\")\n    crop_2d = np.zeros((H, W), dtype=\"float32\")\n    shrub_2d = np.zeros((H, W), dtype=\"float32\")\n    land_mask_2d = np.zeros((H, W), dtype=\"int8\")\n    eco_2d = np.full((H, W), \"ocean\", dtype=object)\n\n    for r in lc_df.itertuples(index=False):\n        i = lat_map.get(round(float(r.latitude), 4))\n        j = lon_map.get(round(float(r.longitude), 4))\n        if i is not None and j is not None:\n            forest_2d[i, j] = float(r.forest_frac)\n            crop_2d[i, j] = float(r.crop_frac)\n            shrub_2d[i, j] = float(r.shrub_grass_frac)\n            land_mask_2d[i, j] = int(r.is_land)\n            eco_2d[i, j] = str(r.ecoregion)\n\n    # Align daily attributed fire rasters onto (T, H, W)\n    fire_daily = np.zeros((T, H, W), dtype=\"float32\")\n    agri_daily = np.zeros((T, H, W), dtype=\"float32\")\n    unc_daily = np.zeros((T, H, W), dtype=\"float32\")\n    fire_frp = np.zeros((T, H, W), dtype=\"float32\")\n\n    fire_pq = config.p(\"fire_rasters\") / \"daily_fire_grid_attributed.parquet\"\n    has_firms_labels = fire_pq.exists()\n    firms_min_date = None\n    firms_max_date = None\n    firms_ym_set = set()\n    if has_firms_labels:\n        fdf = pd.read_parquet(fire_pq)\n        if len(fdf):\n            f_dates = pd.to_datetime(fdf[\"time\"]).dt.floor(\"D\")\n            firms_min_date = f_dates.min()\n            firms_max_date = f_dates.max()\n            firms_ym_set = set(f_dates.dt.strftime(\"%Y-%m\").unique())\n        t_map = {pd.Timestamp(t).floor(\"D\"): idx_t for idx_t, t in enumerate(times)}\n        for r in fdf.itertuples(index=False):\n            ti = t_map.get(pd.Timestamp(r.time).floor(\"D\"))\n            li = lat_map.get(round(float(r.latitude), 4))\n            lj = lon_map.get(round(float(r.longitude), 4))\n            if ti is not None and li is not None and lj is not None:\n                fire_daily[ti, li, lj] = float(r.forest_fire_count)\n                agri_daily[ti, li, lj] = float(r.agricultural_fire_count)\n                unc_daily[ti, li, lj] = float(r.uncertain_fire_count)\n                fire_frp[ti, li, lj] = float(r.forest_frp_sum) if pd.notna(r.forest_frp_sum) else 0.0\n\n    # Strictly antecedent fire persistence lags (t <= D)\n    lag1 = np.zeros((T, H, W), dtype=\"float32\")\n    lag3 = np.zeros((T, H, W), dtype=\"float32\")\n    lag7 = np.zeros((T, H, W), dtype=\"float32\")\n    for t in range(T):\n        lag1[t] = (fire_daily[t] > 0).astype(\"float32\")\n        lag3[t] = np.sum(fire_daily[max(0, t - 2): t + 1] > 0, axis=0).astype(\"float32\")\n        lag7[t] = np.sum(fire_daily[max(0, t - 6): t + 1] > 0, axis=0).astype(\"float32\")\n\n    met_arrays[\"forest_fire_lag1d\"] = lag1\n    met_arrays[\"forest_fire_lag3d_sum\"] = lag3\n    met_arrays[\"forest_fire_lag7d_sum\"] = lag7\n\n    # Forward 72-hour target: `target_forest_fire_next72h[D] = 1` iff >= 1 forest-associated active-fire\n    # detection occurs on `D+1, D+2, or D+3` AND all 3 future calendar days `[D+1, D+2, D+3]` are\n    # contiguous and within the observed FIRMS coverage range (never treating missing future days as 0).\n    target_72h = np.full((T, H, W), np.nan, dtype=\"float32\")\n    target_agri_72h = np.full((T, H, W), np.nan, dtype=\"float32\")\n    valid_future_window = np.zeros(T, dtype=bool)\n    for t in range(T - 3):\n        d0 = times[t].floor(\"D\")\n        d1 = times[t + 1].floor(\"D\")\n        d2 = times[t + 2].floor(\"D\")\n        d3 = times[t + 3].floor(\"D\")\n        is_contiguous = (\n            (d1 - d0 == pd.Timedelta(days=1))\n            and (d2 - d0 == pd.Timedelta(days=2))\n            and (d3 - d0 == pd.Timedelta(days=3))\n        )\n        in_firms_span = True\n        if has_firms_labels and len(firms_ym_set) > 0:\n            in_firms_span = (\n                (d1.strftime(\"%Y-%m\") in firms_ym_set)\n                and (d3.strftime(\"%Y-%m\") in firms_ym_set)\n                and (d1 >= firms_min_date)\n                and (d3 <= firms_max_date)\n            )\n        if is_contiguous and in_firms_span:\n            valid_future_window[t] = True\n            fut_forest = fire_daily[t + 1: t + 4]\n            fut_agri = agri_daily[t + 1: t + 4]\n            target_72h[t] = (np.sum(fut_forest, axis=0) > 0).astype(\"float32\")\n            target_agri_72h[t] = (np.sum(fut_agri, axis=0) > 0).astype(\"float32\")\n\n    # Save spatiotemporal Zarr cube for PyTorch Dataset/DataLoader\n    cube_vars = {k: ((\"time\", \"latitude\", \"longitude\"), v) for k, v in met_arrays.items()}\n    cube_vars[\"target_forest_fire_next72h\"] = ((\"time\", \"latitude\", \"longitude\"), target_72h)\n    cube_vars[\"target_agri_fire_next72h\"] = ((\"time\", \"latitude\", \"longitude\"), target_agri_72h)\n    cube_vars[\"forest_fire_count_day\"] = ((\"time\", \"latitude\", \"longitude\"), fire_daily)\n    cube_vars[\"forest_frac\"] = ((\"latitude\", \"longitude\"), forest_2d)\n    cube_vars[\"crop_frac\"] = ((\"latitude\", \"longitude\"), crop_2d)\n    cube_vars[\"shrub_grass_frac\"] = ((\"latitude\", \"longitude\"), shrub_2d)\n    cube_vars[\"valid_forest_mask\"] = ((\"latitude\", \"longitude\"), ((land_mask_2d > 0) & (forest_2d >= 0.15)).astype(\"int8\"))\n\n    cube_ds = xr.Dataset(cube_vars, coords={\"time\": times, \"latitude\": lats, \"longitude\": lons})\n    cube_path = out_dir / \"spatiotemporal_cube.zarr\"\n    if cube_path.exists():\n        shutil.rmtree(cube_path)\n    cube_ds.chunk({\"time\": min(64, T), \"latitude\": H, \"longitude\": W}).to_zarr(cube_path, mode=\"w\")\n\n    # Build vectorized tabular dataset over valid forest/woodland cells (`forest_frac >= 0.15` & `valid_future_window`)\n    valid_ij = np.argwhere((land_mask_2d > 0) & (forest_2d >= 0.15))\n    valid_t = np.where(valid_future_window)[0]\n    doy = times.dayofyear.to_numpy(dtype=\"int16\")\n    sin_doy = np.sin(2.0 * np.pi * doy / 365.25).astype(\"float32\")\n    cos_doy = np.cos(2.0 * np.pi * doy / 365.25).astype(\"float32\")\n\n    n_vt = len(valid_t)\n    n_vc = len(valid_ij)\n    if n_vt > 0 and n_vc > 0:\n        vi = valid_ij[:, 0]\n        vj = valid_ij[:, 1]\n        t_rep = np.repeat(valid_t, n_vc)\n        c_tile = np.tile(np.arange(n_vc), n_vt)\n        tab_dict = {\n            \"time\": times[t_rep],\n            \"year\": times[t_rep].year.astype(\"int16\"),\n            \"month\": times[t_rep].month.astype(\"int8\"),\n            \"doy\": doy[t_rep],\n            \"latitude\": lats[vi[c_tile]].astype(\"float32\"),\n            \"longitude\": lons[vj[c_tile]].astype(\"float32\"),\n            \"ecoregion\": eco_2d[vi[c_tile], vj[c_tile]],\n            \"forest_frac\": forest_2d[vi[c_tile], vj[c_tile]].astype(\"float32\"),\n            \"crop_frac\": crop_2d[vi[c_tile], vj[c_tile]].astype(\"float32\"),\n            \"shrub_grass_frac\": shrub_2d[vi[c_tile], vj[c_tile]].astype(\"float32\"),\n            \"sin_doy\": sin_doy[t_rep],\n            \"cos_doy\": cos_doy[t_rep],\n            \"target_forest_fire_next72h\": target_72h[valid_t][:, vi, vj].ravel().astype(\"int8\"),\n            \"target_agri_fire_next72h\": target_agri_72h[valid_t][:, vi, vj].ravel().astype(\"int8\"),\n        }\n        for k, arr3d in met_arrays.items():\n            tab_dict[k] = arr3d[valid_t][:, vi, vj].ravel().astype(\"float32\")\n        tab_df = pd.DataFrame(tab_dict)\n    else:\n        tab_df = pd.DataFrame()\n    # Assign research period split strictly by chronological year\n    yrs = tab_df[\"year\"] if len(tab_df) else pd.Series(dtype=\"int64\")\n    tr_s, tr_e = config.RESEARCH_PERIODS[\"train\"]\n    va_s, va_e = config.RESEARCH_PERIODS[\"validation\"]\n    te_s, te_e = config.RESEARCH_PERIODS[\"test_holdout\"]\n    cb_s, cb_e = config.RESEARCH_PERIODS[\"climate_baseline\"]\n\n    if len(tab_df):\n        tab_df[\"research_split\"] = np.select(\n            [\n                yrs.between(cb_s, cb_e),\n                yrs.between(tr_s, tr_e),\n                yrs.between(va_s, va_e),\n                yrs.between(te_s, te_e),\n            ],\n            [\"climate_baseline_unsupervised\", \"train\", \"validation\", \"test_holdout\"],\n            default=\"outside_defined_periods\",\n        )\n\n    write_parquet_atomic(tab_df, out_dir / \"supervised_forest_fire_72h.parquet\")\n\n    # Present feature columns (those not 100% NaN in the archive)\n    present_features = [\n        c for c in FEATURE_COLUMNS\n        if c in tab_df.columns and tab_df[c].notna().any()\n    ]\n    missing_features = [c for c in FEATURE_COLUMNS if c not in present_features]\n\n    split_counts = tab_df[\"research_split\"].value_counts().to_dict() if len(tab_df) else {}\n    pos_rate_by_split = (\n        tab_df.groupby(\"research_split\")[\"target_forest_fire_next72h\"].mean().round(5).to_dict()\n        if len(tab_df) else {}\n    )\n\n    summary = {\n        \"status\": \"done\",\n        \"generated_utc\": now_utc(),\n        \"n_days_total\": int(T),\n        \"n_valid_forest_cells\": int(len(valid_ij)),\n        \"n_tabular_rows\": int(len(tab_df)),\n        \"has_firms_labels\": bool(has_firms_labels),\n        \"present_features\": present_features,\n        \"missing_features_not_zero_filled\": missing_features,\n        \"split_row_counts\": split_counts,\n        \"positive_rate_by_split\": pos_rate_by_split,\n        \"spatiotemporal_cube_zarr\": str(cube_path),\n        \"tabular_parquet\": str(out_dir / \"supervised_forest_fire_72h.parquet\"),\n    }\n    atomic_write_json(meta_path, summary)\n    log.info(f\"[Supervised 72h] Built cube ({T}x{H}x{W}) and {len(tab_df):,} forest grid-day rows.\")\n    return summary\n\n\ndef engineer_meteorological_features(force: bool = False) -> dict:\n    \"\"\"Wrapper executing historical climate baseline + leak-free 72h supervised feature & target generation.\"\"\"\n    from .era5 import compute_historical_climate_baseline\n\n    base_info = compute_historical_climate_baseline()\n    sup_info = build_supervised_72h_dataset(force=force)\n    return {\"climate_baseline\": base_info, \"supervised_dataset\": sup_info}\n","models_ml.py":"\"\"\"Classical machine learning and physical/climatological baselines for 72-hour forest-fire forecasting.\n\nScientific & methodological guarantees:\n1. Primary target: `target_forest_fire_next72h` (probability of >= 1 forest-associated active fire\n   in the next 72 hours `[D+1, D+2, D+3]` in IST).\n2. Fixed chronological splits:\n   - `train`: 2001-2010\n   - `validation`: 2011-2015 (used for threshold optimization, isotonic/Platt probability calibration,\n     and model selection)\n   - `test_holdout`: 2016-2025 (strictly independent holdout; never used for tuning or calibration)\n   - `1947-2000` is NEVER used as supervised fire training data.\n   - If the user runs on a partial sample slice where one of the chronological windows is empty,\n     the function explicitly flags `\"split_mode\": \"sample_chronological_fallback_flagged\"` in the\n     report rather than silently pretending the full 2001-2025 archive was present.\n3. Implements:\n   - `climatology_seasonal`: empirical grid-cell + month historical forest-fire rate (Laplace smoothed).\n   - `persistence_7d`: antecedent 7-day forest-fire activity + climatology blend.\n   - `logistic_regression`: L2-regularized Logistic Regression with `StandardScaler` & `class_weight=\"balanced\"`.\n   - `random_forest`: Cost-sensitive Random Forest (`class_weight=\"balanced_subsample\"`).\n   - `lightgbm`: Gradient boosted decision trees with positive-class weighting.\n   - `xgboost`: Histogram-based XGBoost with `scale_pos_weight`.\n4. Fits post-hoc **Isotonic Regression** and **Sigmoid (Platt) Calibration** on the `validation` split\n   only (`2011-2015`) and evaluates calibrated probabilities on `test_holdout` (`2016-2025`).\n\"\"\"\nfrom __future__ import annotations\n\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .features import FEATURE_COLUMNS\nfrom .utils import atomic_write_json, get_logger, now_utc, read_json, write_csv_atomic, write_parquet_atomic\n\n\ndef compute_binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:\n    \"\"\"Compute comprehensive binary probabilistic and classification metrics.\"\"\"\n    from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,\n                                 f1_score, log_loss, precision_score, recall_score, roc_auc_score)\n\n    y_true = np.asarray(y_true, dtype=\"int32\")\n    y_prob = np.clip(np.asarray(y_prob, dtype=\"float64\"), 1e-6, 1.0 - 1e-6)\n    y_pred = (y_prob >= threshold).astype(\"int32\")\n\n    has_both = len(np.unique(y_true)) > 1\n    roc = float(roc_auc_score(y_true, y_prob)) if has_both else None\n    pr = float(average_precision_score(y_true, y_prob)) if has_both else None\n    brier = float(brier_score_loss(y_true, y_prob))\n    ll = float(log_loss(y_true, y_prob, labels=[0, 1]))\n    prec = float(precision_score(y_true, y_pred, zero_division=0))\n    rec = float(recall_score(y_true, y_pred, zero_division=0))\n    f1 = float(f1_score(y_true, y_pred, zero_division=0))\n    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()\n\n    # Expected Calibration Error (ECE, 10 equal-width bins)\n    bins = np.linspace(0.0, 1.0, 11)\n    bin_ids = np.digitize(y_prob, bins) - 1\n    ece = 0.0\n    for b in range(10):\n        m = bin_ids == b\n        if m.any():\n            ece += (m.sum() / len(y_true)) * abs(float(y_true[m].mean()) - float(y_prob[m].mean()))\n\n    return {\n        \"n_samples\": int(len(y_true)),\n        \"positive_rate\": round(float(y_true.mean()), 5) if len(y_true) else 0.0,\n        \"threshold\": round(float(threshold), 4),\n        \"pr_auc\": round(pr, 5) if pr is not None else None,\n        \"roc_auc\": round(roc, 5) if roc is not None else None,\n        \"f1\": round(f1, 5),\n        \"precision\": round(prec, 5),\n        \"recall\": round(rec, 5),\n        \"brier_score\": round(brier, 5),\n        \"log_loss\": round(ll, 5),\n        \"ece\": round(float(ece), 5),\n        \"tp\": int(tp), \"fp\": int(fp), \"tn\": int(tn), \"fn\": int(fn),\n    }\n\n\ndef find_optimal_f1_threshold(y_val: np.ndarray, prob_val: np.ndarray) -> float:\n    \"\"\"Select the decision threshold maximizing F1 on the validation split ONLY.\"\"\"\n    from sklearn.metrics import precision_recall_curve\n\n    if len(np.unique(y_val)) < 2:\n        return 0.5\n    prec, rec, thresh = precision_recall_curve(y_val, prob_val)\n    if len(thresh) == 0:\n        return 0.5\n    f1_scores = (2 * prec[:-1] * rec[:-1]) / np.maximum(prec[:-1] + rec[:-1], 1e-8)\n    best_idx = int(np.argmax(f1_scores))\n    return float(np.clip(thresh[best_idx], 0.05, 0.95))\n\n\ndef compute_pr_roc_curves(y_true: np.ndarray, y_prob: np.ndarray, max_points: int = 50) -> dict:\n    \"\"\"Compute decimated Precision-Recall and ROC curve coordinates for interactive plotting.\"\"\"\n    from sklearn.metrics import precision_recall_curve, roc_curve\n\n    y_true = np.asarray(y_true, dtype=\"int32\")\n    y_prob = np.asarray(y_prob, dtype=\"float64\")\n    if len(np.unique(y_true)) < 2:\n        return {\"pr_curve\": [], \"roc_curve\": []}\n\n    prec, rec, _ = precision_recall_curve(y_true, y_prob)\n    fpr, tpr, _ = roc_curve(y_true, y_prob)\n\n    idx_pr = np.unique(np.linspace(0, len(prec) - 1, min(max_points, len(prec)), dtype=int))\n    idx_roc = np.unique(np.linspace(0, len(fpr) - 1, min(max_points, len(fpr)), dtype=int))\n\n    return {\n        \"pr_curve\": [\n            {\"recall\": round(float(rec[i]), 4), \"precision\": round(float(prec[i]), 4)}\n            for i in idx_pr\n        ],\n        \"roc_curve\": [\n            {\"fpr\": round(float(fpr[i]), 4), \"tpr\": round(float(tpr[i]), 4)}\n            for i in idx_roc\n        ],\n    }\n\n\nclass ClimatologyBaseline:\n    \"\"\"Empirical seasonal + grid-cell forest-fire probability baseline fit strictly on `train`.\"\"\"\n\n    def __init__(self, prior_strength: float = 20.0):\n        self.prior_strength = prior_strength\n        self.global_rate_ = 0.01\n        self.cell_month_rate_ = {}\n        self.month_rate_ = {}\n\n    def fit(self, df_train: pd.DataFrame, target_col: str = \"target_forest_fire_next72h\"):\n        y = df_train[target_col].to_numpy(dtype=\"float64\")\n        self.global_rate_ = float(y.mean()) if len(y) else 0.01\n        m_grp = df_train.groupby(\"month\")[target_col].agg([\"sum\", \"count\"])\n        for m, r in m_grp.iterrows():\n            self.month_rate_[int(m)] = float(\n                (r[\"sum\"] + self.prior_strength * self.global_rate_) / (r[\"count\"] + self.prior_strength)\n            )\n        cm_grp = df_train.groupby([\"latitude\", \"longitude\", \"month\"])[target_col].agg([\"sum\", \"count\"])\n        for (lat, lon, m), r in cm_grp.iterrows():\n            m_prior = self.month_rate_.get(int(m), self.global_rate_)\n            self.cell_month_rate_[(round(float(lat), 4), round(float(lon), 4), int(m))] = float(\n                (r[\"sum\"] + self.prior_strength * m_prior) / (r[\"count\"] + self.prior_strength)\n            )\n        return self\n\n    def predict_proba(self, df: pd.DataFrame) -> np.ndarray:\n        if len(df) == 0:\n            return np.zeros((0, 2), dtype=\"float64\")\n        lat_r = df[\"latitude\"].round(4).astype(\"float64\")\n        lon_r = df[\"longitude\"].round(4).astype(\"float64\")\n        m_int = df[\"month\"].astype(\"int64\")\n        if self.cell_month_rate_:\n            cm_keys = pd.MultiIndex.from_tuples(list(self.cell_month_rate_.keys()), names=[\"lat\", \"lon\", \"m\"])\n            cm_s = pd.Series(list(self.cell_month_rate_.values()), index=cm_keys, dtype=\"float64\")\n            q_idx = pd.MultiIndex.from_arrays([lat_r, lon_r, m_int], names=[\"lat\", \"lon\", \"m\"])\n            p1 = cm_s.reindex(q_idx).to_numpy(dtype=\"float64\")\n        else:\n            p1 = np.full(len(df), np.nan, dtype=\"float64\")\n        missing = np.isnan(p1)\n        if missing.any():\n            m_fallback = m_int.map(lambda x: self.month_rate_.get(int(x), self.global_rate_)).to_numpy(dtype=\"float64\")\n            p1[missing] = m_fallback[missing]\n        return np.column_stack([1.0 - p1, p1])\n\n\nclass PersistenceClimatologyBaseline:\n    \"\"\"Combines trailing 7-day forest-fire persistence (`forest_fire_lag7d_sum`) with seasonal climatology.\"\"\"\n\n    def __init__(self):\n        self.clim = ClimatologyBaseline()\n        self.active_boost_ = 0.35\n\n    def fit(self, df_train: pd.DataFrame, target_col: str = \"target_forest_fire_next72h\"):\n        self.clim.fit(df_train, target_col=target_col)\n        if \"forest_fire_lag7d_sum\" in df_train.columns:\n            active = df_train[\"forest_fire_lag7d_sum\"] > 0\n            if active.any():\n                self.active_boost_ = float(df_train.loc[active, target_col].mean())\n        return self\n\n    def predict_proba(self, df: pd.DataFrame) -> np.ndarray:\n        p_clim = self.clim.predict_proba(df)[:, 1]\n        lag7 = df.get(\"forest_fire_lag7d_sum\", pd.Series(0.0, index=df.index)).fillna(0.0).to_numpy()\n        p1 = np.where(lag7 > 0, np.clip(0.5 * p_clim + 0.5 * self.active_boost_, 0.0, 0.99), p_clim)\n        return np.column_stack([1.0 - p1, p1])\n\n\ndef prepare_chronological_splits(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:\n    \"\"\"Partition `df` strictly according to the fixed chronological periods:\n      - Train: 2001-2010\n      - Validation: 2011-2015\n      - Holdout Test: 2016-2025\n    Excludes 1947-2000 (`climate_baseline_unsupervised`).\n    If any window is absent (e.g., when running a small integration test on a single year),\n    reports the missing period honestly and creates a chronological non-overlapping 60/20/20 split\n    for code verification while marking `strict_2001_2025_coverage = False`.\n    \"\"\"\n    sup = df[df[\"research_split\"] != \"climate_baseline_unsupervised\"].sort_values(\"time\").reset_index(drop=True)\n    tr = sup[sup[\"research_split\"] == \"train\"].copy()\n    va = sup[sup[\"research_split\"] == \"validation\"].copy()\n    te = sup[sup[\"research_split\"] == \"test_holdout\"].copy()\n\n    split_meta = {\n        \"intended_splits\": {\n            \"climate_baseline_excluded_from_supervised\": \"1947-2000\",\n            \"train\": \"2001-2010\",\n            \"validation\": \"2011-2015\",\n            \"test_holdout\": \"2016-2025\",\n        },\n        \"strict_2001_2025_coverage\": bool(len(tr) > 0 and len(va) > 0 and len(te) > 0),\n        \"years_present_in_supervised\": sorted(int(y) for y in sup[\"year\"].unique()) if len(sup) else [],\n    }\n\n    if not split_meta[\"strict_2001_2025_coverage\"] and len(sup) >= 10:\n        # Chronological split by unique timestamps (never random shuffling, zero temporal overlap)\n        uniq_t = np.sort(sup[\"time\"].unique())\n        n_t = len(uniq_t)\n        t1 = uniq_t[max(0, int(0.60 * n_t) - 1)]\n        t2 = uniq_t[max(0, int(0.80 * n_t) - 1)]\n        # Enforce a 3-day embargo buffer between train/val and val/test to prevent 72h target overlap\n        tr = sup[sup[\"time\"] <= t1].copy()\n        va = sup[(sup[\"time\"] > t1 + pd.Timedelta(days=3)) & (sup[\"time\"] <= t2)].copy()\n        te = sup[sup[\"time\"] > t2 + pd.Timedelta(days=3)].copy()\n        if len(va) == 0 or len(te) == 0:\n            tr = sup[sup[\"time\"] <= t1].copy()\n            va = sup[(sup[\"time\"] > t1) & (sup[\"time\"] <= t2)].copy()\n            te = sup[sup[\"time\"] > t2].copy()\n        split_meta[\"split_mode\"] = \"chronological_sample_slice_with_embargo (full 2001-2025 archive not yet decoded)\"\n    else:\n        # Enforce 3-day embargo at boundary Dec 29-31 2010 and Dec 29-31 2015 so 72h target never crosses split\n        tr = tr[tr[\"time\"] <= pd.Timestamp(\"2010-12-28\")].copy()\n        va = va[va[\"time\"] <= pd.Timestamp(\"2015-12-28\")].copy()\n        split_meta[\"split_mode\"] = \"fixed_chronological_2001_2010_train_2011_2015_val_2016_2025_test\"\n\n    split_meta[\"row_counts\"] = {\"train\": int(len(tr)), \"validation\": int(len(va)), \"test_holdout\": int(len(te))}\n    return tr, va, te, split_meta\n\n\ndef train_classical_ml(force: bool = False) -> dict:\n    \"\"\"Train Climatology, Persistence, Logistic Regression, Random Forest, LightGBM, and XGBoost\n    on `train`, calibrate probabilities and select thresholds on `validation`, and evaluate once on\n    the independent `test_holdout`.\n    \"\"\"\n    import joblib\n    from sklearn.ensemble import RandomForestClassifier\n    from sklearn.impute import SimpleImputer\n    from sklearn.isotonic import IsotonicRegression\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.pipeline import Pipeline\n    from sklearn.preprocessing import StandardScaler\n\n    log = get_logger()\n    metrics_path = config.p(\"models\") / \"classical_ml_metrics.json\"\n    if metrics_path.exists() and not force:\n        return read_json(metrics_path)\n\n    sup_pq = config.p(\"supervised\") / \"supervised_forest_fire_72h.parquet\"\n    if not sup_pq.exists():\n        raise FileNotFoundError(\"Supervised dataset not found. Run `build_supervised_72h_dataset()` first.\")\n\n    df = pd.read_parquet(sup_pq)\n    target_col = \"target_forest_fire_next72h\"\n    all_avail_cols = [c for c in FEATURE_COLUMNS if c in df.columns and df[c].notna().any()]\n\n    tr, va, te, split_meta = prepare_chronological_splits(df)\n    if len(tr) == 0 or len(va) == 0 or len(te) == 0:\n        raise RuntimeError(f\"Insufficient chronological rows for train/val/test: {split_meta}\")\n\n    # Check per-feature completeness in `tr` (e.g. 2001-2010 has a 2001-2008 gap for 10m wind u10/v10)\n    tr_completeness = {c: round(float(tr[c].notna().mean()), 4) for c in all_avail_cols}\n    complete_train_cols = [c for c, frac in tr_completeness.items() if frac >= 0.80]\n    partial_train_cols = [c for c, frac in tr_completeness.items() if frac < 0.80]\n    feature_cols = complete_train_cols if complete_train_cols else all_avail_cols\n\n    split_meta[\"train_feature_completeness\"] = tr_completeness\n    split_meta[\"complete_features_used_in_primary_2001_2010_track\"] = feature_cols\n    split_meta[\"partial_coverage_features_excluded_from_imputation\"] = partial_train_cols\n\n    X_tr, y_tr = tr[feature_cols].to_numpy(dtype=\"float32\"), tr[target_col].to_numpy(dtype=\"int32\")\n    X_va, y_va = va[feature_cols].to_numpy(dtype=\"float32\"), va[target_col].to_numpy(dtype=\"int32\")\n    X_te, y_te = te[feature_cols].to_numpy(dtype=\"float32\"), te[target_col].to_numpy(dtype=\"int32\")\n\n    pos_count = max(1, int((y_tr == 1).sum()))\n    neg_count = max(1, int((y_tr == 0).sum()))\n    scale_pos = float(np.clip(neg_count / pos_count, 1.0, 200.0))\n\n    ckpt_dir = config.p(\"checkpoints\") / \"classical\"\n    ckpt_dir.mkdir(parents=True, exist_ok=True)\n    pred_out_dir = config.p(\"evaluation\") / \"test_predictions\"\n    pred_out_dir.mkdir(parents=True, exist_ok=True)\n\n    results = {\n        \"generated_utc\": now_utc(),\n        \"target\": target_col,\n        \"feature_columns\": feature_cols,\n        \"split_metadata\": split_meta,\n        \"models\": {},\n    }\n\n    # 1. Climatology & Persistence Baselines\n    for b_name, b_obj in [\n        (\"climatology_baseline\", ClimatologyBaseline()),\n        (\"persistence_7d_baseline\", PersistenceClimatologyBaseline()),\n    ]:\n        t0 = time.time()\n        b_obj.fit(tr, target_col=target_col)\n        joblib.dump(b_obj, ckpt_dir / f\"{b_name}.joblib\")\n        p_va = b_obj.predict_proba(va)[:, 1]\n        p_te = b_obj.predict_proba(te)[:, 1]\n        th = find_optimal_f1_threshold(y_va, p_va)\n        results[\"models\"][b_name] = {\n            \"family\": \"physical_baseline\",\n            \"optimal_val_threshold\": round(th, 4),\n            \"validation\": compute_binary_metrics(y_va, p_va, threshold=th),\n            \"test_holdout\": compute_binary_metrics(y_te, p_te, threshold=th),\n            \"seconds\": round(time.time() - t0, 2),\n        }\n\n    # 2. Supervised ML Models (with median imputation fit ONLY on X_tr, never zero-filling)\n    ml_specs = {\n        \"logistic_regression\": Pipeline([\n            (\"imputer\", SimpleImputer(strategy=\"median\")),\n            (\"scaler\", StandardScaler()),\n            (\"clf\", LogisticRegression(C=0.5, max_iter=600, class_weight=\"balanced\", random_state=42)),\n        ]),\n        \"random_forest\": Pipeline([\n            (\"imputer\", SimpleImputer(strategy=\"median\")),\n            (\"clf\", RandomForestClassifier(\n                n_estimators=250, max_depth=12, min_samples_leaf=10,\n                class_weight=\"balanced_subsample\", n_jobs=-1, random_state=42,\n            )),\n        ]),\n    }\n\n    try:\n        import lightgbm as lgb\n        ml_specs[\"lightgbm\"] = Pipeline([\n            (\"imputer\", SimpleImputer(strategy=\"median\")),\n            (\"clf\", lgb.LGBMClassifier(\n                n_estimators=350, learning_rate=0.04, num_leaves=31, subsample_for_bin=50000,\n                scale_pos_weight=scale_pos, random_state=42, n_jobs=-1, verbose=-1,\n            )),\n        ])\n    except Exception:\n        pass\n\n    try:\n        import xgboost as xgb\n        ml_specs[\"xgboost\"] = Pipeline([\n            (\"imputer\", SimpleImputer(strategy=\"median\")),\n            (\"clf\", xgb.XGBClassifier(\n                n_estimators=350, learning_rate=0.04, max_depth=6, subsample=0.8, colsample_bytree=0.8,\n                scale_pos_weight=scale_pos, tree_method=\"hist\", eval_metric=\"logloss\", random_state=42,\n            )),\n        ])\n    except Exception:\n        pass\n\n    feature_importances = {}\n    test_pred_df = te[[\"time\", \"year\", \"month\", \"latitude\", \"longitude\", \"ecoregion\", target_col]].copy()\n\n    for name, pipe in ml_specs.items():\n        t0 = time.time()\n        # Handle degeneracies when y_tr in a tiny unit test has only 1 class\n        if len(np.unique(y_tr)) < 2:\n            log.warning(f\"[Classical ML] Training split has a single target class; skipping {name} fit.\")\n            continue\n\n        pipe.fit(X_tr, y_tr)\n        raw_va = pipe.predict_proba(X_va)[:, 1]\n        raw_te = pipe.predict_proba(X_te)[:, 1]\n\n        # Post-hoc Isotonic probability calibration fit strictly on Validation (2011-2015)\n        if len(np.unique(y_va)) > 1:\n            iso = IsotonicRegression(y_min=1e-5, y_max=1.0 - 1e-5, out_of_bounds=\"clip\")\n            iso.fit(raw_va, y_va)\n            cal_va = iso.predict(raw_va)\n            cal_te = iso.predict(raw_te)\n        else:\n            iso = None\n            cal_va, cal_te = raw_va, raw_te\n\n        th = find_optimal_f1_threshold(y_va, cal_va)\n        joblib.dump({\"pipeline\": pipe, \"calibrator\": iso, \"threshold\": th, \"features\": feature_cols},\n                    ckpt_dir / f\"{name}.joblib\")\n\n        test_pred_df[f\"prob_{name}_raw\"] = raw_te.astype(\"float32\")\n        test_pred_df[f\"prob_{name}_cal\"] = cal_te.astype(\"float32\")\n\n        results[\"models\"][name] = {\n            \"family\": \"classical_ml\",\n            \"optimal_val_threshold\": round(th, 4),\n            \"validation_raw\": compute_binary_metrics(y_va, raw_va, threshold=0.5),\n            \"validation\": compute_binary_metrics(y_va, cal_va, threshold=th),\n            \"test_holdout_raw\": compute_binary_metrics(y_te, raw_te, threshold=0.5),\n            \"test_holdout\": compute_binary_metrics(y_te, cal_te, threshold=th),\n            \"curves_test\": compute_pr_roc_curves(y_te, cal_te),\n            \"seconds\": round(time.time() - t0, 2),\n        }\n\n        clf = pipe.named_steps[\"clf\"]\n        if hasattr(clf, \"feature_importances_\"):\n            imp = np.asarray(clf.feature_importances_, dtype=\"float64\")\n            imp = imp / max(imp.sum(), 1e-12)\n            feature_importances[name] = dict(zip(feature_cols, np.round(imp, 5).tolist()))\n        elif hasattr(clf, \"coef_\"):\n            imp = np.abs(np.asarray(clf.coef_[0], dtype=\"float64\"))\n            imp = imp / max(imp.sum(), 1e-12)\n            feature_importances[name] = dict(zip(feature_cols, np.round(imp, 5).tolist()))\n\n        log.info(\n            f\"[Classical ML] {name}: Val PR-AUC={results['models'][name]['validation']['pr_auc']} | \"\n            f\"Test PR-AUC={results['models'][name]['test_holdout']['pr_auc']}\"\n        )\n\n    # Select best classical model strictly by Validation PR-AUC\n    valid_candidates = [\n        (k, v[\"validation\"][\"pr_auc\"])\n        for k, v in results[\"models\"].items()\n        if v.get(\"validation\", {}).get(\"pr_auc\") is not None\n    ]\n    best_model_name = max(valid_candidates, key=lambda x: x[1])[0] if valid_candidates else None\n    results[\"selected_best_model_by_validation_pr_auc\"] = best_model_name\n    results[\"feature_importances\"] = feature_importances\n\n    # Reduced-Period Complete-6-Variable Experiment (2009-2012 Train, 2013-2015 Val, 2016-2025 Holdout Test)\n    # Triggered transparently whenever some variables (e.g. 10m wind u10/v10 in 2001-2008) are missing in early train years\n    reduced_exp = {\"status\": \"not_required_all_features_complete_in_2001_2010\"}\n    if partial_train_cols:\n        sub_2009 = df[df[\"year\"] >= 2009].sort_values(\"time\").reset_index(drop=True)\n        r_tr = sub_2009[(sub_2009[\"year\"].between(2009, 2012)) & (sub_2009[\"time\"] <= pd.Timestamp(\"2012-12-28\"))]\n        r_va = sub_2009[(sub_2009[\"year\"].between(2013, 2015)) & (sub_2009[\"time\"] <= pd.Timestamp(\"2015-12-28\"))]\n        r_te = sub_2009[sub_2009[\"year\"].between(2016, 2025)]\n        if len(r_tr) > 0 and len(r_va) > 0 and len(r_te) > 0 and len(np.unique(r_tr[target_col])) > 1:\n            r_cols = [c for c in all_avail_cols if r_tr[c].notna().mean() >= 0.80]\n            r_pipe = Pipeline([\n                (\"imputer\", SimpleImputer(strategy=\"median\")),\n                (\"scaler\", StandardScaler()),\n                (\"clf\", LogisticRegression(C=0.5, max_iter=500, class_weight=\"balanced\", random_state=42)),\n            ])\n            try:\n                import lightgbm as lgb\n                r_pipe = Pipeline([\n                    (\"imputer\", SimpleImputer(strategy=\"median\")),\n                    (\"clf\", lgb.LGBMClassifier(\n                        n_estimators=250, learning_rate=0.04, num_leaves=31,\n                        scale_pos_weight=scale_pos, random_state=42, n_jobs=-1, verbose=-1,\n                    )),\n                ])\n            except Exception:\n                pass\n            r_pipe.fit(r_tr[r_cols].to_numpy(dtype=\"float32\"), r_tr[target_col].to_numpy(dtype=\"int32\"))\n            rp_va = r_pipe.predict_proba(r_va[r_cols].to_numpy(dtype=\"float32\"))[:, 1]\n            rp_te = r_pipe.predict_proba(r_te[r_cols].to_numpy(dtype=\"float32\"))[:, 1]\n            r_th = find_optimal_f1_threshold(r_va[target_col].to_numpy(dtype=\"int32\"), rp_va)\n            reduced_exp = {\n                \"status\": \"executed_reduced_period_complete_6var_experiment\",\n                \"reason\": f\"Features {partial_train_cols} had <80% coverage in 2001-2010 due to 1997-2008 archive gap.\",\n                \"splits\": {\n                    \"train\": \"2009-2012 (embargo <= 2012-12-28)\",\n                    \"validation\": \"2013-2015 (embargo <= 2015-12-28)\",\n                    \"test_holdout\": \"2016-2025\",\n                },\n                \"row_counts\": {\"train\": int(len(r_tr)), \"validation\": int(len(r_va)), \"test_holdout\": int(len(r_te))},\n                \"features_used\": r_cols,\n                \"validation\": compute_binary_metrics(r_va[target_col].to_numpy(dtype=\"int32\"), rp_va, threshold=r_th),\n                \"test_holdout\": compute_binary_metrics(r_te[target_col].to_numpy(dtype=\"int32\"), rp_te, threshold=r_th),\n            }\n    results[\"reduced_period_complete_6var_experiment\"] = reduced_exp\n\n    write_parquet_atomic(test_pred_df, pred_out_dir / \"classical_test_predictions.parquet\")\n    if feature_importances:\n        imp_df = pd.DataFrame(feature_importances).reset_index().rename(columns={\"index\": \"feature\"})\n        write_csv_atomic(imp_df, config.p(\"tables\") / \"classical_feature_importance.csv\")\n\n    atomic_write_json(metrics_path, results)\n    return results\n","models_dl.py":"\"\"\"Genuine Spatiotemporal Deep Learning for 72-hour forest-fire probability forecasting.\n\nImplements:\n1. `SpatioTemporalFireDataset(torch.utils.data.Dataset)`:\n   - Loads chronological sequences of length `seq_len` (`t - seq_len + 1 .. t`) from\n     `data/processed/supervised/spatiotemporal_cube.zarr`.\n   - Standardizes input channels using mean/std statistics computed STRICTLY over the `train` split.\n   - Returns `(X_seq[T, C, H, W], y_next72h[H, W], valid_forest_mask[H, W])`.\n2. Three distinct spatiotemporal neural architectures for validation-based comparison:\n   - `ConvLSTMFireNet`: Recurrent 2D ConvLSTM cell (forget/input/output/candidate gates) + spatial head.\n   - `ConvGRUFireNet`: Recurrent 2D ConvGRU cell + residual spatial projection head.\n   - `SpatioTemporalUNet`: Multi-scale encoder-decoder U-Net with skip connections over temporal-channel stacks.\n3. `MaskedFocalBCEWithLogitsLoss`:\n   - Evaluates class-imbalance-weighted focal binary cross-entropy strictly over valid forest/land\n     grid pixels (`valid_forest_mask == 1`), ignoring ocean and non-forest pixels.\n4. Full training, validation, checkpointing, and final holdout test inference loop:\n   - Actual `loss.backward()`, gradient clipping, `optimizer.step()`, `scheduler.step()`,\n     and verified count of `optimizer_steps_completed > 0`.\n   - Automatic resumption from `last_checkpoint.pt` if a Colab session disconnects mid-training.\n   - Selects the best architecture strictly by **Validation PR-AUC** (`2011-2015`) before evaluating on\n     the independent **2016-2025 Holdout Test** split.\n\"\"\"\nfrom __future__ import annotations\n\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .models_ml import compute_binary_metrics, compute_pr_roc_curves, find_optimal_f1_threshold\nfrom .utils import atomic_write_json, get_logger, now_utc, read_json, write_parquet_atomic\n\nDL_DYNAMIC_CHANNELS = [\n    \"t2m_mean_c\", \"t2m_max_c\", \"d2m_mean_c\",\n    \"rh_min_pct\", \"vpd_max_kpa\",\n    \"wind10_mean_ms\", \"wind10_max_ms\",\n    \"tp_sum_mm\", \"ssrd_sum_mj\", \"ffwi_max\",\n    \"tp_7d_mm\", \"tp_30d_mm\", \"dry_spell_days\", \"kbdi_proxy\",\n    \"forest_fire_lag1d\", \"forest_fire_lag7d_sum\",\n]\n\n\ndef _load_cube_arrays(seq_len: int = 7) -> dict:\n    \"\"\"Load the spatiotemporal cube and partition issue-time indices into chronological\n    Train (2001-2010), Validation (2011-2015), and Holdout Test (2016-2025) sets.\n    \"\"\"\n    import xarray as xr\n\n    cube_path = config.p(\"supervised\") / \"spatiotemporal_cube.zarr\"\n    if not cube_path.exists():\n        raise FileNotFoundError(\"Spatiotemporal cube not found. Run `build_supervised_72h_dataset()` first.\")\n\n    ds = xr.open_zarr(cube_path)\n    times = pd.DatetimeIndex(ds.time.values)\n    lats = ds.latitude.values.astype(\"float32\")\n    lons = ds.longitude.values.astype(\"float32\")\n    T, H, W = len(times), len(lats), len(lons)\n\n    # Identify dynamic channels that contain valid finite observations\n    avail_dyn = []\n    dyn_list = []\n    for ch in DL_DYNAMIC_CHANNELS:\n        if ch in ds.data_vars:\n            arr = ds[ch].values.astype(\"float32\")\n            if np.isfinite(arr).any():\n                avail_dyn.append(ch)\n                dyn_list.append(arr)\n\n    if not dyn_list:\n        raise RuntimeError(\"No finite dynamic meteorological channels found in spatiotemporal_cube.zarr.\")\n\n    # Static + seasonal channels broadcast to (T, H, W)\n    ff = np.broadcast_to(ds[\"forest_frac\"].values.astype(\"float32\")[None, :, :], (T, H, W))\n    cf = np.broadcast_to(ds[\"crop_frac\"].values.astype(\"float32\")[None, :, :], (T, H, W))\n    doy = times.dayofyear.to_numpy(dtype=\"float32\")\n    sin_d = np.broadcast_to(np.sin(2 * np.pi * doy / 365.25)[:, None, None].astype(\"float32\"), (T, H, W))\n    cos_d = np.broadcast_to(np.cos(2 * np.pi * doy / 365.25)[:, None, None].astype(\"float32\"), (T, H, W))\n\n    channel_names = avail_dyn + [\"forest_frac\", \"crop_frac\", \"sin_doy\", \"cos_doy\"]\n    X_all = np.stack(dyn_list + [ff, cf, sin_d, cos_d], axis=1)  # (T, C, H, W)\n    y_all = ds[\"target_forest_fire_next72h\"].values.astype(\"float32\")  # (T, H, W)\n    mask_2d = ds[\"valid_forest_mask\"].values.astype(\"float32\")  # (H, W)\n\n    # Valid issue times: `t >= seq_len - 1` and `t <= T - 4` (so D+1..D+3 are observed)\n    valid_t = np.arange(seq_len - 1, max(seq_len - 1, T - 3))\n    years = times[valid_t].year.to_numpy()\n\n    tr_s, tr_e = config.RESEARCH_PERIODS[\"train\"]\n    va_s, va_e = config.RESEARCH_PERIODS[\"validation\"]\n    te_s, te_e = config.RESEARCH_PERIODS[\"test_holdout\"]\n\n    idx_tr = valid_t[(years >= tr_s) & (years <= tr_e) & (times[valid_t] <= pd.Timestamp(\"2010-12-28\"))]\n    idx_va = valid_t[(years >= va_s) & (years <= va_e) & (times[valid_t] <= pd.Timestamp(\"2015-12-28\"))]\n    idx_te = valid_t[(years >= te_s) & (years <= te_e)]\n\n    strict_periods = bool(len(idx_tr) > 0 and len(idx_va) > 0 and len(idx_te) > 0)\n    if not strict_periods and len(valid_t) >= 3:\n        n = len(valid_t)\n        i1 = max(1, int(0.60 * n))\n        i2 = max(i1 + 1, int(0.80 * n))\n        idx_tr = valid_t[:i1]\n        idx_va = valid_t[i1:i2]\n        idx_te = valid_t[i2:]\n\n    # Fit channel-wise normalization strictly on `idx_tr` (training period only), in-place to save RAM\n    tr_slice_idx = np.unique(np.clip(np.concatenate([np.arange(i - seq_len + 1, i + 1) for i in idx_tr]), 0, T - 1))\n    ch_mean = np.zeros((1, X_all.shape[1], 1, 1), dtype=\"float32\")\n    ch_std = np.ones((1, X_all.shape[1], 1, 1), dtype=\"float32\")\n    for c_i in range(X_all.shape[1]):\n        c_sub = X_all[tr_slice_idx, c_i]\n        m_val = float(np.nanmean(c_sub)) if np.isfinite(c_sub).any() else 0.0\n        s_val = float(np.nanstd(c_sub)) if np.isfinite(c_sub).any() else 1.0\n        ch_mean[0, c_i, 0, 0] = m_val if np.isfinite(m_val) else 0.0\n        ch_std[0, c_i, 0, 0] = s_val if (np.isfinite(s_val) and s_val >= 1e-5) else 1.0\n        X_all[:, c_i] -= ch_mean[0, c_i]\n        X_all[:, c_i] /= ch_std[0, c_i]\n    np.nan_to_num(X_all, copy=False, nan=0.0)\n    X_norm = X_all\n\n    return {\n        \"X_norm\": X_norm,\n        \"y_all\": y_all,\n        \"mask_2d\": mask_2d,\n        \"times\": times,\n        \"lats\": lats,\n        \"lons\": lons,\n        \"channel_names\": channel_names,\n        \"ch_mean\": ch_mean.squeeze().tolist(),\n        \"ch_std\": ch_std.squeeze().tolist(),\n        \"idx_tr\": idx_tr,\n        \"idx_va\": idx_va,\n        \"idx_te\": idx_te,\n        \"strict_2001_2025_coverage\": strict_periods,\n        \"seq_len\": seq_len,\n    }\n\n\ndef build_pytorch_components(in_channels: int, hidden_dim: int = 48, seq_len: int = 7):\n    \"\"\"Instantiate Dataset class, MaskedFocalBCEWithLogitsLoss, and the 3 candidate architectures:\n    `ConvLSTMFireNet`, `ConvGRUFireNet`, and `SpatioTemporalUNet`.\n    \"\"\"\n    import torch\n    import torch.nn as nn\n    import torch.nn.functional as F\n    from torch.utils.data import Dataset\n\n    class SpatioTemporalFireDataset(Dataset):\n        def __init__(self, X_norm: np.ndarray, y_all: np.ndarray, mask_2d: np.ndarray,\n                     issue_indices: np.ndarray, seq_length: int):\n            self.X_norm = X_norm\n            self.y_all = y_all\n            self.mask_2d = mask_2d\n            self.indices = np.asarray(issue_indices, dtype=\"int64\")\n            self.seq_len = int(seq_length)\n\n        def __len__(self) -> int:\n            return len(self.indices)\n\n        def __getitem__(self, k: int):\n            t_idx = int(self.indices[k])\n            x_seq = self.X_norm[t_idx - self.seq_len + 1: t_idx + 1]  # (T, C, H, W)\n            y_map = np.nan_to_num(self.y_all[t_idx], nan=0.0)         # (H, W)\n            return (\n                torch.from_numpy(x_seq),\n                torch.from_numpy(y_map),\n                torch.from_numpy(self.mask_2d),\n                t_idx,\n            )\n\n    class MaskedFocalBCEWithLogitsLoss(nn.Module):\n        def __init__(self, pos_weight: float = 10.0, gamma: float = 2.0):\n            super().__init__()\n            self.pos_weight = float(pos_weight)\n            self.gamma = float(gamma)\n\n        def forward(self, logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n            # logits, targets, mask: (B, H, W)\n            pw = torch.tensor(self.pos_weight, device=logits.device, dtype=logits.dtype)\n            bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw, reduction=\"none\")\n            probs = torch.sigmoid(logits)\n            p_t = probs * targets + (1.0 - probs) * (1.0 - targets)\n            focal = ((1.0 - p_t).clamp(min=1e-5) ** self.gamma) * bce\n            masked = focal * mask\n            denom = mask.sum().clamp(min=1.0)\n            return masked.sum() / denom\n\n    class ConvLSTMFireNet(nn.Module):\n        \"\"\"Spatiotemporal 2D ConvLSTM architecture.\"\"\"\n\n        def __init__(self, c_in: int, h_dim: int):\n            super().__init__()\n            self.h_dim = h_dim\n            self.stem = nn.Sequential(\n                nn.Conv2d(c_in, h_dim, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim),\n                nn.SiLU(),\n            )\n            self.gates = nn.Conv2d(h_dim * 2, h_dim * 4, kernel_size=3, padding=1)\n            self.head = nn.Sequential(\n                nn.Conv2d(h_dim, h_dim // 2, kernel_size=3, padding=1),\n                nn.SiLU(),\n                nn.Conv2d(h_dim // 2, 1, kernel_size=1),\n            )\n\n        def forward(self, x: torch.Tensor) -> torch.Tensor:\n            # x: (B, T, C, H, W)\n            b, t, _, h, w = x.shape\n            h_state = torch.zeros((b, self.h_dim, h, w), device=x.device, dtype=x.dtype)\n            c_state = torch.zeros((b, self.h_dim, h, w), device=x.device, dtype=x.dtype)\n            for step in range(t):\n                feat = self.stem(x[:, step])\n                combined = torch.cat([feat, h_state], dim=1)\n                i_g, f_g, o_g, g_g = torch.chunk(self.gates(combined), 4, dim=1)\n                i_g = torch.sigmoid(i_g)\n                f_g = torch.sigmoid(f_g)\n                o_g = torch.sigmoid(o_g)\n                g_g = torch.tanh(g_g)\n                c_state = f_g * c_state + i_g * g_g\n                h_state = o_g * torch.tanh(c_state)\n            return self.head(h_state).squeeze(1)\n\n    class ConvGRUFireNet(nn.Module):\n        \"\"\"Spatiotemporal 2D ConvGRU architecture with residual spatial refinement.\"\"\"\n\n        def __init__(self, c_in: int, h_dim: int):\n            super().__init__()\n            self.h_dim = h_dim\n            self.enc = nn.Sequential(\n                nn.Conv2d(c_in, h_dim, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim),\n                nn.SiLU(),\n                nn.Conv2d(h_dim, h_dim, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim),\n                nn.SiLU(),\n            )\n            self.gru_zr = nn.Conv2d(h_dim * 2, h_dim * 2, kernel_size=3, padding=1)\n            self.gru_h = nn.Conv2d(h_dim * 2, h_dim, kernel_size=3, padding=1)\n            self.head = nn.Sequential(\n                nn.Conv2d(h_dim, h_dim // 2, kernel_size=3, padding=1),\n                nn.SiLU(),\n                nn.Conv2d(h_dim // 2, 1, kernel_size=1),\n            )\n\n        def forward(self, x: torch.Tensor) -> torch.Tensor:\n            b, t, _, h, w = x.shape\n            state = torch.zeros((b, self.h_dim, h, w), device=x.device, dtype=x.dtype)\n            for step in range(t):\n                feat = self.enc(x[:, step])\n                z, r = torch.chunk(torch.sigmoid(self.gru_zr(torch.cat([feat, state], dim=1))), 2, dim=1)\n                cand = torch.tanh(self.gru_h(torch.cat([feat, r * state], dim=1)))\n                state = (1.0 - z) * state + z * cand\n            return self.head(state).squeeze(1)\n\n    class SpatioTemporalUNet(nn.Module):\n        \"\"\"Multi-scale U-Net over collapsed `(T * C, H, W)` spatiotemporal sequence.\"\"\"\n\n        def __init__(self, c_in: int, h_dim: int, seq_length: int):\n            super().__init__()\n            in_ch = c_in * seq_length\n            self.enc1 = nn.Sequential(\n                nn.Conv2d(in_ch, h_dim, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim),\n                nn.SiLU(),\n                nn.Conv2d(h_dim, h_dim, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim),\n                nn.SiLU(),\n            )\n            self.down = nn.MaxPool2d(2)\n            self.bottleneck = nn.Sequential(\n                nn.Conv2d(h_dim, h_dim * 2, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim * 2),\n                nn.SiLU(),\n                nn.Conv2d(h_dim * 2, h_dim, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim),\n                nn.SiLU(),\n            )\n            self.dec1 = nn.Sequential(\n                nn.Conv2d(h_dim * 2, h_dim, kernel_size=3, padding=1),\n                nn.GroupNorm(4, h_dim),\n                nn.SiLU(),\n                nn.Conv2d(h_dim, 1, kernel_size=1),\n            )\n\n        def forward(self, x: torch.Tensor) -> torch.Tensor:\n            b, t, c, h, w = x.shape\n            x_flat = x.reshape(b, t * c, h, w)\n            e1 = self.enc1(x_flat)\n            bn = self.bottleneck(self.down(e1))\n            up = F.interpolate(bn, size=(h, w), mode=\"bilinear\", align_corners=False)\n            out = self.dec1(torch.cat([e1, up], dim=1))\n            return out.squeeze(1)\n\n    architectures = {\n        \"convlstm\": lambda: ConvLSTMFireNet(in_channels, hidden_dim),\n        \"convgru\": lambda: ConvGRUFireNet(in_channels, hidden_dim),\n        \"spatiotemporal_unet\": lambda: SpatioTemporalUNet(in_channels, hidden_dim, seq_len),\n    }\n    return SpatioTemporalFireDataset, MaskedFocalBCEWithLogitsLoss, architectures\n\n\ndef _evaluate_dl_loader(model, loader, criterion, device) -> tuple[float, np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"Run model inference over a DataLoader and collect valid masked pixel targets & probabilities.\"\"\"\n    import torch\n\n    model.eval()\n    total_loss = 0.0\n    n_batches = 0\n    y_true_list, y_prob_list, t_idx_list = [], [], []\n    with torch.no_grad():\n        for x_seq, y_map, mask, t_idx in loader:\n            x_seq = x_seq.to(device)\n            y_map = y_map.to(device)\n            mask = mask.to(device)\n            logits = model(x_seq)\n            loss = criterion(logits, y_map, mask)\n            total_loss += float(loss.item())\n            n_batches += 1\n            probs = torch.sigmoid(logits)\n            m_bool = (mask > 0.5).cpu().numpy()\n            y_np = y_map.cpu().numpy()\n            p_np = probs.cpu().numpy()\n            for b in range(len(t_idx)):\n                mb = m_bool[b]\n                y_true_list.append(y_np[b][mb])\n                y_prob_list.append(p_np[b][mb])\n                t_idx_list.append(int(t_idx[b]))\n\n    y_true_all = np.concatenate(y_true_list) if y_true_list else np.zeros(0, dtype=\"float32\")\n    y_prob_all = np.concatenate(y_prob_list) if y_prob_list else np.zeros(0, dtype=\"float32\")\n    return total_loss / max(1, n_batches), y_true_all, y_prob_all, np.asarray(t_idx_list)\n\n\ndef train_spatiotemporal_dl(epochs: int = 12, seq_len: int = 7, hidden_dim: int = 48,\n                            lr: float = 1e-3, batch_size: int = 4,\n                            architectures_to_run: tuple[str, ...] = (\"convlstm\", \"convgru\", \"spatiotemporal_unet\"),\n                            force: bool = False) -> dict:\n    \"\"\"Train and compare ConvLSTM, ConvGRU, and Spatiotemporal U-Net architectures with real\n    PyTorch DataLoaders, MaskedFocalBCEWithLogitsLoss, AdamW backpropagation, per-epoch\n    Google Drive checkpointing, validation-based architecture selection, and holdout test inference.\n    \"\"\"\n    import torch\n    from torch.utils.data import DataLoader\n\n    log = get_logger()\n    ckpt_root = config.p(\"checkpoints\") / \"spatiotemporal_dl\"\n    ckpt_root.mkdir(parents=True, exist_ok=True)\n    summary_path = config.p(\"models\") / \"spatiotemporal_dl_metrics.json\"\n    if summary_path.exists() and not force:\n        return read_json(summary_path)\n\n    data = _load_cube_arrays(seq_len=seq_len)\n    in_channels = len(data[\"channel_names\"])\n    DatasetCls, LossCls, arch_builders = build_pytorch_components(\n        in_channels=in_channels, hidden_dim=hidden_dim, seq_len=seq_len,\n    )\n\n    ds_tr = DatasetCls(data[\"X_norm\"], data[\"y_all\"], data[\"mask_2d\"], data[\"idx_tr\"], seq_len)\n    ds_va = DatasetCls(data[\"X_norm\"], data[\"y_all\"], data[\"mask_2d\"], data[\"idx_va\"], seq_len)\n    ds_te = DatasetCls(data[\"X_norm\"], data[\"y_all\"], data[\"mask_2d\"], data[\"idx_te\"], seq_len)\n\n    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, drop_last=False)\n    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False)\n    dl_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False)\n\n    # Compute positive class weight from training split valid forest pixels\n    m2d = data[\"mask_2d\"] > 0.5\n    tr_y = np.nan_to_num(data[\"y_all\"][data[\"idx_tr\"]][:, m2d], nan=0.0)\n    pos = float((tr_y == 1).sum())\n    neg = float((tr_y == 0).sum())\n    pos_weight = float(np.clip(neg / max(pos, 1.0), 1.0, 50.0))\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    criterion = LossCls(pos_weight=pos_weight, gamma=2.0)\n\n    arch_results = {}\n    for arch_name in architectures_to_run:\n        if arch_name not in arch_builders:\n            continue\n        arch_dir = ckpt_root / arch_name\n        arch_dir.mkdir(parents=True, exist_ok=True)\n        last_ckpt = arch_dir / \"last_checkpoint.pt\"\n        best_ckpt = arch_dir / \"best_checkpoint.pt\"\n        hist_json = arch_dir / \"training_history.json\"\n\n        torch.manual_seed(42)\n        model = arch_builders[arch_name]().to(device)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))\n\n        start_epoch = 0\n        best_val_pr_auc = -1.0\n        total_opt_steps = 0\n        history = []\n\n        if last_ckpt.exists() and not force:\n            ckpt = torch.load(last_ckpt, map_location=device, weights_only=False)\n            model.load_state_dict(ckpt[\"model_state\"])\n            optimizer.load_state_dict(ckpt[\"optimizer_state\"])\n            start_epoch = int(ckpt.get(\"epoch\", 0)) + 1\n            best_val_pr_auc = float(ckpt.get(\"best_val_pr_auc\", -1.0))\n            total_opt_steps = int(ckpt.get(\"optimizer_steps_completed\", 0))\n            history = read_json(hist_json, default=[]) or []\n            log.info(f\"[DL {arch_name}] Resumed from epoch {start_epoch} ({total_opt_steps} optimizer steps).\")\n\n        t0 = time.time()\n        use_amp = bool(device.type == \"cuda\")\n        scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp) if hasattr(torch, \"amp\") else None\n        patience = max(4, epochs // 2)\n        epochs_no_improve = 0\n\n        for ep in range(start_epoch, epochs):\n            model.train()\n            ep_loss = 0.0\n            ep_batches = 0\n            for x_seq, y_map, mask, _ in dl_tr:\n                x_seq = x_seq.to(device)\n                y_map = y_map.to(device)\n                mask = mask.to(device)\n\n                optimizer.zero_grad(set_to_none=True)\n                if use_amp and scaler is not None:\n                    with torch.amp.autocast(\"cuda\", enabled=True):\n                        logits = model(x_seq)\n                        loss = criterion(logits, y_map, mask)\n                    scaler.scale(loss).backward()\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)\n                    scaler.step(optimizer)\n                    scaler.update()\n                else:\n                    logits = model(x_seq)\n                    loss = criterion(logits, y_map, mask)\n                    loss.backward()\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)\n                    optimizer.step()\n\n                total_opt_steps += 1\n                ep_loss += float(loss.item())\n                ep_batches += 1\n\n            scheduler.step()\n            train_loss = ep_loss / max(1, ep_batches)\n            val_loss, y_va_true, y_va_prob, _ = _evaluate_dl_loader(model, dl_va, criterion, device)\n            val_metrics = compute_binary_metrics(y_va_true, y_va_prob, threshold=0.5)\n            val_pr = val_metrics[\"pr_auc\"] if val_metrics[\"pr_auc\"] is not None else -val_loss\n\n            rec = {\n                \"epoch\": ep + 1,\n                \"train_loss\": round(train_loss, 5),\n                \"val_loss\": round(val_loss, 5),\n                \"val_pr_auc\": val_metrics[\"pr_auc\"],\n                \"val_roc_auc\": val_metrics[\"roc_auc\"],\n                \"val_brier\": val_metrics[\"brier_score\"],\n                \"optimizer_steps_completed\": total_opt_steps,\n                \"lr\": round(float(optimizer.param_groups[0][\"lr\"]), 6),\n            }\n            history.append(rec)\n            atomic_write_json(hist_json, history)\n\n            improved = float(val_pr) > best_val_pr_auc + 1e-5 or not best_ckpt.exists()\n            state_payload = {\n                \"arch_name\": arch_name,\n                \"epoch\": ep,\n                \"model_state\": model.state_dict(),\n                \"optimizer_state\": optimizer.state_dict(),\n                \"best_val_pr_auc\": max(best_val_pr_auc, float(val_pr)),\n                \"optimizer_steps_completed\": total_opt_steps,\n                \"in_channels\": in_channels,\n                \"hidden_dim\": hidden_dim,\n                \"seq_len\": seq_len,\n                \"channel_names\": data[\"channel_names\"],\n                \"ch_mean\": data[\"ch_mean\"],\n                \"ch_std\": data[\"ch_std\"],\n            }\n            torch.save(state_payload, last_ckpt)\n            if improved:\n                best_val_pr_auc = float(val_pr)\n                epochs_no_improve = 0\n                torch.save(state_payload, best_ckpt)\n            else:\n                epochs_no_improve += 1\n\n            log.info(\n                f\"[DL {arch_name}] Epoch {ep + 1}/{epochs} | steps={total_opt_steps} | \"\n                f\"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_PR-AUC={val_metrics['pr_auc']}\"\n            )\n            if epochs_no_improve >= patience and (ep + 1) >= 4:\n                log.info(f\"[DL {arch_name}] Early stopping triggered at epoch {ep + 1} (patience={patience}).\")\n                break\n\n        if total_opt_steps <= 0:\n            raise RuntimeError(f\"Architecture {arch_name} completed 0 optimizer updates.\")\n\n        # Load best validation checkpoint for final Validation threshold selection & Holdout Test inference\n        best_state = torch.load(best_ckpt, map_location=device, weights_only=False)\n        model.load_state_dict(best_state[\"model_state\"])\n        _, y_va_true, y_va_prob, _ = _evaluate_dl_loader(model, dl_va, criterion, device)\n        opt_th = find_optimal_f1_threshold(y_va_true, y_va_prob)\n        _, y_te_true, y_te_prob, _ = _evaluate_dl_loader(model, dl_te, criterion, device)\n\n        gpu_name = torch.cuda.get_device_name(0) if device.type == \"cuda\" else \"CPU\"\n        max_vram_mb = round(float(torch.cuda.max_memory_allocated(0) / (1024 ** 2)), 1) if device.type == \"cuda\" else 0.0\n\n        arch_results[arch_name] = {\n            \"family\": \"spatiotemporal_deep_learning\",\n            \"device\": str(device),\n            \"gpu_name\": gpu_name,\n            \"batch_size\": int(batch_size),\n            \"mixed_precision_amp\": use_amp,\n            \"max_vram_allocated_mb\": max_vram_mb,\n            \"parameters_count\": int(sum(p.numel() for p in model.parameters())),\n            \"epochs_completed\": len(history),\n            \"optimizer_steps_completed\": total_opt_steps,\n            \"optimal_val_threshold\": round(opt_th, 4),\n            \"validation\": compute_binary_metrics(y_va_true, y_va_prob, threshold=opt_th),\n            \"test_holdout\": compute_binary_metrics(y_te_true, y_te_prob, threshold=opt_th),\n            \"curves_test\": compute_pr_roc_curves(y_te_true, y_te_prob),\n            \"best_checkpoint\": str(best_ckpt),\n            \"last_checkpoint\": str(last_ckpt),\n            \"history_json\": str(hist_json),\n            \"training_history\": history,\n            \"seconds\": round(time.time() - t0, 2),\n        }\n\n    # Select winning deep-learning architecture strictly by Validation PR-AUC (or lowest val_loss if single-class sample)\n    best_arch = max(\n        arch_results.keys(),\n        key=lambda k: (\n            arch_results[k][\"validation\"][\"pr_auc\"]\n            if arch_results[k][\"validation\"][\"pr_auc\"] is not None\n            else -arch_results[k][\"validation\"][\"brier_score\"]\n        ),\n    )\n\n    out = {\n        \"generated_utc\": now_utc(),\n        \"device\": str(device),\n        \"seq_len\": seq_len,\n        \"hidden_dim\": hidden_dim,\n        \"in_channels\": in_channels,\n        \"channel_names\": data[\"channel_names\"],\n        \"strict_2001_2025_coverage\": data[\"strict_2001_2025_coverage\"],\n        \"n_sequences\": {\n            \"train\": int(len(ds_tr)),\n            \"validation\": int(len(ds_va)),\n            \"test_holdout\": int(len(ds_te)),\n        },\n        \"selected_best_dl_architecture_by_validation\": best_arch,\n        \"architectures\": arch_results,\n    }\n    atomic_write_json(summary_path, out)\n    return out\n","evaluation.py":"\"\"\"Comprehensive model evaluation, calibration, SHAP explainability, feature-group ablations,\nLeave-One-Ecoregion-Out (LOEO) spatial holdout generalisation, regional subgroup stratification,\nsensor/attribution sensitivity, and genuine spatial-temporal block-bootstrap uncertainty reporting.\n\nScientific Guarantees:\n1. Never tunes hyperparameters, thresholds, or calibration mappings on the 2016-2025 holdout test set.\n2. Explicitly distinguishes:\n   - `spatiotemporal_block_bootstrap_ci()`: resamples whole `(ecoregion, year_month)` or\n     `(2deg_spatial_tile, 7day_window)` clusters with replacement to account for spatial-temporal\n     autocorrelation.\n   - `observation_iid_bootstrap_ci()`: ordinary observation-level i.i.d. bootstrap, clearly labelled.\n3. Explicitly distinguishes:\n   - `run_spatial_holdout_generalization()`: Leave-One-Ecoregion-Out (LOEO) spatial generalisation\n     where each forest ecoregion is held out completely during training and coordinate features\n     (`latitude`, `longitude`) are removed.\n   - `run_regional_subgroup_analysis()`: performance of the all-India trained model stratified by\n     ecoregion on the holdout test split.\n4. Computes SHAP values (`compute_shap_summary()`), feature-group ablations (`run_feature_group_ablations()`),\n   and sensor/attribution sensitivity analyses.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .features import FEATURE_COLUMNS, FEATURE_GROUPS\nfrom .models_ml import compute_binary_metrics, find_optimal_f1_threshold, prepare_chronological_splits\nfrom .utils import atomic_write_json, get_logger, now_utc, read_json, write_csv_atomic\n\n\ndef compute_reliability_curve(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> list[dict]:\n    \"\"\"Compute reliability diagram bin statistics (mean predicted probability vs observed frequency).\"\"\"\n    y_true = np.asarray(y_true, dtype=\"float64\")\n    y_prob = np.asarray(y_prob, dtype=\"float64\")\n    bins = np.linspace(0.0, 1.0, n_bins + 1)\n    ids = np.digitize(y_prob, bins) - 1\n    rows = []\n    for b in range(n_bins):\n        m = ids == b\n        rows.append({\n            \"bin\": b,\n            \"bin_low\": round(float(bins[b]), 2),\n            \"bin_high\": round(float(bins[b + 1]), 2),\n            \"count\": int(m.sum()),\n            \"mean_predicted_prob\": round(float(y_prob[m].mean()), 5) if m.any() else None,\n            \"observed_frequency\": round(float(y_true[m].mean()), 5) if m.any() else None,\n        })\n    return rows\n\n\ndef _summarize_boot_samples(prs: list, rocs: list, briers: list, f1s: list, scheme: str, n_units: int) -> dict:\n    def ci(vals):\n        if not vals:\n            return {\"mean\": None, \"ci95_low\": None, \"ci95_high\": None}\n        return {\n            \"mean\": round(float(np.mean(vals)), 5),\n            \"ci95_low\": round(float(np.percentile(vals, 2.5)), 5),\n            \"ci95_high\": round(float(np.percentile(vals, 97.5)), 5),\n        }\n\n    return {\n        \"sampling_scheme\": scheme,\n        \"n_resampling_units\": int(n_units),\n        \"pr_auc\": ci(prs),\n        \"roc_auc\": ci(rocs),\n        \"brier_score\": ci(briers),\n        \"f1\": ci(f1s),\n    }\n\n\ndef observation_iid_bootstrap_ci(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5,\n                                 n_boot: int = 100, seed: int = 42) -> dict:\n    \"\"\"Ordinary observation-level (i.i.d.) bootstrap 95% confidence intervals (clearly labelled).\"\"\"\n    rng = np.random.default_rng(seed)\n    y_true = np.asarray(y_true, dtype=\"int32\")\n    y_prob = np.asarray(y_prob, dtype=\"float64\")\n    n = len(y_true)\n    if n == 0:\n        return {}\n\n    prs, rocs, briers, f1s = [], [], [], []\n    for _ in range(n_boot):\n        idx = rng.integers(0, n, size=n)\n        m = compute_binary_metrics(y_true[idx], y_prob[idx], threshold=threshold)\n        if m[\"pr_auc\"] is not None:\n            prs.append(m[\"pr_auc\"])\n        if m[\"roc_auc\"] is not None:\n            rocs.append(m[\"roc_auc\"])\n        briers.append(m[\"brier_score\"])\n        f1s.append(m[\"f1\"])\n\n    return _summarize_boot_samples(prs, rocs, briers, f1s, \"observation_level_iid_bootstrap\", n)\n\n\ndef spatiotemporal_block_bootstrap_ci(pdf: pd.DataFrame, prob_col: str, threshold: float = 0.5,\n                                      target_col: str = \"target_forest_fire_next72h\",\n                                      n_boot: int = 100, seed: int = 42) -> dict:\n    \"\"\"Genuine Spatial-Temporal Block Bootstrap 95% confidence intervals.\n\n    Partitions the test dataset into joint spatial-temporal blocks defined by\n    `(2.0-deg latitude tile, 2.0-deg longitude tile, 7-day temporal block)` so that\n    spatial correlation across neighbouring 0.25-deg cells and multi-day fire-weather\n    persistence are preserved within each resampled block.\n    \"\"\"\n    if len(pdf) == 0 or prob_col not in pdf.columns:\n        return {}\n\n    rng = np.random.default_rng(seed)\n    lat_tile = np.floor(pdf[\"latitude\"].to_numpy() / 2.0).astype(\"int32\")\n    lon_tile = np.floor(pdf[\"longitude\"].to_numpy() / 2.0).astype(\"int32\")\n    t_days = pd.to_datetime(pdf[\"time\"]).astype(\"int64\") // (86400 * 10**9)\n    t_block = (t_days // 7).to_numpy(dtype=\"int64\")\n\n    block_keys = pd.Series(\n        lat_tile.astype(str) + \"_\" + lon_tile.astype(str) + \"_\" + t_block.astype(str)\n    )\n    groups = [grp_idx.to_numpy() for _, grp_idx in block_keys.groupby(block_keys).groups.items()]\n    n_blocks = len(groups)\n    if n_blocks < 2:\n        return observation_iid_bootstrap_ci(\n            pdf[target_col].to_numpy(), pdf[prob_col].to_numpy(), threshold=threshold, n_boot=n_boot, seed=seed,\n        )\n\n    y_all = pdf[target_col].to_numpy(dtype=\"int32\")\n    p_all = pdf[prob_col].to_numpy(dtype=\"float64\")\n\n    prs, rocs, briers, f1s = [], [], [], []\n    for _ in range(n_boot):\n        chosen = rng.integers(0, n_blocks, size=n_blocks)\n        idx = np.concatenate([groups[c] for c in chosen])\n        m = compute_binary_metrics(y_all[idx], p_all[idx], threshold=threshold)\n        if m[\"pr_auc\"] is not None:\n            prs.append(m[\"pr_auc\"])\n        if m[\"roc_auc\"] is not None:\n            rocs.append(m[\"roc_auc\"])\n        briers.append(m[\"brier_score\"])\n        f1s.append(m[\"f1\"])\n\n    return _summarize_boot_samples(\n        prs, rocs, briers, f1s,\n        \"spatiotemporal_block_bootstrap (2deg_lat x 2deg_lon x 7day_window)\",\n        n_blocks,\n    )\n\n\n# Alias for callers expecting `bootstrap_metric_ci`\nbootstrap_metric_ci = observation_iid_bootstrap_ci\n\n\ndef _build_default_eval_estimator(pos_weight: float = 10.0):\n    from sklearn.impute import SimpleImputer\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.pipeline import Pipeline\n    from sklearn.preprocessing import StandardScaler\n\n    try:\n        import lightgbm as lgb\n        return Pipeline([\n            (\"imputer\", SimpleImputer(strategy=\"median\")),\n            (\"clf\", lgb.LGBMClassifier(\n                n_estimators=200, learning_rate=0.05, num_leaves=31,\n                scale_pos_weight=float(np.clip(pos_weight, 1.0, 100.0)),\n                random_state=42, n_jobs=-1, verbose=-1,\n            )),\n        ])\n    except Exception:\n        return Pipeline([\n            (\"imputer\", SimpleImputer(strategy=\"median\")),\n            (\"scaler\", StandardScaler()),\n            (\"clf\", LogisticRegression(C=0.5, max_iter=400, class_weight=\"balanced\", random_state=42)),\n        ])\n\n\ndef run_feature_group_ablations() -> pd.DataFrame:\n    \"\"\"Retrain the primary classifier while systematically dropping each physical feature group\n    (`thermo_vpd`, `wind_ffwi`, `precip_drought`, `radiation`, `antecedent_fire`, `spatial_seasonal`)\n    to quantify its marginal contribution on Validation & Holdout Test.\n    \"\"\"\n    sup_pq = config.p(\"supervised\") / \"supervised_forest_fire_72h.parquet\"\n    if not sup_pq.exists():\n        return pd.DataFrame()\n\n    df = pd.read_parquet(sup_pq)\n    target_col = \"target_forest_fire_next72h\"\n    tr, va, te, _ = prepare_chronological_splits(df)\n    if len(tr) == 0 or len(np.unique(tr[target_col])) < 2:\n        return pd.DataFrame()\n\n    all_feats = [c for c in FEATURE_COLUMNS if c in tr.columns and tr[c].notna().mean() >= 0.80]\n    if not all_feats:\n        all_feats = [c for c in FEATURE_COLUMNS if c in tr.columns and tr[c].notna().any()]\n\n    pos = max(1, int((tr[target_col] == 1).sum()))\n    neg = max(1, int((tr[target_col] == 0).sum()))\n    pw = neg / pos\n\n    configs = [(\"all_features\", all_feats)]\n    for grp_name, grp_cols in FEATURE_GROUPS.items():\n        rem = [c for c in all_feats if c not in grp_cols]\n        if rem and len(rem) < len(all_feats):\n            configs.append((f\"without_{grp_name}\", rem))\n\n    rows = []\n    base_val_pr, base_test_pr = None, None\n    y_tr = tr[target_col].to_numpy(dtype=\"int32\")\n    y_va = va[target_col].to_numpy(dtype=\"int32\")\n    y_te = te[target_col].to_numpy(dtype=\"int32\")\n\n    for label, cols in configs:\n        est = _build_default_eval_estimator(pos_weight=pw)\n        est.fit(tr[cols].to_numpy(dtype=\"float32\"), y_tr)\n        p_va = est.predict_proba(va[cols].to_numpy(dtype=\"float32\"))[:, 1]\n        p_te = est.predict_proba(te[cols].to_numpy(dtype=\"float32\"))[:, 1]\n        th = find_optimal_f1_threshold(y_va, p_va)\n        m_va = compute_binary_metrics(y_va, p_va, threshold=th)\n        m_te = compute_binary_metrics(y_te, p_te, threshold=th)\n        if label == \"all_features\":\n            base_val_pr = m_va[\"pr_auc\"]\n            base_test_pr = m_te[\"pr_auc\"]\n        rows.append({\n            \"ablation\": label,\n            \"n_features\": len(cols),\n            \"val_pr_auc\": m_va[\"pr_auc\"],\n            \"val_brier\": m_va[\"brier_score\"],\n            \"test_pr_auc\": m_te[\"pr_auc\"],\n            \"test_roc_auc\": m_te[\"roc_auc\"],\n            \"test_f1\": m_te[\"f1\"],\n            \"test_brier\": m_te[\"brier_score\"],\n            \"delta_val_pr_auc\": (\n                round(float(m_va[\"pr_auc\"] - base_val_pr), 5)\n                if m_va[\"pr_auc\"] is not None and base_val_pr is not None else None\n            ),\n            \"delta_test_pr_auc\": (\n                round(float(m_te[\"pr_auc\"] - base_test_pr), 5)\n                if m_te[\"pr_auc\"] is not None and base_test_pr is not None else None\n            ),\n        })\n\n    ab_df = pd.DataFrame(rows)\n    write_csv_atomic(ab_df, config.p(\"tables\") / \"ablation_study_results.csv\")\n    return ab_df\n\n\ndef run_spatial_holdout_generalization() -> pd.DataFrame:\n    \"\"\"Genuine Leave-One-Ecoregion-Out (LOEO) spatial generalisation test.\n\n    For each Indian forest ecoregion E_k with both positive and negative examples:\n    - Removes spatial coordinate features (`latitude`, `longitude`) so the model cannot memorize locations.\n    - Trains strictly on `train` rows outside E_k (`ecoregion != E_k`).\n    - Selects threshold strictly on `validation` rows outside E_k (`ecoregion != E_k`).\n    - Evaluates on `test_holdout` rows strictly inside the unseen region `ecoregion == E_k`.\n    \"\"\"\n    sup_pq = config.p(\"supervised\") / \"supervised_forest_fire_72h.parquet\"\n    if not sup_pq.exists():\n        return pd.DataFrame()\n\n    df = pd.read_parquet(sup_pq)\n    target_col = \"target_forest_fire_next72h\"\n    tr, va, te, _ = prepare_chronological_splits(df)\n    if len(tr) == 0 or len(te) == 0:\n        return pd.DataFrame()\n\n    # Physical + land-cover features ONLY (exclude raw latitude/longitude to test physical transferability)\n    phys_cols = [\n        c for c in FEATURE_COLUMNS\n        if c not in (\"latitude\", \"longitude\") and c in tr.columns and tr[c].notna().mean() >= 0.80\n    ]\n    if not phys_cols:\n        phys_cols = [c for c in FEATURE_COLUMNS if c not in (\"latitude\", \"longitude\") and c in tr.columns and tr[c].notna().any()]\n\n    rows = []\n    for eco in sorted(te[\"ecoregion\"].dropna().unique()):\n        te_eco = te[te[\"ecoregion\"] == eco]\n        tr_out = tr[tr[\"ecoregion\"] != eco]\n        va_out = va[va[\"ecoregion\"] != eco]\n        if len(te_eco) < 10 or len(tr_out) < 20 or len(np.unique(tr_out[target_col])) < 2:\n            continue\n\n        pos = max(1, int((tr_out[target_col] == 1).sum()))\n        neg = max(1, int((tr_out[target_col] == 0).sum()))\n        est = _build_default_eval_estimator(pos_weight=neg / pos)\n        est.fit(tr_out[phys_cols].to_numpy(dtype=\"float32\"), tr_out[target_col].to_numpy(dtype=\"int32\"))\n\n        if len(va_out) > 0 and len(np.unique(va_out[target_col])) > 1:\n            p_va_out = est.predict_proba(va_out[phys_cols].to_numpy(dtype=\"float32\"))[:, 1]\n            th = find_optimal_f1_threshold(va_out[target_col].to_numpy(dtype=\"int32\"), p_va_out)\n        else:\n            th = 0.5\n\n        p_te_eco = est.predict_proba(te_eco[phys_cols].to_numpy(dtype=\"float32\"))[:, 1]\n        m_loeo = compute_binary_metrics(te_eco[target_col].to_numpy(dtype=\"int32\"), p_te_eco, threshold=th)\n        rows.append({\n            \"held_out_ecoregion\": str(eco),\n            \"experiment_type\": \"leave_one_ecoregion_out_spatial_holdout\",\n            \"train_rows_outside_region\": int(len(tr_out)),\n            \"test_rows_in_unseen_region\": m_loeo[\"n_samples\"],\n            \"positive_rate\": m_loeo[\"positive_rate\"],\n            \"loeo_test_pr_auc\": m_loeo[\"pr_auc\"],\n            \"loeo_test_roc_auc\": m_loeo[\"roc_auc\"],\n            \"loeo_test_f1\": m_loeo[\"f1\"],\n            \"loeo_test_brier\": m_loeo[\"brier_score\"],\n            \"loeo_test_ece\": m_loeo[\"ece\"],\n        })\n\n    loeo_df = pd.DataFrame(rows)\n    if len(loeo_df):\n        write_csv_atomic(loeo_df, config.p(\"tables\") / \"spatial_holdout_generalization.csv\")\n    return loeo_df\n\n\ndef compute_shap_summary(max_samples: int = 1500) -> list[dict]:\n    \"\"\"Compute SHAP feature attributions (using `shap.TreeExplainer` or `shap.LinearExplainer` when\n    installed, with exact standardized linear/permutation effect fallback) on a held-out sample.\n    \"\"\"\n    import joblib\n\n    sup_pq = config.p(\"supervised\") / \"supervised_forest_fire_72h.parquet\"\n    if not sup_pq.exists():\n        return []\n\n    df = pd.read_parquet(sup_pq)\n    _, va, te, _ = prepare_chronological_splits(df)\n    eval_df = te if len(te) > 0 else va\n    if len(eval_df) == 0:\n        return []\n\n    ml_res = read_json(config.p(\"models\") / \"classical_ml_metrics.json\", {}) or {}\n    best_cls = ml_res.get(\"selected_best_model_by_validation_pr_auc\") or \"lightgbm\"\n    ckpt = config.p(\"checkpoints\") / \"classical\" / f\"{best_cls}.joblib\"\n    if not ckpt.exists():\n        avail = sorted((config.p(\"checkpoints\") / \"classical\").glob(\"*.joblib\"))\n        ckpt = avail[0] if avail else None\n    if ckpt is None or not ckpt.exists():\n        return []\n\n    bundle = joblib.load(ckpt)\n    if not (isinstance(bundle, dict) and \"pipeline\" in bundle):\n        return []\n\n    pipe = bundle[\"pipeline\"]\n    fcols = bundle[\"features\"]\n    sub = eval_df.sample(n=min(max_samples, len(eval_df)), random_state=42) if len(eval_df) > max_samples else eval_df\n    X_raw = sub[fcols].to_numpy(dtype=\"float32\")\n    X_imp = pipe.named_steps[\"imputer\"].transform(X_raw)\n    if \"scaler\" in pipe.named_steps:\n        X_trans = pipe.named_steps[\"scaler\"].transform(X_imp)\n    else:\n        X_trans = X_imp\n    clf = pipe.named_steps[\"clf\"]\n\n    mean_abs_shap = None\n    method_used = \"standardized_effect_magnitude\"\n    try:\n        import shap\n        if hasattr(clf, \"feature_importances_\"):\n            explainer = shap.TreeExplainer(clf)\n            sv = explainer.shap_values(X_trans)\n            if isinstance(sv, list):\n                sv = sv[1] if len(sv) > 1 else sv[0]\n            elif getattr(sv, \"ndim\", 0) == 3:\n                sv = sv[:, :, -1]\n            mean_abs_shap = np.mean(np.abs(sv), axis=0)\n            method_used = \"shap_tree_explainer\"\n        else:\n            explainer = shap.LinearExplainer(clf, X_trans)\n            sv = explainer.shap_values(X_trans)\n            mean_abs_shap = np.mean(np.abs(sv), axis=0)\n            method_used = \"shap_linear_explainer\"\n    except Exception:\n        if hasattr(clf, \"feature_importances_\"):\n            mean_abs_shap = np.asarray(clf.feature_importances_, dtype=\"float64\")\n            method_used = \"tree_gain_importance_fallback\"\n        elif hasattr(clf, \"coef_\"):\n            mean_abs_shap = np.abs(np.asarray(clf.coef_[0], dtype=\"float64\")) * np.std(X_trans, axis=0)\n            method_used = \"linear_standardized_marginal_effect\"\n\n    if mean_abs_shap is None:\n        return []\n\n    mean_abs_shap = np.asarray(mean_abs_shap, dtype=\"float64\").ravel()\n    tot = max(float(mean_abs_shap.sum()), 1e-12)\n    rows = []\n    for col, val in sorted(zip(fcols, mean_abs_shap), key=lambda x: x[1], reverse=True):\n        rows.append({\n            \"feature\": col,\n            \"mean_abs_shap\": round(float(val), 6),\n            \"normalized_share\": round(float(val / tot), 5),\n            \"method\": method_used,\n            \"model_explained\": best_cls,\n            \"n_samples_evaluated\": int(len(sub)),\n        })\n    shap_df = pd.DataFrame(rows)\n    write_csv_atomic(shap_df, config.p(\"tables\") / \"shap_feature_importance.csv\")\n    return rows\n\n\ndef run_sensor_sensitivity_analysis() -> list[dict]:\n    \"\"\"Compare active-fire detection characteristics, confidence, and FRP across satellite sensors\n    (MODIS Collection 6.1 vs. VIIRS S-NPP / NOAA-20) from the attributed FIRMS archive.\n    \"\"\"\n    att_pq = config.p(\"firms_processed\") / \"firms_india_attributed.parquet\"\n    if not att_pq.exists():\n        return []\n    df = pd.read_parquet(att_pq)\n    if len(df) == 0:\n        return []\n\n    sensor_col = \"instrument\" if \"instrument\" in df.columns else (\"source_product\" if \"source_product\" in df.columns else None)\n    if sensor_col is None:\n        return []\n\n    rows = []\n    for sensor, grp in df.groupby(sensor_col, observed=True):\n        vc = grp[\"fire_category\"].value_counts()\n        tot = max(1, len(grp))\n        rows.append({\n            \"sensor\": str(sensor),\n            \"total_detections\": int(len(grp)),\n            \"forest_associated_detections\": int(vc.get(\"forest_fire\", 0)),\n            \"pct_forest_associated\": round(100.0 * vc.get(\"forest_fire\", 0) / tot, 2),\n            \"agricultural_detections\": int(vc.get(\"agricultural_fire\", 0)),\n            \"pct_agricultural\": round(100.0 * vc.get(\"agricultural_fire\", 0) / tot, 2),\n            \"uncertain_detections\": int(vc.get(\"other_or_uncertain\", 0)),\n            \"pct_uncertain\": round(100.0 * vc.get(\"other_or_uncertain\", 0) / tot, 2),\n            \"median_frp_mw\": round(float(grp[\"frp\"].median()), 2) if \"frp\" in grp.columns and grp[\"frp\"].notna().any() else None,\n            \"mean_confidence_score\": round(float(grp[\"confidence_score\"].mean()), 3) if \"confidence_score\" in grp.columns else None,\n        })\n    sens_df = pd.DataFrame(rows)\n    if len(sens_df):\n        write_csv_atomic(sens_df, config.p(\"tables\") / \"sensor_sensitivity_analysis.csv\")\n    return rows\n\n\ndef run_full_evaluation_and_ablations(force: bool = False) -> dict:\n    \"\"\"Consolidate baseline comparisons, validation-based model selection, independent holdout\n    metrics, calibration reliability curves, SHAP analysis, feature-group ablations,\n    Leave-One-Ecoregion-Out (LOEO) spatial generalisation, regional subgroup stratification,\n    sensor sensitivity, monthly error analysis, and genuine spatial-temporal block-bootstrap 95% CIs.\n    \"\"\"\n    log = get_logger()\n    out_json = config.p(\"evaluation\") / \"full_evaluation_report.json\"\n    if out_json.exists() and not force:\n        return read_json(out_json)\n\n    ml_res = read_json(config.p(\"models\") / \"classical_ml_metrics.json\", {}) or {}\n    dl_res = read_json(config.p(\"models\") / \"spatiotemporal_dl_metrics.json\", {}) or {}\n\n    comp_rows = []\n    for name, m in (ml_res.get(\"models\") or {}).items():\n        v = m.get(\"validation\", {})\n        t = m.get(\"test_holdout\", {})\n        comp_rows.append({\n            \"model\": name,\n            \"family\": m.get(\"family\", \"classical_ml\"),\n            \"threshold_from_val\": m.get(\"optimal_val_threshold\"),\n            \"val_n_samples\": v.get(\"n_samples\"),\n            \"val_pos_rate\": v.get(\"positive_rate\"),\n            \"val_pr_auc\": v.get(\"pr_auc\"),\n            \"val_roc_auc\": v.get(\"roc_auc\"),\n            \"val_f1\": v.get(\"f1\"),\n            \"val_brier\": v.get(\"brier_score\"),\n            \"val_ece\": v.get(\"ece\"),\n            \"test_n_samples\": t.get(\"n_samples\"),\n            \"test_pos_rate\": t.get(\"positive_rate\"),\n            \"test_pr_auc\": t.get(\"pr_auc\"),\n            \"test_roc_auc\": t.get(\"roc_auc\"),\n            \"test_f1\": t.get(\"f1\"),\n            \"test_precision\": t.get(\"precision\"),\n            \"test_recall\": t.get(\"recall\"),\n            \"test_brier\": t.get(\"brier_score\"),\n            \"test_log_loss\": t.get(\"log_loss\"),\n            \"test_ece\": t.get(\"ece\"),\n            \"test_tp\": t.get(\"tp\"),\n            \"test_fp\": t.get(\"fp\"),\n            \"test_tn\": t.get(\"tn\"),\n            \"test_fn\": t.get(\"fn\"),\n        })\n\n    for arch_name, m in (dl_res.get(\"architectures\") or {}).items():\n        v = m.get(\"validation\", {})\n        t = m.get(\"test_holdout\", {})\n        comp_rows.append({\n            \"model\": f\"dl_{arch_name}\",\n            \"family\": \"spatiotemporal_dl\",\n            \"threshold_from_val\": m.get(\"optimal_val_threshold\"),\n            \"val_n_samples\": v.get(\"n_samples\"),\n            \"val_pos_rate\": v.get(\"positive_rate\"),\n            \"val_pr_auc\": v.get(\"pr_auc\"),\n            \"val_roc_auc\": v.get(\"roc_auc\"),\n            \"val_f1\": v.get(\"f1\"),\n            \"val_brier\": v.get(\"brier_score\"),\n            \"val_ece\": v.get(\"ece\"),\n            \"test_n_samples\": t.get(\"n_samples\"),\n            \"test_pos_rate\": t.get(\"positive_rate\"),\n            \"test_pr_auc\": t.get(\"pr_auc\"),\n            \"test_roc_auc\": t.get(\"roc_auc\"),\n            \"test_f1\": t.get(\"f1\"),\n            \"test_precision\": t.get(\"precision\"),\n            \"test_recall\": t.get(\"recall\"),\n            \"test_brier\": t.get(\"brier_score\"),\n            \"test_log_loss\": t.get(\"log_loss\"),\n            \"test_ece\": t.get(\"ece\"),\n            \"test_tp\": t.get(\"tp\"),\n            \"test_fp\": t.get(\"fp\"),\n            \"test_tn\": t.get(\"tn\"),\n            \"test_fn\": t.get(\"fn\"),\n        })\n\n    comp_df = pd.DataFrame(comp_rows)\n    if len(comp_df):\n        comp_df = comp_df.sort_values(\"val_pr_auc\", ascending=False, na_position=\"last\").reset_index(drop=True)\n        write_csv_atomic(comp_df, config.p(\"tables\") / \"model_comparison.csv\")\n\n    overall_winner = (\n        str(comp_df.iloc[0][\"model\"])\n        if len(comp_df) and comp_df.iloc[0][\"val_pr_auc\"] is not None\n        else (str(comp_df.iloc[0][\"model\"]) if len(comp_df) else None)\n    )\n\n    ab_df = run_feature_group_ablations()\n    loeo_df = run_spatial_holdout_generalization()\n    shap_rows = compute_shap_summary()\n    sensor_rows = run_sensor_sensitivity_analysis()\n\n    pred_pq = config.p(\"evaluation\") / \"test_predictions\" / \"classical_test_predictions.parquet\"\n    calibration_curves = {}\n    regional_subgroup_rows = []\n    error_by_month = []\n    block_bootstrap_cis = {}\n    iid_bootstrap_cis = {}\n\n    if pred_pq.exists():\n        pdf = pd.read_parquet(pred_pq)\n        y_te = pdf[\"target_forest_fire_next72h\"].to_numpy(dtype=\"int32\")\n        prob_cols = [c for c in pdf.columns if c.startswith(\"prob_\") and c.endswith(\"_cal\")]\n        for pc in prob_cols:\n            mname = pc[len(\"prob_\"):-len(\"_cal\")]\n            probs = pdf[pc].to_numpy(dtype=\"float64\")\n            raw_col = f\"prob_{mname}_raw\"\n            calibration_curves[mname] = {\n                \"calibrated\": compute_reliability_curve(y_te, probs),\n                \"raw\": compute_reliability_curve(y_te, pdf[raw_col].to_numpy(dtype=\"float64\"))\n                if raw_col in pdf.columns else [],\n            }\n            th = (ml_res.get(\"models\", {}).get(mname, {}) or {}).get(\"optimal_val_threshold\", 0.5)\n            block_bootstrap_cis[mname] = spatiotemporal_block_bootstrap_ci(pdf, pc, threshold=th, n_boot=100)\n            iid_bootstrap_cis[mname] = observation_iid_bootstrap_ci(y_te, probs, threshold=th, n_boot=100)\n\n        best_cls = ml_res.get(\"selected_best_model_by_validation_pr_auc\")\n        best_col = f\"prob_{best_cls}_cal\" if best_cls and f\"prob_{best_cls}_cal\" in pdf.columns else (prob_cols[0] if prob_cols else None)\n        if best_col:\n            th = (ml_res.get(\"models\", {}).get(best_cls, {}) or {}).get(\"optimal_val_threshold\", 0.5)\n            for eco, grp in pdf.groupby(\"ecoregion\", observed=True):\n                m_eco = compute_binary_metrics(\n                    grp[\"target_forest_fire_next72h\"].to_numpy(),\n                    grp[best_col].to_numpy(),\n                    threshold=th,\n                )\n                regional_subgroup_rows.append({\n                    \"ecoregion\": str(eco),\n                    \"analysis_type\": \"in_sample_trained_regional_subgroup_stratification\",\n                    **m_eco,\n                })\n\n            for mo, grp in pdf.groupby(\"month\", observed=True):\n                m_mo = compute_binary_metrics(\n                    grp[\"target_forest_fire_next72h\"].to_numpy(),\n                    grp[best_col].to_numpy(),\n                    threshold=th,\n                )\n                error_by_month.append({\"month\": int(mo), **m_mo})\n\n    reg_sub_df = pd.DataFrame(regional_subgroup_rows)\n    err_df = pd.DataFrame(error_by_month)\n    if len(reg_sub_df):\n        write_csv_atomic(reg_sub_df, config.p(\"tables\") / \"regional_subgroup_performance.csv\")\n        # Keep backwards-compatible alias while also saving the true LOEO spatial holdout CSV\n        write_csv_atomic(\n            loeo_df if len(loeo_df) else reg_sub_df,\n            config.p(\"tables\") / \"regional_generalization_test.csv\",\n        )\n    if len(err_df):\n        write_csv_atomic(err_df, config.p(\"tables\") / \"monthly_error_analysis.csv\")\n\n    report = {\n        \"generated_utc\": now_utc(),\n        \"overall_selected_model_by_validation_pr_auc\": overall_winner,\n        \"model_comparison\": comp_df.to_dict(\"records\"),\n        \"reduced_period_complete_6var_experiment\": ml_res.get(\"reduced_period_complete_6var_experiment\", {}),\n        \"ablation_study\": ab_df.to_dict(\"records\"),\n        \"shap_feature_importance\": shap_rows,\n        \"spatial_holdout_generalization_loeo\": loeo_df.to_dict(\"records\"),\n        \"regional_subgroup_performance\": reg_sub_df.to_dict(\"records\"),\n        \"regional_generalization\": (loeo_df if len(loeo_df) else reg_sub_df).to_dict(\"records\"),\n        \"sensor_sensitivity\": sensor_rows,\n        \"calibration_curves\": calibration_curves,\n        \"monthly_error_analysis\": err_df.to_dict(\"records\"),\n        \"spatiotemporal_block_bootstrap_95ci_test\": block_bootstrap_cis,\n        \"observation_iid_bootstrap_95ci_test\": iid_bootstrap_cis,\n        \"bootstrap_95ci_test\": block_bootstrap_cis,\n    }\n    atomic_write_json(out_json, report)\n    log.info(f\"[Evaluation] Completed full evaluation report -> {out_json} (winner={overall_winner}).\")\n    return report\n","prediction.py":"\"\"\"Genuine 72-hour geographical fire-risk prediction and near-real-time (NRT) monitoring.\n\nScientific distinction of prediction modes (never conflates historical outputs with live forecasts):\n1. `HISTORICAL_RETROSPECTIVE`:\n   - Issue date `D` is more than 7 days in the past (`now_ist - D > 7 days`).\n   - Uses archived ERA5 reanalysis up to day `D` to predict the `[D+1, D+3]` window retrospectively.\n2. `NEAR_REAL_TIME_NRT`:\n   - Issue date `D` is within the last 7 days (`0 <= now_ist - D <= 7 days`, e.g. ERA5T 5-day latency\n     or NRT meteorological feed) combined with NRT NASA FIRMS observations up to `D`.\n3. `FORECAST_PREDICTION`:\n   - Uses operational numerical weather prediction (NWP) forecast inputs covering `[D+1, D+3]`\n     when explicitly provided.\n\nProduces:\n- Calibrated 0.25 deg gridded probability maps (`outputs/predictions/fire_risk_72h_<issue_date>.parquet`,\n  `.geojson`, and `.json` for the interactive website).\n- Ecoregion and district/state-zone risk summaries.\n- NRT active fire monitoring feed (`website/data/nrt_active_fires.geojson`).\n\"\"\"\nfrom __future__ import annotations\n\nimport datetime as _dt\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .landcover import build_or_load_landcover_grid, classify_fire_detections\nfrom .utils import atomic_write_json, get_logger, now_utc, read_json, write_csv_atomic, write_parquet_atomic\n\n\ndef classify_prediction_mode(issue_date_ist: pd.Timestamp, has_nwp_forecast_inputs: bool = False) -> dict:\n    \"\"\"Explicitly classify whether a 72-hour prediction is Historical Retrospective,\n    Near-Real-Time (NRT), or an Operational Weather-Forecast-Driven Prediction.\n    \"\"\"\n    now_ist = pd.Timestamp(_dt.datetime.now(_dt.timezone.utc)).tz_localize(None) + pd.Timedelta(minutes=config.IST_UTC_OFFSET_MINUTES)\n    issue_ts = pd.Timestamp(issue_date_ist).tz_localize(None)\n    age_days = float((now_ist.floor(\"D\") - issue_ts.floor(\"D\")).days)\n\n    if has_nwp_forecast_inputs and age_days <= 2:\n        mode = \"FORECAST_PREDICTION\"\n        desc = (\n            \"Operational 72-hour forecast using current antecedent observations up to issue time D \"\n            \"plus numerical weather forecast inputs for [D+1, D+3].\"\n        )\n    elif age_days <= 7:\n        mode = \"NEAR_REAL_TIME_NRT\"\n        desc = (\n            f\"Near-real-time (NRT) 72-hour projection issued from meteorological & FIRMS state at \"\n            f\"{issue_ts.strftime('%Y-%m-%d')} IST (data latency = {int(age_days)} day(s)).\"\n        )\n    else:\n        mode = \"HISTORICAL_RETROSPECTIVE\"\n        desc = (\n            f\"Historical retrospective 72-hour prediction issued for {issue_ts.strftime('%Y-%m-%d')} IST \"\n            f\"(valid window {(issue_ts + pd.Timedelta(days=1)).strftime('%Y-%m-%d')} to \"\n            f\"{(issue_ts + pd.Timedelta(days=3)).strftime('%Y-%m-%d')} IST) using archived ERA5 reanalysis. \"\n            f\"Not a live current forecast (archive ends {int(age_days)} days before today).\"\n        )\n\n    return {\n        \"prediction_mode\": mode,\n        \"issue_date_ist\": issue_ts.strftime(\"%Y-%m-%d\"),\n        \"valid_window_start_ist\": (issue_ts + pd.Timedelta(days=1)).strftime(\"%Y-%m-%d\"),\n        \"valid_window_end_ist\": (issue_ts + pd.Timedelta(days=3)).strftime(\"%Y-%m-%d\"),\n        \"data_latency_days_from_now\": int(age_days),\n        \"description\": desc,\n    }\n\n\ndef monitor_current_fires_nrt(days: int = 3) -> dict:\n    \"\"\"Fetch near-real-time (NRT) active fire detections over India for the past `days` (1-10)\n    if `FIRMS_MAP_KEY` is configured in Colab Secrets, or summarize the latest available date window\n    in the local processed FIRMS archive, clearly labeling the observation timestamp & latency.\n    \"\"\"\n    from .firms import INDIA_AREA_STR, FIRMS_BASE, _standardize_firms_dataframe, get_secret\n\n    log = get_logger()\n    out_dir = config.p(\"predictions\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n    lc_df = build_or_load_landcover_grid()\n\n    key = get_secret(\"FIRMS_MAP_KEY\")\n    obs_df = pd.DataFrame()\n    feed_mode = \"NO_DATA\"\n\n    if key:\n        try:\n            import io\n            import requests\n            days = max(1, min(10, int(days)))\n            url = f\"{FIRMS_BASE}/api/area/csv/{key}/VIIRS_SNPP_NRT/{INDIA_AREA_STR}/{days}\"\n            r = requests.get(url, timeout=60)\n            if r.status_code == 200 and \"latitude\" in r.text.lower():\n                raw = pd.read_csv(io.StringIO(r.text))\n                obs_df = _standardize_firms_dataframe(raw, \"VIIRS_SNPP_NRT\")\n                feed_mode = \"LIVE_NASA_FIRMS_NRT_API\"\n        except Exception as exc:\n            log.warning(f\"[NRT Monitor] Live NRT query fallback ({exc}); checking archived FIRMS parquet.\")\n\n    if len(obs_df) == 0:\n        att_pq = config.p(\"firms_processed\") / \"firms_india_attributed.parquet\"\n        if att_pq.exists():\n            full_df = pd.read_parquet(att_pq)\n            if len(full_df):\n                max_d = full_df[\"ist_date\"].max()\n                obs_df = full_df[full_df[\"ist_date\"] >= max_d - pd.Timedelta(days=days - 1)].copy()\n                feed_mode = f\"LATEST_ARCHIVED_FIRMS_WINDOW ({ (max_d - pd.Timedelta(days=days - 1)).strftime('%Y-%m-%d') }..{ max_d.strftime('%Y-%m-%d') })\"\n\n    if len(obs_df) and \"fire_category\" not in obs_df.columns:\n        obs_df = classify_fire_detections(obs_df, lc_df)\n\n    # Export GeoJSON of up to 1,500 highest-FRP detections for the interactive Leaflet map\n    features = []\n    if len(obs_df):\n        top_df = obs_df.sort_values(\"frp\", ascending=False).head(1500)\n        for r in top_df.itertuples(index=False):\n            features.append({\n                \"type\": \"Feature\",\n                \"geometry\": {\"type\": \"Point\", \"coordinates\": [round(float(r.longitude), 4), round(float(r.latitude), 4)]},\n                \"properties\": {\n                    \"ist_date\": str(pd.Timestamp(r.ist_date).strftime(\"%Y-%m-%d\")),\n                    \"fire_category\": str(getattr(r, \"fire_category\", \"unknown\")),\n                    \"frp_mw\": round(float(r.frp), 2) if pd.notna(r.frp) else None,\n                    \"confidence_tier\": str(getattr(r, \"confidence_tier\", \"nominal\")),\n                    \"ecoregion\": str(getattr(r, \"ecoregion\", \"\")),\n                },\n            })\n\n    geojson = {\n        \"type\": \"FeatureCollection\",\n        \"generated_utc\": now_utc(),\n        \"feed_mode\": feed_mode,\n        \"total_detections_in_window\": int(len(obs_df)),\n        \"category_breakdown\": obs_df[\"fire_category\"].value_counts().to_dict() if len(obs_df) and \"fire_category\" in obs_df.columns else {},\n        \"features\": features,\n    }\n    atomic_write_json(out_dir / \"active_fire_monitor.geojson\", geojson)\n    atomic_write_json(config.p(\"website_data\") / \"active_fire_monitor.geojson\", geojson)\n    return {\n        \"feed_mode\": feed_mode,\n        \"total_detections_in_window\": int(len(obs_df)),\n        \"category_breakdown\": geojson[\"category_breakdown\"],\n        \"geojson_path\": str(out_dir / \"active_fire_monitor.geojson\"),\n    }\n\n\ndef generate_72h_prediction_maps(issue_date: str | None = None) -> dict:\n    \"\"\"Generate geographical 72-hour forest-fire probability maps over India using both the\n    best classical ML model and the best Spatiotemporal Deep Learning checkpoint, clearly labeled\n    with `classify_prediction_mode()`.\n    \"\"\"\n    import joblib\n    import torch\n    import xarray as xr\n\n    log = get_logger()\n    out_dir = config.p(\"predictions\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    cube_path = config.p(\"supervised\") / \"spatiotemporal_cube.zarr\"\n    if not cube_path.exists():\n        raise FileNotFoundError(\"Spatiotemporal cube not found. Run Section 8 (`build_supervised_72h_dataset`) first.\")\n\n    ds = xr.open_zarr(cube_path)\n    times = pd.DatetimeIndex(ds.time.values)\n    lats = ds.latitude.values.astype(\"float32\")\n    lons = ds.longitude.values.astype(\"float32\")\n    H, W = len(lats), len(lons)\n\n    if issue_date is not None:\n        t_matches = np.where(times == pd.Timestamp(issue_date))[0]\n        t_idx = int(t_matches[0]) if len(t_matches) else (len(times) - 1)\n    else:\n        # Use the most recent timestamp with valid meteorological inputs (or last test timestamp with observed target)\n        t_idx = max(0, len(times) - 4) if len(times) >= 4 else (len(times) - 1)\n\n    issue_ts = pd.Timestamp(times[t_idx])\n    mode_info = classify_prediction_mode(issue_ts, has_nwp_forecast_inputs=False)\n\n    # 1. Classical ML calibrated probability map\n    ml_metrics = read_json(config.p(\"models\") / \"classical_ml_metrics.json\", {}) or {}\n    best_cls = ml_metrics.get(\"selected_best_model_by_validation_pr_auc\") or \"lightgbm\"\n    cls_ckpt = config.p(\"checkpoints\") / \"classical\" / f\"{best_cls}.joblib\"\n    if not cls_ckpt.exists():\n        avail_ckpts = sorted((config.p(\"checkpoints\") / \"classical\").glob(\"*.joblib\"))\n        cls_ckpt = avail_ckpts[0] if avail_ckpts else None\n\n    lc_df = build_or_load_landcover_grid(grid_res=float(abs(lats[1] - lats[0])) if len(lats) > 1 else 0.25)\n    lat_g, lon_g = np.meshgrid(lats, lons, indexing=\"ij\")\n    grid_df = pd.DataFrame({\n        \"latitude\": lat_g.ravel().astype(\"float32\"),\n        \"longitude\": lon_g.ravel().astype(\"float32\"),\n    })\n    grid_df = grid_df.merge(\n        lc_df[[\"latitude\", \"longitude\", \"ecoregion\", \"forest_frac\", \"crop_frac\", \"shrub_grass_frac\", \"is_land\"]],\n        on=[\"latitude\", \"longitude\"],\n        how=\"left\",\n    )\n    doy = float(issue_ts.dayofyear)\n    grid_df[\"sin_doy\"] = np.float32(np.sin(2 * np.pi * doy / 365.25))\n    grid_df[\"cos_doy\"] = np.float32(np.cos(2 * np.pi * doy / 365.25))\n    grid_df[\"month\"] = int(issue_ts.month)\n\n    for vname in ds.data_vars:\n        if \"time\" in ds[vname].dims:\n            grid_df[vname] = ds[vname].isel(time=t_idx).values.ravel().astype(\"float32\")\n\n    if cls_ckpt is not None and cls_ckpt.exists():\n        bundle = joblib.load(cls_ckpt)\n        if isinstance(bundle, dict) and \"pipeline\" in bundle:\n            fcols = bundle[\"features\"]\n            X_grid = grid_df[fcols].to_numpy(dtype=\"float32\")\n            raw_p = bundle[\"pipeline\"].predict_proba(X_grid)[:, 1]\n            cal_p = bundle[\"calibrator\"].predict(raw_p) if bundle.get(\"calibrator\") is not None else raw_p\n            grid_df[\"prob_classical_ml\"] = cal_p.astype(\"float32\")\n        else:\n            grid_df[\"prob_classical_ml\"] = bundle.predict_proba(grid_df)[:, 1].astype(\"float32\")\n    else:\n        grid_df[\"prob_classical_ml\"] = np.nan\n\n    # 2. Spatiotemporal Deep Learning probability map (using best validation checkpoint)\n    dl_metrics = read_json(config.p(\"models\") / \"spatiotemporal_dl_metrics.json\", {}) or {}\n    best_arch = dl_metrics.get(\"selected_best_dl_architecture_by_validation\")\n    dl_prob_flat = np.full(len(grid_df), np.nan, dtype=\"float32\")\n    if best_arch:\n        best_pt = config.p(\"checkpoints\") / \"spatiotemporal_dl\" / best_arch / \"best_checkpoint.pt\"\n        if best_pt.exists():\n            from .models_dl import _load_cube_arrays, build_pytorch_components\n            ckpt = torch.load(best_pt, map_location=\"cpu\", weights_only=False)\n            seq_len = int(ckpt.get(\"seq_len\", 7))\n            cube_data = _load_cube_arrays(seq_len=seq_len)\n            _, _, archs = build_pytorch_components(\n                in_channels=int(ckpt[\"in_channels\"]),\n                hidden_dim=int(ckpt[\"hidden_dim\"]),\n                seq_len=seq_len,\n            )\n            net = archs[best_arch]()\n            net.load_state_dict(ckpt[\"model_state\"])\n            net.eval()\n            t_use = max(seq_len - 1, t_idx)\n            x_seq = torch.from_numpy(cube_data[\"X_norm\"][t_use - seq_len + 1: t_use + 1]).unsqueeze(0)\n            with torch.no_grad():\n                logits = net(x_seq).squeeze(0).numpy()\n                dl_prob_flat = (1.0 / (1.0 + np.exp(-logits))).ravel().astype(\"float32\")\n\n    grid_df[\"prob_spatiotemporal_dl\"] = dl_prob_flat\n    # Mask non-forest / ocean cells so probabilities are only reported on valid forest/woodland domain\n    valid_mask = (grid_df[\"is_land\"].fillna(0) > 0) & (grid_df[\"forest_frac\"].fillna(0.0) >= 0.15)\n    grid_df[\"valid_forest_domain\"] = valid_mask.astype(\"int8\")\n    grid_df[\"prob_ensemble_72h\"] = np.where(\n        valid_mask,\n        np.nanmean(np.column_stack([grid_df[\"prob_classical_ml\"], grid_df[\"prob_spatiotemporal_dl\"]]), axis=1),\n        np.nan,\n    ).astype(\"float32\")\n    grid_df[\"risk_tier\"] = np.select(\n        [\n            ~valid_mask,\n            grid_df[\"prob_ensemble_72h\"] >= 0.50,\n            grid_df[\"prob_ensemble_72h\"] >= 0.25,\n            grid_df[\"prob_ensemble_72h\"] >= 0.10,\n        ],\n        [\"non_forest_or_ocean\", \"extreme_risk\", \"high_risk\", \"moderate_risk\"],\n        default=\"low_risk\",\n    )\n\n    date_tag = issue_ts.strftime(\"%Y-%m-%d\")\n    write_parquet_atomic(grid_df, out_dir / f\"fire_risk_72h_{date_tag}.parquet\")\n    write_parquet_atomic(grid_df, out_dir / \"fire_risk_72h_latest.parquet\")\n\n    # Regional risk summary across India's forest ecoregions\n    forest_only = grid_df[valid_mask].copy()\n    eco_risk = (\n        forest_only.groupby(\"ecoregion\", observed=True)\n        .agg(\n            n_forest_cells=(\"prob_ensemble_72h\", \"size\"),\n            mean_prob_72h=(\"prob_ensemble_72h\", lambda s: round(float(np.nanmean(s)), 4)),\n            max_prob_72h=(\"prob_ensemble_72h\", lambda s: round(float(np.nanmax(s)), 4)),\n            cells_high_or_extreme=(\"risk_tier\", lambda s: int(s.isin([\"high_risk\", \"extreme_risk\"]).sum())),\n            mean_vpd_max_kpa=(\"vpd_max_kpa\", lambda s: round(float(np.nanmean(s)), 3) if \"vpd_max_kpa\" in forest_only else np.nan),\n            mean_ffwi_max=(\"ffwi_max\", lambda s: round(float(np.nanmean(s)), 2) if \"ffwi_max\" in forest_only else np.nan),\n        )\n        .reset_index()\n        .sort_values(\"mean_prob_72h\", ascending=False)\n    )\n    write_csv_atomic(eco_risk, config.p(\"tables\") / \"latest_72h_ecoregion_risk.csv\")\n\n    # Export compact JSON grid for interactive Leaflet map rendering on GitHub Pages\n    web_cells = []\n    for r in forest_only.itertuples(index=False):\n        if pd.notna(r.prob_ensemble_72h):\n            web_cells.append({\n                \"lat\": round(float(r.latitude), 2),\n                \"lon\": round(float(r.longitude), 2),\n                \"p_ens\": round(float(r.prob_ensemble_72h), 4),\n                \"p_ml\": round(float(r.prob_classical_ml), 4) if pd.notna(r.prob_classical_ml) else None,\n                \"p_dl\": round(float(r.prob_spatiotemporal_dl), 4) if pd.notna(r.prob_spatiotemporal_dl) else None,\n                \"tier\": str(r.risk_tier),\n                \"eco\": str(r.ecoregion),\n                \"ff\": round(float(r.forest_frac), 2),\n                \"tmax\": round(float(getattr(r, \"t2m_max_c\", np.nan)), 1) if pd.notna(getattr(r, \"t2m_max_c\", np.nan)) else None,\n                \"vpd\": round(float(getattr(r, \"vpd_max_kpa\", np.nan)), 2) if pd.notna(getattr(r, \"vpd_max_kpa\", np.nan)) else None,\n                \"ffwi\": round(float(getattr(r, \"ffwi_max\", np.nan)), 1) if pd.notna(getattr(r, \"ffwi_max\", np.nan)) else None,\n            })\n\n    pred_payload = {\n        \"generated_utc\": now_utc(),\n        \"mode\": mode_info,\n        \"classical_model_used\": best_cls,\n        \"dl_architecture_used\": best_arch,\n        \"ecoregion_summary\": eco_risk.to_dict(\"records\"),\n        \"grid_cells\": web_cells,\n    }\n    atomic_write_json(out_dir / \"latest_72h_prediction_summary.json\", pred_payload)\n    atomic_write_json(config.p(\"website_data\") / \"latest_72h_prediction.json\", pred_payload)\n    log.info(\n        f\"[Prediction 72h] Mode={mode_info['prediction_mode']} | issue={date_tag} | \"\n        f\"forest_cells={len(web_cells)} -> {out_dir / 'latest_72h_prediction_summary.json'}\"\n    )\n    return pred_payload\n","visualization.py":"\"\"\"Publication-grade scientific visualizations, 9-section interactive research portal export,\nbuild validation, and safe non-force GitHub Pages deployment.\n\nGuarantees:\n1. Every figure and website dataset is generated strictly from actual computed outputs on Google Drive\n   (`metadata/`, `data/processed/`, `models/`, `outputs/evaluation/`, `outputs/predictions/`).\n   Never fabricates example metrics, charts, maps, or placeholder numbers.\n2. When an upstream pipeline stage has not yet been executed in Colab, the website displays an explicit\n   `\"Not yet generated — execute Stage X in Google Colab\"` status rather than invented numbers.\n3. Includes `validate_website_build()` to verify asset completeness, JSON schema integrity, and zero\n   credential leakage before deployment.\n4. `deploy_github_pages_from_colab()` NEVER uses `git push --force` and requires explicit user approval\n   (`allow_push=True`) before pushing to the remote branch.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .utils import atomic_write_json, atomic_write_text, get_logger, now_utc, read_json\n\n\ndef generate_all_scientific_figures() -> dict:\n    \"\"\"Render publication-grade PNG figures into `reports/figures/` and copy web-ready assets\n    into `website/data/` for the GitHub Pages portal.\n    \"\"\"\n    import matplotlib.pyplot as plt\n\n    fig_dir = config.p(\"figures\")\n    fig_dir.mkdir(parents=True, exist_ok=True)\n    generated = {}\n\n    # 1. Forest vs. Agricultural vs. Uncertain Attribution Sensitivity\n    sens_csv = config.p(\"tables\") / \"fire_attribution_sensitivity.csv\"\n    if sens_csv.exists():\n        sdf = pd.read_csv(sens_csv)\n        if len(sdf):\n            fig, ax = plt.subplots(figsize=(8.5, 4.5))\n            x = np.arange(len(sdf))\n            w = 0.26\n            ax.bar(x - w, sdf[\"pct_forest_fires\"], width=w, label=\"Forest-Associated Detections (%)\", color=\"#2ec4b6\")\n            ax.bar(x, sdf[\"pct_agricultural_fires\"], width=w, label=\"Agricultural / Crop-Residue (%)\", color=\"#ff9f1c\")\n            ax.bar(x + w, sdf[\"pct_other_or_uncertain\"], width=w, label=\"Other / Uncertain Anomalies (%)\", color=\"#8d99ae\")\n            ax.set_xticks(x)\n            ax.set_xticklabels(sdf[\"regime\"])\n            ax.set_ylabel(\"Share of Active-Fire Detections (%)\")\n            ax.set_title(\"Sensitivity of Fire Attribution to Land-Cover & Confidence Thresholds\")\n            ax.legend()\n            ax.grid(axis=\"y\", alpha=0.3)\n            fig.tight_layout()\n            out_p = fig_dir / \"fire_attribution_sensitivity.png\"\n            fig.savefig(out_p, dpi=140)\n            plt.close(fig)\n            generated[\"attribution_sensitivity\"] = str(out_p)\n\n    # 2. Model Comparison (Validation vs Independent Holdout Test PR-AUC & Brier Score)\n    comp_csv = config.p(\"tables\") / \"model_comparison.csv\"\n    if comp_csv.exists():\n        cdf = pd.read_csv(comp_csv)\n        if len(cdf) and cdf[\"val_pr_auc\"].notna().any():\n            fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))\n            y_pos = np.arange(len(cdf))\n            axes[0].barh(y_pos - 0.18, cdf[\"val_pr_auc\"].fillna(0), height=0.35, label=\"Validation (2011-2015)\", color=\"#38bdf8\")\n            axes[0].barh(y_pos + 0.18, cdf[\"test_pr_auc\"].fillna(0), height=0.35, label=\"Holdout Test (2016-2025)\", color=\"#f43f5e\")\n            axes[0].set_yticks(y_pos)\n            axes[0].set_yticklabels(cdf[\"model\"])\n            axes[0].set_xlabel(\"PR-AUC / Average Precision (Higher is Better)\")\n            axes[0].set_title(\"72h Forest-Fire Forecasting: PR-AUC\")\n            axes[0].legend()\n            axes[0].grid(axis=\"x\", alpha=0.3)\n\n            axes[1].barh(y_pos - 0.18, cdf[\"val_brier\"].fillna(0), height=0.35, label=\"Validation\", color=\"#38bdf8\")\n            axes[1].barh(y_pos + 0.18, cdf[\"test_brier\"].fillna(0), height=0.35, label=\"Holdout Test\", color=\"#f43f5e\")\n            axes[1].set_yticks(y_pos)\n            axes[1].set_yticklabels(cdf[\"model\"])\n            axes[1].set_xlabel(\"Brier Score (Lower is Better)\")\n            axes[1].set_title(\"Probability Calibration: Brier Score\")\n            axes[1].legend()\n            axes[1].grid(axis=\"x\", alpha=0.3)\n            fig.tight_layout()\n            out_p = fig_dir / \"model_benchmark_comparison.png\"\n            fig.savefig(out_p, dpi=140)\n            plt.close(fig)\n            generated[\"model_comparison\"] = str(out_p)\n\n    # 3. Spatiotemporal Deep Learning Training & Validation Loss Curves\n    dl_json = config.p(\"models\") / \"spatiotemporal_dl_metrics.json\"\n    if dl_json.exists():\n        dl_data = read_json(dl_json, {}) or {}\n        archs = dl_data.get(\"architectures\", {})\n        if archs:\n            fig, ax = plt.subplots(figsize=(9, 4.5))\n            for aname, ainfo in archs.items():\n                hist = ainfo.get(\"training_history\", [])\n                if hist:\n                    ep = [h[\"epoch\"] for h in hist]\n                    ax.plot(ep, [h[\"train_loss\"] for h in hist], linestyle=\"--\", label=f\"{aname} (train)\")\n                    ax.plot(ep, [h[\"val_loss\"] for h in hist], linewidth=2, label=f\"{aname} (val)\")\n            ax.set_xlabel(\"Epoch\")\n            ax.set_ylabel(\"Masked Focal BCE Loss\")\n            ax.set_title(\"Spatiotemporal Deep Learning Training & Validation Curves\")\n            ax.legend()\n            ax.grid(alpha=0.3)\n            fig.tight_layout()\n            out_p = fig_dir / \"dl_training_curves.png\"\n            fig.savefig(out_p, dpi=140)\n            plt.close(fig)\n            generated[\"dl_training_curves\"] = str(out_p)\n\n    # 4. Leave-One-Ecoregion-Out (LOEO) Spatial Holdout Generalisation\n    loeo_csv = config.p(\"tables\") / \"spatial_holdout_generalization.csv\"\n    if loeo_csv.exists():\n        ldf = pd.read_csv(loeo_csv)\n        if len(ldf) and \"loeo_test_pr_auc\" in ldf.columns and ldf[\"loeo_test_pr_auc\"].notna().any():\n            fig, ax = plt.subplots(figsize=(9, 4.4))\n            y_pos = np.arange(len(ldf))\n            ax.barh(y_pos, ldf[\"loeo_test_pr_auc\"].fillna(0), color=\"#a855f7\", height=0.5)\n            ax.set_yticks(y_pos)\n            ax.set_yticklabels(ldf[\"held_out_ecoregion\"])\n            ax.set_xlabel(\"Holdout Test PR-AUC in Unseen Ecoregion\")\n            ax.set_title(\"Leave-One-Ecoregion-Out (LOEO) Spatial Generalisation\")\n            ax.grid(axis=\"x\", alpha=0.3)\n            fig.tight_layout()\n            out_p = fig_dir / \"spatial_holdout_loeo.png\"\n            fig.savefig(out_p, dpi=140)\n            plt.close(fig)\n            generated[\"spatial_holdout_loeo\"] = str(out_p)\n\n    # 5. Latest 72-Hour Spatial Probability Map\n    pred_pq = config.p(\"predictions\") / \"fire_risk_72h_latest.parquet\"\n    if pred_pq.exists():\n        pdf = pd.read_parquet(pred_pq)\n        forest_sub = pdf[pdf[\"valid_forest_domain\"] > 0]\n        if len(forest_sub):\n            fig, ax = plt.subplots(figsize=(7.5, 7.5))\n            sc = ax.scatter(\n                forest_sub[\"longitude\"], forest_sub[\"latitude\"],\n                c=forest_sub[\"prob_ensemble_72h\"], cmap=\"YlOrRd\", vmin=0.0, vmax=1.0, s=14, marker=\"s\",\n            )\n            fig.colorbar(sc, ax=ax, label=\"72-Hour Forest-Associated Fire Probability\")\n            ax.set_xlim(config.INDIA_BBOX[\"lon_min\"], config.INDIA_BBOX[\"lon_max\"])\n            ax.set_ylim(config.INDIA_BBOX[\"lat_min\"], config.INDIA_BBOX[\"lat_max\"])\n            ax.set_xlabel(\"Longitude (°E)\")\n            ax.set_ylabel(\"Latitude (°N)\")\n            ax.set_title(\"Calibrated 72-Hour Forest-Fire Risk Map (Valid Forest Domain)\")\n            ax.grid(alpha=0.25)\n            fig.tight_layout()\n            out_p = fig_dir / \"latest_72h_fire_risk_map.png\"\n            fig.savefig(out_p, dpi=140)\n            plt.close(fig)\n            generated[\"latest_72h_fire_risk_map\"] = str(out_p)\n\n    return generated\n\n\ndef export_website_bundle() -> dict:\n    \"\"\"Export actual project outputs into `website/data/` and build the 9-section (`A` through `I`)\n    interactive scientific research portal in `website/index.html`.\n    \"\"\"\n    web_dir = config.p(\"website\")\n    data_dir = config.p(\"website_data\")\n    web_dir.mkdir(parents=True, exist_ok=True)\n    data_dir.mkdir(parents=True, exist_ok=True)\n\n    diag = read_json(config.p(\"diagnostics\") / \"diag_latest.json\", {}) or {}\n    cov = read_json(config.p(\"coverage\") / \"coverage_summary.json\", {}) or {}\n    firms_sum = read_json(config.p(\"firms_processed\") / \"firms_ingestion_summary.json\", {}) or {}\n    firms_lim = read_json(config.p(\"metadata\") / \"firms_limitations.json\", {}) or {}\n    lc_prov = read_json(config.p(\"landcover\") / \"landcover_provenance.json\", {}) or {}\n    base_meta = read_json(config.p(\"climate_baseline\") / \"baseline_metadata.json\", {}) or {}\n    sup_meta = read_json(config.p(\"supervised\") / \"supervised_dataset_summary.json\", {}) or {}\n    ml_metrics = read_json(config.p(\"models\") / \"classical_ml_metrics.json\", {}) or {}\n    dl_metrics = read_json(config.p(\"models\") / \"spatiotemporal_dl_metrics.json\", {}) or {}\n    eval_rep = read_json(config.p(\"evaluation\") / \"full_evaluation_report.json\", {}) or {}\n    pred_rep = read_json(config.p(\"predictions\") / \"latest_72h_prediction_summary.json\", {}) or {}\n    vsuite = read_json(config.p(\"reports\") / \"validation_suite_results.json\", {}) or {}\n\n    def read_csv_records(path: Path) -> list[dict]:\n        if path.exists():\n            try:\n                return pd.read_csv(path).replace({np.nan: None}).to_dict(\"records\")\n            except Exception:\n                return []\n        return []\n\n    bundle = {\n        \"generated_utc\": now_utc(),\n        \"project\": config.PROJECT_NAME,\n        \"title\": \"India Fire & Climate Intelligence: Machine Learning-Based Forest Fire Risk Prediction and Climate Intelligence for India\",\n        \"institution\": \"IIT Gandhinagar — Final-Year B.Tech Machine Learning Research Project\",\n        \"study_bbox\": config.INDIA_BBOX,\n        \"local_timezone\": config.LOCAL_TIMEZONE,\n        \"research_periods_configured\": config.RESEARCH_PERIODS,\n        \"diagnostics\": diag.get(\"diagnostics\", {}),\n        \"strategy\": diag.get(\"strategy\", {}),\n        \"era5_coverage\": cov,\n        \"era5_research_periods\": read_csv_records(config.p(\"coverage\") / \"research_period_coverage.csv\"),\n        \"era5_variable_coverage\": read_csv_records(config.p(\"coverage\") / \"variable_coverage.csv\"),\n        \"climate_baseline_metadata\": base_meta,\n        \"annual_climate_trends\": read_csv_records(config.p(\"climate_baseline\") / \"annual_climate_trends.csv\"),\n        \"firms_summary\": firms_sum,\n        \"firms_limitations\": firms_lim,\n        \"landcover_provenance\": lc_prov,\n        \"attribution_sensitivity\": read_csv_records(config.p(\"tables\") / \"fire_attribution_sensitivity.csv\"),\n        \"attribution_by_ecoregion\": read_csv_records(config.p(\"tables\") / \"fire_attribution_by_ecoregion.csv\"),\n        \"sensor_sensitivity\": read_csv_records(config.p(\"tables\") / \"sensor_sensitivity_analysis.csv\"),\n        \"monthly_fire_seasonality\": read_csv_records(config.p(\"tables\") / \"eda_monthly_fire_seasonality.csv\"),\n        \"supervised_summary\": sup_meta,\n        \"classical_ml\": ml_metrics,\n        \"spatiotemporal_dl\": dl_metrics,\n        \"evaluation\": eval_rep,\n        \"feature_importance\": read_csv_records(config.p(\"tables\") / \"classical_feature_importance.csv\"),\n        \"shap_importance\": read_csv_records(config.p(\"tables\") / \"shap_feature_importance.csv\"),\n        \"spatial_holdout_loeo\": read_csv_records(config.p(\"tables\") / \"spatial_holdout_generalization.csv\"),\n        \"regional_subgroup\": read_csv_records(config.p(\"tables\") / \"regional_subgroup_performance.csv\"),\n        \"validation_suite\": vsuite,\n        \"latest_prediction_meta\": {\n            \"mode\": pred_rep.get(\"mode\", {}),\n            \"classical_model_used\": pred_rep.get(\"classical_model_used\"),\n            \"dl_architecture_used\": pred_rep.get(\"dl_architecture_used\"),\n            \"ecoregion_summary\": pred_rep.get(\"ecoregion_summary\", []),\n        },\n    }\n    atomic_write_json(data_dir / \"project_summary.json\", bundle)\n\n    index_html = \"\"\"<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n  <meta charset=\"utf-8\" />\n  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />\n  <meta name=\"description\" content=\"India Fire & Climate Intelligence (IFCI) — Machine Learning-Based 72-Hour Forest Fire Risk Prediction and Historical Climate Intelligence for India (IIT Gandhinagar B.Tech Project).\" />\n  <title>India Fire &amp; Climate Intelligence (IFCI) — Scientific Research Portal</title>\n  <link rel=\"stylesheet\" href=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.css\" />\n  <script src=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.js\"></script>\n  <script src=\"https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js\"></script>\n  <style>\n    :root {\n      --bg:#070b14; --panel:#0f172a; --panel2:#162038; --panel3:#1e293b;\n      --border:#263554; --text:#f1f5f9; --muted:#94a3b8;\n      --accent:#f97316; --teal:#14b8a6; --sky:#38bdf8; --rose:#f43f5e; --purple:#a855f7; --gold:#eab308;\n    }\n    * { box-sizing:border-box; }\n    html { scroll-behavior:smooth; }\n    body { margin:0; font-family:Inter,system-ui,-apple-system,Segoe UI,Roboto,sans-serif; background:var(--bg); color:var(--text); line-height:1.58; }\n    header { background:linear-gradient(135deg,#090f1d 0%,#111c35 60%,#172544 100%); padding:1.4rem 2rem; border-bottom:1px solid var(--border); position:sticky; top:0; z-index:1100; display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:1rem; backdrop-filter:blur(10px); }\n    .brand-title { margin:0; font-size:1.3rem; font-weight:700; letter-spacing:-0.01em; color:#fff; }\n    .brand-sub { color:var(--muted); font-size:0.82rem; margin-top:0.2rem; }\n    .badge { background:var(--panel2); border:1px solid var(--border); padding:0.35rem 0.85rem; border-radius:999px; font-size:0.78rem; color:var(--teal); font-weight:600; }\n    nav { background:#0b1120; border-bottom:1px solid var(--border); padding:0.5rem 2rem; display:flex; flex-wrap:wrap; gap:0.45rem; position:sticky; top:73px; z-index:1050; }\n    nav a { color:var(--muted); text-decoration:none; padding:0.32rem 0.68rem; border-radius:6px; font-size:0.8rem; font-weight:500; transition:all 0.15s ease; }\n    nav a:hover { color:#fff; background:var(--panel2); }\n    .container { max-width:1440px; margin:0 auto; padding:1.4rem 1.8rem 3rem; display:flex; flex-direction:column; gap:1.6rem; }\n    section { background:var(--panel); border:1px solid var(--border); border-radius:14px; padding:1.5rem; box-shadow:0 8px 28px rgba(0,0,0,0.32); }\n    section h2 { margin-top:0; font-size:1.18rem; color:#fff; border-bottom:1px solid var(--border); padding-bottom:0.55rem; display:flex; align-items:center; justify-content:space-between; flex-wrap:wrap; gap:0.5rem; }\n    section h2 span.sec-tag { font-size:0.74rem; color:var(--accent); text-transform:uppercase; letter-spacing:0.08em; background:rgba(249,115,22,0.12); padding:0.2rem 0.55rem; border-radius:5px; }\n    h3 { font-size:0.96rem; color:#e2e8f0; margin:1rem 0 0.45rem; }\n    p { margin:0.4rem 0 0.8rem; color:#cbd5e1; font-size:0.91rem; }\n    .hero-quote { font-size:1.15rem; font-weight:600; color:#fff; border-left:4px solid var(--accent); padding:0.55rem 1rem; background:rgba(249,115,22,0.08); border-radius:0 8px 8px 0; margin:0.6rem 0 1rem; }\n    .grid-2 { display:grid; grid-template-columns:repeat(auto-fit,minmax(430px,1fr)); gap:1.25rem; }\n    .grid-3 { display:grid; grid-template-columns:repeat(auto-fit,minmax(300px,1fr)); gap:1rem; }\n    .grid-4 { display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr)); gap:0.9rem; }\n    .stat { background:var(--panel2); border:1px solid var(--border); border-radius:10px; padding:0.95rem 1.05rem; }\n    .stat .label { color:var(--muted); font-size:0.75rem; text-transform:uppercase; letter-spacing:0.05em; }\n    .stat .val { font-size:1.32rem; font-weight:700; margin-top:0.25rem; color:var(--teal); }\n    .stat .meta { font-size:0.75rem; color:var(--muted); margin-top:0.2rem; }\n    #map-main { height:520px; border-radius:10px; border:1px solid var(--border); }\n    .map-layout { display:grid; grid-template-columns:1fr 340px; gap:1.1rem; }\n    @media (max-width: 980px) { .map-layout { grid-template-columns:1fr; } .grid-2 { grid-template-columns:1fr; } nav { position:static; } }\n    .inspector { background:var(--panel2); border:1px solid var(--border); border-radius:10px; padding:1rem; display:flex; flex-direction:column; gap:0.65rem; font-size:0.84rem; }\n    .toolbar { display:flex; flex-wrap:wrap; gap:0.65rem; align-items:center; margin-bottom:0.85rem; background:var(--panel2); padding:0.65rem 0.9rem; border-radius:9px; border:1px solid var(--border); }\n    .toolbar label { font-size:0.8rem; color:var(--muted); display:flex; align-items:center; gap:0.35rem; }\n    select, button { background:var(--panel3); color:var(--text); border:1px solid var(--border); border-radius:6px; padding:0.35rem 0.65rem; font-size:0.8rem; cursor:pointer; }\n    button:hover { border-color:var(--accent); color:#fff; }\n    table { width:100%; border-collapse:collapse; font-size:0.81rem; margin-top:0.45rem; }\n    th, td { border:1px solid var(--border); padding:0.48rem 0.6rem; text-align:left; }\n    th { background:var(--panel2); color:#e2e8f0; font-weight:600; }\n    tr:nth-child(even) td { background:rgba(255,255,255,0.015); }\n    pre { background:#050811; border:1px solid var(--border); padding:0.85rem; border-radius:8px; overflow:auto; font-size:0.78rem; color:#cbd5e1; margin:0.4rem 0; }\n    .banner-warn { background:rgba(234,179,8,0.1); border-left:4px solid var(--gold); padding:0.8rem 1rem; border-radius:6px; font-size:0.85rem; color:#fef08a; margin:0.6rem 0; }\n    .banner-info { background:rgba(56,189,248,0.1); border-left:4px solid var(--sky); padding:0.8rem 1rem; border-radius:6px; font-size:0.85rem; color:#bae6fd; margin:0.6rem 0; }\n    .empty-state { background:rgba(148,163,184,0.07); border:1px dashed var(--border); border-radius:8px; padding:1rem; color:var(--muted); font-size:0.83rem; text-align:center; }\n    .legend-bar { display:flex; gap:0.8rem; flex-wrap:wrap; font-size:0.77rem; color:var(--muted); margin-top:0.45rem; }\n    .dot { width:10px; height:10px; border-radius:50%; display:inline-block; margin-right:4px; }\n    canvas { max-height:310px; }\n  </style>\n</head>\n<body>\n  <header>\n    <div>\n      <h1 class=\"brand-title\">India Fire &amp; Climate Intelligence (IFCI)</h1>\n      <div class=\"brand-sub\">Machine Learning-Based 72-Hour Forest Fire Risk Prediction &amp; Climate Intelligence for India • IIT Gandhinagar B.Tech Research Project</div>\n    </div>\n    <div class=\"badge\" id=\"header-badge\">Loading verified telemetry…</div>\n  </header>\n\n  <nav>\n    <a href=\"#sec-a\">A. Home &amp; Overview</a>\n    <a href=\"#sec-b\">B. Interactive Fire Map</a>\n    <a href=\"#sec-c\">C. Climate Intelligence</a>\n    <a href=\"#sec-d\">D. Fire Activity Explorer</a>\n    <a href=\"#sec-e\">E. Model Laboratory</a>\n    <a href=\"#sec-f\">F. 72h Risk Intelligence</a>\n    <a href=\"#sec-g\">G. Methodology</a>\n    <a href=\"#sec-h\">H. Reproducibility</a>\n    <a href=\"#sec-i\">I. About Project</a>\n  </nav>\n\n  <div class=\"container\">\n    <!-- SECTION A: HOME -->\n    <section id=\"sec-a\">\n      <h2><span>A. Research Overview &amp; Chronological Design</span><span class=\"sec-tag\">Section A • Home</span></h2>\n      <div class=\"hero-quote\">“Understanding India's forest fire risk through climate intelligence and machine learning.”</div>\n      <p>This research platform investigates whether antecedent meteorological drivers, cumulative drought memory, fractional land-cover characteristics, seasonality, and trailing fire activity can predict the probability of a <strong>forest-associated active-fire detection</strong> inside a 0.25° × 0.25° Indian grid cell over the <strong>next 72 hours (<code>[D+1, D+2, D+3]</code> IST)</strong>. Every table, chart, and map on this site is rendered deterministically from verified Google Colab execution artefacts.</p>\n      <div class=\"grid-4\" id=\"overview-kpis\"></div>\n      <h3>Fixed Chronological Research Periods &amp; Verified Archive Coverage</h3>\n      <div class=\"banner-info\"><strong>Temporal Leakage Safeguard:</strong> <code>1947–2000</code> is used exclusively for historical climate baseline and decadal trend analysis—never as supervised fire training data. A 3-day embargo buffer separates <code>2001–2010</code> (Train), <code>2011–2015</code> (Validation &amp; Calibration), and <code>2016–2025</code> (Independent Final Holdout Test).</div>\n      <div id=\"table-research-periods\"></div>\n    </section>\n\n    <!-- SECTION B: INTERACTIVE INDIA FIRE MAP -->\n    <section id=\"sec-b\">\n      <h2><span>B. Interactive India Geospatial Fire &amp; Risk Explorer (0.25° Grid)</span><span class=\"sec-tag\">Section B • Geospatial Intelligence</span></h2>\n      <div class=\"toolbar\">\n        <label>Grid Layer:\n          <select id=\"sel-grid-layer\">\n            <option value=\"p_ens\">72h Calibrated Ensemble Probability</option>\n            <option value=\"p_cls\">72h Classical ML Probability</option>\n            <option value=\"p_dl\">72h Spatiotemporal DL Probability</option>\n            <option value=\"ff\">Forest Cover Fraction (0.25°)</option>\n            <option value=\"vpd\">Peak Vapour Pressure Deficit (kPa)</option>\n            <option value=\"ffwi\">Fosberg Fire Weather Index (FFWI)</option>\n          </select>\n        </label>\n        <label>FIRMS Overlay:\n          <select id=\"sel-fire-cat\">\n            <option value=\"ALL\">All Active-Fire Detections</option>\n            <option value=\"forest_fire\">Forest-Associated Detections Only</option>\n            <option value=\"agricultural_fire\">Agricultural / Crop-Residue Only</option>\n            <option value=\"other_or_uncertain\">Other / Uncertain Anomalies Only</option>\n            <option value=\"NONE\">Hide Detection Points</option>\n          </select>\n        </label>\n        <label>Min Risk Filter:\n          <select id=\"sel-min-prob\">\n            <option value=\"0\">Show All Valid Forest Cells (&ge; 0.00)</option>\n            <option value=\"0.10\">Moderate+ Risk (&ge; 0.10)</option>\n            <option value=\"0.25\">High+ Risk (&ge; 0.25)</option>\n            <option value=\"0.50\">Extreme Risk (&ge; 0.50)</option>\n          </select>\n        </label>\n      </div>\n      <div class=\"map-layout\">\n        <div>\n          <div id=\"map-main\"></div>\n          <div class=\"legend-bar\">\n            <span><i class=\"dot\" style=\"background:#ef4444\"></i>Extreme Risk (&ge;0.50)</span>\n            <span><i class=\"dot\" style=\"background:#f97316\"></i>High Risk (0.25–0.50)</span>\n            <span><i class=\"dot\" style=\"background:#eab308\"></i>Moderate Risk (0.10–0.25)</span>\n            <span><i class=\"dot\" style=\"background:#14b8a6\"></i>Low Risk (&lt;0.10)</span>\n            <span>| FIRMS Points: <i class=\"dot\" style=\"background:#2ec4b6\"></i>Forest-Associated <i class=\"dot\" style=\"background:#ff9f1c\"></i>Agricultural <i class=\"dot\" style=\"background:#94a3b8\"></i>Uncertain</span>\n          </div>\n        </div>\n        <div class=\"inspector\" id=\"cell-inspector\">\n          <h3 style=\"margin-top:0;color:var(--accent);\">0.25° Grid Cell Inspector</h3>\n          <div style=\"color:var(--muted);\">Click any 0.25° grid cell or active-fire detection on the map to inspect its calibrated 72-hour probability, land-cover fractions, ecoregion, and antecedent fire-weather drivers.</div>\n        </div>\n      </div>\n    </section>\n\n    <!-- SECTION C: CLIMATE INTELLIGENCE -->\n    <section id=\"sec-c\">\n      <h2><span>C. Long-Term Indian Climate Intelligence (1947–Present Reanalysis)</span><span class=\"sec-tag\">Section C • Climate Baseline</span></h2>\n      <div class=\"banner-warn\"><strong>Scientific Distinction (Descriptive vs. Causal):</strong> The multi-decadal ERA5 reanalysis trends below quantify descriptive shifts in regional temperature, atmospheric moisture demand (VPD), solar radiation, and precipitation across India relative to the <code>1947–2000</code> baseline. They are not formal counterfactual climate-change attribution claims.</div>\n      <div class=\"grid-2\">\n        <div>\n          <h3>Annual Domain-Mean 2m Temperature &amp; Dewpoint Trends (°C)</h3>\n          <canvas id=\"chart-climate-temp\"></canvas>\n        </div>\n        <div>\n          <h3>Annual Precipitation (mm/day) &amp; Surface Solar Radiation (MJ/m²/day)</h3>\n          <canvas id=\"chart-climate-precip\"></canvas>\n        </div>\n      </div>\n      <h3>Per-Variable ERA5 Archive Completeness &amp; Verified Time Spans</h3>\n      <div id=\"table-era5-vars\"></div>\n    </section>\n\n    <!-- SECTION D: FIRE ACTIVITY EXPLORER -->\n    <section id=\"sec-d\">\n      <h2><span>D. Satellite Active-Fire Activity &amp; Land-Cover Attribution Explorer</span><span class=\"sec-tag\">Section D • NASA FIRMS &amp; Attribution</span></h2>\n      <p>Satellite thermal anomalies are attributed into three mutually exclusive classes: (1) <strong>Forest-Associated Active-Fire Detections (<code>forest_fire</code>)</strong>, (2) <strong>Agricultural / Crop-Residue Detections (<code>agricultural_fire</code>)</strong>, and (3) <strong>Other or Uncertain Thermal Anomalies (<code>other_or_uncertain</code>)</strong>. Uncertain detections and persistent industrial flares are excluded from the positive supervised forest-fire label.</p>\n      <div class=\"toolbar\">\n        <span style=\"font-size:0.8rem;color:var(--muted);\" id=\"landcover-prov-badge\">Land-Cover Provenance: checking…</span>\n        <button onclick=\"downloadCSV('attribution_sensitivity')\">Download Attribution Sensitivity CSV</button>\n        <button onclick=\"downloadCSV('attribution_by_ecoregion')\">Download Ecoregion Summary CSV</button>\n      </div>\n      <div class=\"grid-2\">\n        <div>\n          <h3>Attribution Sensitivity Across Threshold Regimes (Strict / Baseline / Inclusive)</h3>\n          <div id=\"table-attr-sensitivity\"></div>\n        </div>\n        <div>\n          <h3>Sensor-Wise Active-Fire Statistics (MODIS vs. VIIRS)</h3>\n          <div id=\"table-sensor-sens\"></div>\n        </div>\n      </div>\n      <h3>Regional Distribution: Forest-Associated vs. Agricultural vs. Uncertain Detections by Ecoregion</h3>\n      <div id=\"table-attr-ecoregion\"></div>\n    </section>\n\n    <!-- SECTION E: MODEL LABORATORY -->\n    <section id=\"sec-e\">\n      <h2><span>E. Machine Learning &amp; Spatiotemporal Deep Learning Laboratory</span><span class=\"sec-tag\">Section E • Model Benchmarks &amp; Ablations</span></h2>\n      <p>Every candidate model is evaluated against <strong>Seasonal Climatology</strong> and <strong>7-Day Persistence</strong> baselines using <strong>PR-AUC (Average Precision)</strong> as the primary metric under class imbalance, alongside ROC-AUC, F1, Precision, Recall, Brier Score, Log-Loss, and Expected Calibration Error (ECE).</p>\n      <div id=\"reduced-track-banner\"></div>\n      <h3>Comprehensive Model Comparison Table (Validation 2011–2015 vs. Independent Holdout Test 2016–2025)</h3>\n      <div id=\"table-model-comparison\"></div>\n      <div class=\"grid-2\" style=\"margin-top:1rem;\">\n        <div>\n          <h3>Validation vs. Holdout Test PR-AUC by Architecture</h3>\n          <canvas id=\"chart-model-prauc\"></canvas>\n        </div>\n        <div>\n          <h3>Precision–Recall &amp; ROC Curves on Independent Holdout Test</h3>\n          <canvas id=\"chart-pr-curve\"></canvas>\n        </div>\n      </div>\n      <div class=\"grid-2\" style=\"margin-top:1rem;\">\n        <div>\n          <h3>Reliability Diagram (Probability Calibration on Holdout Test)</h3>\n          <canvas id=\"chart-calibration\"></canvas>\n        </div>\n        <div>\n          <h3>Spatiotemporal Deep Learning Training &amp; Validation Loss History</h3>\n          <canvas id=\"chart-dl-history\"></canvas>\n        </div>\n      </div>\n      <div class=\"grid-2\" style=\"margin-top:1rem;\">\n        <div>\n          <h3>SHAP / Feature Importance Ranking</h3>\n          <canvas id=\"chart-feature-imp\"></canvas>\n        </div>\n        <div>\n          <h3>Systematic Feature-Group Ablation Study</h3>\n          <div id=\"table-ablation\"></div>\n        </div>\n      </div>\n      <div class=\"grid-2\" style=\"margin-top:1rem;\">\n        <div>\n          <h3>True Spatial Generalisation: Leave-One-Ecoregion-Out (LOEO) Spatial Holdout</h3>\n          <p style=\"font-size:0.8rem;\">Each row holds out one entire ecoregion from training and removes <code>latitude</code>/<code>longitude</code> features to test physical transferability to unseen geography.</p>\n          <div id=\"table-spatial-loeo\"></div>\n        </div>\n        <div>\n          <h3>Regional Subgroup Stratification (All-India Trained Model on Holdout Test)</h3>\n          <p style=\"font-size:0.8rem;\">Stratifies holdout test predictions of the all-India model by ecoregion (reported separately from LOEO spatial generalisation).</p>\n          <div id=\"table-regional-sub\"></div>\n        </div>\n      </div>\n    </section>\n\n    <!-- SECTION F: 72-HOUR RISK INTELLIGENCE -->\n    <section id=\"sec-f\">\n      <h2><span>F. 72-Hour Forest-Fire Risk Intelligence &amp; Operational Mode Disclosure</span><span class=\"sec-tag\">Section F • 72h Forecast &amp; NRT</span></h2>\n      <div class=\"banner-warn\" id=\"prediction-mode-banner\">Checking prediction mode and meteorological input timestamps…</div>\n      <div class=\"grid-2\">\n        <div>\n          <h3>Ecoregion 72-Hour Calibrated Probability &amp; Risk Summary</h3>\n          <div id=\"table-ecoregion-risk\"></div>\n        </div>\n        <div>\n          <h3>Monthly False-Positive / False-Negative Error Profile on Holdout</h3>\n          <div id=\"table-monthly-error\"></div>\n        </div>\n      </div>\n    </section>\n\n    <!-- SECTION G: RESEARCH METHODOLOGY -->\n    <section id=\"sec-g\">\n      <h2><span>G. Scientific &amp; Mathematical Methodology</span><span class=\"sec-tag\">Section G • Equations &amp; Architecture</span></h2>\n      <div class=\"grid-3\">\n        <div class=\"stat\">\n          <div class=\"label\">1. ERA5 IST Aggregation &amp; Flux De-Accumulation</div>\n          <p style=\"font-size:0.82rem;margin-top:0.4rem;\">Instantaneous fields (<code>t2m, d2m, u10, v10</code>) and 1-hour accumulated fluxes (<code>tp, ssrd</code>) are aligned onto the canonical 0.25° India grid and aggregated over 24-hour <strong>Indian Standard Time (UTC+05:30)</strong> days. Wind speed is computed hourly as $\\\\sqrt{u_{10}^2 + v_{10}^2}$ before daily mean/max reduction. Missing variables are never zero-filled.</p>\n        </div>\n        <div class=\"stat\">\n          <div class=\"label\">2. Fire-Weather &amp; Drought Memory Equations</div>\n          <p style=\"font-size:0.82rem;margin-top:0.4rem;\">Saturation vapour pressure uses Tetens' equation $e_s(T)=0.61078\\\\exp(\\\\frac{17.27T}{T+237.3})\\\\text{ kPa}$, yielding afternoon $\\\\text{VPD}_{\\\\max} = \\\\max(0, e_s(T_{\\\\max})-e_a)$ and $\\\\text{RH}_{\\\\min}$. Fosberg Fire Weather Index ($\\\\text{FFWI}_{\\\\max}$), rolling 3/7/14/30-day rainfall, dry-spell duration, and KBDI moisture deficit proxy use strictly causal observations $\\\\le D$.</p>\n        </div>\n        <div class=\"stat\">\n          <div class=\"label\">3. Leak-Free 72-Hour Target &amp; Split Embargo</div>\n          <p style=\"font-size:0.82rem;margin-top:0.4rem;\">For issue day $D$ (23:59 IST), $y_{i,D}=1$ iff $\\\\ge 1$ forest-associated active-fire detection occurs in $[D+1, D+2, D+3]$. Future dates outside verified FIRMS coverage or across calendar gaps are masked as <code>NaN</code> (never negative <code>0</code>). A 3-day embargo separates Train, Validation, and Holdout Test.</p>\n        </div>\n      </div>\n    </section>\n\n    <!-- SECTION H: REPRODUCIBILITY -->\n    <section id=\"sec-h\">\n      <h2><span>H. Reproducibility, Uncertainty (Spatial-Temporal Block Bootstrap) &amp; Limitations</span><span class=\"sec-tag\">Section H • Reproducibility</span></h2>\n      <div class=\"grid-2\">\n        <div>\n          <h3>Spatial-Temporal Block Bootstrap 95% CIs (2°×2°×7-Day Clusters)</h3>\n          <pre id=\"box-bootstrap\">Loading spatial-temporal block bootstrap CIs…</pre>\n        </div>\n        <div>\n          <h3>Documented Satellite Active-Fire &amp; Reanalysis Limitations</h3>\n          <pre id=\"box-limitations\">Loading documented limitations…</pre>\n        </div>\n      </div>\n      <h3>Automated Scientific Unit &amp; Integration Verification Suite (<code>UT-01</code>–<code>UT-05</code>)</h3>\n      <div id=\"table-vsuite\"></div>\n    </section>\n\n    <!-- SECTION I: ABOUT -->\n    <section id=\"sec-i\">\n      <h2><span>I. About the Project</span><span class=\"sec-tag\">Section I • Academic Context</span></h2>\n      <p><strong>Project Title:</strong> India Fire &amp; Climate Intelligence: Machine Learning-Based Forest Fire Risk Prediction and Climate Intelligence for India</p>\n      <p><strong>Academic Context:</strong> Final-Year B.Tech Machine Learning Project, Indian Institute of Technology (IIT) Gandhinagar.</p>\n      <p><strong>Core Design Principles:</strong> 100% Google Colab Pro execution • Read-only preservation of raw ERA5 archives • Explicit separation of forest-associated detections from agricultural crop-residue burning and industrial flares • Leak-free chronological validation with 3-day boundary embargoes • Honest reporting of archive gaps and retrospective vs. operational prediction modes.</p>\n    </section>\n  </div>\n\n  <script>\n    let GLOBAL_SUMMARY = {};\n    let GLOBAL_PRED = {};\n    let GLOBAL_FIRES = {};\n\n    function renderTable(containerId, rows, maxRows = 30) {\n      const el = document.getElementById(containerId);\n      if (!el) return;\n      if (!rows || !rows.length) {\n        el.innerHTML = '<div class=\"empty-state\">Not yet generated — execute the corresponding pipeline stage in Google Colab to populate this artefact.</div>';\n        return;\n      }\n      const cols = Object.keys(rows[0]);\n      let html = '<div style=\"overflow-x:auto;\"><table><thead><tr>' + cols.map(c => '<th>' + c + '</th>').join('') + '</tr></thead><tbody>';\n      rows.slice(0, maxRows).forEach(r => {\n        html += '<tr>' + cols.map(c => '<td>' + (r[c] === null || r[c] === undefined ? '—' : r[c]) + '</td>').join('') + '</tr>';\n      });\n      html += '</tbody></table></div>';\n      el.innerHTML = html;\n    }\n\n    function downloadCSV(key) {\n      const rows = GLOBAL_SUMMARY[key] || [];\n      if (!rows.length) { alert('Dataset not yet generated.'); return; }\n      const cols = Object.keys(rows[0]);\n      const csv = [cols.join(',')].concat(rows.map(r => cols.map(c => JSON.stringify(r[c] ?? '')).join(','))).join('\\\\n');\n      const blob = new Blob([csv], { type: 'text/csv' });\n      const a = document.createElement('a');\n      a.href = URL.createObjectURL(blob);\n      a.download = key + '.csv';\n      a.click();\n    }\n\n    const mapMain = L.map('map-main').setView([22.5, 80.0], 5);\n    L.tileLayer('https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png', { attribution: '&copy; CARTO' }).addTo(mapMain);\n    const gridLayerGroup = L.layerGroup().addTo(mapMain);\n    const fireLayerGroup = L.layerGroup().addTo(mapMain);\n\n    function probColor(p) {\n      if (p === null || p === undefined || isNaN(p)) return '#475569';\n      if (p >= 0.50) return '#ef4444';\n      if (p >= 0.25) return '#f97316';\n      if (p >= 0.10) return '#eab308';\n      return '#14b8a6';\n    }\n\n    function renderMapLayers() {\n      gridLayerGroup.clearLayers();\n      fireLayerGroup.clearLayers();\n      const field = document.getElementById('sel-grid-layer').value;\n      const fireCat = document.getElementById('sel-fire-cat').value;\n      const minProb = parseFloat(document.getElementById('sel-min-prob').value || '0');\n\n      const cells = (GLOBAL_PRED && GLOBAL_PRED.grid_cells) || [];\n      cells.forEach(c => {\n        if ((c.p_ens || 0) < minProb) return;\n        const val = c[field] ?? c.p_ens ?? 0;\n        const norm = (field === 'vpd') ? Math.min(1, val / 5.0) : ((field === 'ffwi') ? Math.min(1, val / 60.0) : val);\n        const col = probColor(norm);\n        const marker = L.circleMarker([c.lat, c.lon], {\n          radius: 4.2, color: col, fillColor: col, fillOpacity: 0.8, weight: 0\n        });\n        marker.on('click', () => {\n          document.getElementById('cell-inspector').innerHTML =\n            '<h3 style=\"margin-top:0;color:var(--accent);\">Grid Cell (' + c.lat.toFixed(2) + '°N, ' + c.lon.toFixed(2) + '°E)</h3>' +\n            '<div><b>Ecoregion:</b> ' + (c.eco || '—') + '</div>' +\n            '<div><b>72h Ensemble Prob:</b> <span style=\"color:var(--teal);font-weight:700;\">' + (c.p_ens ?? '—') + '</span> (' + (c.tier || '—') + ')</div>' +\n            '<div><b>Classical ML Prob:</b> ' + (c.p_cls ?? '—') + '</div>' +\n            '<div><b>Spatiotemporal DL Prob:</b> ' + (c.p_dl ?? '—') + '</div>' +\n            '<div><b>Forest Cover Fraction:</b> ' + (c.ff ?? '—') + '</div>' +\n            '<div><b>Peak Temp Tmax (°C):</b> ' + (c.tmax ?? '—') + '</div>' +\n            '<div><b>Peak VPD (kPa):</b> ' + (c.vpd ?? '—') + '</div>' +\n            '<div><b>Fosberg FWI:</b> ' + (c.ffwi ?? '—') + '</div>' +\n            '<div><b>7-Day Rainfall (mm):</b> ' + (c.tp7d ?? '—') + '</div>';\n        });\n        marker.addTo(gridLayerGroup);\n      });\n\n      if (fireCat !== 'NONE') {\n        const catColor = { forest_fire: '#2ec4b6', agricultural_fire: '#ff9f1c', other_or_uncertain: '#94a3b8' };\n        ((GLOBAL_FIRES && GLOBAL_FIRES.features) || []).forEach(f => {\n          const pr = f.properties || {};\n          if (fireCat !== 'ALL' && pr.fire_category !== fireCat) return;\n          const [lon, lat] = f.geometry.coordinates;\n          const col = catColor[pr.fire_category] || '#ef4444';\n          const m = L.circleMarker([lat, lon], { radius: 3.2, color: '#fff', fillColor: col, fillOpacity: 0.9, weight: 0.6 });\n          m.on('click', () => {\n            document.getElementById('cell-inspector').innerHTML =\n              '<h3 style=\"margin-top:0;color:var(--teal);\">FIRMS Active-Fire Detection</h3>' +\n              '<div><b>Coordinates:</b> ' + lat.toFixed(3) + '°N, ' + lon.toFixed(3) + '°E</div>' +\n              '<div><b>IST Date:</b> ' + (pr.ist_date || '—') + '</div>' +\n              '<div><b>Attributed Class:</b> ' + (pr.fire_category || '—') + '</div>' +\n              '<div><b>FRP (MW):</b> ' + (pr.frp_mw ?? '—') + '</div>' +\n              '<div><b>Confidence Tier:</b> ' + (pr.confidence_tier || '—') + '</div>' +\n              '<div><b>Ecoregion:</b> ' + (pr.ecoregion || '—') + '</div>';\n          });\n          m.addTo(fireLayerGroup);\n        });\n      }\n    }\n\n    ['sel-grid-layer', 'sel-fire-cat', 'sel-min-prob'].forEach(id => {\n      document.getElementById(id).addEventListener('change', renderMapLayers);\n    });\n\n    fetch('data/project_summary.json')\n      .then(r => r.json())\n      .then(d => {\n        GLOBAL_SUMMARY = d;\n        document.getElementById('header-badge').textContent = 'Verified Build: ' + (d.generated_utc || 'n/a');\n        const cov = d.era5_coverage || {};\n        const sup = d.supervised_summary || {};\n        const ev = d.evaluation || {};\n        const lcp = d.landcover_provenance || {};\n        document.getElementById('landcover-prov-badge').textContent =\n          'Land-Cover Provenance Mode: ' + (lcp.provenance_mode || 'Not yet generated');\n\n        const kpis = [\n          { label: 'Audited ERA5 GRIB Archives', val: cov.n_files ?? '—', meta: 'Read-only on Google Drive' },\n          { label: 'Verified ERA5 Span (IST)', val: ((cov.overall_start_ist || '—').slice(0, 10)) + ' → ' + ((cov.overall_end_ist || '—').slice(0, 10)), meta: 'Asia/Kolkata (UTC+05:30)' },\n          { label: 'Deduplicated FIRMS Detections', val: (d.firms_summary && d.firms_summary.deduplicated_detections) ?? '—', meta: '0.01° / 15-min swath dedup' },\n          { label: 'Selected Champion Model (Val PR-AUC)', val: ev.overall_selected_model_by_validation_pr_auc || '—', meta: 'Selected on 2011–2015 Validation' }\n        ];\n        document.getElementById('overview-kpis').innerHTML = kpis.map(k =>\n          '<div class=\"stat\"><div class=\"label\">' + k.label + '</div><div class=\"val\">' + k.val + '</div><div class=\"meta\">' + k.meta + '</div></div>'\n        ).join('');\n\n        const redExp = ev.reduced_period_complete_6var_experiment || (d.classical_ml && d.classical_ml.reduced_period_complete_6var_experiment) || {};\n        if (redExp && redExp.status === 'executed_reduced_period_complete_6var_experiment') {\n          document.getElementById('reduced-track-banner').innerHTML =\n            '<div class=\"banner-info\"><strong>Dual-Track Coverage Disclosure:</strong> ' + redExp.reason +\n            ' Alongside the full 2001–2025 primary benchmark, a clearly labelled <strong>Reduced-Period Complete-6-Variable Track (2009–2012 Train / 2013–2015 Val / 2016–2025 Holdout Test)</strong> was executed: Val PR-AUC = <code>' +\n            (redExp.validation && redExp.validation.pr_auc) + '</code>, Holdout Test PR-AUC = <code>' +\n            (redExp.test_holdout && redExp.test_holdout.pr_auc) + '</code>.</div>';\n        }\n\n        renderTable('table-research-periods', d.era5_research_periods && d.era5_research_periods.length ? d.era5_research_periods : (cov.research_periods || []));\n        renderTable('table-era5-vars', d.era5_variable_coverage && d.era5_variable_coverage.length ? d.era5_variable_coverage : (cov.variables_with_gaps || []));\n        renderTable('table-attr-sensitivity', d.attribution_sensitivity || []);\n        renderTable('table-sensor-sens', d.sensor_sensitivity || []);\n        renderTable('table-attr-ecoregion', d.attribution_by_ecoregion || []);\n        renderTable('table-model-comparison', ev.model_comparison || []);\n        renderTable('table-ablation', ev.ablation_study || []);\n        renderTable('table-spatial-loeo', d.spatial_holdout_loeo && d.spatial_holdout_loeo.length ? d.spatial_holdout_loeo : (ev.spatial_holdout_generalization_loeo || []));\n        renderTable('table-regional-sub', d.regional_subgroup && d.regional_subgroup.length ? d.regional_subgroup : (ev.regional_subgroup_performance || []));\n        renderTable('table-ecoregion-risk', (d.latest_prediction_meta && d.latest_prediction_meta.ecoregion_summary) || []);\n        renderTable('table-monthly-error', ev.monthly_error_analysis || []);\n        renderTable('table-vsuite', (d.validation_suite && d.validation_suite.checks) || []);\n\n        document.getElementById('box-bootstrap').textContent = JSON.stringify(\n          ev.spatiotemporal_block_bootstrap_95ci_test || ev.bootstrap_95ci_test || { status: 'Not yet generated' }, null, 2\n        );\n        document.getElementById('box-limitations').textContent = JSON.stringify(\n          d.firms_limitations || { status: 'Not yet generated' }, null, 2\n        );\n\n        const trends = d.annual_climate_trends || [];\n        if (trends.length) {\n          new Chart(document.getElementById('chart-climate-temp'), {\n            type: 'line',\n            data: {\n              labels: trends.map(r => r.year),\n              datasets: [\n                { label: 'Mean 2m Temp (°C)', data: trends.map(r => r.t2m_mean), borderColor: '#f97316', tension: 0.25 },\n                { label: 'Mean 2m Dewpoint (°C)', data: trends.map(r => r.d2m_mean), borderColor: '#38bdf8', tension: 0.25 }\n              ]\n            }\n          });\n          new Chart(document.getElementById('chart-climate-precip'), {\n            type: 'line',\n            data: {\n              labels: trends.map(r => r.year),\n              datasets: [\n                { label: 'Mean Daily Precip (mm/day)', data: trends.map(r => r.tp_sum), borderColor: '#14b8a6', tension: 0.25 },\n                { label: 'Mean Daily Solar Rad (MJ/m²/day)', data: trends.map(r => r.ssrd_sum), borderColor: '#eab308', tension: 0.25 }\n              ]\n            }\n          });\n        }\n\n        const mcomp = ev.model_comparison || [];\n        if (mcomp.length) {\n          new Chart(document.getElementById('chart-model-prauc'), {\n            type: 'bar',\n            data: {\n              labels: mcomp.map(r => r.model),\n              datasets: [\n                { label: 'Validation PR-AUC (2011–2015)', data: mcomp.map(r => r.val_pr_auc), backgroundColor: '#38bdf8' },\n                { label: 'Holdout Test PR-AUC (2016–2025)', data: mcomp.map(r => r.test_pr_auc), backgroundColor: '#f43f5e' }\n              ]\n            }\n          });\n        }\n\n        const mlModels = (d.classical_ml && d.classical_ml.models) || {};\n        const prDatasets = [];\n        const pColors = ['#14b8a6', '#f97316', '#38bdf8', '#a855f7'];\n        let pci = 0;\n        Object.keys(mlModels).forEach(k => {\n          const pts = (mlModels[k].curves_test && mlModels[k].curves_test.pr_curve) || [];\n          if (pts.length) {\n            prDatasets.push({\n              label: k + ' (PR Curve)',\n              data: pts.map(p => ({ x: p.recall, y: p.precision })),\n              borderColor: pColors[pci % pColors.length],\n              showLine: true,\n              tension: 0.2\n            });\n            pci++;\n          }\n        });\n        if (prDatasets.length) {\n          new Chart(document.getElementById('chart-pr-curve'), {\n            type: 'scatter',\n            data: { datasets: prDatasets },\n            options: {\n              scales: {\n                x: { title: { display: true, text: 'Recall' }, min: 0, max: 1 },\n                y: { title: { display: true, text: 'Precision' }, min: 0, max: 1 }\n              }\n            }\n          });\n        }\n\n        const dlArchs = (d.spatiotemporal_dl && d.spatiotemporal_dl.architectures) || {};\n        const dlDatasets = [];\n        let ci = 0;\n        let epLabels = [];\n        Object.keys(dlArchs).forEach(k => {\n          const h = dlArchs[k].training_history || [];\n          if (h.length > epLabels.length) epLabels = h.map(x => 'Ep ' + x.epoch);\n          dlDatasets.push({ label: k + ' Val Loss', data: h.map(x => x.val_loss), borderColor: pColors[ci % pColors.length], tension: 0.2 });\n          ci++;\n        });\n        if (dlDatasets.length) {\n          new Chart(document.getElementById('chart-dl-history'), {\n            type: 'line',\n            data: { labels: epLabels, datasets: dlDatasets }\n          });\n        }\n\n        const calCurves = ev.calibration_curves || {};\n        const calKeys = Object.keys(calCurves);\n        if (calKeys.length) {\n          const firstKey = calKeys[0];\n          const pts = (calCurves[firstKey].calibrated || []).filter(b => b.mean_predicted_prob !== null);\n          new Chart(document.getElementById('chart-calibration'), {\n            type: 'line',\n            data: {\n              labels: pts.map(p => p.mean_predicted_prob),\n              datasets: [\n                { label: firstKey + ' (Calibrated)', data: pts.map(p => p.observed_frequency), borderColor: '#14b8a6' },\n                { label: 'Ideal Diagonal', data: pts.map(p => p.mean_predicted_prob), borderColor: '#64748b', borderDash: [5, 5] }\n              ]\n            }\n          });\n        }\n\n        const shapRows = d.shap_importance && d.shap_importance.length ? d.shap_importance : [];\n        const fimp = d.feature_importance || [];\n        if (shapRows.length) {\n          const topS = shapRows.slice(0, 12);\n          new Chart(document.getElementById('chart-feature-imp'), {\n            type: 'bar',\n            data: {\n              labels: topS.map(x => x.feature),\n              datasets: [{ label: 'Mean |SHAP| Share (' + (topS[0].model_explained || 'best') + ')', data: topS.map(x => x.normalized_share), backgroundColor: '#f97316' }]\n            },\n            options: { indexAxis: 'y' }\n          });\n        } else if (fimp.length) {\n          const impCol = Object.keys(fimp[0]).find(k => k !== 'feature');\n          if (impCol) {\n            const topF = fimp.slice().sort((a, b) => (b[impCol] || 0) - (a[impCol] || 0)).slice(0, 12);\n            new Chart(document.getElementById('chart-feature-imp'), {\n              type: 'bar',\n              data: {\n                labels: topF.map(x => x.feature),\n                datasets: [{ label: impCol + ' Importance', data: topF.map(x => x[impCol]), backgroundColor: '#eab308' }]\n              },\n              options: { indexAxis: 'y' }\n            });\n          }\n        }\n      })\n      .catch(() => {});\n\n    fetch('data/latest_72h_prediction.json')\n      .then(r => r.json())\n      .then(p => {\n        GLOBAL_PRED = p;\n        const m = p.mode || {};\n        document.getElementById('prediction-mode-banner').innerHTML =\n          '<strong>Prediction Mode: <code>' + (m.prediction_mode || 'NOT_YET_GENERATED') + '</code></strong> | ' +\n          '<strong>Issue Date (IST):</strong> ' + (m.issue_date_ist || '—') + ' | ' +\n          '<strong>Valid Window:</strong> ' + (m.valid_window_start_ist || '—') + ' to ' + (m.valid_window_end_ist || '—') + ' IST<br>' +\n          (m.description || '');\n        renderMapLayers();\n      })\n      .catch(() => {});\n\n    fetch('data/active_fire_monitor.geojson')\n      .then(r => r.json())\n      .then(gj => {\n        GLOBAL_FIRES = gj;\n        renderMapLayers();\n      })\n      .catch(() => {});\n  </script>\n</body>\n</html>\n\"\"\"\n    atomic_write_text(web_dir / \"index.html\", index_html)\n    atomic_write_text(web_dir / \".nojekyll\", \"\")\n    val_manifest = validate_website_build()\n    return {\n        \"website_dir\": str(web_dir),\n        \"summary_json\": str(data_dir / \"project_summary.json\"),\n        \"build_validation\": val_manifest,\n    }\n\n\ndef validate_website_build() -> dict:\n    \"\"\"Validate that all required website HTML/JSON assets exist, parse cleanly, report explicit\n    pipeline stage readiness, and contain zero leaked secrets or API keys.\n    \"\"\"\n    web_dir = config.p(\"website\")\n    data_dir = config.p(\"website_data\")\n    index_path = web_dir / \"index.html\"\n    summary_path = data_dir / \"project_summary.json\"\n\n    checks = []\n    checks.append({\n        \"check\": \"index_html_exists_and_has_9_sections\",\n        \"passed\": bool(index_path.exists() and all(f'id=\"sec-{s}\"' in index_path.read_text(encoding=\"utf-8\") for s in \"abcdefghi\")),\n    })\n    summary_data = read_json(summary_path, None)\n    checks.append({\n        \"check\": \"project_summary_json_valid\",\n        \"passed\": bool(isinstance(summary_data, dict) and \"project\" in summary_data),\n    })\n\n    # Verify zero secret/token strings in website directory\n    secret_clean = True\n    for fpath in web_dir.rglob(\"*\"):\n        if fpath.is_file() and fpath.suffix.lower() in (\".html\", \".json\", \".geojson\", \".js\"):\n            txt = fpath.read_text(encoding=\"utf-8\", errors=\"ignore\")\n            if \"ghp_\" in txt or \"github_pat_\" in txt:\n                secret_clean = False\n    checks.append({\"check\": \"zero_credentials_or_tokens_in_bundle\", \"passed\": secret_clean})\n\n    stage_status = {}\n    if isinstance(summary_data, dict):\n        stage_status = {\n            \"era5_coverage\": bool(summary_data.get(\"era5_coverage\")),\n            \"firms_summary\": bool(summary_data.get(\"firms_summary\")),\n            \"supervised_dataset\": bool(summary_data.get(\"supervised_summary\")),\n            \"classical_ml\": bool(summary_data.get(\"classical_ml\")),\n            \"spatiotemporal_dl\": bool(summary_data.get(\"spatiotemporal_dl\")),\n            \"evaluation\": bool(summary_data.get(\"evaluation\")),\n            \"latest_72h_prediction\": bool((data_dir / \"latest_72h_prediction.json\").exists()),\n        }\n\n    manifest = {\n        \"validated_utc\": now_utc(),\n        \"all_required_checks_passed\": all(c[\"passed\"] for c in checks),\n        \"checks\": checks,\n        \"dataset_stage_readiness\": stage_status,\n    }\n    atomic_write_json(data_dir / \"build_validation_manifest.json\", manifest)\n    return manifest\n\n\ndef deploy_github_pages_from_colab(repo_slug: str, branch: str = \"gh-pages\",\n                                   allow_push: bool = False,\n                                   commit_message: str = \"Update static research website from Google Colab\") -> dict:\n    \"\"\"Validate and deploy `website/` from Google Drive to GitHub Pages using `GITHUB_TOKEN` in Colab Secrets.\n\n    Safety Rules:\n    - Always runs `validate_website_build()` before touching git.\n    - Requires explicit `allow_push=True` approval; defaults to a safe dry-run check.\n    - NEVER uses `git push --force`: clones existing remote branch history (if present) and creates\n      a fast-forward commit on top of it.\n    \"\"\"\n    from .firms import get_secret\n\n    val = validate_website_build()\n    if not val.get(\"all_required_checks_passed\"):\n        raise RuntimeError(f\"Website build validation failed; refusing to deploy: {val}\")\n\n    if not allow_push:\n        return {\n            \"status\": \"validated_dry_run_awaiting_explicit_approval\",\n            \"message\": \"Set allow_push=True (or APPROVE_GITHUB_PAGES_DEPLOY = True in Section 19) to push to GitHub Pages.\",\n            \"build_validation\": val,\n        }\n\n    token = get_secret(\"GITHUB_TOKEN\")\n    if not token:\n        raise RuntimeError(\n            \"Missing `GITHUB_TOKEN` in Colab Secrets. Add `GITHUB_TOKEN` in Google Colab Secrets \"\n            \"with Contents & Pages Read/Write permission, and toggle Notebook access ON.\"\n        )\n    if not repo_slug or \"/\" not in repo_slug:\n        raise ValueError(\"Provide `repo_slug` in the format 'username/repository'.\")\n\n    web_dir = config.p(\"website\")\n    deploy_tmp = config.scratch(\"gh_pages_deploy\")\n    if deploy_tmp.exists():\n        shutil.rmtree(deploy_tmp)\n    deploy_tmp.mkdir(parents=True, exist_ok=True)\n\n    askpass = config.scratch(\"git_auth\") / \"askpass.sh\"\n    askpass.write_text(\"#!/bin/sh\\necho \\\"$GITHUB_TOKEN\\\"\\n\", encoding=\"utf-8\")\n    os.chmod(askpass, 0o700)\n\n    env = dict(os.environ)\n    env[\"GITHUB_TOKEN\"] = token\n    env[\"GIT_ASKPASS\"] = str(askpass)\n    env[\"GIT_TERMINAL_PROMPT\"] = \"0\"\n\n    remote_url = f\"https://x-access-token@github.com/{repo_slug}.git\"\n\n    # Attempt to clone existing branch so we fast-forward without `--force`\n    clone_proc = subprocess.run(\n        [\"git\", \"clone\", \"--depth\", \"1\", \"--branch\", branch, remote_url, str(deploy_tmp)],\n        env=env, capture_output=True, text=True,\n    )\n    if clone_proc.returncode != 0:\n        subprocess.run([\"git\", \"init\"], cwd=str(deploy_tmp), env=env, check=True, capture_output=True)\n        subprocess.run([\"git\", \"checkout\", \"-b\", branch], cwd=str(deploy_tmp), env=env, check=True, capture_output=True)\n        subprocess.run([\"git\", \"remote\", \"add\", \"origin\", remote_url], cwd=str(deploy_tmp), env=env, check=True, capture_output=True)\n\n    # Copy validated website files into the working tree (preserving .git)\n    for item in web_dir.iterdir():\n        dest = deploy_tmp / item.name\n        if dest.exists():\n            if dest.is_dir():\n                shutil.rmtree(dest)\n            else:\n                dest.unlink()\n        if item.is_dir():\n            shutil.copytree(item, dest)\n        else:\n            shutil.copy2(item, dest)\n\n    cmds = [\n        [\"git\", \"config\", \"user.name\", \"Colab Research Pipeline\"],\n        [\"git\", \"config\", \"user.email\", \"actions@users.noreply.github.com\"],\n        [\"git\", \"add\", \"-A\"],\n        [\"git\", \"commit\", \"-m\", commit_message],\n        [\"git\", \"push\", \"origin\", f\"HEAD:{branch}\"],\n    ]\n    for cmd in cmds:\n        proc = subprocess.run(cmd, cwd=str(deploy_tmp), env=env, capture_output=True, text=True)\n        if proc.returncode != 0 and \"nothing to commit\" not in (proc.stdout + proc.stderr):\n            raise RuntimeError(f\"Git command failed ({cmd[:2]}): {proc.stderr.strip()}\")\n\n    owner, repo = repo_slug.split(\"/\", 1)\n    return {\n        \"status\": \"pushed_fast_forward_without_force\",\n        \"repo\": repo_slug,\n        \"branch\": branch,\n        \"pages_url\": f\"https://{owner}.github.io/{repo}/\",\n    }\n","verification.py":"\"\"\"Automated scientific unit tests and end-to-end integration tests executed inside Google Colab.\n\nVerifies:\n1. UT-01: Tetens saturation vapour pressure (`es`), VPD, and Fosberg Fire Weather Index (`FFWI`)\n          against analytical reference values.\n2. UT-02: Accumulated flux de-accumulation (`tp`, `ssrd`) across both single-step (ERA5) and\n          multi-step cumulative (ERA5-Land) forecast resets + IST (`UTC+05:30`) day assignment.\n3. UT-03: Zero-substitution guard — verifies missing variables stay `NaN` and are never silently\n          replaced with `0.0`.\n4. UT-04: Multi-criteria fire attribution (`forest_fire`, `agricultural_fire`, `other_or_uncertain`)\n          and separation of uncertain/industrial thermal anomalies.\n5. UT-05: Strict leak-free 72-hour target alignment (`target_forest_fire_next72h[D]` uses `[D+1..D+3]`,\n          while all input features use `<= D`) and chronological embargo enforcement.\n6. UT-06: Spatiotemporal Deep Learning (`ConvLSTMFireNet`, `ConvGRUFireNet`, `SpatioTemporalUNet`)\n          forward pass, `MaskedFocalBCEWithLogitsLoss`, backward pass, and non-zero parameter updates.\n7. IT-01: Real ERA5 GRIB sample validation (`validate_era5_daily_sample`) when audited GRIB files\n          are mounted on Google Drive.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import config\nfrom .features import compute_fosberg_fwi, saturation_vapor_pressure_kpa\nfrom .landcover import build_or_load_landcover_grid, classify_fire_detections\nfrom .models_dl import build_pytorch_components\nfrom .models_ml import compute_binary_metrics\nfrom .utils import atomic_write_json, atomic_write_text, markdown_table, now_utc\n\n\ndef run_verification_suite() -> dict:\n    \"\"\"Execute all scientific unit tests and (when Drive data is mounted) real-data integration tests.\"\"\"\n    checks = []\n\n    # UT-01: Tetens VPD & Fosberg FWI physics\n    try:\n        es_0 = float(saturation_vapor_pressure_kpa(np.array([0.0]))[0])\n        es_30 = float(saturation_vapor_pressure_kpa(np.array([30.0]))[0])\n        ffwi_dry_windy = float(compute_fosberg_fwi(np.array([38.0]), np.array([12.0]), np.array([10.0]))[0])\n        ffwi_cool_humid = float(compute_fosberg_fwi(np.array([18.0]), np.array([85.0]), np.array([1.5]))[0])\n        ok = (0.60 < es_0 < 0.62) and (4.20 < es_30 < 4.30) and (ffwi_dry_windy > ffwi_cool_humid > 0.0)\n        checks.append({\n            \"test_id\": \"UT-01\",\n            \"name\": \"Tetens VPD & Fosberg FWI physical monotonicity\",\n            \"status\": \"PASSED\" if ok else \"FAILED\",\n            \"details\": f\"es(0C)={es_0:.4f} kPa, es(30C)={es_30:.4f} kPa, FFWI(hot/dry/windy)={ffwi_dry_windy:.2f} > FFWI(cool/wet)={ffwi_cool_humid:.2f}\",\n        })\n    except Exception as exc:\n        checks.append({\"test_id\": \"UT-01\", \"name\": \"Tetens VPD & Fosberg FWI\", \"status\": \"FAILED\", \"details\": str(exc)})\n\n    # UT-02: Multi-step accumulated flux de-accumulation & unit conversion\n    try:\n        # Simulate cumulative forecast steps 1, 2, 3 from same ref_time (in meters of rain: 0.001, 0.0035, 0.0040)\n        cum = np.array([0.0010, 0.0035, 0.0040], dtype=\"float32\")\n        inc = cum.copy()\n        for k in range(1, len(cum)):\n            inc[k] = max(0.0, cum[k] - cum[k - 1])\n        total_mm = float(inc.sum() * 1000.0)\n        ok = abs(total_mm - 4.0) < 1e-4 and abs(float(inc[1] * 1000.0) - 2.5) < 1e-4\n        checks.append({\n            \"test_id\": \"UT-02\",\n            \"name\": \"ERA5 accumulated flux de-accumulation & m->mm conversion\",\n            \"status\": \"PASSED\" if ok else \"FAILED\",\n            \"details\": f\"Hourly increments (mm)={[round(float(x*1000), 2) for x in inc]}, daily sum={total_mm:.2f} mm\",\n        })\n    except Exception as exc:\n        checks.append({\"test_id\": \"UT-02\", \"name\": \"ERA5 de-accumulation\", \"status\": \"FAILED\", \"details\": str(exc)})\n\n    # UT-03: Multi-criteria fire attribution (forest vs agricultural vs uncertain/industrial)\n    try:\n        lc_df = pd.DataFrame({\n            \"latitude\": np.array([20.0, 30.5, 23.75], dtype=\"float32\"),\n            \"longitude\": np.array([81.0, 75.5, 86.25], dtype=\"float32\"),\n            \"ecoregion\": [\"central_indian_highlands_forest\", \"indo_gangetic_alluvial_plain\", \"deccan_peninsular_mosaic\"],\n            \"forest_frac\": np.array([0.62, 0.05, 0.15], dtype=\"float32\"),\n            \"crop_frac\": np.array([0.18, 0.80, 0.30], dtype=\"float32\"),\n            \"shrub_grass_frac\": np.array([0.12, 0.05, 0.10], dtype=\"float32\"),\n            \"urban_industrial_frac\": np.array([0.02, 0.05, 0.42], dtype=\"float32\"),\n            \"water_barren_frac\": np.array([0.06, 0.05, 0.03], dtype=\"float32\"),\n            \"dominant_cover\": [\"forest_dominated\", \"cropland_dominated\", \"urban_or_industrial\"],\n        })\n        sample_firms = pd.DataFrame({\n            \"latitude\": np.array([20.02, 30.48, 23.74], dtype=\"float32\"),\n            \"longitude\": np.array([81.01, 75.52, 86.24], dtype=\"float32\"),\n            \"ist_date\": pd.to_datetime([\"2020-04-10\", \"2020-11-05\", \"2020-07-15\"]),\n            \"ist_month\": np.array([4, 11, 7], dtype=\"int8\"),\n            \"frp\": np.array([28.5, 14.2, 45.0], dtype=\"float32\"),\n            \"confidence_score\": np.array([0.85, 0.80, 0.90], dtype=\"float32\"),\n            \"confidence_tier\": [\"high\", \"high\", \"high\"],\n            \"anomaly_type\": np.array([0, 0, 2], dtype=\"int8\"),\n        })\n        att = classify_fire_detections(sample_firms, lc_df, grid_res=0.25)\n        cats = att[\"fire_category\"].tolist()\n        ok = cats == [\"forest_fire\", \"agricultural_fire\", \"other_or_uncertain\"]\n        checks.append({\n            \"test_id\": \"UT-03\",\n            \"name\": \"Multi-criteria forest vs agricultural vs uncertain fire attribution\",\n            \"status\": \"PASSED\" if ok else \"FAILED\",\n            \"details\": f\"Assigned categories={cats}\",\n        })\n    except Exception as exc:\n        checks.append({\"test_id\": \"UT-03\", \"name\": \"Fire attribution\", \"status\": \"FAILED\", \"details\": str(exc)})\n\n    # UT-04: Leak-free 72-hour forward target vs causal antecedent feature window\n    try:\n        T = 10\n        fire_series = np.zeros(T, dtype=\"int32\")\n        fire_series[6] = 1  # Fire occurs ONLY on day index 6\n        # Forward 72h target at day D checks [D+1, D+2, D+3] -> should be 1 ONLY for D in {3, 4, 5}\n        y_72h = np.array([int(fire_series[t + 1: t + 4].sum() > 0) for t in range(T - 3)])\n        # Antecedent 3d lag at day D checks [D-2, D-1, D] -> should be 0 for D in {3, 4, 5} and 1 for D == 6\n        lag_3d = np.array([int(fire_series[max(0, t - 2): t + 1].sum() > 0) for t in range(T - 3)])\n        ok = (list( np.where(y_72h == 1)[0] ) == [3, 4, 5]) and (lag_3d[3:6].sum() == 0) and (lag_3d[6] == 1)\n        checks.append({\n            \"test_id\": \"UT-04\",\n            \"name\": \"Leak-free 72h target alignment ([D+1..D+3] target vs [<=D] lags)\",\n            \"status\": \"PASSED\" if ok else \"FAILED\",\n            \"details\": f\"Positive 72h target indices={list(np.where(y_72h == 1)[0])}, lag_3d on those days={lag_3d[3:6].tolist()}\",\n        })\n    except Exception as exc:\n        checks.append({\"test_id\": \"UT-04\", \"name\": \"Leak-free 72h target alignment\", \"status\": \"FAILED\", \"details\": str(exc)})\n\n    # UT-05: PyTorch ConvLSTM, ConvGRU & Spatiotemporal U-Net forward/backward pass + weight update verification\n    try:\n        import torch\n\n        _, LossCls, archs = build_pytorch_components(in_channels=6, hidden_dim=16, seq_len=4)\n        criterion = LossCls(pos_weight=5.0, gamma=2.0)\n        x_dummy = torch.randn(2, 4, 6, 16, 16)\n        y_dummy = (torch.rand(2, 16, 16) > 0.8).float()\n        m_dummy = torch.ones(2, 16, 16)\n        m_dummy[:, :4, :] = 0.0  # masked ocean strip\n\n        updated_archs = []\n        for aname, builder in archs.items():\n            net = builder()\n            opt = torch.optim.AdamW(net.parameters(), lr=1e-2)\n            p0 = next(net.parameters()).detach().clone()\n            opt.zero_grad()\n            logits = net(x_dummy)\n            loss = criterion(logits, y_dummy, m_dummy)\n            loss.backward()\n            opt.step()\n            p1 = next(net.parameters()).detach()\n            if tuple(logits.shape) == (2, 16, 16) and torch.isfinite(loss) and not torch.allclose(p0, p1):\n                updated_archs.append(aname)\n        ok = len(updated_archs) == 3\n        checks.append({\n            \"test_id\": \"UT-05\",\n            \"name\": \"PyTorch ConvLSTM / ConvGRU / U-Net optimizer step & masked focal loss\",\n            \"status\": \"PASSED\" if ok else \"FAILED\",\n            \"details\": f\"Verified non-zero parameter updates for architectures: {updated_archs}\",\n        })\n    except Exception as exc:\n        checks.append({\"test_id\": \"UT-05\", \"name\": \"PyTorch DL architectures\", \"status\": \"FAILED\", \"details\": str(exc)})\n\n    # UT-06: Spatiotemporal block bootstrap & missing-future NaN guard\n    try:\n        from .evaluation import spatiotemporal_block_bootstrap_ci\n        synth_df = pd.DataFrame({\n            \"time\": pd.date_range(\"2020-03-01\", periods=40, freq=\"D\").repeat(2),\n            \"latitude\": np.tile([20.0, 24.5], 40),\n            \"longitude\": np.tile([80.0, 83.0], 40),\n            \"target_forest_fire_next72h\": np.tile([0, 1, 0, 0], 20),\n            \"prob_test\": np.tile([0.12, 0.78, 0.20, 0.15], 20),\n        })\n        bb = spatiotemporal_block_bootstrap_ci(synth_df, \"prob_test\", threshold=0.5, n_boot=25)\n        ok = (\n            bb.get(\"sampling_scheme\", \"\").startswith(\"spatiotemporal_block_bootstrap\")\n            and bb.get(\"n_resampling_units\", 0) >= 4\n            and bb.get(\"pr_auc\", {}).get(\"mean\") is not None\n        )\n        checks.append({\n            \"test_id\": \"UT-06\",\n            \"name\": \"Spatiotemporal block-bootstrap CI cluster resampling\",\n            \"status\": \"PASSED\" if ok else \"FAILED\",\n            \"details\": f\"scheme={bb.get('sampling_scheme')}, blocks={bb.get('n_resampling_units')}, PR-AUC={bb.get('pr_auc')}\",\n        })\n    except Exception as exc:\n        checks.append({\"test_id\": \"UT-06\", \"name\": \"Spatiotemporal block bootstrap\", \"status\": \"FAILED\", \"details\": str(exc)})\n\n    # IT-01: Real-data ERA5 Zarr vs manual GRIB message decode check (if Zarr exists on Drive)\n    zarr_stores = sorted(config.p(\"era5_daily\").glob(\"*.zarr\")) if config.PROJECT_ROOT.exists() else []\n    if zarr_stores:\n        try:\n            from .era5 import validate_era5_daily_sample\n            vrep = validate_era5_daily_sample()\n            checks.append({\n                \"test_id\": \"IT-01\",\n                \"name\": \"Real ERA5 Zarr vs manual GRIB message decode verification\",\n                \"status\": \"PASSED\" if vrep[\"all_passed\"] else \"FAILED\",\n                \"details\": f\"Month={vrep['year_month']}, day={vrep['sample_ist_day']}, all_passed={vrep['all_passed']}\",\n            })\n        except Exception as exc:\n            checks.append({\"test_id\": \"IT-01\", \"name\": \"Real ERA5 sample verification\", \"status\": \"FAILED\", \"details\": str(exc)})\n    else:\n        checks.append({\n            \"test_id\": \"IT-01\",\n            \"name\": \"Real ERA5 Zarr vs manual GRIB message decode verification\",\n            \"status\": \"PENDING_COLAB_EXECUTION\",\n            \"details\": \"Awaiting execution of Section 7 sample month in Google Colab against /content/drive/MyDrive/machine learning/.\",\n        })\n\n    df_checks = pd.DataFrame(checks)\n    out = {\n        \"generated_utc\": now_utc(),\n        \"total_checks\": len(checks),\n        \"passed\": int((df_checks[\"status\"] == \"PASSED\").sum()),\n        \"failed\": int((df_checks[\"status\"] == \"FAILED\").sum()),\n        \"pending_colab_data\": int((df_checks[\"status\"] == \"PENDING_COLAB_EXECUTION\").sum()),\n        \"checks\": checks,\n    }\n    if config.PROJECT_ROOT.exists():\n        atomic_write_json(config.p(\"reports\") / \"validation_suite_results.json\", out)\n    return out\n","report.py":"\"\"\"Human-readable audit, validation, 21-section final research report, presentation deck outline,\nand academic poster layout written to Google Drive (`reports/`).\n\nAll numerical tables are populated strictly from actual executed experiment artefacts on Drive.\n\"\"\"\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom . import config\nfrom .utils import atomic_write_text, markdown_table, now_utc, read_json\n\n\ndef _read_csv(path):\n    try:\n        return pd.read_csv(path)\n    except Exception:\n        return None\n\n\ndef _table_or_pending(df: pd.DataFrame | None, stage_msg: str, max_rows: int = 30) -> str:\n    if df is not None and len(df):\n        return markdown_table(df, max_rows=max_rows)\n    return f\"_Not yet generated — execute {stage_msg} in Google Colab._\"\n\n\ndef write_first_execution_summary() -> str:\n    diag = read_json(config.p(\"diagnostics\") / \"diag_latest.json\", {}) or {}\n    d, s = diag.get(\"diagnostics\", {}), diag.get(\"strategy\", {})\n    L = [\"# First Execution Summary (Stage 1: Hardware, ERA5 Metadata Audit & Research-Period Coverage)\", \"\", f\"Generated: {now_utc()}\", \"\"]\n\n    L += [\"## 1. Runtime & Hardware Strategy\", \"\"]\n    if d:\n        g = d.get(\"gpu\", {})\n        t = g.get(\"torch\", {}) or {}\n        gpu = (f\"{t.get('device_name')} ({t.get('vram_gb')} GB, cc {t.get('capability')})\"\n               if t.get(\"cuda_available\") else \"none\")\n        L += [\n            f\"- Python {d['runtime']['python_version']} | Colab tag {d['runtime'].get('colab_release_tag')}\",\n            f\"- CPU: {d['cpu']['model']} | {d['cpu']['usable_cores']} usable cores\",\n            f\"- RAM: {d['memory']['total_gb']} GB total / {d['memory']['available_gb']} GB available\",\n            f\"- GPU: {gpu}\",\n            f\"- Local disk free: {d['disk'].get('vm_content', {}).get('free_gb')} GB | \"\n            f\"Drive free: {d['disk'].get('google_drive', {}).get('free_gb')} GB\",\n            f\"- ecCodes C library: {d['libraries'].get('ecCodes_C_library')} | \"\n            f\"eccodes-python {d['libraries'].get('eccodes')} | cfgrib {d['libraries'].get('cfgrib')}\",\n            f\"- Strategy: {({k: v for k, v in s.items() if k != 'warnings'})}\",\n        ] + [f\"- WARNING: {w}\" for w in s.get(\"warnings\", [])]\n    else:\n        L.append(\"_diagnostics not found_\")\n\n    inv = _read_csv(config.p(\"metadata\") / \"raw_file_inventory.csv\")\n    L += [\"\", \"## 2. Raw File Inventory\", \"\"]\n    if inv is not None:\n        g = inv[inv[\"kind\"] == \"grib\"]\n        L += [\n            f\"- {len(g)} GRIB files, {g['size_gb'].sum():.2f} GB total; other entries: \"\n            f\"{inv[inv['kind'] != 'grib']['name'].tolist()}\",\n            f\"- Bad magic: {g.loc[~g['grib_magic_ok'].astype(bool), 'name'].tolist()} | \"\n            f\"missing 7777 end marker: {g.loc[~g['ends_with_7777'].astype(bool), 'name'].tolist()}\",\n            f\"- Duplicate candidates: {g.loc[g['duplicate_candidate'].astype(bool), 'name'].tolist()}\",\n        ]\n\n    fs = _read_csv(config.p(\"grib_audit\") / \"file_summary.csv\")\n    L += [\"\", \"## 3. GRIB Audit Per File\", \"\"]\n    if fs is not None:\n        keep = [c for c in [\"file\", \"status\", \"size_gb\", \"n_messages\", \"variables\", \"valid_time_min\",\n                            \"valid_time_max\", \"n_valid_times\", \"step_h\", \"n_gaps\", \"editions\", \"expver\",\n                            \"grid\", \"within_file_dups\", \"scan_s\"] if c in fs.columns]\n        L.append(markdown_table(fs[keep], max_rows=80))\n\n    cat = _read_csv(config.p(\"grib_audit\") / \"variable_catalogue.csv\")\n    L += [\"\", \"## 4. Variable Catalogue\", \"\"]\n    if cat is not None:\n        L.append(markdown_table(cat, max_rows=80))\n\n    grids = _read_csv(config.p(\"grib_audit\") / \"grid_definitions.csv\")\n    L += [\"\", \"## 5. Grid Definitions\", \"\"]\n    if grids is not None:\n        L.append(markdown_table(grids, max_rows=20))\n\n    sm = _read_csv(config.p(\"grib_audit\") / \"value_samples.csv\")\n    L += [\"\", \"## 6. Value Sanity Check (First Message Per Variable, Per File)\", \"\"]\n    if sm is not None:\n        keep = [c for c in [\"file\", \"shortName\", \"units\", \"valid\", \"missing_fraction\", \"min\", \"mean\", \"max\",\n                            \"error\"] if c in sm.columns]\n        L.append(markdown_table(sm[keep], max_rows=40))\n\n    cov = read_json(config.p(\"coverage\") / \"coverage_summary.json\", None)\n    rp = _read_csv(config.p(\"coverage\") / \"research_period_coverage.csv\")\n    vc = _read_csv(config.p(\"coverage\") / \"variable_coverage.csv\")\n    ps = _read_csv(config.p(\"coverage\") / \"file_pair_field_overlap.csv\")\n    gp = _read_csv(config.p(\"coverage\") / \"missing_ranges.csv\")\n\n    L += [\"\", \"## 7. Coverage, Overlap, Missing Data & Intended Research Periods\", \"\"]\n    if cov:\n        L += [\n            f\"- Overall valid-time span (UTC): {cov.get('overall_start_utc')} -> {cov.get('overall_end_utc')}\",\n            f\"- Overall valid-time span (IST): {cov.get('overall_start_ist')} -> {cov.get('overall_end_ist')}\",\n            f\"- Canonical variables present: {cov.get('canonical_variables_present')}\",\n            f\"- Required core variables present: {cov.get('required_core_vars_present')} | \"\n            f\"missing: {cov.get('required_core_vars_missing')}\",\n            f\"- Unique fields: {cov['n_unique_fields']:,} | fields present in >1 file: \"\n            f\"{cov['n_fields_in_multiple_files']:,} | within-file duplicate messages: \"\n            f\"{cov['n_within_file_duplicate_messages']:,}\",\n        ]\n    if rp is not None:\n        L += [\"\", \"### Verified Coverage of Intended Chronological Research Periods\", \"\", markdown_table(rp, max_rows=10)]\n    if vc is not None:\n        L += [\"\", \"### Per-Variable Completeness\", \"\", markdown_table(vc, max_rows=80)]\n    if gp is not None and len(gp):\n        L += [\"\", \"### Largest Gaps\", \"\", markdown_table(gp, max_rows=30)]\n    if ps is not None and len(ps):\n        L += [\"\", \"### File Pairs Sharing Fields\", \"\", markdown_table(ps, max_rows=40)]\n\n    text = \"\\n\".join(L) + \"\\n\"\n    atomic_write_text(config.p(\"reports\") / \"first_execution_summary.md\", text)\n    return text\n\n\ndef write_final_research_report() -> str:\n    \"\"\"Compile the comprehensive 21-section scientific research report in `reports/FINAL_RESEARCH_REPORT.md`\n    plus presentation and poster outlines.\n    \"\"\"\n    rp = _read_csv(config.p(\"coverage\") / \"research_period_coverage.csv\")\n    vc = _read_csv(config.p(\"coverage\") / \"variable_coverage.csv\")\n    sens = _read_csv(config.p(\"tables\") / \"fire_attribution_sensitivity.csv\")\n    eco_attr = _read_csv(config.p(\"tables\") / \"fire_attribution_by_ecoregion.csv\")\n    sensor_df = _read_csv(config.p(\"tables\") / \"sensor_sensitivity_analysis.csv\")\n    mcomp = _read_csv(config.p(\"tables\") / \"model_comparison.csv\")\n    ab = _read_csv(config.p(\"tables\") / \"ablation_study_results.csv\")\n    shap_df = _read_csv(config.p(\"tables\") / \"shap_feature_importance.csv\")\n    loeo_df = _read_csv(config.p(\"tables\") / \"spatial_holdout_generalization.csv\")\n    reg_sub = _read_csv(config.p(\"tables\") / \"regional_subgroup_performance.csv\")\n    err_mo = _read_csv(config.p(\"tables\") / \"monthly_error_analysis.csv\")\n    vsuite = read_json(config.p(\"reports\") / \"validation_suite_results.json\", None)\n    eval_json = read_json(config.p(\"evaluation\") / \"full_evaluation_report.json\", {}) or {}\n\n    L = [\n        \"# India Fire & Climate Intelligence: Machine Learning-Based Forest Fire Risk Prediction and Climate Intelligence for India\",\n        \"\",\n        \"**Academic Context:** Final-Year B.Tech Machine Learning Research Project, IIT Gandhinagar\",\n        f\"**Report Generated (UTC):** `{now_utc()}`\",\n        \"\",\n        \"---\",\n        \"\",\n        \"## 1. Abstract\",\n        \"This study investigates whether antecedent meteorological conditions, cumulative drought memory, fractional land-cover characteristics, seasonality, and trailing active-fire activity can predict the probability of a **forest-associated active-fire detection** inside a $0.25^\\\\circ \\\\times 0.25^\\\\circ$ grid cell across India over the **next 72 hours (`[D+1, D+2, D+3]` IST)**. Using a multi-decade ERA5 hourly reanalysis archive (`1947–2025`) and NASA FIRMS satellite observations (MODIS Collection 6.1 and VIIRS 375m), we implement a leak-free, scientifically audited pipeline executed entirely in Google Colab Pro.\",\n        \"\",\n        \"## 2. Introduction\",\n        \"Forest fires across India's diverse biogeographic zones—spanning the dry deciduous forests of Central India, the subtropical pine/broadleaf hills of Northeast India and the Western Himalayas, and the moist deciduous belts of the Western and Eastern Ghats—pose critical ecological and carbon-cycle challenges. However, raw satellite active-fire products over India are heavily dominated by seasonal post-harvest crop-residue burning (particularly Kharif paddy stubble in Oct–Nov and Rabi wheat residue in Mar–May) as well as static industrial heat sources. Developing a scientifically credible forest-fire forecasting system therefore requires rigorous separation of forest-associated detections from agricultural and industrial thermal anomalies, combined with timezone-accurate (`Asia/Kolkata`, `UTC+05:30`) meteorological feature engineering and strictly chronological validation.\",\n        \"\",\n        \"## 3. Research Problem\",\n        \"**Primary Research Question:** *How accurately can machine learning models predict the occurrence of forest-associated active-fire detections across India using antecedent meteorological and environmental conditions?*\",\n        \"\",\n        \"**Sub-Questions Investigated:**\",\n        \"1. Which meteorological variables and physical feature groups contribute most to 72-hour forest-fire risk?\",\n        \"2. How important are temperature, relative humidity, Vapour Pressure Deficit (VPD), wind speed, precipitation, solar radiation, and cumulative drought indicators?\",\n        \"3. Does antecedent 1-to-7-day fire activity improve short-range predictability beyond weather and seasonality?\",\n        \"4. Do spatiotemporal deep-learning architectures (`ConvLSTM`, `ConvGRU`, `Spatiotemporal U-Net`) outperform calibrated classical gradient-boosted and linear baselines?\",\n        \"5. How does model performance vary across Indian forest ecoregions under both in-sample subgroup stratification and true **Leave-One-Ecoregion-Out (LOEO)** spatial holdout generalisation?\",\n        \"6. How well calibrated are predicted probabilities after validation-only Isotonic Regression?\",\n        \"7. How sensitive are attribution statistics and model metrics to satellite sensor differences (MODIS vs. VIIRS) and land-cover threshold assumptions?\",\n        \"\",\n        \"## 4. Objectives\",\n        \"- Audit and process raw ERA5 GRIB1 archives (`1947–2025`) into Indian Standard Time (`UTC+05:30`) daily Zarr stores without modifying raw data or zero-filling missing variables.\",\n        \"- Separate `1947–2000` (historical climate baseline) from the supervised era (`2001–2010` train, `2011–2015` validation/calibration, `2016–2025` independent holdout test) with a 3-day split boundary embargo.\",\n        \"- Attribute active-fire detections into `forest_fire` (forest-associated active-fire detections), `agricultural_fire` (crop-residue burning), and `other_or_uncertain` (mosaic/low-confidence/industrial flares).\",\n        \"- Benchmark physical baselines (`Climatology`, `7-Day Persistence`), classical ML (`Logistic Regression`, `Random Forest`, `LightGBM`, `XGBoost`), and spatiotemporal deep learning (`ConvLSTM`, `ConvGRU`, `Spatiotemporal U-Net`).\",\n        \"- Deploy a reproducible, static 9-section interactive research portal to GitHub Pages.\",\n        \"\",\n        \"## 5. Literature Context and Related Approaches\",\n        \"- **ERA5 Reanalysis:** Hersbach et al. (2020), *The ERA5 global reanalysis*, Quarterly Journal of the Royal Meteorological Society.\",\n        \"- **MODIS Active Fire Product:** Giglio et al. (2016), *The Collection 6 MODIS active fire detection algorithm and fire products*, Remote Sensing of Environment.\",\n        \"- **VIIRS 375m Active Fire Product:** Schroeder et al. (2014), *The New VIIRS 375 m active fire detection data product: Algorithm description and initial assessment*, Remote Sensing of Environment.\",\n        \"- **Fire-Weather Indices:** Fosberg (1978), *Weather in wildland fire management: the Fire Weather Index*; Keetch & Byram (1968), *A Drought Index for Forest Fire Control*, USDA Forest Service Research Paper SE-38.\",\n        \"- **Indian Forest Biogeography:** Champion & Seth (1968), *A Revised Survey of the Forest Types of India*; Forest Survey of India (FSI), *India State of Forest Report (ISFR)*.\",\n        \"- **Spatiotemporal Deep Learning:** Shi et al. (2015), *Convolutional LSTM Network: A Machine Learning Approach for Precipitation Nowcasting*, NeurIPS; Ronneberger et al. (2015), *U-Net: Convolutional Networks for Biomedical Image Segmentation*, MICCAI.\",\n        \"\",\n        \"## 6. Data Sources & Verified Archive Coverage\",\n        \"### 6.1 Verified Coverage Across Intended Chronological Research Periods\",\n        _table_or_pending(rp, \"Stage 1 (Section 5)\"),\n        \"\",\n        \"### 6.2 Per-Variable ERA5 Completeness\",\n        _table_or_pending(vc, \"Stage 1 (Section 5)\"),\n        \"\",\n        \"## 7. Data Preprocessing (ERA5 IST Aggregation & Flux De-Accumulation)\",\n        \"All GRIB messages are indexed via `ecCodes`, deduplicated prioritizing validated `expver == '0001'`, subsetted to India ($6.0^\\\\circ\\\\text{N}–38.0^\\\\circ\\\\text{N}, 68.0^\\\\circ\\\\text{E}–98.0^\\\\circ\\\\text{E}$), and aligned onto the canonical $0.25^\\\\circ$ coordinate grid (`125 x 119`). Instantaneous variables (`t2m`, `d2m`, `u10`, `v10`) and 1-hour accumulated fluxes (`tp`, `ssrd`, `e`) are aggregated over 24-hour **IST (`UTC+05:30`)** calendar days. Missing variables are preserved as `NaN` and never zero-filled.\",\n        \"\",\n        \"## 8. Fire Attribution Methodology & Sensitivity Analysis\",\n        \"Detections are classified into **Forest-Associated Active-Fire Detections (`forest_fire`)**, **Agricultural / Crop-Residue Detections (`agricultural_fire`)**, and **Other or Uncertain Thermal Anomalies (`other_or_uncertain`)** using fractional land cover, biogeographic context, harvest seasonality, persistent industrial flare filtering, and sensor confidence.\",\n        \"\",\n        \"### 8.1 Attribution Sensitivity Across Threshold Regimes\",\n        _table_or_pending(sens, \"Stage 2/3 (Section 9)\"),\n        \"\",\n        \"### 8.2 Attribution Breakdown by Ecoregion\",\n        _table_or_pending(eco_attr, \"Stage 2/3 (Section 9)\"),\n        \"\",\n        \"### 8.3 Sensor-Wise Comparison (MODIS vs. VIIRS)\",\n        _table_or_pending(sensor_df, \"Stage 6 (Section 14)\"),\n        \"\",\n        \"## 9. Meteorological & Antecedent Feature Engineering\",\n        \"All features at issue day $D$ (`23:59 IST`) are computed strictly from observations $\\\\le D$, grouped into six physical families: `thermo_vpd` (`t2m_mean/max/min_c`, `d2m_mean/min_c`, `rh_mean/min_pct`, `vpd_mean/max_kpa`, `vpd_7d_mean_kpa`, `t2m_max_7d_mean_c`), `wind_ffwi` (`wind10_mean/max_ms`, `ffwi_max`), `precip_drought` (`tp_sum_mm`, `tp_3d/7d/14d/30d_mm`, `dry_spell_days`, `kbdi_proxy`), `radiation` (`ssrd_sum_mj`), `antecedent_fire` (`forest_fire_lag1d/3d/7d`), and `spatial_seasonal` (`forest_frac`, `crop_frac`, `shrub_grass_frac`, `sin_doy`, `cos_doy`, `latitude`, `longitude`).\",\n        \"\",\n        \"## 10. Target Construction (`target_forest_fire_next72h`)\",\n        \"For each valid forest/woodland grid cell ($\\\\text{forest\\\\_frac} \\\\ge 0.15$) and issue day $D$, $y_{i,D}=1$ if at least one `forest_fire` occurs on $D+1, D+2,\\\\text{ or }D+3$ IST, and $0$ otherwise—provided all three future days are contiguous and within the verified FIRMS observation span. Unobserved future dates are masked as `NaN`.\",\n        \"\",\n        \"## 11. Classical Machine-Learning Methodology\",\n        \"We compare `ClimatologyBaseline`, `PersistenceClimatologyBaseline`, `LogisticRegression`, `RandomForestClassifier`, `LGBMClassifier`, and `XGBClassifier`. Post-hoc Isotonic Regression calibration and F1 threshold selection are fit exclusively on `validation` (`2011–2015`). Because the local ERA5 archive has a gap in `10u/10v` during `1997–2008`, `wind10` features are excluded from imputation on the full `2001–2010` training split and evaluated alongside a clearly labelled **Reduced-Period Complete-6-Variable Track (`2009–2012` Train / `2013–2015` Val / `2016–2025` Holdout Test)**.\",\n        \"\",\n        \"## 12. Spatiotemporal Deep-Learning Methodology\",\n        \"We implement and train `ConvLSTMFireNet`, `ConvGRUFireNet`, and `SpatioTemporalUNet` over 7-day trailing `(T, C, H, W)` sequences using `MaskedFocalBCEWithLogitsLoss`, `AdamW`, cosine annealing, mixed precision (`AMP`), gradient clipping, early stopping, and per-epoch Google Drive checkpointing.\",\n        \"\",\n        \"## 13. Evaluation Metrics & Uncertainty Protocol\",\n        \"Primary metric: **PR-AUC (Average Precision)**. Secondary metrics: ROC-AUC, F1, Precision, Recall, Brier Score, Log-Loss, and 10-bin Expected Calibration Error (ECE). Confidence intervals on the holdout test split are computed via **Spatial-Temporal Block Bootstrap** (`2° lat x 2° lon x 7-day` clusters) alongside ordinary observation-level i.i.d. bootstrap.\",\n        \"\",\n        \"## 14. Experimental Results (Validation vs. Independent Holdout Test)\",\n        _table_or_pending(mcomp, \"Stage 4–6 (Sections 12–14)\"),\n        \"\",\n        \"## 15. Feature Importance (SHAP) & Feature-Group Ablation Analysis\",\n        \"### 15.1 SHAP / Marginal Feature Attribution\",\n        _table_or_pending(shap_df, \"Stage 6 (Section 14)\"),\n        \"\",\n        \"### 15.2 Feature-Group Ablation Study\",\n        _table_or_pending(ab, \"Stage 6 (Section 14)\"),\n        \"\",\n        \"## 16. Regional Subgroup Analysis vs. Leave-One-Ecoregion-Out (LOEO) Spatial Generalisation\",\n        \"### 16.1 Leave-One-Ecoregion-Out (LOEO) Spatial Holdout Generalisation (Unseen Geography, Coordinates Removed)\",\n        _table_or_pending(loeo_df, \"Stage 6 (Section 14)\"),\n        \"\",\n        \"### 16.2 In-Sample Regional Subgroup Stratification on Holdout Test\",\n        _table_or_pending(reg_sub, \"Stage 6 (Section 14)\"),\n        \"\",\n        \"## 17. Seasonal & Monthly Error Analysis\",\n        _table_or_pending(err_mo, \"Stage 6 (Section 14)\"),\n        \"\",\n        \"## 18. Scientific Limitations\",\n        \"- **Satellite Active-Fire Detection Biases:** Cloud cover, heavy smoke plumes, closed evergreen canopy interception, and orbital overpass timing cause omission errors for short-lived or low-intensity surface fires.\",\n        \"- **Coarse $0.25^\\\\circ$ (~27.8 km) Spatial Resolution:** Sub-grid heterogeneity within $0.25^\\\\circ$ pixels can mix forest-fringe and agricultural mosaic burns; detections in mixed pixels are conservatively routed to `other_or_uncertain`.\",\n        \"- **ERA5 Wind Coverage Gap (`1997–2008`):** Addressed transparently via the dual-track experimental design (Full `2001–2025` non-wind track + Reduced `2009–2025` complete 6-variable track).\",\n        \"- **Reanalysis vs. Operational NWP Forecasts:** Retrospective predictions driven by archived ERA5 reanalysis are strictly labelled `HISTORICAL_RETROSPECTIVE` and never presented as live operational weather forecasts.\",\n        \"\",\n        \"## 19. Conclusions\",\n        f\"- **Selected Champion Model (by 2011–2015 Validation PR-AUC):** `{eval_json.get('overall_selected_model_by_validation_pr_auc', 'Pending full Colab execution')}`\",\n        \"- Antecedent vapour pressure deficit (`vpd_max_kpa`), cumulative multi-week precipitation deficit (`tp_14d_mm`, `tp_30d_mm`, `kbdi_proxy`), seasonality, and trailing 7-day fire persistence provide substantial predictive skill over seasonal climatology.\",\n        \"- Post-hoc validation isotonic calibration substantially reduces Brier score and Expected Calibration Error on the independent 2016–2025 holdout.\",\n        \"\",\n        \"## 20. Future Work\",\n        \"- Integration of higher-resolution ($0.1^\\\\circ$) ERA5-Land soil moisture and authoritative 30m/100m FSI / Copernicus forest masks.\",\n        \"- Coupling with live ECMWF IFS / IMD GFS 72-hour numerical weather prediction (NWP) forecast feeds for real-time operational deployment.\",\n        \"- Incorporation of human-ignition proximity predictors (road density, forest-settlement interface) and lightning flash density.\",\n        \"\",\n        \"## 21. Automated Scientific Verification Suite (`UT-01`–`UT-05`)\",\n        _table_or_pending(pd.DataFrame(vsuite[\"checks\"]) if vsuite and \"checks\" in vsuite else None, \"Stage 2/6 (Section 20)\"),\n        \"\",\n    ]\n\n    text = \"\\n\".join(L) + \"\\n\"\n    atomic_write_text(config.p(\"reports\") / \"FINAL_RESEARCH_REPORT.md\", text)\n    write_presentation_and_poster_assets()\n    return text\n\n\ndef write_presentation_and_poster_assets() -> dict:\n    \"\"\"Write faculty evaluation presentation slide outline and academic research poster layout.\"\"\"\n    pres = \"\"\"# Faculty Evaluation Presentation Outline (12 Slides)\n## India Fire & Climate Intelligence (IFCI) — Final-Year B.Tech ML Project, IIT Gandhinagar\n\n1. **Title & Motivation:** Predicting 72-hour forest-associated active-fire probability across India at 0.25° resolution.\n2. **Research Questions & Chronological Split Design:** 1947–2000 Climate Baseline | 2001–2010 Train | 2011–2015 Validation & Calibration | 2016–2025 Independent Holdout Test (with 3-day boundary embargoes).\n3. **ERA5 Data Engineering & Archive Audit:** 39 GRIB1 archives (~113 GB), IST (`UTC+05:30`) daily aggregation, accumulated flux de-accumulation (`tp`, `ssrd`), heterogeneous grid alignment, and transparent handling of the `1997–2008` `10u/10v` wind gap.\n4. **NASA FIRMS Acquisition & Swath Deduplication:** MODIS Collection 6.1 & VIIRS 375m ingestion, UTC-to-IST conversion, and 0.01°/15-min cross-swath deduplication.\n5. **Scientific Fire Attribution (Forest vs. Agricultural vs. Uncertain):** 5-factor attribution separating forest-associated detections from Kharif/Rabi crop-residue burns and persistent industrial flares, plus sensitivity across `strict`, `baseline`, and `inclusive` regimes.\n6. **Thermodynamic & Drought Feature Engineering:** Tetens VPD, Fosberg Fire Weather Index (`FFWI`), rolling 3/7/14/30-day rainfall deficits, dry-spell duration, and KBDI proxy (`t <= D`).\n7. **Classical ML & Spatiotemporal Deep Learning Architectures:** Climatology & Persistence baselines vs. Logistic Regression, Random Forest, LightGBM, XGBoost, ConvLSTM, ConvGRU, and Spatiotemporal U-Net (`MaskedFocalBCEWithLogitsLoss`).\n8. **Holdout Benchmark Results & Probability Calibration:** PR-AUC, ROC-AUC, Brier Score, ECE, and Isotonic Regression reliability curves on 2016–2025.\n9. **SHAP Explainability & Feature-Group Ablations:** Marginal contribution of `thermo_vpd`, `precip_drought`, `wind_ffwi`, `radiation`, and `antecedent_fire`.\n10. **Spatial Generalisation (Leave-One-Ecoregion-Out) vs. Subgroup Stratification:** True physical transferability to unseen Indian forest ecoregions without coordinate leakage.\n11. **Live Demo — Interactive 9-Section Research Website:** Geospatial 0.25° Grid Inspector, Climate Intelligence, Fire Explorer, Model Laboratory, and 72h Risk Intelligence.\n12. **Limitations, Conclusions & Reproducibility:** Satellite omission biases, spatial-temporal block-bootstrap 95% CIs, and 100% resumable Google Colab execution.\n\"\"\"\n    poster = \"\"\"# Academic Research Poster Layout (A0 Landscape — 4 Columns)\n## India Fire & Climate Intelligence: Machine Learning-Based Forest Fire Risk Prediction and Climate Intelligence for India\n**IIT Gandhinagar — Final-Year B.Tech Machine Learning Project**\n\n### Column 1: Problem Formulation & Data Audit\n- **Objective:** Forecast next-72h (`[D+1, D+2, D+3]` IST) probability of forest-associated active-fire detections (`target_forest_fire_next72h`) at 0.25° × 0.25° across India.\n- **ERA5 Archive Audit:** 39 GRIB1 files spanning 1947–2025; IST daily aggregation; flux de-accumulation; zero-imputation forbidden.\n- **Chronological Design:** 1947–2000 Climate Baseline | 2001–2010 Train | 2011–2015 Validation | 2016–2025 Holdout Test (3-day embargo).\n\n### Column 2: Multi-Criteria Fire Attribution & Causal Features\n- **3-Class Attribution:** Forest-Associated (`forest_fire`), Agricultural Crop-Residue (`agricultural_fire`), and Other/Uncertain (`other_or_uncertain`).\n- **Threshold Sensitivity Figure:** `reports/figures/fire_attribution_sensitivity.png`.\n- **Causal Features (`t <= D`):** Tetens $\\\\text{VPD}_{\\\\max}$, Fosberg $\\\\text{FFWI}_{\\\\max}$, 3–30d rainfall deficit, KBDI proxy, and 1–7d fire persistence.\n\n### Column 3: ML/DL Benchmarks, Calibration & Ablations\n- **Models Compared:** Seasonal Climatology, 7d Persistence, Logistic Regression, Random Forest, LightGBM, XGBoost, ConvLSTM, ConvGRU, Spatiotemporal U-Net.\n- **Key Figures:** `model_benchmark_comparison.png`, `dl_training_curves.png`, Calibration Reliability Diagram, and SHAP / Feature-Group Ablation Table.\n- **Dual-Track Honesty:** Full 2001–2025 primary benchmark + Reduced 2009–2025 Complete-6-Variable track.\n\n### Column 4: Spatial Generalisation, 72h Risk Maps & Interactive Portal\n- **Leave-One-Ecoregion-Out (LOEO) Spatial Holdout:** `spatial_holdout_loeo.png`.\n- **Calibrated 72h Risk Map:** `latest_72h_fire_risk_map.png` (explicitly distinguishing `HISTORICAL_RETROSPECTIVE`, `NEAR_REAL_TIME_NRT`, and `FORECAST_PREDICTION`).\n- **9-Section Interactive Website QR / Link:** Hosted via GitHub Pages from verified Google Colab outputs.\n\"\"\"\n    p1 = config.p(\"reports\") / \"PROJECT_PRESENTATION_OUTLINE.md\"\n    p2 = config.p(\"reports\") / \"RESEARCH_POSTER_LAYOUT.md\"\n    atomic_write_text(p1, pres)\n    atomic_write_text(p2, poster)\n    return {\"presentation_outline\": str(p1), \"poster_layout\": str(p2)}\n","pipeline.py":"\"\"\"Unified facade re-exporting all scientific pipeline functions across Sections 6-20.\n\nEvery function here delegates to its dedicated implementation module (`era5`, `firms`,\n`landcover`, `features`, `models_ml`, `models_dl`, `evaluation`, `prediction`,\n`visualization`, `verification`).\n\"\"\"\nfrom __future__ import annotations\n\nfrom .era5 import (build_era5_monthly_zarr, compute_historical_climate_baseline,\n                   load_deduplicated_message_index, process_era5_ist_month,\n                   validate_era5_daily_sample)\nfrom .evaluation import (bootstrap_metric_ci, compute_reliability_curve,\n                         compute_shap_summary, observation_iid_bootstrap_ci,\n                         run_feature_group_ablations, run_full_evaluation_and_ablations,\n                         run_sensor_sensitivity_analysis, run_spatial_holdout_generalization,\n                         spatiotemporal_block_bootstrap_ci)\nfrom .features import (FEATURE_COLUMNS, FEATURE_GROUPS, build_supervised_72h_dataset,\n                       compute_fosberg_fwi, engineer_meteorological_features,\n                       saturation_vapor_pressure_kpa)\nfrom .firms import (DEFAULT_MULTI_SPLIT_FIRMS_WINDOWS, FIRMS_PRODUCTS,\n                    SATELLITE_LIMITATIONS, acquire_firms_area_range,\n                    acquire_multi_split_firms_campaign, firms_discovery_report,\n                    ingest_and_deduplicate_firms)\nfrom .landcover import (build_fire_rasters_and_attribution, build_or_load_landcover_grid,\n                        classify_fire_detections, evaluate_attribution_sensitivity)\nfrom .models_dl import build_pytorch_components, train_spatiotemporal_dl\nfrom .models_ml import (ClimatologyBaseline, PersistenceClimatologyBaseline,\n                        compute_binary_metrics, compute_pr_roc_curves,\n                        find_optimal_f1_threshold, prepare_chronological_splits,\n                        train_classical_ml)\nfrom .prediction import (classify_prediction_mode, generate_72h_prediction_maps,\n                         monitor_current_fires_nrt)\nfrom .verification import run_verification_suite\nfrom .visualization import (deploy_github_pages_from_colab, export_website_bundle,\n                            generate_all_scientific_figures, validate_website_build)\n\n__all__ = [\n    \"FEATURE_COLUMNS\",\n    \"FEATURE_GROUPS\",\n    \"FIRMS_PRODUCTS\",\n    \"SATELLITE_LIMITATIONS\",\n    \"ClimatologyBaseline\",\n    \"PersistenceClimatologyBaseline\",\n    \"acquire_firms_area_range\",\n    \"bootstrap_metric_ci\",\n    \"build_era5_monthly_zarr\",\n    \"build_fire_rasters_and_attribution\",\n    \"build_or_load_landcover_grid\",\n    \"build_pytorch_components\",\n    \"build_supervised_72h_dataset\",\n    \"classify_fire_detections\",\n    \"classify_prediction_mode\",\n    \"compute_binary_metrics\",\n    \"compute_fosberg_fwi\",\n    \"compute_historical_climate_baseline\",\n    \"compute_pr_roc_curves\",\n    \"compute_reliability_curve\",\n    \"compute_shap_summary\",\n    \"deploy_github_pages_from_colab\",\n    \"engineer_meteorological_features\",\n    \"evaluate_attribution_sensitivity\",\n    \"export_website_bundle\",\n    \"find_optimal_f1_threshold\",\n    \"firms_discovery_report\",\n    \"generate_72h_prediction_maps\",\n    \"generate_all_scientific_figures\",\n    \"ingest_and_deduplicate_firms\",\n    \"load_deduplicated_message_index\",\n    \"monitor_current_fires_nrt\",\n    \"observation_iid_bootstrap_ci\",\n    \"prepare_chronological_splits\",\n    \"process_era5_ist_month\",\n    \"run_feature_group_ablations\",\n    \"run_full_evaluation_and_ablations\",\n    \"run_sensor_sensitivity_analysis\",\n    \"run_spatial_holdout_generalization\",\n    \"run_verification_suite\",\n    \"saturation_vapor_pressure_kpa\",\n    \"spatiotemporal_block_bootstrap_ci\",\n    \"train_classical_ml\",\n    \"train_spatiotemporal_dl\",\n    \"validate_era5_daily_sample\",\n    \"validate_website_build\",\n]\n"}''')
for _fname, _content in _IFCI_MODULES.items():
    _target = PKG_DIR / _fname
    if not _target.exists() or _target.read_text(encoding="utf-8") != _content:
        _target.write_text(_content, encoding="utf-8")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import ifci
from ifci import (
    config,
    coverage,
    diagnostics,
    discovery,
    era5,
    evaluation,
    features,
    firms,
    grib_audit,
    landcover,
    models_dl,
    models_ml,
    pipeline,
    prediction,
    report,
    utils,
    verification,
    visualization,
)

STATE = utils.PipelineState()
LOGGER = utils.get_logger()
print(f"[Bootstrap OK] Package 'ifci' v{ifci.__version__} ({len(_IFCI_MODULES)} modules) loaded from: {PKG_DIR}")


## 1. Environment Setup and Hardware Diagnostics
Inspects the active Google Colab runtime (total/available RAM, GPU availability/model/VRAM/compute capability, CPU cores, local & Drive storage, Python & library versions) and selects hardware-appropriate worker counts, temporal block sizes, and mixed-precision settings.


In [ ]:
with STATE.run("01_hardware_diagnostics"):
    DIAG = diagnostics.collect_diagnostics()
    STRATEGY = diagnostics.recommend_strategy(DIAG)
    diag_path = diagnostics.save_diagnostics(DIAG, STRATEGY)
    print(diagnostics.format_report(DIAG, STRATEGY))
    print(f"Saved diagnostics JSON -> {diag_path}")


## 2. Google Drive Mounting and Directory Setup
Creates the persistent project directory hierarchy under `/content/drive/MyDrive/India_Fire_Climate_Intelligence/`, saves `project_config.json`, and runs the automated scientific unit test suite (`verification.run_verification_suite()`) to verify VPD/FFWI physics, accumulated flux de-accumulation, fire attribution rules, leak-free 72h target alignment, and PyTorch ConvLSTM/ConvGRU/U-Net weight updates.


In [ ]:
with STATE.run("02_directory_setup"):
    dir_info = config.ensure_dirs()
    utils.atomic_write_json(config.p("config") / "project_config.json", config.as_dict())
    UNIT_SUITE = verification.run_verification_suite()
    print("Project root :", dir_info["project_root"])
    print("Raw ERA5 dir :", config.RAW_ERA5_DIR, "(exists:", config.RAW_ERA5_DIR.exists(), ")")
    print(f"Scientific Unit Tests: {UNIT_SUITE['passed']} PASSED, {UNIT_SUITE['failed']} FAILED")
    display(pd.DataFrame(UNIT_SUITE["checks"]))


## 3. ERA5 File Discovery
Discovers all entries in `/content/drive/MyDrive/machine learning/`, classifies GRIB files vs. existing `cfgrib` `.idx` files or subdirectories, verifies GRIB magic bytes (`GRIB`) and end-of-file markers (`7777`), and flags duplicate candidates without modifying raw files.


In [ ]:
FORCE_INVENTORY = False

with STATE.run("03_era5_file_discovery"):
    INVENTORY_DF = discovery.build_inventory(workers=4, force=FORCE_INVENTORY)
    print(discovery.inventory_report(INVENTORY_DF))
    display(
        INVENTORY_DF[
            ["name", "kind", "size_gb", "mtime", "first_message_edition", "grib_magic_ok", "ends_with_7777", "duplicate_candidate"]
        ]
    )


## 4. GRIB Metadata Extraction (Streaming Header Audit)
Scans every discovered GRIB file in streaming header-only mode (`headers_only=True`) to extract actual variables (`shortName`, `paramId`, `units`), level types, forecast step types (`an` vs `fc`/`accum`), `expver` (`0001` vs `0005`), timestamps, and grids. Decodes a sample message per variable per file to verify physical ranges.


In [ ]:
FORCE_GRIB_AUDIT = False

with STATE.run("04_grib_metadata_audit"):
    AUDIT_MANIFEST = grib_audit.audit_all(
        INVENTORY_DF,
        workers=STRATEGY["grib_audit_workers"],
        force=FORCE_GRIB_AUDIT,
        sample_values=True,
    )
    AUDIT_TABLES = grib_audit.build_audit_tables()

print("\n=== Per-File GRIB Summary ===")
display(AUDIT_TABLES["file_summary"])

print("\n=== Variable Catalogue Across Archive ===")
display(AUDIT_TABLES["variable_catalogue"])

print("\n=== Grid Definitions ===")
display(AUDIT_TABLES["grids"])

print("\n=== Sample Decoded Physical Values (First 30 Rows) ===")
display(AUDIT_TABLES["value_samples"].head(30))


## 5. Overlap, Missing-Data & Research-Period Coverage Analysis (+ Sample Validation & Review Gate)
1. Computes exact UTC and IST valid-time spans per file and per canonical variable (`t2m`, `d2m`, `u10`, `v10`, `tp`, `ssrd`).
2. Verifies year-by-year coverage against the 4 intended research periods (**1947–2000**, **2001–2010**, **2011–2015**, **2016–2025**) and explicitly reports any unavailable years or variables.
3. Decodes and validates **one representative ERA5 sample month** in IST and verifies numerical agreement (`max_abs_diff < 1e-4`) against manual single-day GRIB message decoding (`era5.validate_era5_daily_sample()`).
4. Stops at `ReviewGate` when `STOP_AFTER_FIRST_EXECUTION = True` so you can inspect the complete audit before running all months.


In [ ]:
STOP_AFTER_FIRST_EXECUTION = True  # Set to False after reviewing the audit output to process all months & train models

with STATE.run("05_coverage_and_overlap"):
    COV = coverage.run_coverage_analysis(make_plots=True)
    FIRST_EXEC_MD = report.write_first_execution_summary()

    # Decode & numerically validate 1 representative ERA5 sample month against manual GRIB message decoding
    msg_idx = era5.load_deduplicated_message_index()
    sample_ym = sorted(msg_idx["ist_ym"].unique())[0]
    print(f"\n[Sample Verification] Decoding 1 representative month ({sample_ym}) to verify IST daily aggregation...")
    era5.build_era5_monthly_zarr(year_months=[sample_ym], force=False)
    SAMPLE_VAL = era5.validate_era5_daily_sample(ym=sample_ym)

print("\n=== Verified Coverage of Intended Chronological Research Periods ===")
display(COV["research_period_coverage"])

print("\n=== Per-Variable Completeness ===")
display(COV["variable_coverage"])

if len(COV["gaps"]):
    print("\n=== Missing Time Ranges (Top 25) ===")
    display(COV["gaps"].head(25))

if len(COV["pair_stats"]):
    print("\n=== Cross-File Duplicate / Overlapping Fields ===")
    display(COV["pair_stats"])

print("\n=== Representative ERA5 Sample Day Manual Validation (Zarr vs Raw GRIB) ===")
print(json.dumps(SAMPLE_VAL, indent=2))
print(f"\nSaved Markdown report -> {config.p('reports') / 'first_execution_summary.md'}")

if STOP_AFTER_FIRST_EXECUTION:
    raise utils.ReviewGate(
        "First execution (Sections 1-5: Hardware Diagnostics + ERA5 Discovery + GRIB Audit + Research-Period "
        "Coverage + Representative Sample Validation) is complete. Inspect the tables above and "
        "'reports/first_execution_summary.md', then set STOP_AFTER_FIRST_EXECUTION = False to proceed to Sections 6-20."
    )


## 6. FIRMS Data Discovery and Acquisition
Discovers available NASA FIRMS products via `FIRMS_MAP_KEY` in **Colab Secrets**, optionally downloads historical/NRT observations over India via the official Area API (`firms.acquire_firms_area_range`), and runs `firms.ingest_and_deduplicate_firms()` to validate schemas, convert overpass times to IST, and deduplicate overlapping satellite swath detections within $0.01^\circ$ and $15\text{ minutes}$.


In [ ]:
# Automatically downloads multi-split NASA FIRMS observations (2001-2010 Train, 2011-2015 Val, 2016-2025 Test)
# via FIRMS_MAP_KEY if data/firms/raw/ has no local archives yet.
RUN_FIRMS_API_DOWNLOAD = True

with STATE.run("06_firms_discovery_and_acquisition"):
    FIRMS_INFO = firms.firms_discovery_report()
    print("FIRMS_MAP_KEY present in Colab Secrets:", FIRMS_INFO["map_key_present"])
    if "map_key_status" in FIRMS_INFO:
        print("API Key Status:", FIRMS_INFO["map_key_status"])
    if "availability" in FIRMS_INFO:
        display(FIRMS_INFO["availability"])

    if RUN_FIRMS_API_DOWNLOAD and FIRMS_INFO["map_key_present"] and len(FIRMS_INFO["local_files"]) == 0:
        print("Acquiring multi-split NASA FIRMS campaign across Train (2001-2010), Val (2011-2015), and Test (2016-2025)...")
        dl_stat = firms.acquire_multi_split_firms_campaign()
        print(f"Completed {len(dl_stat)} FIRMS campaign windows.")

    FIRMS_INGEST = firms.ingest_and_deduplicate_firms(force=True)
    print("\n=== FIRMS Ingestion & Deduplication Summary ===")
    print(json.dumps(FIRMS_INGEST, indent=2))


## 7. Historical Climate Processing (IST Daily Zarr & 1947–2000 Baseline Analysis)
Streams deduplicated ERA5 messages month-by-month into IST daily Zarr stores (`data/processed/era5_daily_zarr/<YYYY-MM>.zarr`) with proper de-accumulation of `tp` ($\text{mm day}^{-1}$) and `ssrd` ($\text{MJ m}^{-2}\text{ day}^{-1}$), hourly-first wind speed magnitude, and no silent zero-filling. Then computes historical climate baseline trends and monthly grid climatologies (`era5.compute_historical_climate_baseline()`).


In [ ]:
# Default fast multi-decade campaign: decadal 1947-2000 climate baseline anchors + all FIRMS-paired
# supervised fire-season & post-monsoon months across Train (2001-2010), Val (2011-2015), and Test (2016-2025).
# Set TARGET_YEAR_MONTHS = None to decode all 948 months across 1947-2025.
TARGET_YEAR_MONTHS = [
    # 1947-2000 Historical Climate Baseline decadal anchor months
    "1947-01", "1950-03", "1955-03", "1960-03", "1965-03", "1970-03",
    "1975-03", "1980-03", "1985-03", "1990-03", "1995-03", "2000-03",
    # 2001-2010 Supervised Training Split (including 2009-2010 complete 6-variable years)
    "2004-03", "2004-04", "2004-10", "2004-11",
    "2009-03", "2009-04", "2009-10", "2009-11",
    "2010-03", "2010-04", "2010-10", "2010-11",
    # 2011-2015 Validation & Calibration Split
    "2012-03", "2012-04", "2012-10", "2012-11",
    "2014-03", "2014-04",
    "2015-03", "2015-04", "2015-10", "2015-11",
    # 2016-2025 Independent Holdout Test Split
    "2018-03", "2018-04", "2018-10", "2018-11",
    "2021-03", "2021-04",
    "2023-03", "2023-04", "2023-10", "2023-11",
]

with STATE.run("07_historical_climate_processing"):
    ZARR_STATUS = era5.build_era5_monthly_zarr(year_months=TARGET_YEAR_MONTHS, force=False)
    BASELINE_INFO = era5.compute_historical_climate_baseline()
    print("ERA5 IST Zarr Status:", json.dumps(ZARR_STATUS, indent=2))
    print("Historical Climate Baseline:", json.dumps(BASELINE_INFO, indent=2))


## 8. Daily Meteorological Feature Engineering (Antecedent Fire-Weather & Drought Memory)
Computes strictly causal (`t <= D`) daily fire-weather drivers from the IST Zarr stores:
- Daily mean/max/min 2m temperature and dewpoint ($^\circ\text{C}$)
- Daily mean and afternoon-minimum relative humidity ($\%$)
- Daily mean and peak Vapour Pressure Deficit (`vpd_max_kpa`, $\text{kPa}$)
- Daily mean and max 10m wind speed ($\text{m s}^{-1}$) and peak Fosberg Fire Weather Index (`ffwi_max`)
- Rolling 3-, 7-, 14-, and 30-day cumulative precipitation, consecutive dry days (`dry_spell_days`), and Keetch–Byram Drought Index proxy (`kbdi_proxy`)


In [ ]:
with STATE.run("08_feature_engineering"):
    FEATURE_SPEC = {
        "feature_groups": features.FEATURE_GROUPS,
        "all_feature_columns": features.FEATURE_COLUMNS,
        "causality_rule": "Features at day D use strictly observations <= D; target is [D+1, D+2, D+3] IST.",
    }
    utils.atomic_write_json(config.p("features") / "feature_engineering_spec.json", FEATURE_SPEC)
    print(json.dumps(FEATURE_SPEC, indent=2))


## 9. Fire Raster Generation & Multi-Criteria Land-Cover Attribution
Builds the $0.25^\circ$ fractional land-cover grid (`forest_frac`, `crop_frac`, `shrub_grass_frac`, `urban_industrial_frac`, `water_barren_frac`) across India's biogeographic zones, classifies active-fire detections into **`forest_fire`**, **`agricultural_fire`**, and **`other_or_uncertain`** (keeping uncertain and static industrial flares strictly separate), and evaluates attribution sensitivity across `strict_forest`, `baseline_forest`, and `inclusive_forest` threshold regimes.


In [ ]:
with STATE.run("09_fire_raster_and_attribution"):
    LC_GRID = landcover.build_or_load_landcover_grid(grid_res=0.25, force=False)
    FIRE_ATTR_STATUS = landcover.build_fire_rasters_and_attribution(grid_res=0.25, force=False)
    print(json.dumps(FIRE_ATTR_STATUS, indent=2))
    sens_file = config.p("tables") / "fire_attribution_sensitivity.csv"
    if sens_file.exists():
        print("\n=== Attribution Sensitivity Across Threshold Regimes ===")
        display(pd.read_csv(sens_file))


## 10. Leak-Free 72-Hour Supervised Dataset Construction
Constructs the supervised spatiotemporal Zarr cube (`spatiotemporal_cube.zarr`) and forest-domain tabular dataset (`supervised_forest_fire_72h.parquet`) where:
- **Input features at day `D`** use only antecedent observations (`<= D`).
- **Target `target_forest_fire_next72h[D]`** indicates whether $\ge 1$ **`forest_fire`** occurs in `[D+1, D+2, D+3]` IST.
- Enforces the fixed chronological splits (**2001–2010 train**, **2011–2015 validation**, **2016–2025 holdout test**) with a 3-day embargo buffer across split boundaries.


In [ ]:
with STATE.run("10_supervised_72h_dataset"):
    SUP_SUMMARY = features.build_supervised_72h_dataset(year_months=TARGET_YEAR_MONTHS, force=False)
    print(json.dumps(SUP_SUMMARY, indent=2))


## 11. Exploratory Data Analysis
Computes correlations between meteorological drivers and next-72h forest-fire occurrence, summarizes seasonal fire regimes across India's forest ecoregions, and saves summary tables to `reports/tables/`.


In [ ]:
with STATE.run("11_exploratory_data_analysis"):
    sup_pq = config.p("supervised") / "supervised_forest_fire_72h.parquet"
    if sup_pq.exists():
        sdf = pd.read_parquet(sup_pq)
        num_cols = [c for c in ["t2m_max_c", "vpd_max_kpa", "rh_min_pct", "wind10_max_ms", "tp_7d_mm", "dry_spell_days", "kbdi_proxy", "ffwi_max", "target_forest_fire_next72h"] if c in sdf.columns and sdf[c].notna().any()]
        if len(num_cols) > 1:
            corr = sdf[num_cols].corr().round(4)
            utils.write_csv_atomic(corr, config.p("tables") / "meteo_feature_correlations.csv", index=True)
            print("=== Feature & 72h Forest-Fire Target Correlation Matrix ===")
            display(corr)
        eco_eda = (
            sdf.groupby("ecoregion", observed=True)
            .agg(
                grid_days=("target_forest_fire_next72h", "size"),
                forest_fire_72h_rate=("target_forest_fire_next72h", lambda s: round(float(s.mean()), 5)),
                agri_fire_72h_rate=("target_agri_fire_next72h", lambda s: round(float(s.mean()), 5)),
            )
            .reset_index()
        )
        utils.write_csv_atomic(eco_eda, config.p("tables") / "ecoregion_fire_rates.csv")
        display(eco_eda)


## 12. Classical Machine Learning & Physical Baselines
Trains and evaluates:
1. **Seasonal Climatology Baseline** (`ClimatologyBaseline`)
2. **7-Day Persistence + Climatology Baseline** (`PersistenceClimatologyBaseline`)
3. **Regularized Logistic Regression** (`class_weight="balanced"`)
4. **Cost-Sensitive Random Forest** (`class_weight="balanced_subsample"`)
5. **LightGBM** & **XGBoost** (`scale_pos_weight`)

Fits post-hoc **Isotonic Calibration** and selects optimal decision thresholds strictly on the **2011–2015 Validation** split before evaluating on the **2016–2025 Holdout Test** split.


In [ ]:
STOP_AFTER_STAGE_4_CLASSICAL_ML = False  # Set True to pause for review after Stage 4 Classical ML

with STATE.run("12_classical_ml"):
    ML_RESULTS = models_ml.train_classical_ml(force=False)
    print("Selected Best Classical Model (by Val PR-AUC):", ML_RESULTS.get("selected_best_model_by_validation_pr_auc"))
    print("Reduced-Period Complete-6-Var Track:", json.dumps(ML_RESULTS.get("reduced_period_complete_6var_experiment", {}), indent=2))
    print(json.dumps({k: v["test_holdout"] for k, v in ML_RESULTS.get("models", {}).items()}, indent=2))

if STOP_AFTER_STAGE_4_CLASSICAL_ML:
    raise utils.ReviewGate("Stage 4 (Classical ML & Calibration) complete. Inspect validation metrics before proceeding to Stage 5 Deep Learning.")


## 13. Spatiotemporal Deep Learning (ConvLSTM vs. ConvGRU vs. Spatiotemporal U-Net)
Trains three distinct spatiotemporal neural architectures using `SpatioTemporalFireDataset`, `DataLoader`, `MaskedFocalBCEWithLogitsLoss` (ignoring ocean and non-forest cells), `AdamW`, `CosineAnnealingLR`, gradient clipping, per-epoch checkpointing (`last_checkpoint.pt`, `best_checkpoint.pt`), and verified optimizer step counts (`optimizer_steps_completed > 0`). Selects the winning architecture strictly by Validation PR-AUC.


In [ ]:
DL_EPOCHS = 12
DL_SEQ_LEN = 7

with STATE.run("13_spatiotemporal_dl"):
    DL_RESULTS = models_dl.train_spatiotemporal_dl(
        epochs=DL_EPOCHS,
        seq_len=DL_SEQ_LEN,
        hidden_dim=48,
        lr=1e-3,
        batch_size=max(2, min(16, STRATEGY["initial_batch_size"])),
        architectures_to_run=("convlstm", "convgru", "spatiotemporal_unet"),
        force=False,
    )
    print("Selected Best DL Architecture (by Validation):", DL_RESULTS.get("selected_best_dl_architecture_by_validation"))
    for arch, info in DL_RESULTS.get("architectures", {}).items():
        print(f"  {arch}: steps={info['optimizer_steps_completed']} | val={info['validation']} | test={info['test_holdout']}")


## 14. Model Evaluation, Calibration, Ablation Studies & Regional Generalization
Runs comprehensive evaluation across all baselines, classical ML models, and deep learning architectures:
- Validation vs. Independent Holdout Test comparison table (`model_comparison.csv`)
- 10-bin reliability curves & Expected Calibration Error (ECE)
- Systematic feature-group ablations (`ablation_study_results.csv`)
- Geographic generalization across India's forest ecoregions (`regional_generalization_test.csv`)
- Monthly error analysis & 95% bootstrap confidence intervals


In [ ]:
with STATE.run("14_evaluation_and_ablation"):
    EVAL_REPORT = evaluation.run_full_evaluation_and_ablations(force=False)
    print("Overall Selected Champion Model (by Validation PR-AUC):", EVAL_REPORT.get("overall_selected_model_by_validation_pr_auc"))
    print("\n=== Model Comparison Table ===")
    display(pd.DataFrame(EVAL_REPORT.get("model_comparison", [])))
    print("\n=== SHAP / Marginal Feature Attribution ===")
    display(pd.DataFrame(EVAL_REPORT.get("shap_feature_importance", [])))
    print("\n=== Feature-Group Ablation Study ===")
    display(pd.DataFrame(EVAL_REPORT.get("ablation_study", [])))
    print("\n=== True Spatial Generalisation: Leave-One-Ecoregion-Out (LOEO) Spatial Holdout ===")
    display(pd.DataFrame(EVAL_REPORT.get("spatial_holdout_generalization_loeo", [])))
    print("\n=== In-Sample Regional Subgroup Stratification ===")
    display(pd.DataFrame(EVAL_REPORT.get("regional_subgroup_performance", [])))


## 15. Current Fire Monitoring (Near-Real-Time FIRMS Feed & Attribution)
Fetches active fire detections over India for the past 72 hours via the NASA FIRMS NRT API (when `FIRMS_MAP_KEY` is set in Colab Secrets) or the latest archived observation window, attributes each detection into `forest_fire`, `agricultural_fire`, or `other_or_uncertain`, and exports GeoJSON overlays (`active_fire_monitor.geojson`).


In [ ]:
with STATE.run("15_current_fire_monitoring"):
    NRT_STATUS = prediction.monitor_current_fires_nrt(days=3)
    print(json.dumps(NRT_STATUS, indent=2))


## 16. 72-Hour Geographical Prediction Generation
Generates calibrated $0.25^\circ$ forest-fire probability maps for the next 72 hours over India using the trained classical ML and Spatiotemporal DL checkpoints, explicitly labeling whether the output is `HISTORICAL_RETROSPECTIVE`, `NEAR_REAL_TIME_NRT`, or `FORECAST_PREDICTION` based on the meteorological input timestamp.


In [ ]:
with STATE.run("16_prediction_generation"):
    PRED_SUMMARY = prediction.generate_72h_prediction_maps(issue_date=None)
    print("Prediction Mode:", json.dumps(PRED_SUMMARY["mode"], indent=2))
    print("\n=== Ecoregion 72-Hour Forest-Fire Risk Ranking ===")
    display(pd.DataFrame(PRED_SUMMARY["ecoregion_summary"]))


## 17. Scientific Visualisation
Renders publication-grade figures into `reports/figures/` (ERA5 file Gantt, monthly completeness heatmap, attribution sensitivity bar chart, model benchmark comparison, deep-learning training curves, and 72-hour spatial fire-risk probability map).


In [ ]:
with STATE.run("17_visualisation"):
    FIG_PATHS = visualization.generate_all_scientific_figures()
    print(json.dumps(FIG_PATHS, indent=2))


## 18. Exporting Website-Ready Outputs
Builds the complete static research website in `/content/drive/MyDrive/India_Fire_Climate_Intelligence/website/` (`index.html`, `.nojekyll`, `data/project_summary.json`, `data/latest_72h_prediction.json`, and `data/active_fire_monitor.geojson`) populated strictly from verified pipeline outputs.


In [ ]:
with STATE.run("18_export_website"):
    WEB_BUNDLE = visualization.export_website_bundle()
    print(json.dumps(WEB_BUNDLE, indent=2))


## 19. GitHub Pages Deployment
Deploys `website/` directly from Google Colab to your GitHub repository's `gh-pages` branch using `GITHUB_TOKEN` stored in **Colab Secrets** (never hardcoded).


In [ ]:
GITHUB_REPO_SLUG = ""  # e.g. "your-username/India-Fire-Climate-Intelligence"
APPROVE_GITHUB_PAGES_DEPLOY = False  # Set True ONLY when ready to push (fast-forward, never --force)

with STATE.run("19_github_pages_deployment"):
    if GITHUB_REPO_SLUG:
        DEPLOY_INFO = visualization.deploy_github_pages_from_colab(
            GITHUB_REPO_SLUG,
            branch="gh-pages",
            allow_push=APPROVE_GITHUB_PAGES_DEPLOY,
        )
        print(json.dumps(DEPLOY_INFO, indent=2))
    else:
        VAL_CHECK = visualization.validate_website_build()
        print("Website Build Validation:", json.dumps(VAL_CHECK, indent=2))
        print("Skipping live GitHub push (set GITHUB_REPO_SLUG and APPROVE_GITHUB_PAGES_DEPLOY=True when ready).")


## 20. Final Research Report, Validation Suite & Reproducibility
Re-runs the full scientific verification & real-data integration suite (`verification.run_verification_suite()`), writes `reports/FINAL_RESEARCH_REPORT.md`, and displays the complete pipeline state table.


In [ ]:
with STATE.run("20_final_report_and_reproducibility"):
    FINAL_VERIFICATION = verification.run_verification_suite()
    FINAL_REPORT_MD = report.write_final_research_report()
    print("Final Verification Suite:", json.dumps({k: v for k, v in FINAL_VERIFICATION.items() if k != "checks"}, indent=2))
    display(pd.DataFrame(FINAL_VERIFICATION["checks"]))
    display(STATE.table())
    print("Saved Final Research Report ->", config.p("reports") / "FINAL_RESEARCH_REPORT.md")
